In [ ]:
import networkx as nx
import json
import math
import pandas as pd
import numpy as np
import random
import os
import re
import gc
import time
import sys
import sqlite3
import warnings
from abc import ABC, abstractmethod
from scipy import sparse
from scipy.spatial.distance import cdist
from scipy.interpolate import griddata
from PIL import Image
from datetime import datetime, timedelta

# [修改 1] 引入 joblib 用于 Jupyter 环境下的并行计算
from joblib import Parallel, delayed 

# 禁用插值产生的可能的警告
warnings.filterwarnings("ignore")

# ==============================================================================
# [全局配置] 并行计算环境设置
# ==============================================================================
# 限制 Numpy/OpenBLAS 的底层线程数。
# 原因：在 Joblib 多进程模式下，如果每个子进程再自动开启多线程进行矩阵运算，
# 会导致 CPU 资源争抢（Oversubscription）甚至死锁，显著降低仿真效率。
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

# ==============================================================================
# 1. 策略模式：定义扰动（降雨）行为
# ==============================================================================
class DisturbanceStrategy(ABC):
    """
    抽象基类：定义系统受到的外部干扰策略。
    
    Methods:
        get_intensity: 获取特定时空的干扰强度。
        get_speed_impact_factor: 获取干扰对系统运行速度的影响系数。
    """
    @abstractmethod
    def get_intensity(self, time_step, position): pass
    @abstractmethod
    def get_speed_impact_factor(self, intensity, edge_type, **kwargs): pass

class ExtremeRainfall(DisturbanceStrategy):
    """
    极端降雨策略模型：模拟一个移动的暴雨中心及其对地铁站点的积水影响。
    
    包含物理过程：
    1. 芝加哥雨型（Chicago Design Storm）生成时间序列降雨。
    2. 移动风暴中心（Moving Storm Center）。
    3. 空间衰减（ARF）计算。
    4. SCS-CN 径流模型计算地表积水。
    """
    def __init__(self, base_path, max_intensity=201.9, rainfall_type='chicago', P=100, r=0.405, rain_duration=180):
        """
        初始化降雨模型参数。
        
        Args:
            max_intensity (float): 降雨峰值强度参数 (mm/h 或相关系数)。
            P (int): 降雨重现期 (Return Period, e.g., 100年一遇)。
            r (float): 雨峰位置系数 (Peak Position parameter)，0-1之间，0.4代表峰值出现在前40%。
            rain_duration (int): 降雨持续总时长 (分钟)。
        """
        # 设定模拟的空间范围（上海经纬度大致范围）
        self.lat_range = (30.6984, 31.5139)
        self.lon_range = (120.849, 121.966)
        
        # 初始化暴雨中心起始位置和移动速度
        self.center_loc, self.velocity = self._init_random_boundary_start()
        
        self.Q0 = max_intensity  # 降雨强度基准
        self.base_path = base_path
        self.rainfall_type = rainfall_type
        self.P_period = P        # 重现期
        self.r_peak = r          # 雨峰系数
        self.rain_duration = rain_duration 
        
        self.chicago_profile = None 
        self.img_array = None
        self._load_drainage_image() # 加载区域排水能力图
        self._distance_cache = {}   # 缓存站点到雨中心的距离，减少重复三角函数计算
        
        # 如果是芝加哥雨型，预计算随时间变化的强度曲线
        if self.rainfall_type == 'chicago': self._precompute_chicago_profile()

    def _init_random_boundary_start(self):
        """
        随机初始化降雨中心。
        逻辑：选择地图四个边界之一作为起点，目标点设为地图中心附近，以此计算移动矢量。
        """
        lat_min, lat_max = self.lat_range
        lon_min, lon_max = self.lon_range
        side = random.choice([0, 1, 2, 3]) # 0:上(北), 1:下(南), 2:左(西), 3:右(东)
        
        # 设定目标点为区域中心附近略带随机偏移
        target_lat = (lat_min + lat_max) / 2 + random.uniform(-0.1, 0.1)
        target_lon = (lon_min + lon_max) / 2 + random.uniform(-0.1, 0.1)
        
        if side == 0:   start_lat, start_lon = lat_max, random.uniform(lon_min, lon_max)
        elif side == 1: start_lat, start_lon = lat_min, random.uniform(lon_min, lon_max)
        elif side == 2: start_lat, start_lon = random.uniform(lat_min, lat_max), lon_min
        else:           start_lat, start_lon = random.uniform(lat_min, lat_max), lon_max
        
        speed_kmh = random.uniform(25, 40) # 降雨系统移动速度 (km/h)
        speed_km_min = speed_kmh / 60.0
        
        # 计算移动单位向量（转换为经纬度增量）
        delta_lat_km = (target_lat - start_lat) * 111.0
        delta_lon_km = (target_lon - start_lon) * 96.0
        dist_km = math.sqrt(delta_lat_km**2 + delta_lon_km**2) or 1
        
        v_lat_deg = ((delta_lat_km / dist_km) * speed_km_min) / 111.0
        v_lon_deg = ((delta_lon_km / dist_km) * speed_km_min) / 96.0
        return (start_lat, start_lon), (v_lat_deg, v_lon_deg)

    def move_step(self):
        """每一分钟移动一次降雨中心，并清空距离缓存（因为中心变了，所有距离需重算）"""
        curr_lat, curr_lon = self.center_loc
        v_lat, v_lon = self.velocity
        self.center_loc = (curr_lat + v_lat, curr_lon + v_lon)
        self._distance_cache = {} 

    def _precompute_chicago_profile(self):
        """
        预计算芝加哥雨型（Chicago Design Storm）的时间分布。
        公式来源：Keifer & Chu (1957)，参数 A, n, b 针对上海地区暴雨强度公式适配。
        """
        times = np.arange(0, self.rain_duration + 1, 1) 
        intensities = []
        t_peak_time = self.rain_duration * self.r_peak
        
        # A, n, b 为上海地区的降雨公式参数（需根据研究区域调整）
        A = 9.581 * (1 + 0.846 * np.log10(self.P_period))
        n = 0.656
        b = 7
        
        for t in times:
            # 峰前时刻公式
            if t <= t_peak_time:
                t_b = t_peak_time - t
                num = A * ((1 - n) * (t_b / self.r_peak) + b)
                den = math.pow((t_b / self.r_peak) + b, 1 + n)
            # 峰后时刻公式
            else:
                t_a = t - t_peak_time
                num = A * ((1 - n) * (t_a / (1 - self.r_peak)) + b)
                den = math.pow((t_a / (1 - self.r_peak)) + b, 1 + n)
            intensities.append(num / den)
        
        # 模拟雨后由于排水不畅导致的“长尾效应”（指数衰减）
        tail_duration = 120  
        if intensities:
            last_val = intensities[-1]
            for t_ex in range(1, tail_duration + 1):
                val = last_val * math.exp(-0.1 * t_ex)
                intensities.append(val if val > 0.001 else 0.0)
        
        # 存储为累计降雨量 (Cumulative Rainfall)，方便后续计算径流
        self.chicago_profile = np.cumsum(intensities)

    def _load_drainage_image(self):
        """加载排水能力栅格图，用于根据地理位置查询排水效率"""
        path = os.path.join(self.base_path, "排水能力_简化.png")
        try:
            img = Image.open(path)
            self.img_array = np.array(img)
        except FileNotFoundError:
            # 如果没图，生成一个随机的（调试用）
            self.img_array = np.random.randint(0, 255, (1000, 1000, 3), dtype=np.uint8)

    def _haversine(self, lat1, lon1, lat2, lon2):
        """计算地球表面两点间的大圆距离 (km)"""
        R = 6371.0 
        dlat = math.radians(lat2 - lat1)
        dlon = math.radians(lon2 - lon1)
        a = math.sin(dlat/2)**2 + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(dlon/2)**2
        return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))

    def _lat_lon_to_pixel(self, lat, lon):
        """经纬度坐标与图像像素坐标的转换"""
        if self.img_array is None: return 0, 0
        h, w = self.img_array.shape[:2]
        lat_min, lat_max = self.lat_range
        lon_min, lon_max = self.lon_range
        px = int((lon - lon_min) / (lon_max - lon_min) * w)
        py = int((lat_max - lat) / (lat_max - lat_min) * h)
        return min(max(px, 0), w - 1), min(max(py, 0), h - 1)

    def _get_drainage_from_image(self, lat, lon):
        """从像素颜色值映射到实际排水能力 (mm/h)"""
        px, py = self._lat_lon_to_pixel(lat, lon)
        # 获取灰度值或R通道值
        r_val = self.img_array[py, px][0] if self.img_array.ndim == 3 else self.img_array[py, px]
        r_norm = r_val / 255.0
        # 简单的阈值映射规则
        if r_val == 0: return 27
        elif r_norm <= 0.9: return 36
        elif r_norm <= 0.99: return 27
        else: return 50

    def _calculate_arf_rainfall(self, distance):
        """空间衰减计算：根据距雨中心距离计算面雨量折减系数 (Area Reduction Factor, ARF)"""
        area = math.pi * (distance**2)
        # ARF 经验公式
        factor = -0.058 * math.log(area) + 0.9949 if area >= 1 else 1.0
        Q_temp = self.Q0 * factor
        # 增加空间随机扰动
        return max(0, Q_temp + np.random.normal(0, 0.3 * abs(Q_temp)))

    def get_instantaneous_intensity(self, time_step, position, precomputed_dist=None):
        """
        获取某时刻、某位置的瞬时降雨强度 (mm/h)。
        主要用于地上线路的能见度/运行风险判断。
        [修改说明] 引入60分钟滑动平均，解决芝加哥雨型瞬时峰值过高的问题。
        """
        if precomputed_dist is not None: dist = precomputed_dist
        else:
            if position in self._distance_cache: dist = self._distance_cache[position]
            else:
                dist = self._haversine(self.center_loc[0], self.center_loc[1], position[0], position[1])
                self._distance_cache[position] = dist
                
        spatial_val = self._calculate_arf_rainfall(dist)
        
        if self.rainfall_type == 'chicago':
            if time_step >= len(self.chicago_profile) - 1: return 0.0
            scale = spatial_val / self.Q0 if self.Q0 > 0 else 0
            
            # [核心修改] 使用 60分钟滑动窗口计算小时雨强
            idx_now = min(int(time_step), len(self.chicago_profile) - 1)
            window_size = 60 
            idx_prev = max(0, idx_now - window_size)
            
            # 这一时段内的总雨量 (mm)
            rain_depth_in_window = self.chicago_profile[idx_now] - self.chicago_profile[idx_prev]
            
            # 时段长度 (min)
            duration = idx_now - idx_prev
            if duration <= 0: return 0.0
            
            # 换算成小时雨强 (mm/h)
            intensity_hourly = (rain_depth_in_window / duration) * 60.0
            
            return max(0, intensity_hourly * scale)
        else:
            # 简单均匀降雨模式
            return max(0, spatial_val / (self.rain_duration / 60.0)) if self.rain_duration > 0 else 0

    def get_intensity(self, time_step, position, precomputed_dist=None):
        """
        核心计算：计算地表净积水深度 (mm)。
        使用了 SCS-CN (Soil Conservation Service Curve Number) 径流模型的简化形式。
        Logic: 积水 = 降雨径流(Runoff) - 城市管网排水(Drainage)
        """
        lat, lon = position
        if precomputed_dist is not None: dist = precomputed_dist
        else:
            if position in self._distance_cache: dist = self._distance_cache[position]
            else:
                dist = self._haversine(self.center_loc[0], self.center_loc[1], lat, lon)
                self._distance_cache[position] = dist
        spatial_val = self._calculate_arf_rainfall(dist)
        
        # 1. 计算累计降雨量 (Cumulative Rainfall)
        if self.rainfall_type == 'chicago':
            scale = spatial_val / self.Q0 if self.Q0 > 0 else 0
            idx = min(int(time_step), len(self.chicago_profile) - 1)
            cum_rain = self.chicago_profile[idx] * scale
        else:
            progress = min(time_step / float(self.rain_duration), 1.0)
            cum_rain = spatial_val * progress
            
        # 2. SCS 径流转换
        S = 15.7 # 潜在最大入渗量 (Potential Maximum Retention)，与土壤/地表类型有关
        # SCS方程：Q = (P - 0.2S)^2 / (P + 0.8S)
        runoff = (cum_rain - 0.05 * S)**2 / (cum_rain + 0.95 * S) if cum_rain > 0.05 * S else 0
        
        # 3. 扣除区域排水
        drainage = (self._get_drainage_from_image(lat, lon) / 60.0) * time_step
        return max(0, runoff - drainage) # 返回值为净积水深度

    def get_speed_impact_factor(self, water_depth, edge_type, line_context=None):
        """
        计算速度折减系数：将物理积水深度转换为对系统运行/步行速度的影响 (0.0 - 1.0)。
        1.0 表示正常速度，0.0 表示完全中断。
        """
        # [修改] 提取线路整体最大积水深度，用于判断是否触发全线停运
        # 如果 max_water_total > 450，意味着线路上某处已严重积水，该线路应整体停运
        max_water_line = 0.0
        if line_context:
            max_water_line = line_context.get('max_water_total', 0.0)

        # [修改] 核心逻辑变更：
        # 如果当前站点水深 > 450mm OR 线路任意位置水深 > 450mm (由 line_context 传入)，
        # 则视为线路瘫痪。
        is_line_failed = (water_depth > 450) or (max_water_line > 450)

        if edge_type == 'boarding': 
            # 进站/步行速度折减 (基于经验研究的步速-水深函数，如 Ishigaki et al.)
            # [修改] 若线路瘫痪（不论是本站淹水还是线路其他站淹水），进站系数设为 0.0001
            if is_line_failed:
                return 0.0001
            
            if water_depth <= 0: return 1.0
            if water_depth <= 250: return max(0.0001, 0.9 - (water_depth / 250.0) * 0.1)
            elif water_depth <= 450: return 0.0001
            else: return 0.0001 # 兜底逻辑，实际上 is_line_failed 已覆盖
            
        elif edge_type == 'riding': 
            # 乘车运行速度折减
            if line_context:
                is_above = line_context.get('is_above_ground', 0)
                max_rain = line_context.get('max_rain_above', 0.0)
                # max_water_line 已经在上面提取了
                
                if is_above == 1: # 地上线主要受瞬时强降雨（能见度）影响
                    if max_rain > 100: return 0.0001 # [修改] 极强降雨停运，因子改为 0.0001
                    elif max_rain >= 80: return 0.5 # 强降雨限速
                else: # 地下线主要受区间进水影响
                    # [修改] 若线路上最大积水超过450mm，列车停运，影响因子变为0.0001
                    if max_water_line > 450: return 0.0001
            return 1.0

# ==============================================================================
# [新增] 真实数据降雨策略类
# ==============================================================================
class RealDataRainfall(DisturbanceStrategy):
    """
    真实数据降雨策略：基于 SQLite 数据库中的历史/实时降雨数据进行空间插值。
    模拟 2025-07-29 等特定日期的真实降雨分布。
    """
    def __init__(self, base_path, db_name="shanghai_rain_history.db", start_datetime_str="2025-07-29 00:00"):
        self.base_path = base_path
        self.db_path = os.path.join(base_path, db_name)
        self.start_dt = datetime.strptime(start_datetime_str, "%Y-%m-%d %H:%M")
        self.center_loc = (31.2304, 121.4737) # 默认中心设为上海市中心，适配接口（真实数据无中心概念）
        
        # 缓存机制，避免每分钟每节点都去查库插值
        self._last_query_time_str = None
        self._cached_grid_interpolator = None # 存储 griddata 的结果或KDTree
        self._cached_df = None
        
        # 上海区域网格参数
        self.SH_LON_MIN, self.SH_LON_MAX = 120.8, 122.3
        self.SH_LAT_MIN, self.SH_LAT_MAX = 30.6, 31.9
        
        self.img_array = None
        self._load_drainage_image()

    def _load_drainage_image(self):
        """复用父类的排水图加载逻辑"""
        path = os.path.join(self.base_path, "排水能力_简化.png")
        try:
            img = Image.open(path)
            self.img_array = np.array(img)
        except:
            self.img_array = np.random.randint(0, 255, (1000, 1000, 3), dtype=np.uint8)

    def _lat_lon_to_pixel(self, lat, lon):
        """复用父类的坐标转换逻辑"""
        if self.img_array is None: return 0, 0
        h, w = self.img_array.shape[:2]
        # 注意：这里需要与 ExtremeRainfall 的范围保持一致，或者使用本类定义的范围
        # 假设排水图是覆盖 self.SH_LAT_MIN/MAX 范围的
        lat_min, lat_max = 30.6984, 31.5139 # 这里沿用 ExtremeRainfall 的范围设定以匹配图片
        lon_min, lon_max = 120.849, 121.966
        px = int((lon - lon_min) / (lon_max - lon_min) * w)
        py = int((lat_max - lat) / (lat_max - lat_min) * h)
        return min(max(px, 0), w - 1), min(max(py, 0), h - 1)

    def _get_drainage_from_image(self, lat, lon):
        """复用父类的排水能力获取逻辑"""
        px, py = self._lat_lon_to_pixel(lat, lon)
        r_val = self.img_array[py, px][0] if self.img_array.ndim == 3 else self.img_array[py, px]
        r_norm = r_val / 255.0
        if r_val == 0: return 27
        elif r_norm <= 0.9: return 36
        elif r_norm <= 0.99: return 27
        else: return 50
    
    # 距离计算方法
    def _haversine(self, lat1, lon1, lat2, lon2):
        """计算地球表面两点间的大圆距离 (km)"""
        R = 6371.0 
        dlat = math.radians(lat2 - lat1)
        dlon = math.radians(lon2 - lon1)
        a = math.sin(dlat/2)**2 + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(dlon/2)**2
        return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))
    
    def move_step(self):
        """真实数据不需要移动风暴中心，此方法留空适配接口"""
        pass

    def _fetch_and_interpolate(self, current_time_str):
        """从数据库读取特定时间的数据并构建插值函数"""
        # 如果时间没变，直接返回缓存
        if current_time_str == self._last_query_time_str and self._cached_df is not None:
            return
        
        self._last_query_time_str = current_time_str
        conn = None
        try:
            conn = sqlite3.connect(self.db_path)
            # 使用 LIKE 模糊匹配时间 (e.g., '2025-07-29 05:00%')
            query = """
                SELECT s.lon, s.lat, r.rain_value 
                FROM rain_data r
                JOIN stations s ON r.station_id = s.station_id
                WHERE r.datetime LIKE ?
            """
            df = pd.read_sql_query(query, conn, params=(f"{current_time_str}%",))
            
            if len(df) < 3:
                # 数据不足时，假设无雨
                self._cached_df = pd.DataFrame({'lon': [], 'lat': [], 'rain_value': []})
            else:
                self._cached_df = df
        except Exception as e:
            # print(f"DB Error: {e}") 
            self._cached_df = pd.DataFrame({'lon': [], 'lat': [], 'rain_value': []})
        finally:
            if conn: conn.close()

    def get_instantaneous_intensity(self, time_step, position):
        """获取瞬时雨强 (mm/h)"""
        # 计算当前绝对时间
        curr_dt = self.start_dt + timedelta(minutes=time_step)
        # 数据库是每小时数据 (e.g. 05:00, 06:00)，这里简单取最近的小时，或者按分钟插值
        # 为了性能，这里取整点小时 (format: YYYY-MM-DD HH)
        # 如果需要更精细，可以修改为 HH:MM 并确保数据库有该精度，或者在两个小时之间做线性插值
        # 这里采用：取当前小时的数据（阶梯状变化）
        target_time_str = curr_dt.strftime("%Y-%m-%d %H") # 匹配数据库的 '2025-07-29 05' 开头
        
        self._fetch_and_interpolate(target_time_str)
        
        if self._cached_df is None or len(self._cached_df) == 0:
            return 0.0
        
        # 使用 Nearest 或 Linear 插值获取特定位置的雨量
        # 由于每次调用 griddata 开销大，这里针对单点计算：
        # 实际上 griddata 适合生成网格。对于单点查询，计算距离最近的几个站做 IDW 更快。
        # 这里简化使用 scipy.interpolate.griddata 对单点插值
        points = self._cached_df[['lat', 'lon']].values
        values = self._cached_df['rain_value'].values
        
        # 注意：griddata 输入是 (points, values, xi)
        # points: (N, 2), xi: (M, 2)
        try:
            val = griddata(points, values, np.array([position]), method='nearest')[0]
            return max(0, val)
        except:
            return 0.0

    def get_intensity(self, time_step, position, precomputed_dist=None):
        """
        计算地表积水深度 (mm)。
        对于真实数据，我们假设数据库里存的是“小时雨量 (mm)”。
        为了计算积水，我们需要累计雨量。
        注意：这是一个简化处理。真实的积水模拟需要连续的时间积分。
        这里我们假设：积水 = 当前时刻瞬时雨强累积 - 排水。
        """
        lat, lon = position
        
        # 获取瞬时雨强 (假设数据库存储的是 mm/h 或者该小时的累计雨量)
        intensity_mm_h = self.get_instantaneous_intensity(time_step, position)
        
        # 计算当前时间步（1分钟）带来的新增雨量
        rain_inc = intensity_mm_h / 60.0 
        
        # 在这里，DisturbanceStrategy 是无状态的(stateless)关于“积水历史”的吗？
        # ExtremeRainfall 类里是用 cumulative profile 算的。
        # 如果我们没有累计曲线，我们只能近似。
        # 方案：假设过去 N 小时的雨都下下来了，扣除排水。
        # 但这非常慢。
        # 妥协方案：SubwayDIIMSystem 的 G.nodes[n]['water'] 是每一帧刷新的。
        # 如果 ExtremeRainfall 是每次重算 total accumulation，那这里也得这么做。
        # 但查询历史所有小时的数据太慢了。
        
        # *修正*：SubwayDIIMSystem.step() 里是直接赋值：
        # self.G.nodes[n]['water'] = self.strategy.get_intensity(...)
        # 所以 get_intensity 必须返回“当前积水深度”。
        
        # 对于 RealData，我们难以每次从 0时刻开始积分。
        # 建议：仅返回瞬时雨强导致的“潜在积水”，或者基于简单的“当前雨强 * 因子”来近似积水风险。
        # 或者，为了更准确，我们在 SimulationSystem 里维护积水状态，而不是在这里计算 total。
        # 但为了不改动 System 的逻辑（System 假设 Strategy 返回积水深度），
        # 我们这里做一个近似： 积水深度 ~= 最近 1 小时的雨量 * 径流系数 - 排水 * 1小时
        
        S = 15.7
        # 假设 effective rain 是过去 60 分钟的累积（即当前 hourly value）
        cum_rain = intensity_mm_h # 粗略假设
        
        runoff = (cum_rain - 0.05 * S)**2 / (cum_rain + 0.95 * S) if cum_rain > 0.05 * S else 0
        
        # 排水能力 (mm/h)
        drainage_rate = self._get_drainage_from_image(lat, lon)
        
        # 估算积水：径流 - 排水（假设排水一直在工作）
        # 这里仅仅是一个状态快照，无法完美模拟时序累积，除非重构 System
        water_depth = max(0, runoff - drainage_rate) 
        
        return water_depth

    def get_speed_impact_factor(self, water_depth, edge_type, line_context=None):
        """复用父类的速度折减逻辑"""
        # [修改] 提取线路整体最大积水深度
        max_water_line = 0.0
        if line_context:
            max_water_line = line_context.get('max_water_total', 0.0)

        # [修改] 判断是否全线停运
        is_line_failed = (water_depth > 450) or (max_water_line > 450)

        if edge_type == 'boarding': 
            # [修改] 若线路瘫痪，进站系数设为 0.0001
            if is_line_failed:
                return 0.0001
                
            if water_depth <= 0: return 1.0
            if water_depth <= 250: return max(0.01, 0.9 - (water_depth / 250.0) * 0.1)
            elif water_depth <= 450: return max(0.0001, 0.8 - ((water_depth - 250) / 200.0) * 0.7)
            else: return 0.0001 
            
        elif edge_type == 'riding': 
            if line_context:
                is_above = line_context.get('is_above_ground', 0)
                max_rain = line_context.get('max_rain_above', 0.0)
                
                if is_above == 1: 
                    if max_rain > 100: return 0.0001  # [修改] 极强降雨停运
                    elif max_rain >= 80: return 0.5 
                else: 
                    if max_water_line > 450: return 0.0001 # [修改] 地下线受淹停运
            return 1.0


# ==============================================================================
# 2. 核心系统：地铁网络拓扑与 DIIM (动态进程间影响模型) 模拟
# ==============================================================================
class SubwayDIIMSystem:
    """
    地铁动态影响模拟系统 (Dynamic Inoperability Input-Output Model System)
    
    主要功能：
    1. 构建双层网络图（Station Layer + Line Layer）。
    2. 加载静态数据（拓扑、时刻表、客流）。
    3. 执行时间步进仿真，计算 DIIM 状态更新。
    4. 计算网络性能指标（加权全局效率）。
    """
    def __init__(self, base_path, disturbance_strategy, sim_start_time='05:30', real_flow_df=None):
        self.base_path = base_path
        self.strategy = disturbance_strategy
        self.real_flow_df = real_flow_df # [新增] 接收真实客流数据 DataFrame
        self.G = nx.MultiGraph() # 使用多重图构建分层网络，允许两个节点间存在多种关系的边
        
        try:
            t_obj = datetime.strptime(sim_start_time, "%H:%M")
            self.sim_start_min = t_obj.hour * 60 + t_obj.minute
            self.sim_start_dt_obj = t_obj # 保存 datetime 对象用于计算
        except: 
            self.sim_start_min = 330
            self.sim_start_dt_obj = datetime(2000, 1, 1, 5, 30)
        
        # 内部缓存与索引映射
        self.id_map_long_to_short = {}
        self.line_stations_index = {}
        self.schedule_rules = [] 
        self.schedule_lookup = {}
        self.station_pos_cache = {}
        self.fast_edge_data = [] # 扁平化的边数据列表，用于极速遍历（避开NX图查询开销）
        self.node_idx_map = {}
        self.node_pos_list = []
        self.node_dists_list = []
        self.node_st_names = []
        
        # 顺序执行初始化流程
        self._load_id_mapping()           # 1. 映射线路ID (Excel -> Dict)
        self._load_json_topology()        # 2. 加载基础拓扑 (JSON)
        self._load_detailed_schedule()    # 3. 加载列车时刻/发车间隔
        self._build_station_index()       # 4. 构建索引
        self._build_graph_topology()      # 5. 构建分层网络图 (Graph Building)
        self._apply_ground_elevation_status() # 6. 标记地上/地下线
        self._load_passenger_flow_data()  # 7. 加载客流权重 (静态初始值)
        self._build_single_layer_network()# 8. 构建单层简化网 (用于辅助计算)
        self._init_diim_matrices()        # 9. 初始化动态影响矩阵 (DIIM Init)
        
        # 韧性恢复参数（设定恢复速度）
        # c(t) 为受损程度，恢复过程遵循指数衰减
        T_rec = 15.0 
        self.k_recovery = (1.0 / T_rec) * math.log(1.0 / 0.01)
        
        self._init_flow_weights()         # 初始化流权重
        self._precompute_static_simulation_data() # 预处理静态数据加速计算
        
        # 计算初始（无故障）状态下的加权网络效率，作为后续归一化的分母
        self.Ef0 = self.calculate_weighted_efficiency(self.sim_start_min, 'Weekday')
        self.congestion_warning_triggered = False

    def _load_id_mapping(self):
        """从Excel加载线路名称与短ID（如：1号线 -> "1"）的映射关系"""
        path = os.path.join(self.base_path, "线路id与名称对应-20251210.xlsx")
        try:
            if os.path.exists(path): df = pd.read_excel(path)
            else: df = pd.read_csv(path + " - Sheet1.csv")
            for _, row in df.iterrows():
                if pd.isna(row['Line ID']): continue
                long_id = str(int(row['Line ID'])) if isinstance(row['Line ID'], float) else str(row['Line ID'])
                line_name = str(row['Line Name'])
                short_id = None
                if "浦江" in line_name: short_id = "PJ"
                elif "磁浮" in line_name: short_id = "Maglev"
                elif "号线" in line_name:
                    match = re.search(r'(\d+)', line_name)
                    if match: short_id = match.group(1)
                if short_id: self.id_map_long_to_short[long_id] = short_id
        except: pass

    def _load_json_topology(self):
        """读取地铁拓扑结构 JSON 文件"""
        path = os.path.join(self.base_path, "subway.json")
        with open(path, "r", encoding="utf-8") as f: self.raw_json = json.load(f)

    def _load_detailed_schedule(self):
        """加载详细的发车间隔规则（区分早晚高峰、工作日/周末、区间车）"""
        path = os.path.join(self.base_path, "line_schedule_detailed.xlsx")
        try:
            df = pd.read_excel(path)
            df['LineID'] = df['LineID'].apply(lambda x: str(int(x)) if isinstance(x, float) else str(x).strip())
            for _, row in df.iterrows():
                s_dt = pd.to_datetime(str(row['Start_Time']), format='%H:%M:%S', errors='coerce')
                e_dt = pd.to_datetime(str(row['End_Time']), format='%H:%M:%S', errors='coerce')
                rule = {
                    'line_id': row['LineID'], 'day_type': str(row['DayType']).strip(),
                    'start_min': s_dt.hour*60+s_dt.minute if pd.notnull(s_dt) else 0,
                    'end_min': e_dt.hour*60+e_dt.minute if pd.notnull(e_dt) else 1440,
                    'from_st': str(row['From_Station']).strip(), 'to_st': str(row['To_Station']).strip(),
                    'interval': float(row['Interval_Sec'])
                }
                self.schedule_rules.append(rule)
                key = (rule['line_id'], rule['day_type'])
                if key not in self.schedule_lookup: self.schedule_lookup[key] = []
                self.schedule_lookup[key].append(rule)
        except: pass

    def _build_station_index(self):
        """构建车站在线路中的顺序索引，用于后续判断某车站是否在区间车的覆盖范围内"""
        for line_info in self.raw_json['l']:
            short_id = self.id_map_long_to_short.get(str(line_info['ls']))
            if not short_id: continue
            st_names = [st['n'] for st in line_info['st']]
            if short_id not in self.line_stations_index: 
                self.line_stations_index[short_id] = []
                self.station_pos_cache[short_id] = {}
            for st in st_names:
                if st not in self.line_stations_index[short_id]: 
                    self.line_stations_index[short_id].append(st)
                    self.station_pos_cache[short_id][st] = len(self.line_stations_index[short_id]) - 1

    def _is_station_in_section(self, line_id, station_name, from_st, to_st):
        """逻辑判定：特定车站是否位于指定的调度区间（from_st 到 to_st）内"""
        line_cache = self.station_pos_cache.get(line_id)
        if not line_cache: return False
        idx_s, idx_f, idx_t = line_cache.get(station_name), line_cache.get(from_st), line_cache.get(to_st)
        if idx_s is None or idx_f is None or idx_t is None: return False
        return min(idx_f, idx_t) <= idx_s <= max(idx_f, idx_t)

    def _build_graph_topology(self):
        """
        构建三维分层模型 (Space-L space)：
        1. 车站层 (Station Layer): 物理车站位置。
        2. 线路层 (Line Layer): 特定线路的站台。
        3. 边 (Edges):
           - boarding: 车站层 <-> 线路层 (进出站)。
           - riding: 线路层 <-> 线路层 (乘车)。
        """
        for line_info in self.raw_json['l']:
            short_id = self.id_map_long_to_short.get(str(line_info['ls']))
            if not short_id: continue
            sts = line_info['st']
            for st in sts:
                pos = tuple(map(float, st['sl'].split(',')))[::-1] # 转为 (lat, lon)
                node_st = f"{st['n']}_车站层"
                node_ln = f"{st['n']}_线路层_{short_id}"
                
                # 添加节点
                if not self.G.has_node(node_st): self.G.add_node(node_st, layer='station', pos=pos, flow=1000)
                self.G.add_node(node_ln, layer='line', pos=pos, line_id=short_id, station_name=st['n'])
                
                # 添加进站边 (Boarding/Alighting Link)
                self.G.add_edge(node_st, node_ln, type='boarding', weight=0)
            
            # 添加运行区间边 (Riding Link)
            for i in range(len(sts) - 1):
                u, v = f"{sts[i]['n']}_线路层_{short_id}", f"{sts[i+1]['n']}_线路层_{short_id}"
                if self.G.has_node(u) and self.G.has_node(v):
                    self.G.add_edge(u, v, type='riding', weight=0, line_id=short_id)

    def _apply_ground_elevation_status(self):
        """识别地铁的地上区间，这对于评估降雨暴露风险非常关键（地下区间受淹水影响，地上区间受雨强影响）"""
        file_path = os.path.join(self.base_path, "上海地铁地上线归纳.txt")
        above_set = set()
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                for line in f:
                    if '：' not in line: continue
                    parts = line.strip().split('：')
                    # 解析线路ID
                    match = re.search(r'(\d+)', parts[0])
                    lid = match.group(1) if match else ("PJ" if "浦江" in parts[0] else None)
                    if not lid: continue
                    # 解析区间
                    for group in parts[1].split('，'):
                        sts = group.strip().split('-')
                        for i in range(len(sts)-1):
                            pair = tuple(sorted((sts[i].strip(), sts[i+1].strip())))
                            above_set.add((lid, pair[0], pair[1]))
        except: pass
        
        # 遍历图中的边，标记 is_above_ground 属性
        for u, v, k, attr in self.G.edges(keys=True, data=True):
            if attr.get('type') == 'riding':
                lid = attr.get('line_id')
                stu, stv = self.G.nodes[u].get('station_name'), self.G.nodes[v].get('station_name')
                pair = tuple(sorted((stu, stv))) if stu and stv else None
                attr['is_above_ground'] = 1 if pair and (lid, pair[0], pair[1]) in above_set else 0

    def _load_passenger_flow_data(self):
        """从外部文件加载客流强度，用于后续加权网络效率的计算（客流大的站点权重更高）"""
        file_path = os.path.join(self.base_path, "进站客流.xlsx")
        p_flow = {}
        sum_f, num_f = 0, 0
        try:
            xls = pd.ExcelFile(file_path)
            for sheet in xls.sheet_names:
                for _, row in xls.parse(sheet, header=None).iterrows():
                    val = ((row[1] or 0) + (row[2] or 0)) / 2
                    p_flow[f"{row[0]}_车站层"] = p_flow.get(f"{row[0]}_车站层", 0) + val
                    sum_f += val; num_f += 1
        except: pass
        
        # 缺失值填充（使用平均值）
        avg_flow = sum_f / max(num_f, 1) or 1000
        avg_deg = np.mean([d for n, d in self.G.degree() if self.G.nodes[n].get('layer')=='station']) or 1
        
        for n, d in self.G.nodes(data=True):
            if d.get('layer') == 'station':
                d['flow'] = p_flow.get(n, (avg_flow/avg_deg)*self.G.degree(n))

    def _build_single_layer_network(self):
        """构建一个简化的单层站点间网络，用于辅助拓扑特征计算"""
        self.single_layer_net = nx.Graph()
        for u, v, attr in self.G.edges(data=True):
            if attr.get('type') == 'riding':
                n1 = self.G.nodes[u].get('station_name', u.split('_')[0])
                n2 = self.G.nodes[v].get('station_name', v.split('_')[0])
                if n1 != n2: self.single_layer_net.add_edge(n1, n2)

    def _init_flow_weights(self):
        """初始化边权：将节点流分配到相邻的进站边和运行边上"""
        ln_weights = {}
        # 1. 进站边权重
        for u, v, k, attr in self.G.edges(keys=True, data=True):
            if attr['type'] == 'boarding':
                st = u if self.G.nodes[u].get('layer')=='station' else v
                ln = v if st == u else u
                w = self.G.nodes[st].get('flow', 1000) / max(1, self.G.degree[st])
                attr['flow_weight'] = w
                ln_weights[ln] = w
        # 2. 运行边权重（取两端节点平均）
        for u, v, k, attr in self.G.edges(keys=True, data=True):
            if attr['type'] == 'riding':
                attr['flow_weight'] = (ln_weights.get(u, 100)+ln_weights.get(v, 100))/2

    def _precompute_static_simulation_data(self):
        """
        [极速仿真优化] 预计算所有不会随时间改变的数据结构。
        将图数据扁平化为 List/Array，避免在 step() 循环中频繁调用 NetworkX 的 lookup 方法。
        """
        nodes = list(self.G.nodes())
        self.node_idx_map = {n: i for i, n in enumerate(nodes)}
        self.node_dists_list = [0.0] * len(nodes)
        for i, n in enumerate(nodes):
            self.node_pos_list.append(self.G.nodes[n]['pos'])
            self.node_st_names.append(self.G.nodes[n].get('station_name', n.split('_')[0]))
        
        target_day_type = 'Weekday'
        for u, v, k in self.edge_list:
            attr = self.G.edges[u, v, k]
            u_idx, v_idx = self.node_idx_map[u], self.node_idx_map[v]
            lid = attr.get('line_id') or self.G.nodes[u].get('line_id') or self.G.nodes[v].get('line_id')
            est = {
                'u_idx': u_idx, 'v_idx': v_idx, 'type': attr['type'], 'line_id': lid,
                'is_above_ground': attr.get('is_above_ground', 0),
                'flow_weight': attr.get('flow_weight', 1.0),
                'boarding_dist': 0.0, 'boarding_node_flow': 1000.0,
                'riding_pos_mid': None, 'riding_applicable_rules': []
            }
            # 预绑定该边适用的调度规则（避免运行时搜索）
            if lid:
                rules = self.schedule_lookup.get((str(lid), target_day_type), [])
                st_u, st_v = self.node_st_names[u_idx], self.node_st_names[v_idx]
                check_st = st_u if attr['type'] == 'boarding' else None
                for r in rules:
                    if check_st:
                        if self._is_station_in_section(str(lid), check_st, r['from_st'], r['to_st']):
                             est['riding_applicable_rules'].append(r)
                    else:
                        if self._is_station_in_section(str(lid), st_u, r['from_st'], r['to_st']) and \
                           self._is_station_in_section(str(lid), st_v, r['from_st'], r['to_st']):
                             est['riding_applicable_rules'].append(r)
            if attr['type'] == 'boarding':
                st = self.node_st_names[u_idx]
                deg = self.single_layer_net.degree[st] if st in self.single_layer_net else 1
                est['boarding_dist'] = deg * 100.0 # 模拟车站内步行距离
                est['boarding_node_flow'] = self.G.nodes[u if self.G.nodes[u]['layer']=='station' else v].get('flow', 1000)
            else:
                p1, p2 = self.G.nodes[u]['pos'], self.G.nodes[v]['pos']
                est['riding_pos_mid'] = ((p1[0]+p2[0])/2, (p1[1]+p2[1])/2)
            self.fast_edge_data.append(est)

    def calculate_weighted_efficiency(self, current_time_min, day_type, line_stats=None):
        """
        计算效率，并包含终极路径追踪。
        """
        node_waters = [self.G.nodes[n].get('water', 0) for n in self.G.nodes()]
        
        # 1. 计算所有边权重
        for i, (u, v, k) in enumerate(self.edge_list):
            sd = self.fast_edge_data[i]
            
            # Rain
            avg_w = (node_waters[sd['u_idx']] + node_waters[sd['v_idx']]) / 2.0
            t_curr, _, _ = self._calc_times_fast(sd, avg_w, current_time_min, line_stats)
            
            # Base
            t_baseline, _, _ = self._calc_times_fast(sd, 0.0, current_time_min, line_stats=None)
            
            w = sd['flow_weight']
            
            # 这里的权重已经是 "时间 / 流量权重"
            weight_rain = (t_curr / w) if w > 0 and t_curr != float('inf') else float('inf')
            weight_base = (t_baseline / w) if w > 0 and t_baseline != float('inf') else float('inf')
            
            self.G.edges[u,v,k]['tau'] = weight_rain
            self.G.edges[u,v,k]['tau_baseline'] = weight_base
            
        st_nodes = [n for n, d in self.G.nodes(data=True) if d.get('layer') == 'station']
        st_set = set(st_nodes)
        N = len(st_nodes)
        if N < 2: return 0.0
        
        sum_inv_curr = 0.0
        sum_inv_base = 0.0
        
        found_anomaly_path = False
        
        # 2. 计算最短路径
        for src in st_nodes:
            # 获取 Rain 场景下的所有路径长度
            try:
                lens_rain = nx.single_source_dijkstra_path_length(self.G, src, weight='tau')
            except: lens_rain = {}
            
            # 获取 Base 场景下的所有路径长度
            try:
                lens_base = nx.single_source_dijkstra_path_length(self.G, src, weight='tau_baseline')
            except: lens_base = {}
            
            # 累加并检查
            for t in st_set:
                if t == src: continue
                
                l_r = lens_rain.get(t, float('inf'))
                l_b = lens_base.get(t, float('inf'))
                
                if l_r > 0: sum_inv_curr += 1.0/l_r
                if l_b > 0: sum_inv_base += 1.0/l_b
                
                # === 捕获异常路径 ===
                # 如果 Rain 距离 < Base 距离，说明 Rain 路径更短（更优），这是不可能的
                if l_r < l_b - 1e-6 and not found_anomaly_path:
                    print(f"\n🔥🔥🔥 [PATH ANOMALY DETECTED] at Time {current_time_min} 🔥🔥🔥")
                    print(f"Source: {src} -> Target: {t}")
                    print(f"Dist Rain: {l_r:.6f} < Dist Base: {l_b:.6f}")
                    
                    # 追踪 Rain 路径
                    try:
                        path = nx.shortest_path(self.G, src, t, weight='tau')
                        print("--- Rain Path Breakdown ---")
                        total_chk = 0
                        for i in range(len(path)-1):
                            u_p, v_p = path[i], path[i+1]
                            # 获取边数据 (取最小权重的那个key)
                            edge_data = self.G.get_edge_data(u_p, v_p)
                            # 找到 weight='tau' 最小的那个 key
                            best_k = min(edge_data, key=lambda k: edge_data[k]['tau'])
                            attr = edge_data[best_k]
                            
                            w_r = attr['tau']
                            w_b = attr['tau_baseline']
                            print(f"   {u_p} -> {v_p} | RainW={w_r:.4f} vs BaseW={w_b:.4f} | {'ERROR' if w_r < w_b else 'OK'}")
                            total_chk += w_r
                        print(f"   Check Sum: {total_chk:.4f}")
                    except:
                        print("   (Could not reconstruct path)")
                    
                    found_anomaly_path = True
        
        Ef = (1/(N*(N-1))) * sum_inv_curr
        Ef0 = (1/(N*(N-1))) * sum_inv_base
        
        return Ef / Ef0 if Ef0 > 0 else 1.0

    def _init_diim_matrices(self):
        """初始化动态进程间影响模型 (DIIM) 矩阵结构"""
        self.edge_list = list(self.G.edges(keys=True))
        self.edge_idx = {e: i for i, e in enumerate(self.edge_list)}
        n = len(self.edge_list)
        self.q_vector = np.zeros(n) # 存储各边的受损失效率 q
        self.A_matrix = sparse.csr_matrix((n, n))
        self.node_to_boarding_indices = {}
        for n_st in [n for n, d in self.G.nodes(data=True) if d['layer']=='station']:
            idxs = []
            for u, v, k in self.G.edges(n_st, keys=True):
                if self.G.edges[u,v,k]['type'] == 'boarding':
                    k_t = (u,v,k) if (u,v,k) in self.edge_idx else (v,u,k)
                    if k_t in self.edge_idx: idxs.append(self.edge_idx[k_t])
            self.node_to_boarding_indices[n_st] = idxs

    def _calc_times_fast(self, static_data, water_depth, current_time_min, line_stats=None):
        """
        计算特定边在特定积水深度下的瞬时通行时间。
        Returns: (当前受灾时间 t_curr, 正常理想时间 t_norm, 调试信息字典 debug_info)
        """
        lookup_time = current_time_min % 1440
        etype = static_data['type']
        
        # 准备调试信息
        debug_info = {}

        lid = static_data['line_id']
        ctx = None
        if lid and line_stats and str(lid) in line_stats:
            st_data = line_stats[str(lid)]
            ctx = {'is_above_ground': static_data.get('is_above_ground', 0), 
                   'max_rain_above': st_data['max_rain_above'], 
                   'max_water_total': st_data['max_water_total']}

        if etype == 'boarding':
            dist = static_data['boarding_dist']
            
            # 1. 速度计算
            speed_factor = self.strategy.get_speed_impact_factor(water_depth, 'boarding', line_context=ctx)
            v_curr = 1.4 * speed_factor
            
            # 记录调试信息
            debug_info['speed_factor'] = speed_factor
            debug_info['v_curr'] = v_curr
            debug_info['dist'] = dist

            if v_curr < 0.01: return float('inf'), 100.0, debug_info
            
            tw_c = dist / v_curr
            tw_n = dist / 1.4
            
            t_b = 30 # 列车停靠时间

            # 2. 等待时间计算
            t_wait_c, t_wait_n = 0, 0
            if static_data['line_id']:
                t_base = 300.0
                for r in static_data['riding_applicable_rules']:
                    if r['start_min'] <= lookup_time < r['end_min']:
                        t_base = r['interval']; break
                
                hw_n = t_base
                # 计算受影响后的发车间隔
                riding_factor = self.strategy.get_speed_impact_factor(water_depth, 'riding', line_context=ctx)
                hw_c = hw_n / max(riding_factor, 0.0001)
                
                # 排队论参数
                lam = max(static_data['boarding_node_flow'] / 3600.0, 0.0001)
                
                # 记录更多调试信息
                debug_info['hw_n'] = hw_n
                debug_info['hw_c'] = hw_c
                debug_info['riding_factor'] = riding_factor
                debug_info['lambda'] = lam

                # [修正开始] 定义单调性约束的排队等待时间计算函数
                def calc_w(EV, L):
                    # 1. 计算饱和度 (Saturation Ratio)
                    # 原分母公式 D = 2 * (1/L - 2*EV)
                    # 饱和状态意味着 1/L = 2*EV => L * 2 * EV = 1
                    # 因此定义 rho = L * 2 * EV
                    rho = L * 2 * EV
                    
                    # 设定安全阈值，超过此阈值认为进入拥堵/过饱和区
                    # 在此区域不再使用原公式（原公式有奇点且非单调），改用幂函数惩罚
                    RHO_THRESH = 0.9

                    # 计算阈值处的理论等待时间 (作为锚点)
                    # 当 rho = 0.9 时:
                    # denom_limit = 2 * (1/L - 2*EV) 
                    #             = 2/L * (1 - 2*EV*L) 
                    #             = 2/L * (1 - 0.9) = 0.2 / L
                    # 这是一个近似计算，为了性能，我们直接用硬逻辑判断
                    
                    if rho < RHO_THRESH:
                        # === 正常区域：使用原排队论公式 ===
                        denom = 2 * ((1.0/L) - 2*EV)
                        # 虽然理论上 rho < 0.9 保证 denom > 0，但加个由衷的兜底
                        if denom <= 1e-6: 
                            val = 1800.0 # 理论上不应走到这里
                        else:
                            val = 2.5*EV + (t_b**2 + 3*EV**2)/denom
                        
                        # 即使在正常区域，也允许适当的上限，但不要强制截断导致非单调
                        # 如果计算值确实很大（极罕见），让它保持很大，不要 clamp 到 1800
                        return min(val, 3600.0) 
                    
                    else:
                        # === 拥堵/过饱和区域：使用单调惩罚函数 ===
                        # 这里的逻辑是：既然已经堵了，那么发车间隔(EV)越大，rho越大，等待时间必须越长。
                        # 我们构造一个平滑过渡函数。
                        # 先算出在 RHO_THRESH (0.9) 时的边界值 T_limit
                        denom_limit = 2 * ((1.0/L) * (1 - RHO_THRESH))
                        t_limit = 2.5*EV + (t_b**2 + 3*EV**2)/denom_limit
                        
                        # 惩罚因子：(当前饱和度 / 阈值饱和度) 的平方
                        # 平方项确保了"越堵越慢"的非线性增长，有效拉开 Rain 和 Base 的差距
                        penalty_factor = (rho / RHO_THRESH) ** 2
                        
                        # 最终时间 = 边界时间 * 惩罚因子
                        # 这样即使 Base 很堵 (rho=0.95), Rain 更堵 (rho=1.1), 
                        # Rain 的结果也一定比 Base 大。
                        final_wait = t_limit * penalty_factor
                        
                        # 即使是非常极端的情况，也建议保留一个物理极限（如 2小时），
                        # 但这个极限必须足够大，以免再次出现 Base > Rain 的截断悖论
                        return min(final_wait, 7200.0) 

                # [修正结束]
                
                t_wait_n = calc_w(hw_n+t_b, lam)
                t_wait_c = calc_w(hw_c+t_b, lam)
                
                debug_info['t_wait_n'] = t_wait_n
                debug_info['t_wait_c'] = t_wait_c

            return tw_c + t_wait_c, tw_n + t_wait_n, debug_info
            
        elif etype == 'riding':
            t_base = 180.0
            for r in static_data['riding_applicable_rules']:
                if r['start_min'] <= lookup_time < r['end_min']:
                    t_base = r['interval']; break
            
            fac = self.strategy.get_speed_impact_factor(water_depth, 'riding', line_context=ctx)
            
            debug_info['t_base_interval'] = t_base
            debug_info['riding_factor'] = fac
            
            if fac <= 0.0001: return float('inf'), t_base, debug_info
            return t_base / fac, t_base, debug_info
            
        return 0, 0, debug_info

    def step(self, t_minute, day_type='Weekday'):
        """
        推进模拟时间步，并包含全网权重暴力一致性检查。
        """
        curr_clk = self.sim_start_min + t_minute
        
        # 1. 动态更新客流 (保持原逻辑)
        if self.real_flow_df is not None:
            curr_abs_time = None
            if hasattr(self.strategy, 'start_dt'):
                curr_abs_time = self.strategy.start_dt + timedelta(minutes=t_minute)
            
            if curr_abs_time:
                lookup_h = curr_abs_time.replace(minute=0, second=0, microsecond=0)
                try:
                    if curr_abs_time.minute == 0 or t_minute == 0:
                        batch = self.real_flow_df[self.real_flow_df['time'] == lookup_h]
                        if not batch.empty:
                            for _, row in batch.iterrows():
                                st_name = row['station']
                                flow_val = (row['inbound'] + row['outbound']) / 2
                                node_st = f"{st_name}_车站层"
                                if self.G.has_node(node_st):
                                    self.G.nodes[node_st]['flow'] = flow_val
                                    
                            for est in self.fast_edge_data:
                                if est['type'] == 'boarding':
                                    st_node_idx = est['u_idx'] if self.G.nodes[list(self.G.nodes())[est['u_idx']]].get('layer') == 'station' else est['v_idx']
                                    st_node_name = list(self.G.nodes())[st_node_idx]
                                    est['boarding_node_flow'] = self.G.nodes[st_node_name].get('flow', 1000)
                except Exception as e:
                    pass

        self.strategy.move_step()
        clat, clon = self.strategy.center_loc
        
        # 2. 更新积水
        for i, (lat, lon) in enumerate(self.node_pos_list):
            self.node_dists_list[i] = self.strategy._haversine(clat, clon, lat, lon)
        for i, n in enumerate(self.G.nodes()):
            w = self.strategy.get_intensity(t_minute, self.G.nodes[n]['pos'], precomputed_dist=self.node_dists_list[i])
            self.G.nodes[n]['water'] = w
            
        # 3. 统计线路风险
        line_stats = {}
        node_waters = [self.G.nodes[n].get('water', 0) for n in self.G.nodes()]
        for sd in self.fast_edge_data:
            if sd['type'] == 'riding' and sd['line_id']:
                lid = str(sd['line_id'])
                if lid not in line_stats: line_stats[lid] = {'max_rain_above': 0.0, 'max_water_total': 0.0}
                aw = (node_waters[sd['u_idx']] + node_waters[sd['v_idx']]) / 2
                if aw > line_stats[lid]['max_water_total']: line_stats[lid]['max_water_total'] = aw
                if sd['is_above_ground']:
                    r = self.strategy.get_instantaneous_intensity(t_minute, sd['riding_pos_mid'])
                    if r > line_stats[lid]['max_rain_above']: line_stats[lid]['max_rain_above'] = r
                    
        # 4. DIIM 更新与全网扫描
        curr_c = np.zeros(len(self.edge_list))
        
        # [暴力扫描] 统计计数器
        scan_count_rain_faster = 0
        scan_total_edges = 0
        scan_debug_msg = ""

        for i, sd in enumerate(self.fast_edge_data):
            aw = (node_waters[sd['u_idx']] + node_waters[sd['v_idx']]) / 2
            
            # 计算 Rain 时间
            tc, tn, _ = self._calc_times_fast(sd, aw, curr_clk, line_stats)
            
            # 计算 Base 时间 (强制无雨)
            tb, _, _ = self._calc_times_fast(sd, 0.0, curr_clk, line_stats=None)
            
            # --- 核心检查点 ---
            # 只要 Rain 比 Base 快超过 0.0001秒，就记录下来
            if tc < tb - 0.0001:
                scan_count_rain_faster += 1
                if scan_count_rain_faster <= 3: # 只记录前3个，防止刷屏
                    scan_debug_msg += f"\n   [BAD EDGE] {self.node_st_names[sd['u_idx']]}->{self.node_st_names[sd['v_idx']]} ({sd['type']}) | Rain={tc:.4f} < Base={tb:.4f}"

            # DIIM 逻辑
            val = 0.0
            if tc == float('inf'): val = 1.0 
            elif tc > 0: val = 1.0 - (tn / tc)
            curr_c[i] = max(0.0, min(1.0, val)) 
            scan_total_edges += 1
        
        # [打印扫描结果]
        if scan_count_rain_faster > 0:
            print(f"\n🚨 [CRITICAL SCAN] Time {t_minute}: Found {scan_count_rain_faster}/{scan_total_edges} edges where Rain is FASTER!")
            print(scan_debug_msg)
        else:
            # 如果没有发现异常边，打印一条确认信息
            # print(f"✅ [SCAN OK] Time {t_minute}: All edges Rain >= Base.")
            pass

        q_next = (1 - self.k_recovery) * self.q_vector + self.k_recovery * curr_c
        self.q_vector = np.clip(q_next, 0, 1)
        
        for i, (u,v,k) in enumerate(self.edge_list): self.G.edges[u,v,k]['q_value'] = self.q_vector[i]
        aff_lines = set()
        for i, q in enumerate(self.q_vector):
            if q > 0.1:
                lid = self.fast_edge_data[i]['line_id']
                if lid: aff_lines.add(lid)
        node_risks = []
        for n, idxs in self.node_to_boarding_indices.items():
            if idxs: node_risks.append(max(self.q_vector[i] for i in idxs))
            else: node_risks.append(0)
        
        self.current_line_stats = line_stats
        avg_water = np.mean(list(node_waters))
        max_n_risk = max(node_risks) if node_risks else 0
        
        avg_q_safe = np.nanmean(self.q_vector) if len(self.q_vector) > 0 else 0.0
        avg_q_safe = np.nan_to_num(avg_q_safe)
        
        return avg_q_safe, avg_water, len(aff_lines), max_n_risk

# ==============================================================================
# 3. 蒙特卡洛数据记录器：收集并保存多次随机实验的数据
# ==============================================================================
class MonteCarloRecorder:
    def __init__(self, time_steps):
        self.times = time_steps
        self.df_eff = pd.DataFrame({'Time': time_steps})
        self.df_risk = pd.DataFrame({'Time': time_steps})
        self.df_water = pd.DataFrame({'Time': time_steps})
        self.df_intensity = pd.DataFrame({'Time': time_steps})
        self.df_lat = pd.DataFrame({'Time': time_steps})
        self.df_lon = pd.DataFrame({'Time': time_steps})
        self.df_aff_lines = pd.DataFrame({'Time': time_steps})
        self.df_max_n_risk = pd.DataFrame({'Time': time_steps})

    def add_run_data(self, run_id, eff, risk, water, inten, lats, lons, aff_ln, max_nr):
        col_name = f"Run_{run_id}"
        self.df_eff[col_name] = eff
        self.df_risk[col_name] = risk
        self.df_water[col_name] = water
        self.df_intensity[col_name] = inten
        self.df_lat[col_name] = lats
        self.df_lon[col_name] = lons
        self.df_aff_lines[col_name] = aff_ln
        self.df_max_n_risk[col_name] = max_nr

    def save_to_excel(self, base_path):
        fname = os.path.join(base_path, f"MonteCarlo_Results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.xlsx")
        print(f"💾 正在导出蒙特卡洛数据到: {fname}")
        try:
            with pd.ExcelWriter(fname, engine='openpyxl') as writer:
                self.df_eff.to_excel(writer, sheet_name='Efficiency', index=False)
                self.df_risk.to_excel(writer, sheet_name='Risk_Avg_Q', index=False)
                self.df_water.to_excel(writer, sheet_name='Water_Level_Avg', index=False)
                self.df_intensity.to_excel(writer, sheet_name='Intensity_Center', index=False)
                self.df_lat.to_excel(writer, sheet_name='Location_Lat', index=False)
                self.df_lon.to_excel(writer, sheet_name='Location_Lon', index=False)
                self.df_aff_lines.to_excel(writer, sheet_name='Affected_Line_Count', index=False)
                self.df_max_n_risk.to_excel(writer, sheet_name='Max_Node_Risk', index=False)
            print("✅ 导出完成。")
        except Exception as e:
            print(f"❌ 导出失败: {e}")

# ==========================================
# 5. [Worker] 独立仿真任务：供多进程并行调用
# 注意：该函数必须定义在顶层，以便 Joblib 进行序列化 (Pickle)
# ==========================================
def simulation_worker(run_id, base_path, intensity, period, rain_dur, sim_dur, start_str, 
                      use_real_data=False, real_db_name=None, real_flow_data=None):
    try:
        # [修改] 支持真实数据模式初始化
        if use_real_data:
            strategy = RealDataRainfall(base_path, db_name=real_db_name, start_datetime_str=start_str)
            # 在真实模式下，我们传入客流 DataFrame
            sys_obj = SubwayDIIMSystem(base_path, strategy, sim_start_time=start_str, real_flow_df=real_flow_data)
        else:
            strategy = ExtremeRainfall(base_path, max_intensity=intensity, 
                                       rainfall_type='chicago', P=period, 
                                       rain_duration=rain_dur)
            
            # [修改] 为 ExtremeRainfall 模式注入客流数据和基准时间
            # 我们将模拟日期固定在无雨的 2025-08-08，以便使用这天的真实客流
            base_date_str = "2025-08-08"
            full_start_str = f"{base_date_str} {start_str}"
            try:
                # 手动给 strategy 绑定 start_dt，以便 SubwayDIIMSystem 能够计算绝对时间并查询客流
                strategy.start_dt = datetime.strptime(full_start_str, "%Y-%m-%d %H:%M")
            except ValueError:
                pass
                
            # 将外部传入的 real_flow_data (08.08的客流) 传给系统
            sys_obj = SubwayDIIMSystem(base_path, strategy, sim_start_time=start_str, real_flow_df=real_flow_data)
            
        t_obj = datetime.strptime(start_str, "%Y-%m-%d %H:%M") if len(start_str) > 5 else datetime.strptime(start_str, "%H:%M")
        start_min = t_obj.hour * 60 + t_obj.minute
        time_steps = list(range(sim_dur))
        r_eff, r_risk, r_water, r_inten, r_lat, r_lon, r_aff, r_mnr = [], [], [], [], [], [], [], []
        
        for t in time_steps:
            avg_q, avg_w, n_lines, max_nr = sys_obj.step(t)
            curr_clk = start_min + t
            # 计算该时间步的网络加权效率
            ef = sys_obj.calculate_weighted_efficiency(curr_clk, 'Weekday', line_stats=getattr(sys_obj, 'current_line_stats', None))
            
            # 对于真实数据，位置固定，返回 0 或默认中心
            c_lat, c_lon = (0, 0)
            if hasattr(sys_obj.strategy, 'center_loc'):
                c_lat, c_lon = sys_obj.strategy.center_loc
                
            # 瞬时雨强，如果是真实数据，传一个中心点坐标去获取，或者取平均？
            # 这里取 (31.23, 121.47) 上海中心点作为参考
            ref_pos = (31.2304, 121.4737)
            inten = sys_obj.strategy.get_instantaneous_intensity(t, ref_pos)
            
            r_eff.append(ef)
            r_risk.append(avg_q)
            r_water.append(avg_w)
            r_inten.append(inten)
            r_lat.append(c_lat)
            r_lon.append(c_lon)
            r_aff.append(n_lines)
            r_mnr.append(max_nr)
        
        # 强制垃圾回收，防止内存泄漏
        del sys_obj
        del strategy
        gc.collect() 
        return (run_id, r_eff, r_risk, r_water, r_inten, r_lat, r_lon, r_aff, r_mnr)
    except Exception as e:
        return (run_id, None, str(e), None, None, None, None, None, None)

# [新增] 数据一致性校验函数
def verify_data_consistency(base_path, flow_df):
    """
    检查 Excel 客流数据、JSON 拓扑数据、DB 降雨数据中的车站名称一致性。
    """
    print("\n🔍 正在进行数据一致性校验...")
    
    # 1. 获取 Excel 中的车站列表
    excel_stations = set(flow_df['station'].unique())
    
    # 2. 获取 JSON 中的车站列表
    json_path = os.path.join(base_path, "subway.json")
    json_stations = set()
    try:
        with open(json_path, "r", encoding="utf-8") as f: 
            data = json.load(f)
            for l in data['l']:
                for st in l['st']:
                    json_stations.add(st['n'])
    except:
        print("⚠️ 无法读取 subway.json，跳过拓扑校验。")

    # 3. 打印比对结果 (Excel vs JSON)
    if json_stations:
        missing_in_json = excel_stations - json_stations
        missing_in_excel = json_stations - excel_stations
        
        print(f"  - Excel 中有 {len(excel_stations)} 个车站")
        print(f"  - JSON 中有 {len(json_stations)} 个车站")
        
        if missing_in_json:
            print(f"⚠️ 警告: Excel 中有 {len(missing_in_json)} 个车站在 JSON 拓扑中找不到 (可能无法更新其客流):")
            print(f"    示例: {list(missing_in_json)[:5]} ...")
        else:
            print("✅ Excel 所有车站均已在 JSON 拓扑中找到。")
            
        if missing_in_excel:
            # 这通常不是问题，因为 Excel 可能只包含部分车站数据
            pass

    # 4. 获取 DB 中的车站列表 (简单检查)
    db_path = os.path.join(base_path, "shanghai_rain_history.db")
    if os.path.exists(db_path):
        try:
            conn = sqlite3.connect(db_path)
            cursor = conn.cursor()
            cursor.execute("SELECT DISTINCT station_name FROM stations") # 假设表名 stations，列名 station_name
            db_stations = set([r[0] for r in cursor.fetchall()])
            conn.close()
            
            print(f"  - DB 中有 {len(db_stations)} 个气象监测站")
            # 气象站和地铁站通常不同名，这里仅做数量确认，不做集合比对
        except:
            print("⚠️ 无法读取数据库中的站点信息，跳过校验。")
    else:
        print("⚠️ 数据库文件不存在。")
    print("--------------------------------------------------\n")


# ==========================================
# 6. 主程序入口 (Joblib 并行执行版)
# ==========================================
def main():
    # === 参数配置区域 ===
    BASE_PATH = "E:\\BaiHao\\20260103算例补充\\输入与输出数据"
    
    # --- 模式开关 ---
    USE_REAL_DATA = False   # [开关] 是否使用 2025.07.29-08.01 的真实数据
    # ----------------
    
    if USE_REAL_DATA:
        # 真实数据模式配置
        MC_ITERATIONS = 1         # 真实数据是确定性的，通常跑 1 次即可 (或者跑多次以测试算法稳定性)
        START_TIME_STR = "2025-07-29 05:00" # 仿真开始时间
        END_TIME_STR   = "2025-08-01 23:00" # 仿真结束时间
        #START_TIME_STR = "2025-07-29 07:00" # 仿真开始时间
        #END_TIME_STR   = "2025-07-29 08:00" # 仿真结束时间
        
        # 计算总分钟数
        t_start = datetime.strptime(START_TIME_STR, "%Y-%m-%d %H:%M")
        t_end   = datetime.strptime(END_TIME_STR, "%Y-%m-%d %H:%M")
        SIM_DURATION = int((t_end - t_start).total_seconds() / 60)
        
        DB_NAME = "shanghai_rain_history.db"
        FLOW_FILE = "shanghai_metro_flow_final_complete.xlsx" # 支持 csv 或 xlsx
        
        print(f"\n🚀 启动真实数据仿真模式 ({START_TIME_STR} -> {END_TIME_STR}, Duration={SIM_DURATION}min)")
        
        # 加载客流数据 (Pre-load flow data)
        flow_path = os.path.join(BASE_PATH, FLOW_FILE)
        print(f"📂 加载客流文件: {flow_path}")
        try:
            if flow_path.endswith('.csv'):
                flow_df = pd.read_csv(flow_path)
            else:
                flow_df = pd.read_excel(flow_path)

            # 1. 基础清洗：转字符串、去空格、统一符号
            flow_df['station'] = flow_df['station'].astype(str).str.strip().str.replace('●', '·').str.replace(' ', '')

            # 2. 特殊处理：将“浦电路”拆分为4号线和6号线，客流平分
            # 筛选出原始名称为 "浦电路" 的所有行
            pudian_mask = (flow_df['station'] == '浦电路')
            
            if pudian_mask.any():
                # 提取这些行
                pudian_rows = flow_df[pudian_mask].copy()
                
                # 构造4号线数据 (取名为 JSON 中对应的 '浦电路(4号线)', 客流除以2)
                pudian_4 = pudian_rows.copy()
                pudian_4['station'] = '浦电路(4号线)'
                pudian_4['inbound'] = pudian_4['inbound'] / 2.0
                pudian_4['outbound'] = pudian_4['outbound'] / 2.0
                
                # 构造6号线数据 (取名为 JSON 中对应的 '浦电路(6号线)', 客流除以2)
                pudian_6 = pudian_rows.copy()
                pudian_6['station'] = '浦电路(6号线)'
                pudian_6['inbound'] = pudian_6['inbound'] / 2.0
                pudian_6['outbound'] = pudian_6['outbound'] / 2.0
                
                # 从原表中删除“浦电路”，并追加拆分后的两部分数据
                flow_df = flow_df[~pudian_mask] # 删除原行
                flow_df = pd.concat([flow_df, pudian_4, pudian_6], ignore_index=True)
                print(f"ℹ️ 已将 {len(pudian_rows)} 条'浦电路'数据拆分为4号线与6号线 (客流平分)。")

            # 【新增修复】 简单的站点名称清洗，解决匹配警告
            def clean_station_name(name):
                name = str(name).strip()
                # 1. 替换特殊符号 '●' 为标准符号 '·' 
                name = name.replace('●', '·').replace(' ', '')
                
                # 2. 修正已知的 Excel 录入笔误及旧站名
                fixes = {
                    # --- 笔误修正 ---
                    '淞淞发路': '淞发路',
                    '淞淞虹路': '淞虹路',
                    '淞淞滨路': '淞滨路',  # [新增] 修正笔误
                    '临洮洮路': '临洮路',  # [新增] 修正笔误
                    '颛颛桥': '颛桥',
                    '佘佘山': '佘山',
                    # --- 站名变更/映射 ---               
                    # --- 暂未开通/无拓扑车站 ---
                    '龙居路': '龙居路',    # 14号线暂缓开通，拓扑中无此站，保留原名即可
                    '严御路': '康恒路',    # [新增] 严御路已更名为康恒路(11号线) 。但是拓扑里面没它。
                }
                return fixes.get(name, name)

            flow_df['station'] = flow_df['station'].apply(clean_station_name)

            # 预处理：转换时间列，方便后续查询
            flow_df['time'] = pd.to_datetime(flow_df['time'])
            
            # 执行一致性校验
            verify_data_consistency(BASE_PATH, flow_df)
            
        except Exception as e:
            print(f"❌ 读取客流文件失败: {e}")
            return

    else:
        # 原始蒙特卡洛模式配置
        MC_ITERATIONS = 100       
        RAIN_P_PERIOD = 2       
        RAIN_DURATION = 180       
        SIM_DURATION = 240        
        START_TIME_STR = "09:00"     
        print(f"\n🚀 启动随机蒙特卡洛仿真模式 (Runs={MC_ITERATIONS})")
        DB_NAME = None
        
        # [修改] 读取 08.08 的基准客流数据 (全量 CSV)
        flow_file_full = "shanghai_metro_flow_with_code_final_20250802-20250831_v4.csv"
        flow_path = os.path.join(BASE_PATH, flow_file_full)
        print(f"📂 加载基准客流文件 (2025.08.08): {flow_path}")
        
        try:
            # 读取 CSV
            full_df = pd.read_csv(flow_path)
            
            # 过滤 2025-08-08 的数据
            # [修改] 原代码的 format='%Y-%m-%d %H' 无法解析 ":00:00" 部分，改为自动推断
            full_df['time'] = pd.to_datetime(full_df['time']) 
            
            target_date = pd.Timestamp("2025-08-08").date()
            flow_df = full_df[full_df['time'].dt.date == target_date].copy()
            
            # 重命名列以匹配 SubwayDIIMSystem
            # csv: inbound, outbound, station_name_mapped
            # system expects: inbound, outbound, station
            flow_df.rename(columns={'station_name_mapped': 'station'}, inplace=True)
            
            # 站点名称清洗 (确保与 if 块中的逻辑一致)
            def clean_station_name(name):
                name = str(name).strip()
                name = name.replace('●', '·').replace(' ', '')
                fixes = {
                    '淞淞发路': '淞发路',
                    '淞淞虹路': '淞虹路',
                    '淞淞滨路': '淞滨路',
                    '临洮洮路': '临洮路', 
                    '颛颛桥': '颛桥',
                    '佘佘山': '佘山',
                    '严御路': '康恒路',
                }
                return fixes.get(name, name)
            flow_df['station'] = flow_df['station'].apply(clean_station_name)
            
            # 处理浦电路拆分 (简单起见，这里复用逻辑)
            pudian_mask = (flow_df['station'] == '浦电路')
            if pudian_mask.any():
                pudian_rows = flow_df[pudian_mask].copy()
                pudian_4 = pudian_rows.copy(); pudian_4['station'] = '浦电路(4号线)'; pudian_4[['inbound','outbound']] /= 2
                pudian_6 = pudian_rows.copy(); pudian_6['station'] = '浦电路(6号线)'; pudian_6[['inbound','outbound']] /= 2
                flow_df = flow_df[~pudian_mask]
                flow_df = pd.concat([flow_df, pudian_4, pudian_6], ignore_index=True)
            
            print(f"✅ 已加载 2025.08.08 客流数据，共 {len(flow_df)} 条记录")
            
        except Exception as e:
            print(f"❌ 读取客流文件失败: {e}")
            flow_df = None

    # N_JOBS 控制并行核心数
    # N_JOBS = 10 if USE_REAL_DATA else 30 # 真实数据读取DB开销大，建议减少并发数或设为1
    N_JOBS = 40

    recorder = MonteCarloRecorder(list(range(SIM_DURATION)))
    
    try:
        # 使用 joblib.Parallel 并行派发 simulation_worker 任务
        results = Parallel(n_jobs=N_JOBS, backend='loky', verbose=10)(
            delayed(simulation_worker)(
                run+1, BASE_PATH, 250, 
                RAIN_P_PERIOD, # period (unused in real mode)
                RAIN_DURATION, # rain_dur (unused in real mode)
                SIM_DURATION, START_TIME_STR,
                use_real_data=USE_REAL_DATA,
                real_db_name=DB_NAME,
                real_flow_data=flow_df # [修改] 将读取到的 flow_df 传递进去
            ) for run in range(MC_ITERATIONS)
        )
        
        # 遍历并行结果并存入记录器
        for res in results:
            if res[1] is None:
                print(f"❌ [Error] Run {res[0]} failed: {res[2]}")
            else:
                recorder.add_run_data(*res)
                
    except Exception as e:
        print(f"❌ [Critical] Parallel execution failed: {e}")

    print("\n🏁 仿真结束，正在保存数据...")
    recorder.save_to_excel(BASE_PATH)

if __name__ == "__main__":
    main()

In [ ]:
import networkx as nx
import json
import math
import pandas as pd
import numpy as np
import random
import os
import re
import gc
import time
import sys
import sqlite3
import warnings
from abc import ABC, abstractmethod
from scipy import sparse
from scipy.spatial.distance import cdist
from scipy.interpolate import griddata
from PIL import Image
from datetime import datetime, timedelta

# [修改 1] 引入 joblib 用于 Jupyter 环境下的并行计算
from joblib import Parallel, delayed 

# 禁用插值产生的可能的警告
warnings.filterwarnings("ignore")

# ==============================================================================
# [全局配置] 并行计算环境设置
# ==============================================================================
# 限制 Numpy/OpenBLAS 的底层线程数。
# 原因：在 Joblib 多进程模式下，如果每个子进程再自动开启多线程进行矩阵运算，
# 会导致 CPU 资源争抢（Oversubscription）甚至死锁，显著降低仿真效率。
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

# ==============================================================================
# 1. 策略模式：定义扰动（降雨）行为
# ==============================================================================
class DisturbanceStrategy(ABC):
    """
    抽象基类：定义系统受到的外部干扰策略。
    
    Methods:
        get_intensity: 获取特定时空的干扰强度。
        get_speed_impact_factor: 获取干扰对系统运行速度的影响系数。
    """
    @abstractmethod
    def get_intensity(self, time_step, position): pass
    @abstractmethod
    def get_speed_impact_factor(self, intensity, edge_type, **kwargs): pass

class ExtremeRainfall(DisturbanceStrategy):
    """
    极端降雨策略模型：模拟一个移动的暴雨中心及其对地铁站点的积水影响。
    
    包含物理过程：
    1. 芝加哥雨型（Chicago Design Storm）生成时间序列降雨。
    2. 移动风暴中心（Moving Storm Center）。
    3. 空间衰减（ARF）计算。
    4. SCS-CN 径流模型计算地表积水。
    """
    def __init__(self, base_path, max_intensity=201.9, rainfall_type='chicago', P=100, r=0.405, rain_duration=180):
        """
        初始化降雨模型参数。
        
        Args:
            max_intensity (float): 降雨峰值强度参数 (mm/h 或相关系数)。
            P (int): 降雨重现期 (Return Period, e.g., 100年一遇)。
            r (float): 雨峰位置系数 (Peak Position parameter)，0-1之间，0.4代表峰值出现在前40%。
            rain_duration (int): 降雨持续总时长 (分钟)。
        """
        # 设定模拟的空间范围（上海经纬度大致范围）
        self.lat_range = (30.6984, 31.5139)
        self.lon_range = (120.849, 121.966)
        
        # 初始化暴雨中心起始位置和移动速度
        self.center_loc, self.velocity = self._init_random_boundary_start()
        
        self.Q0 = max_intensity  # 降雨强度基准
        self.base_path = base_path
        self.rainfall_type = rainfall_type
        self.P_period = P        # 重现期
        self.r_peak = r          # 雨峰系数
        self.rain_duration = rain_duration 
        
        self.chicago_profile = None 
        self.img_array = None
        self._load_drainage_image() # 加载区域排水能力图
        self._distance_cache = {}   # 缓存站点到雨中心的距离，减少重复三角函数计算
        
        # 如果是芝加哥雨型，预计算随时间变化的强度曲线
        if self.rainfall_type == 'chicago': self._precompute_chicago_profile()

    def _init_random_boundary_start(self):
        """
        随机初始化降雨中心。
        逻辑：选择地图四个边界之一作为起点，目标点设为地图中心附近，以此计算移动矢量。
        """
        lat_min, lat_max = self.lat_range
        lon_min, lon_max = self.lon_range
        side = random.choice([0, 1, 2, 3]) # 0:上(北), 1:下(南), 2:左(西), 3:右(东)
        
        # 设定目标点为区域中心附近略带随机偏移
        target_lat = (lat_min + lat_max) / 2 + random.uniform(-0.1, 0.1)
        target_lon = (lon_min + lon_max) / 2 + random.uniform(-0.1, 0.1)
        
        if side == 0:   start_lat, start_lon = lat_max, random.uniform(lon_min, lon_max)
        elif side == 1: start_lat, start_lon = lat_min, random.uniform(lon_min, lon_max)
        elif side == 2: start_lat, start_lon = random.uniform(lat_min, lat_max), lon_min
        else:           start_lat, start_lon = random.uniform(lat_min, lat_max), lon_max
        
        speed_kmh = random.uniform(25, 40) # 降雨系统移动速度 (km/h)
        speed_km_min = speed_kmh / 60.0
        
        # 计算移动单位向量（转换为经纬度增量）
        delta_lat_km = (target_lat - start_lat) * 111.0
        delta_lon_km = (target_lon - start_lon) * 96.0
        dist_km = math.sqrt(delta_lat_km**2 + delta_lon_km**2) or 1
        
        v_lat_deg = ((delta_lat_km / dist_km) * speed_km_min) / 111.0
        v_lon_deg = ((delta_lon_km / dist_km) * speed_km_min) / 96.0
        return (start_lat, start_lon), (v_lat_deg, v_lon_deg)

    def move_step(self):
        """每一分钟移动一次降雨中心，并清空距离缓存（因为中心变了，所有距离需重算）"""
        curr_lat, curr_lon = self.center_loc
        v_lat, v_lon = self.velocity
        self.center_loc = (curr_lat + v_lat, curr_lon + v_lon)
        self._distance_cache = {} 

    def _precompute_chicago_profile(self):
        """
        预计算芝加哥雨型（Chicago Design Storm）的时间分布。
        公式来源：Keifer & Chu (1957)，参数 A, n, b 针对上海地区暴雨强度公式适配。
        """
        times = np.arange(0, self.rain_duration + 1, 1) 
        intensities = []
        t_peak_time = self.rain_duration * self.r_peak
        
        # A, n, b 为上海地区的降雨公式参数（需根据研究区域调整）
        A = 9.581 * (1 + 0.846 * np.log10(self.P_period))
        n = 0.656
        b = 7
        
        for t in times:
            # 峰前时刻公式
            if t <= t_peak_time:
                t_b = t_peak_time - t
                num = A * ((1 - n) * (t_b / self.r_peak) + b)
                den = math.pow((t_b / self.r_peak) + b, 1 + n)
            # 峰后时刻公式
            else:
                t_a = t - t_peak_time
                num = A * ((1 - n) * (t_a / (1 - self.r_peak)) + b)
                den = math.pow((t_a / (1 - self.r_peak)) + b, 1 + n)
            intensities.append(num / den)
        
        # 模拟雨后由于排水不畅导致的“长尾效应”（指数衰减）
        tail_duration = 120  
        if intensities:
            last_val = intensities[-1]
            for t_ex in range(1, tail_duration + 1):
                val = last_val * math.exp(-0.1 * t_ex)
                intensities.append(val if val > 0.001 else 0.0)
        
        # 存储为累计降雨量 (Cumulative Rainfall)，方便后续计算径流
        self.chicago_profile = np.cumsum(intensities)

    def _load_drainage_image(self):
        """加载排水能力栅格图，用于根据地理位置查询排水效率"""
        path = os.path.join(self.base_path, "排水能力_简化.png")
        try:
            img = Image.open(path)
            self.img_array = np.array(img)
        except FileNotFoundError:
            # 如果没图，生成一个随机的（调试用）
            self.img_array = np.random.randint(0, 255, (1000, 1000, 3), dtype=np.uint8)

    def _haversine(self, lat1, lon1, lat2, lon2):
        """计算地球表面两点间的大圆距离 (km)"""
        R = 6371.0 
        dlat = math.radians(lat2 - lat1)
        dlon = math.radians(lon2 - lon1)
        a = math.sin(dlat/2)**2 + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(dlon/2)**2
        return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))

    def _lat_lon_to_pixel(self, lat, lon):
        """经纬度坐标与图像像素坐标的转换"""
        if self.img_array is None: return 0, 0
        h, w = self.img_array.shape[:2]
        lat_min, lat_max = self.lat_range
        lon_min, lon_max = self.lon_range
        px = int((lon - lon_min) / (lon_max - lon_min) * w)
        py = int((lat_max - lat) / (lat_max - lat_min) * h)
        return min(max(px, 0), w - 1), min(max(py, 0), h - 1)

    def _get_drainage_from_image(self, lat, lon):
        """从像素颜色值映射到实际排水能力 (mm/h)"""
        px, py = self._lat_lon_to_pixel(lat, lon)
        # 获取灰度值或R通道值
        r_val = self.img_array[py, px][0] if self.img_array.ndim == 3 else self.img_array[py, px]
        r_norm = r_val / 255.0
        # 简单的阈值映射规则
        if r_val == 0: return 27
        elif r_norm <= 0.9: return 36
        elif r_norm <= 0.99: return 27
        else: return 50

    def _calculate_arf_rainfall(self, distance):
        """空间衰减计算：根据距雨中心距离计算面雨量折减系数 (Area Reduction Factor, ARF)"""
        area = math.pi * (distance**2)
        # ARF 经验公式
        factor = -0.058 * math.log(area) + 0.9949 if area >= 1 else 1.0
        Q_temp = self.Q0 * factor
        # 增加空间随机扰动
        return max(0, Q_temp + np.random.normal(0, 0.3 * abs(Q_temp)))

    def get_instantaneous_intensity(self, time_step, position, precomputed_dist=None):
        """
        获取某时刻、某位置的瞬时降雨强度 (mm/h)。
        主要用于地上线路的能见度/运行风险判断。
        [修改说明] 引入60分钟滑动平均，解决芝加哥雨型瞬时峰值过高的问题。
        """
        if precomputed_dist is not None: dist = precomputed_dist
        else:
            if position in self._distance_cache: dist = self._distance_cache[position]
            else:
                dist = self._haversine(self.center_loc[0], self.center_loc[1], position[0], position[1])
                self._distance_cache[position] = dist
                
        spatial_val = self._calculate_arf_rainfall(dist)
        
        if self.rainfall_type == 'chicago':
            if time_step >= len(self.chicago_profile) - 1: return 0.0
            scale = spatial_val / self.Q0 if self.Q0 > 0 else 0
            
            # [核心修改] 使用 60分钟滑动窗口计算小时雨强
            idx_now = min(int(time_step), len(self.chicago_profile) - 1)
            window_size = 60 
            idx_prev = max(0, idx_now - window_size)
            
            # 这一时段内的总雨量 (mm)
            rain_depth_in_window = self.chicago_profile[idx_now] - self.chicago_profile[idx_prev]
            
            # 时段长度 (min)
            duration = idx_now - idx_prev
            if duration <= 0: return 0.0
            
            # 换算成小时雨强 (mm/h)
            intensity_hourly = (rain_depth_in_window / duration) * 60.0
            
            return max(0, intensity_hourly * scale)
        else:
            # 简单均匀降雨模式
            return max(0, spatial_val / (self.rain_duration / 60.0)) if self.rain_duration > 0 else 0

    def get_intensity(self, time_step, position, precomputed_dist=None):
        """
        核心计算：计算地表净积水深度 (mm)。
        使用了 SCS-CN (Soil Conservation Service Curve Number) 径流模型的简化形式。
        Logic: 积水 = 降雨径流(Runoff) - 城市管网排水(Drainage)
        """
        lat, lon = position
        if precomputed_dist is not None: dist = precomputed_dist
        else:
            if position in self._distance_cache: dist = self._distance_cache[position]
            else:
                dist = self._haversine(self.center_loc[0], self.center_loc[1], lat, lon)
                self._distance_cache[position] = dist
        spatial_val = self._calculate_arf_rainfall(dist)
        
        # 1. 计算累计降雨量 (Cumulative Rainfall)
        if self.rainfall_type == 'chicago':
            scale = spatial_val / self.Q0 if self.Q0 > 0 else 0
            idx = min(int(time_step), len(self.chicago_profile) - 1)
            cum_rain = self.chicago_profile[idx] * scale
        else:
            progress = min(time_step / float(self.rain_duration), 1.0)
            cum_rain = spatial_val * progress
            
        # 2. SCS 径流转换
        S = 15.7 # 潜在最大入渗量 (Potential Maximum Retention)，与土壤/地表类型有关
        # SCS方程：Q = (P - 0.2S)^2 / (P + 0.8S)
        runoff = (cum_rain - 0.05 * S)**2 / (cum_rain + 0.95 * S) if cum_rain > 0.05 * S else 0
        
        # 3. 扣除区域排水
        drainage = (self._get_drainage_from_image(lat, lon) / 60.0) * time_step
        return max(0, runoff - drainage) # 返回值为净积水深度

    def get_speed_impact_factor(self, water_depth, edge_type, line_context=None):
        """
        计算速度折减系数：将物理积水深度转换为对系统运行/步行速度的影响 (0.0 - 1.0)。
        1.0 表示正常速度，0.0 表示完全中断。
        """
        # [修改] 提取线路整体最大积水深度，用于判断是否触发全线停运
        # 如果 max_water_total > 450，意味着线路上某处已严重积水，该线路应整体停运
        max_water_line = 0.0
        if line_context:
            max_water_line = line_context.get('max_water_total', 0.0)

        # [修改] 核心逻辑变更：
        # 如果当前站点水深 > 450mm OR 线路任意位置水深 > 450mm (由 line_context 传入)，
        # 则视为线路瘫痪。
        is_line_failed = (water_depth > 450) or (max_water_line > 450)

        if edge_type == 'boarding': 
            # 进站/步行速度折减 (基于经验研究的步速-水深函数，如 Ishigaki et al.)
            # [修改] 若线路瘫痪（不论是本站淹水还是线路其他站淹水），进站系数设为 0.0001
            if is_line_failed:
                return 0.0001
            
            if water_depth <= 0: return 1.0
            if water_depth <= 250: return max(0.0001, 0.9 - (water_depth / 250.0) * 0.1)
            elif water_depth <= 450: return 0.0001
            else: return 0.0001 # 兜底逻辑，实际上 is_line_failed 已覆盖
            
        elif edge_type == 'riding': 
            # 乘车运行速度折减
            if line_context:
                is_above = line_context.get('is_above_ground', 0)
                max_rain = line_context.get('max_rain_above', 0.0)
                # max_water_line 已经在上面提取了
                
                if is_above == 1: # 地上线主要受瞬时强降雨（能见度）影响
                    if max_rain > 100: return 0.0001 # [修改] 极强降雨停运，因子改为 0.0001
                    elif max_rain >= 80: return 0.5 # 强降雨限速
                else: # 地下线主要受区间进水影响
                    # [修改] 若线路上最大积水超过450mm，列车停运，影响因子变为0.0001
                    if max_water_line > 450: return 0.0001
            return 1.0

# ==============================================================================
# [新增] 真实数据降雨策略类
# ==============================================================================
class RealDataRainfall(DisturbanceStrategy):
    """
    真实数据降雨策略：基于 SQLite 数据库中的历史/实时降雨数据进行空间插值。
    模拟 2025-07-29 等特定日期的真实降雨分布。
    """
    def __init__(self, base_path, db_name="shanghai_rain_history.db", start_datetime_str="2025-07-29 00:00"):
        self.base_path = base_path
        self.db_path = os.path.join(base_path, db_name)
        self.start_dt = datetime.strptime(start_datetime_str, "%Y-%m-%d %H:%M")
        self.center_loc = (31.2304, 121.4737) # 默认中心设为上海市中心，适配接口（真实数据无中心概念）
        
        # 缓存机制，避免每分钟每节点都去查库插值
        self._last_query_time_str = None
        self._cached_grid_interpolator = None # 存储 griddata 的结果或KDTree
        self._cached_df = None
        
        # 上海区域网格参数
        self.SH_LON_MIN, self.SH_LON_MAX = 120.8, 122.3
        self.SH_LAT_MIN, self.SH_LAT_MAX = 30.6, 31.9
        
        self.img_array = None
        self._load_drainage_image()

    def _load_drainage_image(self):
        """复用父类的排水图加载逻辑"""
        path = os.path.join(self.base_path, "排水能力_简化.png")
        try:
            img = Image.open(path)
            self.img_array = np.array(img)
        except:
            self.img_array = np.random.randint(0, 255, (1000, 1000, 3), dtype=np.uint8)

    def _lat_lon_to_pixel(self, lat, lon):
        """复用父类的坐标转换逻辑"""
        if self.img_array is None: return 0, 0
        h, w = self.img_array.shape[:2]
        # 注意：这里需要与 ExtremeRainfall 的范围保持一致，或者使用本类定义的范围
        # 假设排水图是覆盖 self.SH_LAT_MIN/MAX 范围的
        lat_min, lat_max = 30.6984, 31.5139 # 这里沿用 ExtremeRainfall 的范围设定以匹配图片
        lon_min, lon_max = 120.849, 121.966
        px = int((lon - lon_min) / (lon_max - lon_min) * w)
        py = int((lat_max - lat) / (lat_max - lat_min) * h)
        return min(max(px, 0), w - 1), min(max(py, 0), h - 1)

    def _get_drainage_from_image(self, lat, lon):
        """复用父类的排水能力获取逻辑"""
        px, py = self._lat_lon_to_pixel(lat, lon)
        r_val = self.img_array[py, px][0] if self.img_array.ndim == 3 else self.img_array[py, px]
        r_norm = r_val / 255.0
        if r_val == 0: return 27
        elif r_norm <= 0.9: return 36
        elif r_norm <= 0.99: return 27
        else: return 50
    
    # 距离计算方法
    def _haversine(self, lat1, lon1, lat2, lon2):
        """计算地球表面两点间的大圆距离 (km)"""
        R = 6371.0 
        dlat = math.radians(lat2 - lat1)
        dlon = math.radians(lon2 - lon1)
        a = math.sin(dlat/2)**2 + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(dlon/2)**2
        return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))
    
    def move_step(self):
        """真实数据不需要移动风暴中心，此方法留空适配接口"""
        pass

    def _fetch_and_interpolate(self, current_time_str):
        """从数据库读取特定时间的数据并构建插值函数"""
        # 如果时间没变，直接返回缓存
        if current_time_str == self._last_query_time_str and self._cached_df is not None:
            return
        
        self._last_query_time_str = current_time_str
        conn = None
        try:
            conn = sqlite3.connect(self.db_path)
            # 使用 LIKE 模糊匹配时间 (e.g., '2025-07-29 05:00%')
            query = """
                SELECT s.lon, s.lat, r.rain_value 
                FROM rain_data r
                JOIN stations s ON r.station_id = s.station_id
                WHERE r.datetime LIKE ?
            """
            df = pd.read_sql_query(query, conn, params=(f"{current_time_str}%",))
            
            if len(df) < 3:
                # 数据不足时，假设无雨
                self._cached_df = pd.DataFrame({'lon': [], 'lat': [], 'rain_value': []})
            else:
                self._cached_df = df
        except Exception as e:
            # print(f"DB Error: {e}") 
            self._cached_df = pd.DataFrame({'lon': [], 'lat': [], 'rain_value': []})
        finally:
            if conn: conn.close()

    def get_instantaneous_intensity(self, time_step, position):
        """获取瞬时雨强 (mm/h)"""
        # 计算当前绝对时间
        curr_dt = self.start_dt + timedelta(minutes=time_step)
        # 数据库是每小时数据 (e.g. 05:00, 06:00)，这里简单取最近的小时，或者按分钟插值
        # 为了性能，这里取整点小时 (format: YYYY-MM-DD HH)
        # 如果需要更精细，可以修改为 HH:MM 并确保数据库有该精度，或者在两个小时之间做线性插值
        # 这里采用：取当前小时的数据（阶梯状变化）
        target_time_str = curr_dt.strftime("%Y-%m-%d %H") # 匹配数据库的 '2025-07-29 05' 开头
        
        self._fetch_and_interpolate(target_time_str)
        
        if self._cached_df is None or len(self._cached_df) == 0:
            return 0.0
        
        # 使用 Nearest 或 Linear 插值获取特定位置的雨量
        # 由于每次调用 griddata 开销大，这里针对单点计算：
        # 实际上 griddata 适合生成网格。对于单点查询，计算距离最近的几个站做 IDW 更快。
        # 这里简化使用 scipy.interpolate.griddata 对单点插值
        points = self._cached_df[['lat', 'lon']].values
        values = self._cached_df['rain_value'].values
        
        # 注意：griddata 输入是 (points, values, xi)
        # points: (N, 2), xi: (M, 2)
        try:
            val = griddata(points, values, np.array([position]), method='nearest')[0]
            return max(0, val)
        except:
            return 0.0

    def get_intensity(self, time_step, position, precomputed_dist=None):
        """
        计算地表积水深度 (mm)。
        对于真实数据，我们假设数据库里存的是“小时雨量 (mm)”。
        为了计算积水，我们需要累计雨量。
        注意：这是一个简化处理。真实的积水模拟需要连续的时间积分。
        这里我们假设：积水 = 当前时刻瞬时雨强累积 - 排水。
        """
        lat, lon = position
        
        # 获取瞬时雨强 (假设数据库存储的是 mm/h 或者该小时的累计雨量)
        intensity_mm_h = self.get_instantaneous_intensity(time_step, position)
        
        # 计算当前时间步（1分钟）带来的新增雨量
        rain_inc = intensity_mm_h / 60.0 
        
        # 在这里，DisturbanceStrategy 是无状态的(stateless)关于“积水历史”的吗？
        # ExtremeRainfall 类里是用 cumulative profile 算的。
        # 如果我们没有累计曲线，我们只能近似。
        # 方案：假设过去 N 小时的雨都下下来了，扣除排水。
        # 但这非常慢。
        # 妥协方案：SubwayDIIMSystem 的 G.nodes[n]['water'] 是每一帧刷新的。
        # 如果 ExtremeRainfall 是每次重算 total accumulation，那这里也得这么做。
        # 但查询历史所有小时的数据太慢了。
        
        # *修正*：SubwayDIIMSystem.step() 里是直接赋值：
        # self.G.nodes[n]['water'] = self.strategy.get_intensity(...)
        # 所以 get_intensity 必须返回“当前积水深度”。
        
        # 对于 RealData，我们难以每次从 0时刻开始积分。
        # 建议：仅返回瞬时雨强导致的“潜在积水”，或者基于简单的“当前雨强 * 因子”来近似积水风险。
        # 或者，为了更准确，我们在 SimulationSystem 里维护积水状态，而不是在这里计算 total。
        # 但为了不改动 System 的逻辑（System 假设 Strategy 返回积水深度），
        # 我们这里做一个近似： 积水深度 ~= 最近 1 小时的雨量 * 径流系数 - 排水 * 1小时
        
        S = 15.7
        # 假设 effective rain 是过去 60 分钟的累积（即当前 hourly value）
        cum_rain = intensity_mm_h # 粗略假设
        
        runoff = (cum_rain - 0.05 * S)**2 / (cum_rain + 0.95 * S) if cum_rain > 0.05 * S else 0
        
        # 排水能力 (mm/h)
        drainage_rate = self._get_drainage_from_image(lat, lon)
        
        # 估算积水：径流 - 排水（假设排水一直在工作）
        # 这里仅仅是一个状态快照，无法完美模拟时序累积，除非重构 System
        water_depth = max(0, runoff - drainage_rate) 
        
        return water_depth

    def get_speed_impact_factor(self, water_depth, edge_type, line_context=None):
        """复用父类的速度折减逻辑"""
        # [修改] 提取线路整体最大积水深度
        max_water_line = 0.0
        if line_context:
            max_water_line = line_context.get('max_water_total', 0.0)

        # [修改] 判断是否全线停运
        is_line_failed = (water_depth > 450) or (max_water_line > 450)

        if edge_type == 'boarding': 
            # [修改] 若线路瘫痪，进站系数设为 0.0001
            if is_line_failed:
                return 0.0001
                
            if water_depth <= 0: return 1.0
            if water_depth <= 250: return max(0.01, 0.9 - (water_depth / 250.0) * 0.1)
            elif water_depth <= 450: return max(0.0001, 0.8 - ((water_depth - 250) / 200.0) * 0.7)
            else: return 0.0001 
            
        elif edge_type == 'riding': 
            if line_context:
                is_above = line_context.get('is_above_ground', 0)
                max_rain = line_context.get('max_rain_above', 0.0)
                
                if is_above == 1: 
                    if max_rain > 100: return 0.0001  # [修改] 极强降雨停运
                    elif max_rain >= 80: return 0.5 
                else: 
                    if max_water_line > 450: return 0.0001 # [修改] 地下线受淹停运
            return 1.0


# ==============================================================================
# 2. 核心系统：地铁网络拓扑与 DIIM (动态进程间影响模型) 模拟
# ==============================================================================
class SubwayDIIMSystem:
    """
    地铁动态影响模拟系统 (Dynamic Inoperability Input-Output Model System)
    
    主要功能：
    1. 构建双层网络图（Station Layer + Line Layer）。
    2. 加载静态数据（拓扑、时刻表、客流）。
    3. 执行时间步进仿真，计算 DIIM 状态更新。
    4. 计算网络性能指标（加权全局效率）。
    """
    def __init__(self, base_path, disturbance_strategy, sim_start_time='05:30', real_flow_df=None):
        self.base_path = base_path
        self.strategy = disturbance_strategy
        self.real_flow_df = real_flow_df # [新增] 接收真实客流数据 DataFrame
        self.G = nx.MultiGraph() # 使用多重图构建分层网络，允许两个节点间存在多种关系的边
        
        try:
            t_obj = datetime.strptime(sim_start_time, "%H:%M")
            self.sim_start_min = t_obj.hour * 60 + t_obj.minute
            self.sim_start_dt_obj = t_obj # 保存 datetime 对象用于计算
        except: 
            self.sim_start_min = 330
            self.sim_start_dt_obj = datetime(2000, 1, 1, 5, 30)
        
        # 内部缓存与索引映射
        self.id_map_long_to_short = {}
        self.line_stations_index = {}
        self.schedule_rules = [] 
        self.schedule_lookup = {}
        self.station_pos_cache = {}
        self.fast_edge_data = [] # 扁平化的边数据列表，用于极速遍历（避开NX图查询开销）
        self.node_idx_map = {}
        self.node_pos_list = []
        self.node_dists_list = []
        self.node_st_names = []
        
        # 顺序执行初始化流程
        self._load_id_mapping()           # 1. 映射线路ID (Excel -> Dict)
        self._load_json_topology()        # 2. 加载基础拓扑 (JSON)
        self._load_detailed_schedule()    # 3. 加载列车时刻/发车间隔
        self._build_station_index()       # 4. 构建索引
        self._build_graph_topology()      # 5. 构建分层网络图 (Graph Building)
        self._apply_ground_elevation_status() # 6. 标记地上/地下线
        self._load_passenger_flow_data()  # 7. 加载客流权重 (静态初始值)
        self._build_single_layer_network()# 8. 构建单层简化网 (用于辅助计算)
        self._init_diim_matrices()        # 9. 初始化动态影响矩阵 (DIIM Init)
        
        # 韧性恢复参数（设定恢复速度）
        # c(t) 为受损程度，恢复过程遵循指数衰减
        T_rec = 15.0 
        self.k_recovery = (1.0 / T_rec) * math.log(1.0 / 0.01)
        
        self._init_flow_weights()         # 初始化流权重
        self._precompute_static_simulation_data() # 预处理静态数据加速计算
        
        # 计算初始（无故障）状态下的加权网络效率，作为后续归一化的分母
        self.Ef0 = self.calculate_weighted_efficiency(self.sim_start_min, 'Weekday')
        self.congestion_warning_triggered = False

    def _load_id_mapping(self):
        """从Excel加载线路名称与短ID（如：1号线 -> "1"）的映射关系"""
        path = os.path.join(self.base_path, "线路id与名称对应-20251210.xlsx")
        try:
            if os.path.exists(path): df = pd.read_excel(path)
            else: df = pd.read_csv(path + " - Sheet1.csv")
            for _, row in df.iterrows():
                if pd.isna(row['Line ID']): continue
                long_id = str(int(row['Line ID'])) if isinstance(row['Line ID'], float) else str(row['Line ID'])
                line_name = str(row['Line Name'])
                short_id = None
                if "浦江" in line_name: short_id = "PJ"
                elif "磁浮" in line_name: short_id = "Maglev"
                elif "号线" in line_name:
                    match = re.search(r'(\d+)', line_name)
                    if match: short_id = match.group(1)
                if short_id: self.id_map_long_to_short[long_id] = short_id
        except: pass

    def _load_json_topology(self):
        """读取地铁拓扑结构 JSON 文件"""
        path = os.path.join(self.base_path, "subway.json")
        with open(path, "r", encoding="utf-8") as f: self.raw_json = json.load(f)

    def _load_detailed_schedule(self):
        """加载详细的发车间隔规则（区分早晚高峰、工作日/周末、区间车）"""
        path = os.path.join(self.base_path, "line_schedule_detailed.xlsx")
        try:
            df = pd.read_excel(path)
            df['LineID'] = df['LineID'].apply(lambda x: str(int(x)) if isinstance(x, float) else str(x).strip())
            for _, row in df.iterrows():
                s_dt = pd.to_datetime(str(row['Start_Time']), format='%H:%M:%S', errors='coerce')
                e_dt = pd.to_datetime(str(row['End_Time']), format='%H:%M:%S', errors='coerce')
                rule = {
                    'line_id': row['LineID'], 'day_type': str(row['DayType']).strip(),
                    'start_min': s_dt.hour*60+s_dt.minute if pd.notnull(s_dt) else 0,
                    'end_min': e_dt.hour*60+e_dt.minute if pd.notnull(e_dt) else 1440,
                    'from_st': str(row['From_Station']).strip(), 'to_st': str(row['To_Station']).strip(),
                    'interval': float(row['Interval_Sec'])
                }
                self.schedule_rules.append(rule)
                key = (rule['line_id'], rule['day_type'])
                if key not in self.schedule_lookup: self.schedule_lookup[key] = []
                self.schedule_lookup[key].append(rule)
        except: pass

    def _build_station_index(self):
        """构建车站在线路中的顺序索引，用于后续判断某车站是否在区间车的覆盖范围内"""
        for line_info in self.raw_json['l']:
            short_id = self.id_map_long_to_short.get(str(line_info['ls']))
            if not short_id: continue
            st_names = [st['n'] for st in line_info['st']]
            if short_id not in self.line_stations_index: 
                self.line_stations_index[short_id] = []
                self.station_pos_cache[short_id] = {}
            for st in st_names:
                if st not in self.line_stations_index[short_id]: 
                    self.line_stations_index[short_id].append(st)
                    self.station_pos_cache[short_id][st] = len(self.line_stations_index[short_id]) - 1

    def _is_station_in_section(self, line_id, station_name, from_st, to_st):
        """逻辑判定：特定车站是否位于指定的调度区间（from_st 到 to_st）内"""
        line_cache = self.station_pos_cache.get(line_id)
        if not line_cache: return False
        idx_s, idx_f, idx_t = line_cache.get(station_name), line_cache.get(from_st), line_cache.get(to_st)
        if idx_s is None or idx_f is None or idx_t is None: return False
        return min(idx_f, idx_t) <= idx_s <= max(idx_f, idx_t)

    def _build_graph_topology(self):
        """
        构建三维分层模型 (Space-L space)：
        1. 车站层 (Station Layer): 物理车站位置。
        2. 线路层 (Line Layer): 特定线路的站台。
        3. 边 (Edges):
           - boarding: 车站层 <-> 线路层 (进出站)。
           - riding: 线路层 <-> 线路层 (乘车)。
        """
        for line_info in self.raw_json['l']:
            short_id = self.id_map_long_to_short.get(str(line_info['ls']))
            if not short_id: continue
            sts = line_info['st']
            for st in sts:
                pos = tuple(map(float, st['sl'].split(',')))[::-1] # 转为 (lat, lon)
                node_st = f"{st['n']}_车站层"
                node_ln = f"{st['n']}_线路层_{short_id}"
                
                # 添加节点
                if not self.G.has_node(node_st): self.G.add_node(node_st, layer='station', pos=pos, flow=1000)
                self.G.add_node(node_ln, layer='line', pos=pos, line_id=short_id, station_name=st['n'])
                
                # 添加进站边 (Boarding/Alighting Link)
                self.G.add_edge(node_st, node_ln, type='boarding', weight=0)
            
            # 添加运行区间边 (Riding Link)
            for i in range(len(sts) - 1):
                u, v = f"{sts[i]['n']}_线路层_{short_id}", f"{sts[i+1]['n']}_线路层_{short_id}"
                if self.G.has_node(u) and self.G.has_node(v):
                    self.G.add_edge(u, v, type='riding', weight=0, line_id=short_id)

    def _apply_ground_elevation_status(self):
        """识别地铁的地上区间，这对于评估降雨暴露风险非常关键（地下区间受淹水影响，地上区间受雨强影响）"""
        file_path = os.path.join(self.base_path, "上海地铁地上线归纳.txt")
        above_set = set()
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                for line in f:
                    if '：' not in line: continue
                    parts = line.strip().split('：')
                    # 解析线路ID
                    match = re.search(r'(\d+)', parts[0])
                    lid = match.group(1) if match else ("PJ" if "浦江" in parts[0] else None)
                    if not lid: continue
                    # 解析区间
                    for group in parts[1].split('，'):
                        sts = group.strip().split('-')
                        for i in range(len(sts)-1):
                            pair = tuple(sorted((sts[i].strip(), sts[i+1].strip())))
                            above_set.add((lid, pair[0], pair[1]))
        except: pass
        
        # 遍历图中的边，标记 is_above_ground 属性
        for u, v, k, attr in self.G.edges(keys=True, data=True):
            if attr.get('type') == 'riding':
                lid = attr.get('line_id')
                stu, stv = self.G.nodes[u].get('station_name'), self.G.nodes[v].get('station_name')
                pair = tuple(sorted((stu, stv))) if stu and stv else None
                attr['is_above_ground'] = 1 if pair and (lid, pair[0], pair[1]) in above_set else 0

    def _load_passenger_flow_data(self):
        """从外部文件加载客流强度，用于后续加权网络效率的计算（客流大的站点权重更高）"""
        file_path = os.path.join(self.base_path, "进站客流.xlsx")
        p_flow = {}
        sum_f, num_f = 0, 0
        try:
            xls = pd.ExcelFile(file_path)
            for sheet in xls.sheet_names:
                for _, row in xls.parse(sheet, header=None).iterrows():
                    val = ((row[1] or 0) + (row[2] or 0)) / 2
                    p_flow[f"{row[0]}_车站层"] = p_flow.get(f"{row[0]}_车站层", 0) + val
                    sum_f += val; num_f += 1
        except: pass
        
        # 缺失值填充（使用平均值）
        avg_flow = sum_f / max(num_f, 1) or 1000
        avg_deg = np.mean([d for n, d in self.G.degree() if self.G.nodes[n].get('layer')=='station']) or 1
        
        for n, d in self.G.nodes(data=True):
            if d.get('layer') == 'station':
                d['flow'] = p_flow.get(n, (avg_flow/avg_deg)*self.G.degree(n))

    def _build_single_layer_network(self):
        """构建一个简化的单层站点间网络，用于辅助拓扑特征计算"""
        self.single_layer_net = nx.Graph()
        for u, v, attr in self.G.edges(data=True):
            if attr.get('type') == 'riding':
                n1 = self.G.nodes[u].get('station_name', u.split('_')[0])
                n2 = self.G.nodes[v].get('station_name', v.split('_')[0])
                if n1 != n2: self.single_layer_net.add_edge(n1, n2)

    def _init_flow_weights(self):
        """初始化边权：将节点流分配到相邻的进站边和运行边上"""
        ln_weights = {}
        # 1. 进站边权重
        for u, v, k, attr in self.G.edges(keys=True, data=True):
            if attr['type'] == 'boarding':
                st = u if self.G.nodes[u].get('layer')=='station' else v
                ln = v if st == u else u
                w = self.G.nodes[st].get('flow', 1000) / max(1, self.G.degree[st])
                attr['flow_weight'] = w
                ln_weights[ln] = w
        # 2. 运行边权重（取两端节点平均）
        for u, v, k, attr in self.G.edges(keys=True, data=True):
            if attr['type'] == 'riding':
                attr['flow_weight'] = (ln_weights.get(u, 100)+ln_weights.get(v, 100))/2

    def _precompute_static_simulation_data(self):
        """
        [极速仿真优化] 预计算所有不会随时间改变的数据结构。
        将图数据扁平化为 List/Array，避免在 step() 循环中频繁调用 NetworkX 的 lookup 方法。
        """
        nodes = list(self.G.nodes())
        self.node_idx_map = {n: i for i, n in enumerate(nodes)}
        self.node_dists_list = [0.0] * len(nodes)
        for i, n in enumerate(nodes):
            self.node_pos_list.append(self.G.nodes[n]['pos'])
            self.node_st_names.append(self.G.nodes[n].get('station_name', n.split('_')[0]))
        
        target_day_type = 'Weekday'
        for u, v, k in self.edge_list:
            attr = self.G.edges[u, v, k]
            u_idx, v_idx = self.node_idx_map[u], self.node_idx_map[v]
            lid = attr.get('line_id') or self.G.nodes[u].get('line_id') or self.G.nodes[v].get('line_id')
            est = {
                'u_idx': u_idx, 'v_idx': v_idx, 'type': attr['type'], 'line_id': lid,
                'is_above_ground': attr.get('is_above_ground', 0),
                'flow_weight': attr.get('flow_weight', 1.0),
                'boarding_dist': 0.0, 'boarding_node_flow': 1000.0,
                'riding_pos_mid': None, 'riding_applicable_rules': []
            }
            # 预绑定该边适用的调度规则（避免运行时搜索）
            if lid:
                rules = self.schedule_lookup.get((str(lid), target_day_type), [])
                st_u, st_v = self.node_st_names[u_idx], self.node_st_names[v_idx]
                check_st = st_u if attr['type'] == 'boarding' else None
                for r in rules:
                    if check_st:
                        if self._is_station_in_section(str(lid), check_st, r['from_st'], r['to_st']):
                             est['riding_applicable_rules'].append(r)
                    else:
                        if self._is_station_in_section(str(lid), st_u, r['from_st'], r['to_st']) and \
                           self._is_station_in_section(str(lid), st_v, r['from_st'], r['to_st']):
                             est['riding_applicable_rules'].append(r)
            if attr['type'] == 'boarding':
                st = self.node_st_names[u_idx]
                deg = self.single_layer_net.degree[st] if st in self.single_layer_net else 1
                est['boarding_dist'] = deg * 100.0 # 模拟车站内步行距离
                est['boarding_node_flow'] = self.G.nodes[u if self.G.nodes[u]['layer']=='station' else v].get('flow', 1000)
            else:
                p1, p2 = self.G.nodes[u]['pos'], self.G.nodes[v]['pos']
                est['riding_pos_mid'] = ((p1[0]+p2[0])/2, (p1[1]+p2[1])/2)
            self.fast_edge_data.append(est)

    def calculate_weighted_efficiency(self, current_time_min, day_type, line_stats=None):
        """
        计算效率，并包含终极路径追踪。
        """
        node_waters = [self.G.nodes[n].get('water', 0) for n in self.G.nodes()]
        
        # 1. 计算所有边权重
        for i, (u, v, k) in enumerate(self.edge_list):
            sd = self.fast_edge_data[i]
            
            # Rain
            avg_w = (node_waters[sd['u_idx']] + node_waters[sd['v_idx']]) / 2.0
            t_curr, _, _ = self._calc_times_fast(sd, avg_w, current_time_min, line_stats)
            
            # Base
            t_baseline, _, _ = self._calc_times_fast(sd, 0.0, current_time_min, line_stats=None)
            
            w = sd['flow_weight']
            
            # 这里的权重已经是 "时间 / 流量权重"
            weight_rain = (t_curr / w) if w > 0 and t_curr != float('inf') else float('inf')
            weight_base = (t_baseline / w) if w > 0 and t_baseline != float('inf') else float('inf')
            
            self.G.edges[u,v,k]['tau'] = weight_rain
            self.G.edges[u,v,k]['tau_baseline'] = weight_base
            
        st_nodes = [n for n, d in self.G.nodes(data=True) if d.get('layer') == 'station']
        st_set = set(st_nodes)
        N = len(st_nodes)
        if N < 2: return 0.0
        
        sum_inv_curr = 0.0
        sum_inv_base = 0.0
        
        found_anomaly_path = False
        
        # 2. 计算最短路径
        for src in st_nodes:
            # 获取 Rain 场景下的所有路径长度
            try:
                lens_rain = nx.single_source_dijkstra_path_length(self.G, src, weight='tau')
            except: lens_rain = {}
            
            # 获取 Base 场景下的所有路径长度
            try:
                lens_base = nx.single_source_dijkstra_path_length(self.G, src, weight='tau_baseline')
            except: lens_base = {}
            
            # 累加并检查
            for t in st_set:
                if t == src: continue
                
                l_r = lens_rain.get(t, float('inf'))
                l_b = lens_base.get(t, float('inf'))
                
                if l_r > 0: sum_inv_curr += 1.0/l_r
                if l_b > 0: sum_inv_base += 1.0/l_b
                
                # === 捕获异常路径 ===
                # 如果 Rain 距离 < Base 距离，说明 Rain 路径更短（更优），这是不可能的
                if l_r < l_b - 1e-6 and not found_anomaly_path:
                    print(f"\n🔥🔥🔥 [PATH ANOMALY DETECTED] at Time {current_time_min} 🔥🔥🔥")
                    print(f"Source: {src} -> Target: {t}")
                    print(f"Dist Rain: {l_r:.6f} < Dist Base: {l_b:.6f}")
                    
                    # 追踪 Rain 路径
                    try:
                        path = nx.shortest_path(self.G, src, t, weight='tau')
                        print("--- Rain Path Breakdown ---")
                        total_chk = 0
                        for i in range(len(path)-1):
                            u_p, v_p = path[i], path[i+1]
                            # 获取边数据 (取最小权重的那个key)
                            edge_data = self.G.get_edge_data(u_p, v_p)
                            # 找到 weight='tau' 最小的那个 key
                            best_k = min(edge_data, key=lambda k: edge_data[k]['tau'])
                            attr = edge_data[best_k]
                            
                            w_r = attr['tau']
                            w_b = attr['tau_baseline']
                            print(f"   {u_p} -> {v_p} | RainW={w_r:.4f} vs BaseW={w_b:.4f} | {'ERROR' if w_r < w_b else 'OK'}")
                            total_chk += w_r
                        print(f"   Check Sum: {total_chk:.4f}")
                    except:
                        print("   (Could not reconstruct path)")
                    
                    found_anomaly_path = True
        
        Ef = (1/(N*(N-1))) * sum_inv_curr
        Ef0 = (1/(N*(N-1))) * sum_inv_base
        
        return Ef / Ef0 if Ef0 > 0 else 1.0

    def _init_diim_matrices(self):
        """初始化动态进程间影响模型 (DIIM) 矩阵结构"""
        self.edge_list = list(self.G.edges(keys=True))
        self.edge_idx = {e: i for i, e in enumerate(self.edge_list)}
        n = len(self.edge_list)
        self.q_vector = np.zeros(n) # 存储各边的受损失效率 q
        self.A_matrix = sparse.csr_matrix((n, n))
        self.node_to_boarding_indices = {}
        for n_st in [n for n, d in self.G.nodes(data=True) if d['layer']=='station']:
            idxs = []
            for u, v, k in self.G.edges(n_st, keys=True):
                if self.G.edges[u,v,k]['type'] == 'boarding':
                    k_t = (u,v,k) if (u,v,k) in self.edge_idx else (v,u,k)
                    if k_t in self.edge_idx: idxs.append(self.edge_idx[k_t])
            self.node_to_boarding_indices[n_st] = idxs

    def _calc_times_fast(self, static_data, water_depth, current_time_min, line_stats=None):
        """
        计算特定边在特定积水深度下的瞬时通行时间。
        Returns: (当前受灾时间 t_curr, 正常理想时间 t_norm, 调试信息字典 debug_info)
        """
        lookup_time = current_time_min % 1440
        etype = static_data['type']
        
        # 准备调试信息
        debug_info = {}

        lid = static_data['line_id']
        ctx = None
        if lid and line_stats and str(lid) in line_stats:
            st_data = line_stats[str(lid)]
            ctx = {'is_above_ground': static_data.get('is_above_ground', 0), 
                   'max_rain_above': st_data['max_rain_above'], 
                   'max_water_total': st_data['max_water_total']}

        if etype == 'boarding':
            dist = static_data['boarding_dist']
            
            # 1. 速度计算
            speed_factor = self.strategy.get_speed_impact_factor(water_depth, 'boarding', line_context=ctx)
            v_curr = 1.4 * speed_factor
            
            # 记录调试信息
            debug_info['speed_factor'] = speed_factor
            debug_info['v_curr'] = v_curr
            debug_info['dist'] = dist

            if v_curr < 0.01: return float('inf'), 100.0, debug_info
            
            tw_c = dist / v_curr
            tw_n = dist / 1.4
            
            t_b = 30 # 列车停靠时间

            # 2. 等待时间计算
            t_wait_c, t_wait_n = 0, 0
            if static_data['line_id']:
                t_base = 300.0
                for r in static_data['riding_applicable_rules']:
                    if r['start_min'] <= lookup_time < r['end_min']:
                        t_base = r['interval']; break
                
                hw_n = t_base
                # 计算受影响后的发车间隔
                riding_factor = self.strategy.get_speed_impact_factor(water_depth, 'riding', line_context=ctx)
                hw_c = hw_n / max(riding_factor, 0.0001)
                
                # 排队论参数
                lam = max(static_data['boarding_node_flow'] / 3600.0, 0.0001)
                
                # 记录更多调试信息
                debug_info['hw_n'] = hw_n
                debug_info['hw_c'] = hw_c
                debug_info['riding_factor'] = riding_factor
                debug_info['lambda'] = lam

                # [修正开始] 定义单调性约束的排队等待时间计算函数
                def calc_w(EV, L):
                    # 1. 计算饱和度 (Saturation Ratio)
                    # 原分母公式 D = 2 * (1/L - 2*EV)
                    # 饱和状态意味着 1/L = 2*EV => L * 2 * EV = 1
                    # 因此定义 rho = L * 2 * EV
                    rho = L * 2 * EV
                    
                    # 设定安全阈值，超过此阈值认为进入拥堵/过饱和区
                    # 在此区域不再使用原公式（原公式有奇点且非单调），改用幂函数惩罚
                    RHO_THRESH = 0.9

                    # 计算阈值处的理论等待时间 (作为锚点)
                    # 当 rho = 0.9 时:
                    # denom_limit = 2 * (1/L - 2*EV) 
                    #             = 2/L * (1 - 2*EV*L) 
                    #             = 2/L * (1 - 0.9) = 0.2 / L
                    # 这是一个近似计算，为了性能，我们直接用硬逻辑判断
                    
                    if rho < RHO_THRESH:
                        # === 正常区域：使用原排队论公式 ===
                        denom = 2 * ((1.0/L) - 2*EV)
                        # 虽然理论上 rho < 0.9 保证 denom > 0，但加个由衷的兜底
                        if denom <= 1e-6: 
                            val = 1800.0 # 理论上不应走到这里
                        else:
                            val = 2.5*EV + (t_b**2 + 3*EV**2)/denom
                        
                        # 即使在正常区域，也允许适当的上限，但不要强制截断导致非单调
                        # 如果计算值确实很大（极罕见），让它保持很大，不要 clamp 到 1800
                        return min(val, 3600.0) 
                    
                    else:
                        # === 拥堵/过饱和区域：使用单调惩罚函数 ===
                        # 这里的逻辑是：既然已经堵了，那么发车间隔(EV)越大，rho越大，等待时间必须越长。
                        # 我们构造一个平滑过渡函数。
                        # 先算出在 RHO_THRESH (0.9) 时的边界值 T_limit
                        denom_limit = 2 * ((1.0/L) * (1 - RHO_THRESH))
                        t_limit = 2.5*EV + (t_b**2 + 3*EV**2)/denom_limit
                        
                        # 惩罚因子：(当前饱和度 / 阈值饱和度) 的平方
                        # 平方项确保了"越堵越慢"的非线性增长，有效拉开 Rain 和 Base 的差距
                        penalty_factor = (rho / RHO_THRESH) ** 2
                        
                        # 最终时间 = 边界时间 * 惩罚因子
                        # 这样即使 Base 很堵 (rho=0.95), Rain 更堵 (rho=1.1), 
                        # Rain 的结果也一定比 Base 大。
                        final_wait = t_limit * penalty_factor
                        
                        # 即使是非常极端的情况，也建议保留一个物理极限（如 2小时），
                        # 但这个极限必须足够大，以免再次出现 Base > Rain 的截断悖论
                        return min(final_wait, 7200.0) 

                # [修正结束]
                
                t_wait_n = calc_w(hw_n+t_b, lam)
                t_wait_c = calc_w(hw_c+t_b, lam)
                
                debug_info['t_wait_n'] = t_wait_n
                debug_info['t_wait_c'] = t_wait_c

            return tw_c + t_wait_c, tw_n + t_wait_n, debug_info
            
        elif etype == 'riding':
            t_base = 180.0
            for r in static_data['riding_applicable_rules']:
                if r['start_min'] <= lookup_time < r['end_min']:
                    t_base = r['interval']; break
            
            fac = self.strategy.get_speed_impact_factor(water_depth, 'riding', line_context=ctx)
            
            debug_info['t_base_interval'] = t_base
            debug_info['riding_factor'] = fac
            
            if fac <= 0.0001: return float('inf'), t_base, debug_info
            return t_base / fac, t_base, debug_info
            
        return 0, 0, debug_info

    def step(self, t_minute, day_type='Weekday'):
        """
        推进模拟时间步，并包含全网权重暴力一致性检查。
        """
        curr_clk = self.sim_start_min + t_minute
        
        # 1. 动态更新客流 (保持原逻辑)
        if self.real_flow_df is not None:
            curr_abs_time = None
            if hasattr(self.strategy, 'start_dt'):
                curr_abs_time = self.strategy.start_dt + timedelta(minutes=t_minute)
            
            if curr_abs_time:
                lookup_h = curr_abs_time.replace(minute=0, second=0, microsecond=0)
                try:
                    if curr_abs_time.minute == 0 or t_minute == 0:
                        batch = self.real_flow_df[self.real_flow_df['time'] == lookup_h]
                        if not batch.empty:
                            for _, row in batch.iterrows():
                                st_name = row['station']
                                flow_val = (row['inbound'] + row['outbound']) / 2
                                node_st = f"{st_name}_车站层"
                                if self.G.has_node(node_st):
                                    self.G.nodes[node_st]['flow'] = flow_val
                                    
                            for est in self.fast_edge_data:
                                if est['type'] == 'boarding':
                                    st_node_idx = est['u_idx'] if self.G.nodes[list(self.G.nodes())[est['u_idx']]].get('layer') == 'station' else est['v_idx']
                                    st_node_name = list(self.G.nodes())[st_node_idx]
                                    est['boarding_node_flow'] = self.G.nodes[st_node_name].get('flow', 1000)
                except Exception as e:
                    pass

        self.strategy.move_step()
        clat, clon = self.strategy.center_loc
        
        # 2. 更新积水
        for i, (lat, lon) in enumerate(self.node_pos_list):
            self.node_dists_list[i] = self.strategy._haversine(clat, clon, lat, lon)
        for i, n in enumerate(self.G.nodes()):
            w = self.strategy.get_intensity(t_minute, self.G.nodes[n]['pos'], precomputed_dist=self.node_dists_list[i])
            self.G.nodes[n]['water'] = w
            
        # 3. 统计线路风险
        line_stats = {}
        node_waters = [self.G.nodes[n].get('water', 0) for n in self.G.nodes()]
        for sd in self.fast_edge_data:
            if sd['type'] == 'riding' and sd['line_id']:
                lid = str(sd['line_id'])
                if lid not in line_stats: line_stats[lid] = {'max_rain_above': 0.0, 'max_water_total': 0.0}
                aw = (node_waters[sd['u_idx']] + node_waters[sd['v_idx']]) / 2
                if aw > line_stats[lid]['max_water_total']: line_stats[lid]['max_water_total'] = aw
                if sd['is_above_ground']:
                    r = self.strategy.get_instantaneous_intensity(t_minute, sd['riding_pos_mid'])
                    if r > line_stats[lid]['max_rain_above']: line_stats[lid]['max_rain_above'] = r
                    
        # 4. DIIM 更新与全网扫描
        curr_c = np.zeros(len(self.edge_list))
        
        # [暴力扫描] 统计计数器
        scan_count_rain_faster = 0
        scan_total_edges = 0
        scan_debug_msg = ""

        for i, sd in enumerate(self.fast_edge_data):
            aw = (node_waters[sd['u_idx']] + node_waters[sd['v_idx']]) / 2
            
            # 计算 Rain 时间
            tc, tn, _ = self._calc_times_fast(sd, aw, curr_clk, line_stats)
            
            # 计算 Base 时间 (强制无雨)
            tb, _, _ = self._calc_times_fast(sd, 0.0, curr_clk, line_stats=None)
            
            # --- 核心检查点 ---
            # 只要 Rain 比 Base 快超过 0.0001秒，就记录下来
            if tc < tb - 0.0001:
                scan_count_rain_faster += 1
                if scan_count_rain_faster <= 3: # 只记录前3个，防止刷屏
                    scan_debug_msg += f"\n   [BAD EDGE] {self.node_st_names[sd['u_idx']]}->{self.node_st_names[sd['v_idx']]} ({sd['type']}) | Rain={tc:.4f} < Base={tb:.4f}"

            # DIIM 逻辑
            val = 0.0
            if tc == float('inf'): val = 1.0 
            elif tc > 0: val = 1.0 - (tn / tc)
            curr_c[i] = max(0.0, min(1.0, val)) 
            scan_total_edges += 1
        
        # [打印扫描结果]
        if scan_count_rain_faster > 0:
            print(f"\n🚨 [CRITICAL SCAN] Time {t_minute}: Found {scan_count_rain_faster}/{scan_total_edges} edges where Rain is FASTER!")
            print(scan_debug_msg)
        else:
            # 如果没有发现异常边，打印一条确认信息
            # print(f"✅ [SCAN OK] Time {t_minute}: All edges Rain >= Base.")
            pass

        q_next = (1 - self.k_recovery) * self.q_vector + self.k_recovery * curr_c
        self.q_vector = np.clip(q_next, 0, 1)
        
        for i, (u,v,k) in enumerate(self.edge_list): self.G.edges[u,v,k]['q_value'] = self.q_vector[i]
        aff_lines = set()
        for i, q in enumerate(self.q_vector):
            if q > 0.1:
                lid = self.fast_edge_data[i]['line_id']
                if lid: aff_lines.add(lid)
        node_risks = []
        for n, idxs in self.node_to_boarding_indices.items():
            if idxs: node_risks.append(max(self.q_vector[i] for i in idxs))
            else: node_risks.append(0)
        
        self.current_line_stats = line_stats
        avg_water = np.mean(list(node_waters))
        max_n_risk = max(node_risks) if node_risks else 0
        
        avg_q_safe = np.nanmean(self.q_vector) if len(self.q_vector) > 0 else 0.0
        avg_q_safe = np.nan_to_num(avg_q_safe)
        
        return avg_q_safe, avg_water, len(aff_lines), max_n_risk

# ==============================================================================
# 3. 蒙特卡洛数据记录器：收集并保存多次随机实验的数据
# ==============================================================================
class MonteCarloRecorder:
    def __init__(self, time_steps):
        self.times = time_steps
        self.df_eff = pd.DataFrame({'Time': time_steps})
        self.df_risk = pd.DataFrame({'Time': time_steps})
        self.df_water = pd.DataFrame({'Time': time_steps})
        self.df_intensity = pd.DataFrame({'Time': time_steps})
        self.df_lat = pd.DataFrame({'Time': time_steps})
        self.df_lon = pd.DataFrame({'Time': time_steps})
        self.df_aff_lines = pd.DataFrame({'Time': time_steps})
        self.df_max_n_risk = pd.DataFrame({'Time': time_steps})

    def add_run_data(self, run_id, eff, risk, water, inten, lats, lons, aff_ln, max_nr):
        col_name = f"Run_{run_id}"
        self.df_eff[col_name] = eff
        self.df_risk[col_name] = risk
        self.df_water[col_name] = water
        self.df_intensity[col_name] = inten
        self.df_lat[col_name] = lats
        self.df_lon[col_name] = lons
        self.df_aff_lines[col_name] = aff_ln
        self.df_max_n_risk[col_name] = max_nr

    def save_to_excel(self, base_path):
        fname = os.path.join(base_path, f"MonteCarlo_Results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.xlsx")
        print(f"💾 正在导出蒙特卡洛数据到: {fname}")
        try:
            with pd.ExcelWriter(fname, engine='openpyxl') as writer:
                self.df_eff.to_excel(writer, sheet_name='Efficiency', index=False)
                self.df_risk.to_excel(writer, sheet_name='Risk_Avg_Q', index=False)
                self.df_water.to_excel(writer, sheet_name='Water_Level_Avg', index=False)
                self.df_intensity.to_excel(writer, sheet_name='Intensity_Center', index=False)
                self.df_lat.to_excel(writer, sheet_name='Location_Lat', index=False)
                self.df_lon.to_excel(writer, sheet_name='Location_Lon', index=False)
                self.df_aff_lines.to_excel(writer, sheet_name='Affected_Line_Count', index=False)
                self.df_max_n_risk.to_excel(writer, sheet_name='Max_Node_Risk', index=False)
            print("✅ 导出完成。")
        except Exception as e:
            print(f"❌ 导出失败: {e}")

# ==========================================
# 5. [Worker] 独立仿真任务：供多进程并行调用
# 注意：该函数必须定义在顶层，以便 Joblib 进行序列化 (Pickle)
# ==========================================
def simulation_worker(run_id, base_path, intensity, period, rain_dur, sim_dur, start_str, 
                      use_real_data=False, real_db_name=None, real_flow_data=None):
    try:
        # [修改] 支持真实数据模式初始化
        if use_real_data:
            strategy = RealDataRainfall(base_path, db_name=real_db_name, start_datetime_str=start_str)
            # 在真实模式下，我们传入客流 DataFrame
            sys_obj = SubwayDIIMSystem(base_path, strategy, sim_start_time=start_str, real_flow_df=real_flow_data)
        else:
            strategy = ExtremeRainfall(base_path, max_intensity=intensity, 
                                       rainfall_type='chicago', P=period, 
                                       rain_duration=rain_dur)
            
            # [修改] 为 ExtremeRainfall 模式注入客流数据和基准时间
            # 我们将模拟日期固定在无雨的 2025-08-08，以便使用这天的真实客流
            base_date_str = "2025-08-08"
            full_start_str = f"{base_date_str} {start_str}"
            try:
                # 手动给 strategy 绑定 start_dt，以便 SubwayDIIMSystem 能够计算绝对时间并查询客流
                strategy.start_dt = datetime.strptime(full_start_str, "%Y-%m-%d %H:%M")
            except ValueError:
                pass
                
            # 将外部传入的 real_flow_data (08.08的客流) 传给系统
            sys_obj = SubwayDIIMSystem(base_path, strategy, sim_start_time=start_str, real_flow_df=real_flow_data)
            
        t_obj = datetime.strptime(start_str, "%Y-%m-%d %H:%M") if len(start_str) > 5 else datetime.strptime(start_str, "%H:%M")
        start_min = t_obj.hour * 60 + t_obj.minute
        time_steps = list(range(sim_dur))
        r_eff, r_risk, r_water, r_inten, r_lat, r_lon, r_aff, r_mnr = [], [], [], [], [], [], [], []
        
        for t in time_steps:
            avg_q, avg_w, n_lines, max_nr = sys_obj.step(t)
            curr_clk = start_min + t
            # 计算该时间步的网络加权效率
            ef = sys_obj.calculate_weighted_efficiency(curr_clk, 'Weekday', line_stats=getattr(sys_obj, 'current_line_stats', None))
            
            # 对于真实数据，位置固定，返回 0 或默认中心
            c_lat, c_lon = (0, 0)
            if hasattr(sys_obj.strategy, 'center_loc'):
                c_lat, c_lon = sys_obj.strategy.center_loc
                
            # 瞬时雨强，如果是真实数据，传一个中心点坐标去获取，或者取平均？
            # 这里取 (31.23, 121.47) 上海中心点作为参考
            ref_pos = (31.2304, 121.4737)
            inten = sys_obj.strategy.get_instantaneous_intensity(t, ref_pos)
            
            r_eff.append(ef)
            r_risk.append(avg_q)
            r_water.append(avg_w)
            r_inten.append(inten)
            r_lat.append(c_lat)
            r_lon.append(c_lon)
            r_aff.append(n_lines)
            r_mnr.append(max_nr)
        
        # 强制垃圾回收，防止内存泄漏
        del sys_obj
        del strategy
        gc.collect() 
        return (run_id, r_eff, r_risk, r_water, r_inten, r_lat, r_lon, r_aff, r_mnr)
    except Exception as e:
        return (run_id, None, str(e), None, None, None, None, None, None)

# [新增] 数据一致性校验函数
def verify_data_consistency(base_path, flow_df):
    """
    检查 Excel 客流数据、JSON 拓扑数据、DB 降雨数据中的车站名称一致性。
    """
    print("\n🔍 正在进行数据一致性校验...")
    
    # 1. 获取 Excel 中的车站列表
    excel_stations = set(flow_df['station'].unique())
    
    # 2. 获取 JSON 中的车站列表
    json_path = os.path.join(base_path, "subway.json")
    json_stations = set()
    try:
        with open(json_path, "r", encoding="utf-8") as f: 
            data = json.load(f)
            for l in data['l']:
                for st in l['st']:
                    json_stations.add(st['n'])
    except:
        print("⚠️ 无法读取 subway.json，跳过拓扑校验。")

    # 3. 打印比对结果 (Excel vs JSON)
    if json_stations:
        missing_in_json = excel_stations - json_stations
        missing_in_excel = json_stations - excel_stations
        
        print(f"  - Excel 中有 {len(excel_stations)} 个车站")
        print(f"  - JSON 中有 {len(json_stations)} 个车站")
        
        if missing_in_json:
            print(f"⚠️ 警告: Excel 中有 {len(missing_in_json)} 个车站在 JSON 拓扑中找不到 (可能无法更新其客流):")
            print(f"    示例: {list(missing_in_json)[:5]} ...")
        else:
            print("✅ Excel 所有车站均已在 JSON 拓扑中找到。")
            
        if missing_in_excel:
            # 这通常不是问题，因为 Excel 可能只包含部分车站数据
            pass

    # 4. 获取 DB 中的车站列表 (简单检查)
    db_path = os.path.join(base_path, "shanghai_rain_history.db")
    if os.path.exists(db_path):
        try:
            conn = sqlite3.connect(db_path)
            cursor = conn.cursor()
            cursor.execute("SELECT DISTINCT station_name FROM stations") # 假设表名 stations，列名 station_name
            db_stations = set([r[0] for r in cursor.fetchall()])
            conn.close()
            
            print(f"  - DB 中有 {len(db_stations)} 个气象监测站")
            # 气象站和地铁站通常不同名，这里仅做数量确认，不做集合比对
        except:
            print("⚠️ 无法读取数据库中的站点信息，跳过校验。")
    else:
        print("⚠️ 数据库文件不存在。")
    print("--------------------------------------------------\n")


# ==========================================
# 6. 主程序入口 (Joblib 并行执行版)
# ==========================================
def main():
    # === 参数配置区域 ===
    BASE_PATH = "E:\\BaiHao\\20260103算例补充\\输入与输出数据"
    
    # --- 模式开关 ---
    USE_REAL_DATA = False   # [开关] 是否使用 2025.07.29-08.01 的真实数据
    # ----------------
    
    if USE_REAL_DATA:
        # 真实数据模式配置
        MC_ITERATIONS = 1         # 真实数据是确定性的，通常跑 1 次即可 (或者跑多次以测试算法稳定性)
        START_TIME_STR = "2025-07-29 05:00" # 仿真开始时间
        END_TIME_STR   = "2025-08-01 23:00" # 仿真结束时间
        #START_TIME_STR = "2025-07-29 07:00" # 仿真开始时间
        #END_TIME_STR   = "2025-07-29 08:00" # 仿真结束时间
        
        # 计算总分钟数
        t_start = datetime.strptime(START_TIME_STR, "%Y-%m-%d %H:%M")
        t_end   = datetime.strptime(END_TIME_STR, "%Y-%m-%d %H:%M")
        SIM_DURATION = int((t_end - t_start).total_seconds() / 60)
        
        DB_NAME = "shanghai_rain_history.db"
        FLOW_FILE = "shanghai_metro_flow_final_complete.xlsx" # 支持 csv 或 xlsx
        
        print(f"\n🚀 启动真实数据仿真模式 ({START_TIME_STR} -> {END_TIME_STR}, Duration={SIM_DURATION}min)")
        
        # 加载客流数据 (Pre-load flow data)
        flow_path = os.path.join(BASE_PATH, FLOW_FILE)
        print(f"📂 加载客流文件: {flow_path}")
        try:
            if flow_path.endswith('.csv'):
                flow_df = pd.read_csv(flow_path)
            else:
                flow_df = pd.read_excel(flow_path)

            # 1. 基础清洗：转字符串、去空格、统一符号
            flow_df['station'] = flow_df['station'].astype(str).str.strip().str.replace('●', '·').str.replace(' ', '')

            # 2. 特殊处理：将“浦电路”拆分为4号线和6号线，客流平分
            # 筛选出原始名称为 "浦电路" 的所有行
            pudian_mask = (flow_df['station'] == '浦电路')
            
            if pudian_mask.any():
                # 提取这些行
                pudian_rows = flow_df[pudian_mask].copy()
                
                # 构造4号线数据 (取名为 JSON 中对应的 '浦电路(4号线)', 客流除以2)
                pudian_4 = pudian_rows.copy()
                pudian_4['station'] = '浦电路(4号线)'
                pudian_4['inbound'] = pudian_4['inbound'] / 2.0
                pudian_4['outbound'] = pudian_4['outbound'] / 2.0
                
                # 构造6号线数据 (取名为 JSON 中对应的 '浦电路(6号线)', 客流除以2)
                pudian_6 = pudian_rows.copy()
                pudian_6['station'] = '浦电路(6号线)'
                pudian_6['inbound'] = pudian_6['inbound'] / 2.0
                pudian_6['outbound'] = pudian_6['outbound'] / 2.0
                
                # 从原表中删除“浦电路”，并追加拆分后的两部分数据
                flow_df = flow_df[~pudian_mask] # 删除原行
                flow_df = pd.concat([flow_df, pudian_4, pudian_6], ignore_index=True)
                print(f"ℹ️ 已将 {len(pudian_rows)} 条'浦电路'数据拆分为4号线与6号线 (客流平分)。")

            # 【新增修复】 简单的站点名称清洗，解决匹配警告
            def clean_station_name(name):
                name = str(name).strip()
                # 1. 替换特殊符号 '●' 为标准符号 '·' 
                name = name.replace('●', '·').replace(' ', '')
                
                # 2. 修正已知的 Excel 录入笔误及旧站名
                fixes = {
                    # --- 笔误修正 ---
                    '淞淞发路': '淞发路',
                    '淞淞虹路': '淞虹路',
                    '淞淞滨路': '淞滨路',  # [新增] 修正笔误
                    '临洮洮路': '临洮路',  # [新增] 修正笔误
                    '颛颛桥': '颛桥',
                    '佘佘山': '佘山',
                    # --- 站名变更/映射 ---               
                    # --- 暂未开通/无拓扑车站 ---
                    '龙居路': '龙居路',    # 14号线暂缓开通，拓扑中无此站，保留原名即可
                    '严御路': '康恒路',    # [新增] 严御路已更名为康恒路(11号线) 。但是拓扑里面没它。
                }
                return fixes.get(name, name)

            flow_df['station'] = flow_df['station'].apply(clean_station_name)

            # 预处理：转换时间列，方便后续查询
            flow_df['time'] = pd.to_datetime(flow_df['time'])
            
            # 执行一致性校验
            verify_data_consistency(BASE_PATH, flow_df)
            
        except Exception as e:
            print(f"❌ 读取客流文件失败: {e}")
            return

    else:
        # 原始蒙特卡洛模式配置
        MC_ITERATIONS = 100       
        RAIN_P_PERIOD = 5       
        RAIN_DURATION = 180       
        SIM_DURATION = 240        
        START_TIME_STR = "09:00"     
        print(f"\n🚀 启动随机蒙特卡洛仿真模式 (Runs={MC_ITERATIONS})")
        DB_NAME = None
        
        # [修改] 读取 08.08 的基准客流数据 (全量 CSV)
        flow_file_full = "shanghai_metro_flow_with_code_final_20250802-20250831_v4.csv"
        flow_path = os.path.join(BASE_PATH, flow_file_full)
        print(f"📂 加载基准客流文件 (2025.08.08): {flow_path}")
        
        try:
            # 读取 CSV
            full_df = pd.read_csv(flow_path)
            
            # 过滤 2025-08-08 的数据
            # [修改] 原代码的 format='%Y-%m-%d %H' 无法解析 ":00:00" 部分，改为自动推断
            full_df['time'] = pd.to_datetime(full_df['time']) 
            
            target_date = pd.Timestamp("2025-08-08").date()
            flow_df = full_df[full_df['time'].dt.date == target_date].copy()
            
            # 重命名列以匹配 SubwayDIIMSystem
            # csv: inbound, outbound, station_name_mapped
            # system expects: inbound, outbound, station
            flow_df.rename(columns={'station_name_mapped': 'station'}, inplace=True)
            
            # 站点名称清洗 (确保与 if 块中的逻辑一致)
            def clean_station_name(name):
                name = str(name).strip()
                name = name.replace('●', '·').replace(' ', '')
                fixes = {
                    '淞淞发路': '淞发路',
                    '淞淞虹路': '淞虹路',
                    '淞淞滨路': '淞滨路',
                    '临洮洮路': '临洮路', 
                    '颛颛桥': '颛桥',
                    '佘佘山': '佘山',
                    '严御路': '康恒路',
                }
                return fixes.get(name, name)
            flow_df['station'] = flow_df['station'].apply(clean_station_name)
            
            # 处理浦电路拆分 (简单起见，这里复用逻辑)
            pudian_mask = (flow_df['station'] == '浦电路')
            if pudian_mask.any():
                pudian_rows = flow_df[pudian_mask].copy()
                pudian_4 = pudian_rows.copy(); pudian_4['station'] = '浦电路(4号线)'; pudian_4[['inbound','outbound']] /= 2
                pudian_6 = pudian_rows.copy(); pudian_6['station'] = '浦电路(6号线)'; pudian_6[['inbound','outbound']] /= 2
                flow_df = flow_df[~pudian_mask]
                flow_df = pd.concat([flow_df, pudian_4, pudian_6], ignore_index=True)
            
            print(f"✅ 已加载 2025.08.08 客流数据，共 {len(flow_df)} 条记录")
            
        except Exception as e:
            print(f"❌ 读取客流文件失败: {e}")
            flow_df = None

    # N_JOBS 控制并行核心数
    # N_JOBS = 10 if USE_REAL_DATA else 30 # 真实数据读取DB开销大，建议减少并发数或设为1
    N_JOBS = 40

    recorder = MonteCarloRecorder(list(range(SIM_DURATION)))
    
    try:
        # 使用 joblib.Parallel 并行派发 simulation_worker 任务
        results = Parallel(n_jobs=N_JOBS, backend='loky', verbose=10)(
            delayed(simulation_worker)(
                run+1, BASE_PATH, 250, 
                RAIN_P_PERIOD, # period (unused in real mode)
                RAIN_DURATION, # rain_dur (unused in real mode)
                SIM_DURATION, START_TIME_STR,
                use_real_data=USE_REAL_DATA,
                real_db_name=DB_NAME,
                real_flow_data=flow_df # [修改] 将读取到的 flow_df 传递进去
            ) for run in range(MC_ITERATIONS)
        )
        
        # 遍历并行结果并存入记录器
        for res in results:
            if res[1] is None:
                print(f"❌ [Error] Run {res[0]} failed: {res[2]}")
            else:
                recorder.add_run_data(*res)
                
    except Exception as e:
        print(f"❌ [Critical] Parallel execution failed: {e}")

    print("\n🏁 仿真结束，正在保存数据...")
    recorder.save_to_excel(BASE_PATH)

if __name__ == "__main__":
    main()

In [ ]:
import networkx as nx
import json
import math
import pandas as pd
import numpy as np
import random
import os
import re
import gc
import time
import sys
import sqlite3
import warnings
from abc import ABC, abstractmethod
from scipy import sparse
from scipy.spatial.distance import cdist
from scipy.interpolate import griddata
from PIL import Image
from datetime import datetime, timedelta

# [修改 1] 引入 joblib 用于 Jupyter 环境下的并行计算
from joblib import Parallel, delayed 

# 禁用插值产生的可能的警告
warnings.filterwarnings("ignore")

# ==============================================================================
# [全局配置] 并行计算环境设置
# ==============================================================================
# 限制 Numpy/OpenBLAS 的底层线程数。
# 原因：在 Joblib 多进程模式下，如果每个子进程再自动开启多线程进行矩阵运算，
# 会导致 CPU 资源争抢（Oversubscription）甚至死锁，显著降低仿真效率。
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

# ==============================================================================
# 1. 策略模式：定义扰动（降雨）行为
# ==============================================================================
class DisturbanceStrategy(ABC):
    """
    抽象基类：定义系统受到的外部干扰策略。
    
    Methods:
        get_intensity: 获取特定时空的干扰强度。
        get_speed_impact_factor: 获取干扰对系统运行速度的影响系数。
    """
    @abstractmethod
    def get_intensity(self, time_step, position): pass
    @abstractmethod
    def get_speed_impact_factor(self, intensity, edge_type, **kwargs): pass

class ExtremeRainfall(DisturbanceStrategy):
    """
    极端降雨策略模型：模拟一个移动的暴雨中心及其对地铁站点的积水影响。
    
    包含物理过程：
    1. 芝加哥雨型（Chicago Design Storm）生成时间序列降雨。
    2. 移动风暴中心（Moving Storm Center）。
    3. 空间衰减（ARF）计算。
    4. SCS-CN 径流模型计算地表积水。
    """
    def __init__(self, base_path, max_intensity=201.9, rainfall_type='chicago', P=100, r=0.405, rain_duration=180):
        """
        初始化降雨模型参数。
        
        Args:
            max_intensity (float): 降雨峰值强度参数 (mm/h 或相关系数)。
            P (int): 降雨重现期 (Return Period, e.g., 100年一遇)。
            r (float): 雨峰位置系数 (Peak Position parameter)，0-1之间，0.4代表峰值出现在前40%。
            rain_duration (int): 降雨持续总时长 (分钟)。
        """
        # 设定模拟的空间范围（上海经纬度大致范围）
        self.lat_range = (30.6984, 31.5139)
        self.lon_range = (120.849, 121.966)
        
        # 初始化暴雨中心起始位置和移动速度
        self.center_loc, self.velocity = self._init_random_boundary_start()
        
        self.Q0 = max_intensity  # 降雨强度基准
        self.base_path = base_path
        self.rainfall_type = rainfall_type
        self.P_period = P        # 重现期
        self.r_peak = r          # 雨峰系数
        self.rain_duration = rain_duration 
        
        self.chicago_profile = None 
        self.img_array = None
        self._load_drainage_image() # 加载区域排水能力图
        self._distance_cache = {}   # 缓存站点到雨中心的距离，减少重复三角函数计算
        
        # 如果是芝加哥雨型，预计算随时间变化的强度曲线
        if self.rainfall_type == 'chicago': self._precompute_chicago_profile()

    def _init_random_boundary_start(self):
        """
        随机初始化降雨中心。
        逻辑：选择地图四个边界之一作为起点，目标点设为地图中心附近，以此计算移动矢量。
        """
        lat_min, lat_max = self.lat_range
        lon_min, lon_max = self.lon_range
        side = random.choice([0, 1, 2, 3]) # 0:上(北), 1:下(南), 2:左(西), 3:右(东)
        
        # 设定目标点为区域中心附近略带随机偏移
        target_lat = (lat_min + lat_max) / 2 + random.uniform(-0.1, 0.1)
        target_lon = (lon_min + lon_max) / 2 + random.uniform(-0.1, 0.1)
        
        if side == 0:   start_lat, start_lon = lat_max, random.uniform(lon_min, lon_max)
        elif side == 1: start_lat, start_lon = lat_min, random.uniform(lon_min, lon_max)
        elif side == 2: start_lat, start_lon = random.uniform(lat_min, lat_max), lon_min
        else:           start_lat, start_lon = random.uniform(lat_min, lat_max), lon_max
        
        speed_kmh = random.uniform(25, 40) # 降雨系统移动速度 (km/h)
        speed_km_min = speed_kmh / 60.0
        
        # 计算移动单位向量（转换为经纬度增量）
        delta_lat_km = (target_lat - start_lat) * 111.0
        delta_lon_km = (target_lon - start_lon) * 96.0
        dist_km = math.sqrt(delta_lat_km**2 + delta_lon_km**2) or 1
        
        v_lat_deg = ((delta_lat_km / dist_km) * speed_km_min) / 111.0
        v_lon_deg = ((delta_lon_km / dist_km) * speed_km_min) / 96.0
        return (start_lat, start_lon), (v_lat_deg, v_lon_deg)

    def move_step(self):
        """每一分钟移动一次降雨中心，并清空距离缓存（因为中心变了，所有距离需重算）"""
        curr_lat, curr_lon = self.center_loc
        v_lat, v_lon = self.velocity
        self.center_loc = (curr_lat + v_lat, curr_lon + v_lon)
        self._distance_cache = {} 

    def _precompute_chicago_profile(self):
        """
        预计算芝加哥雨型（Chicago Design Storm）的时间分布。
        公式来源：Keifer & Chu (1957)，参数 A, n, b 针对上海地区暴雨强度公式适配。
        """
        times = np.arange(0, self.rain_duration + 1, 1) 
        intensities = []
        t_peak_time = self.rain_duration * self.r_peak
        
        # A, n, b 为上海地区的降雨公式参数（需根据研究区域调整）
        A = 9.581 * (1 + 0.846 * np.log10(self.P_period))
        n = 0.656
        b = 7
        
        for t in times:
            # 峰前时刻公式
            if t <= t_peak_time:
                t_b = t_peak_time - t
                num = A * ((1 - n) * (t_b / self.r_peak) + b)
                den = math.pow((t_b / self.r_peak) + b, 1 + n)
            # 峰后时刻公式
            else:
                t_a = t - t_peak_time
                num = A * ((1 - n) * (t_a / (1 - self.r_peak)) + b)
                den = math.pow((t_a / (1 - self.r_peak)) + b, 1 + n)
            intensities.append(num / den)
        
        # 模拟雨后由于排水不畅导致的“长尾效应”（指数衰减）
        tail_duration = 120  
        if intensities:
            last_val = intensities[-1]
            for t_ex in range(1, tail_duration + 1):
                val = last_val * math.exp(-0.1 * t_ex)
                intensities.append(val if val > 0.001 else 0.0)
        
        # 存储为累计降雨量 (Cumulative Rainfall)，方便后续计算径流
        self.chicago_profile = np.cumsum(intensities)

    def _load_drainage_image(self):
        """加载排水能力栅格图，用于根据地理位置查询排水效率"""
        path = os.path.join(self.base_path, "排水能力_简化.png")
        try:
            img = Image.open(path)
            self.img_array = np.array(img)
        except FileNotFoundError:
            # 如果没图，生成一个随机的（调试用）
            self.img_array = np.random.randint(0, 255, (1000, 1000, 3), dtype=np.uint8)

    def _haversine(self, lat1, lon1, lat2, lon2):
        """计算地球表面两点间的大圆距离 (km)"""
        R = 6371.0 
        dlat = math.radians(lat2 - lat1)
        dlon = math.radians(lon2 - lon1)
        a = math.sin(dlat/2)**2 + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(dlon/2)**2
        return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))

    def _lat_lon_to_pixel(self, lat, lon):
        """经纬度坐标与图像像素坐标的转换"""
        if self.img_array is None: return 0, 0
        h, w = self.img_array.shape[:2]
        lat_min, lat_max = self.lat_range
        lon_min, lon_max = self.lon_range
        px = int((lon - lon_min) / (lon_max - lon_min) * w)
        py = int((lat_max - lat) / (lat_max - lat_min) * h)
        return min(max(px, 0), w - 1), min(max(py, 0), h - 1)

    def _get_drainage_from_image(self, lat, lon):
        """从像素颜色值映射到实际排水能力 (mm/h)"""
        px, py = self._lat_lon_to_pixel(lat, lon)
        # 获取灰度值或R通道值
        r_val = self.img_array[py, px][0] if self.img_array.ndim == 3 else self.img_array[py, px]
        r_norm = r_val / 255.0
        # 简单的阈值映射规则
        if r_val == 0: return 27
        elif r_norm <= 0.9: return 36
        elif r_norm <= 0.99: return 27
        else: return 50

    def _calculate_arf_rainfall(self, distance):
        """空间衰减计算：根据距雨中心距离计算面雨量折减系数 (Area Reduction Factor, ARF)"""
        area = math.pi * (distance**2)
        # ARF 经验公式
        factor = -0.058 * math.log(area) + 0.9949 if area >= 1 else 1.0
        Q_temp = self.Q0 * factor
        # 增加空间随机扰动
        return max(0, Q_temp + np.random.normal(0, 0.3 * abs(Q_temp)))

    def get_instantaneous_intensity(self, time_step, position, precomputed_dist=None):
        """
        获取某时刻、某位置的瞬时降雨强度 (mm/h)。
        主要用于地上线路的能见度/运行风险判断。
        [修改说明] 引入60分钟滑动平均，解决芝加哥雨型瞬时峰值过高的问题。
        """
        if precomputed_dist is not None: dist = precomputed_dist
        else:
            if position in self._distance_cache: dist = self._distance_cache[position]
            else:
                dist = self._haversine(self.center_loc[0], self.center_loc[1], position[0], position[1])
                self._distance_cache[position] = dist
                
        spatial_val = self._calculate_arf_rainfall(dist)
        
        if self.rainfall_type == 'chicago':
            if time_step >= len(self.chicago_profile) - 1: return 0.0
            scale = spatial_val / self.Q0 if self.Q0 > 0 else 0
            
            # [核心修改] 使用 60分钟滑动窗口计算小时雨强
            idx_now = min(int(time_step), len(self.chicago_profile) - 1)
            window_size = 60 
            idx_prev = max(0, idx_now - window_size)
            
            # 这一时段内的总雨量 (mm)
            rain_depth_in_window = self.chicago_profile[idx_now] - self.chicago_profile[idx_prev]
            
            # 时段长度 (min)
            duration = idx_now - idx_prev
            if duration <= 0: return 0.0
            
            # 换算成小时雨强 (mm/h)
            intensity_hourly = (rain_depth_in_window / duration) * 60.0
            
            return max(0, intensity_hourly * scale)
        else:
            # 简单均匀降雨模式
            return max(0, spatial_val / (self.rain_duration / 60.0)) if self.rain_duration > 0 else 0

    def get_intensity(self, time_step, position, precomputed_dist=None):
        """
        核心计算：计算地表净积水深度 (mm)。
        使用了 SCS-CN (Soil Conservation Service Curve Number) 径流模型的简化形式。
        Logic: 积水 = 降雨径流(Runoff) - 城市管网排水(Drainage)
        """
        lat, lon = position
        if precomputed_dist is not None: dist = precomputed_dist
        else:
            if position in self._distance_cache: dist = self._distance_cache[position]
            else:
                dist = self._haversine(self.center_loc[0], self.center_loc[1], lat, lon)
                self._distance_cache[position] = dist
        spatial_val = self._calculate_arf_rainfall(dist)
        
        # 1. 计算累计降雨量 (Cumulative Rainfall)
        if self.rainfall_type == 'chicago':
            scale = spatial_val / self.Q0 if self.Q0 > 0 else 0
            idx = min(int(time_step), len(self.chicago_profile) - 1)
            cum_rain = self.chicago_profile[idx] * scale
        else:
            progress = min(time_step / float(self.rain_duration), 1.0)
            cum_rain = spatial_val * progress
            
        # 2. SCS 径流转换
        S = 15.7 # 潜在最大入渗量 (Potential Maximum Retention)，与土壤/地表类型有关
        # SCS方程：Q = (P - 0.2S)^2 / (P + 0.8S)
        runoff = (cum_rain - 0.05 * S)**2 / (cum_rain + 0.95 * S) if cum_rain > 0.05 * S else 0
        
        # 3. 扣除区域排水
        drainage = (self._get_drainage_from_image(lat, lon) / 60.0) * time_step
        return max(0, runoff - drainage) # 返回值为净积水深度

    def get_speed_impact_factor(self, water_depth, edge_type, line_context=None):
        """
        计算速度折减系数：将物理积水深度转换为对系统运行/步行速度的影响 (0.0 - 1.0)。
        1.0 表示正常速度，0.0 表示完全中断。
        """
        # [修改] 提取线路整体最大积水深度，用于判断是否触发全线停运
        # 如果 max_water_total > 450，意味着线路上某处已严重积水，该线路应整体停运
        max_water_line = 0.0
        if line_context:
            max_water_line = line_context.get('max_water_total', 0.0)

        # [修改] 核心逻辑变更：
        # 如果当前站点水深 > 450mm OR 线路任意位置水深 > 450mm (由 line_context 传入)，
        # 则视为线路瘫痪。
        is_line_failed = (water_depth > 450) or (max_water_line > 450)

        if edge_type == 'boarding': 
            # 进站/步行速度折减 (基于经验研究的步速-水深函数，如 Ishigaki et al.)
            # [修改] 若线路瘫痪（不论是本站淹水还是线路其他站淹水），进站系数设为 0.0001
            if is_line_failed:
                return 0.0001
            
            if water_depth <= 0: return 1.0
            if water_depth <= 250: return max(0.0001, 0.9 - (water_depth / 250.0) * 0.1)
            elif water_depth <= 450: return 0.0001
            else: return 0.0001 # 兜底逻辑，实际上 is_line_failed 已覆盖
            
        elif edge_type == 'riding': 
            # 乘车运行速度折减
            if line_context:
                is_above = line_context.get('is_above_ground', 0)
                max_rain = line_context.get('max_rain_above', 0.0)
                # max_water_line 已经在上面提取了
                
                if is_above == 1: # 地上线主要受瞬时强降雨（能见度）影响
                    if max_rain > 100: return 0.0001 # [修改] 极强降雨停运，因子改为 0.0001
                    elif max_rain >= 80: return 0.5 # 强降雨限速
                else: # 地下线主要受区间进水影响
                    # [修改] 若线路上最大积水超过450mm，列车停运，影响因子变为0.0001
                    if max_water_line > 450: return 0.0001
            return 1.0

# ==============================================================================
# [新增] 真实数据降雨策略类
# ==============================================================================
class RealDataRainfall(DisturbanceStrategy):
    """
    真实数据降雨策略：基于 SQLite 数据库中的历史/实时降雨数据进行空间插值。
    模拟 2025-07-29 等特定日期的真实降雨分布。
    """
    def __init__(self, base_path, db_name="shanghai_rain_history.db", start_datetime_str="2025-07-29 00:00"):
        self.base_path = base_path
        self.db_path = os.path.join(base_path, db_name)
        self.start_dt = datetime.strptime(start_datetime_str, "%Y-%m-%d %H:%M")
        self.center_loc = (31.2304, 121.4737) # 默认中心设为上海市中心，适配接口（真实数据无中心概念）
        
        # 缓存机制，避免每分钟每节点都去查库插值
        self._last_query_time_str = None
        self._cached_grid_interpolator = None # 存储 griddata 的结果或KDTree
        self._cached_df = None
        
        # 上海区域网格参数
        self.SH_LON_MIN, self.SH_LON_MAX = 120.8, 122.3
        self.SH_LAT_MIN, self.SH_LAT_MAX = 30.6, 31.9
        
        self.img_array = None
        self._load_drainage_image()

    def _load_drainage_image(self):
        """复用父类的排水图加载逻辑"""
        path = os.path.join(self.base_path, "排水能力_简化.png")
        try:
            img = Image.open(path)
            self.img_array = np.array(img)
        except:
            self.img_array = np.random.randint(0, 255, (1000, 1000, 3), dtype=np.uint8)

    def _lat_lon_to_pixel(self, lat, lon):
        """复用父类的坐标转换逻辑"""
        if self.img_array is None: return 0, 0
        h, w = self.img_array.shape[:2]
        # 注意：这里需要与 ExtremeRainfall 的范围保持一致，或者使用本类定义的范围
        # 假设排水图是覆盖 self.SH_LAT_MIN/MAX 范围的
        lat_min, lat_max = 30.6984, 31.5139 # 这里沿用 ExtremeRainfall 的范围设定以匹配图片
        lon_min, lon_max = 120.849, 121.966
        px = int((lon - lon_min) / (lon_max - lon_min) * w)
        py = int((lat_max - lat) / (lat_max - lat_min) * h)
        return min(max(px, 0), w - 1), min(max(py, 0), h - 1)

    def _get_drainage_from_image(self, lat, lon):
        """复用父类的排水能力获取逻辑"""
        px, py = self._lat_lon_to_pixel(lat, lon)
        r_val = self.img_array[py, px][0] if self.img_array.ndim == 3 else self.img_array[py, px]
        r_norm = r_val / 255.0
        if r_val == 0: return 27
        elif r_norm <= 0.9: return 36
        elif r_norm <= 0.99: return 27
        else: return 50
    
    # 距离计算方法
    def _haversine(self, lat1, lon1, lat2, lon2):
        """计算地球表面两点间的大圆距离 (km)"""
        R = 6371.0 
        dlat = math.radians(lat2 - lat1)
        dlon = math.radians(lon2 - lon1)
        a = math.sin(dlat/2)**2 + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(dlon/2)**2
        return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))
    
    def move_step(self):
        """真实数据不需要移动风暴中心，此方法留空适配接口"""
        pass

    def _fetch_and_interpolate(self, current_time_str):
        """从数据库读取特定时间的数据并构建插值函数"""
        # 如果时间没变，直接返回缓存
        if current_time_str == self._last_query_time_str and self._cached_df is not None:
            return
        
        self._last_query_time_str = current_time_str
        conn = None
        try:
            conn = sqlite3.connect(self.db_path)
            # 使用 LIKE 模糊匹配时间 (e.g., '2025-07-29 05:00%')
            query = """
                SELECT s.lon, s.lat, r.rain_value 
                FROM rain_data r
                JOIN stations s ON r.station_id = s.station_id
                WHERE r.datetime LIKE ?
            """
            df = pd.read_sql_query(query, conn, params=(f"{current_time_str}%",))
            
            if len(df) < 3:
                # 数据不足时，假设无雨
                self._cached_df = pd.DataFrame({'lon': [], 'lat': [], 'rain_value': []})
            else:
                self._cached_df = df
        except Exception as e:
            # print(f"DB Error: {e}") 
            self._cached_df = pd.DataFrame({'lon': [], 'lat': [], 'rain_value': []})
        finally:
            if conn: conn.close()

    def get_instantaneous_intensity(self, time_step, position):
        """获取瞬时雨强 (mm/h)"""
        # 计算当前绝对时间
        curr_dt = self.start_dt + timedelta(minutes=time_step)
        # 数据库是每小时数据 (e.g. 05:00, 06:00)，这里简单取最近的小时，或者按分钟插值
        # 为了性能，这里取整点小时 (format: YYYY-MM-DD HH)
        # 如果需要更精细，可以修改为 HH:MM 并确保数据库有该精度，或者在两个小时之间做线性插值
        # 这里采用：取当前小时的数据（阶梯状变化）
        target_time_str = curr_dt.strftime("%Y-%m-%d %H") # 匹配数据库的 '2025-07-29 05' 开头
        
        self._fetch_and_interpolate(target_time_str)
        
        if self._cached_df is None or len(self._cached_df) == 0:
            return 0.0
        
        # 使用 Nearest 或 Linear 插值获取特定位置的雨量
        # 由于每次调用 griddata 开销大，这里针对单点计算：
        # 实际上 griddata 适合生成网格。对于单点查询，计算距离最近的几个站做 IDW 更快。
        # 这里简化使用 scipy.interpolate.griddata 对单点插值
        points = self._cached_df[['lat', 'lon']].values
        values = self._cached_df['rain_value'].values
        
        # 注意：griddata 输入是 (points, values, xi)
        # points: (N, 2), xi: (M, 2)
        try:
            val = griddata(points, values, np.array([position]), method='nearest')[0]
            return max(0, val)
        except:
            return 0.0

    def get_intensity(self, time_step, position, precomputed_dist=None):
        """
        计算地表积水深度 (mm)。
        对于真实数据，我们假设数据库里存的是“小时雨量 (mm)”。
        为了计算积水，我们需要累计雨量。
        注意：这是一个简化处理。真实的积水模拟需要连续的时间积分。
        这里我们假设：积水 = 当前时刻瞬时雨强累积 - 排水。
        """
        lat, lon = position
        
        # 获取瞬时雨强 (假设数据库存储的是 mm/h 或者该小时的累计雨量)
        intensity_mm_h = self.get_instantaneous_intensity(time_step, position)
        
        # 计算当前时间步（1分钟）带来的新增雨量
        rain_inc = intensity_mm_h / 60.0 
        
        # 在这里，DisturbanceStrategy 是无状态的(stateless)关于“积水历史”的吗？
        # ExtremeRainfall 类里是用 cumulative profile 算的。
        # 如果我们没有累计曲线，我们只能近似。
        # 方案：假设过去 N 小时的雨都下下来了，扣除排水。
        # 但这非常慢。
        # 妥协方案：SubwayDIIMSystem 的 G.nodes[n]['water'] 是每一帧刷新的。
        # 如果 ExtremeRainfall 是每次重算 total accumulation，那这里也得这么做。
        # 但查询历史所有小时的数据太慢了。
        
        # *修正*：SubwayDIIMSystem.step() 里是直接赋值：
        # self.G.nodes[n]['water'] = self.strategy.get_intensity(...)
        # 所以 get_intensity 必须返回“当前积水深度”。
        
        # 对于 RealData，我们难以每次从 0时刻开始积分。
        # 建议：仅返回瞬时雨强导致的“潜在积水”，或者基于简单的“当前雨强 * 因子”来近似积水风险。
        # 或者，为了更准确，我们在 SimulationSystem 里维护积水状态，而不是在这里计算 total。
        # 但为了不改动 System 的逻辑（System 假设 Strategy 返回积水深度），
        # 我们这里做一个近似： 积水深度 ~= 最近 1 小时的雨量 * 径流系数 - 排水 * 1小时
        
        S = 15.7
        # 假设 effective rain 是过去 60 分钟的累积（即当前 hourly value）
        cum_rain = intensity_mm_h # 粗略假设
        
        runoff = (cum_rain - 0.05 * S)**2 / (cum_rain + 0.95 * S) if cum_rain > 0.05 * S else 0
        
        # 排水能力 (mm/h)
        drainage_rate = self._get_drainage_from_image(lat, lon)
        
        # 估算积水：径流 - 排水（假设排水一直在工作）
        # 这里仅仅是一个状态快照，无法完美模拟时序累积，除非重构 System
        water_depth = max(0, runoff - drainage_rate) 
        
        return water_depth

    def get_speed_impact_factor(self, water_depth, edge_type, line_context=None):
        """复用父类的速度折减逻辑"""
        # [修改] 提取线路整体最大积水深度
        max_water_line = 0.0
        if line_context:
            max_water_line = line_context.get('max_water_total', 0.0)

        # [修改] 判断是否全线停运
        is_line_failed = (water_depth > 450) or (max_water_line > 450)

        if edge_type == 'boarding': 
            # [修改] 若线路瘫痪，进站系数设为 0.0001
            if is_line_failed:
                return 0.0001
                
            if water_depth <= 0: return 1.0
            if water_depth <= 250: return max(0.01, 0.9 - (water_depth / 250.0) * 0.1)
            elif water_depth <= 450: return max(0.0001, 0.8 - ((water_depth - 250) / 200.0) * 0.7)
            else: return 0.0001 
            
        elif edge_type == 'riding': 
            if line_context:
                is_above = line_context.get('is_above_ground', 0)
                max_rain = line_context.get('max_rain_above', 0.0)
                
                if is_above == 1: 
                    if max_rain > 100: return 0.0001  # [修改] 极强降雨停运
                    elif max_rain >= 80: return 0.5 
                else: 
                    if max_water_line > 450: return 0.0001 # [修改] 地下线受淹停运
            return 1.0


# ==============================================================================
# 2. 核心系统：地铁网络拓扑与 DIIM (动态进程间影响模型) 模拟
# ==============================================================================
class SubwayDIIMSystem:
    """
    地铁动态影响模拟系统 (Dynamic Inoperability Input-Output Model System)
    
    主要功能：
    1. 构建双层网络图（Station Layer + Line Layer）。
    2. 加载静态数据（拓扑、时刻表、客流）。
    3. 执行时间步进仿真，计算 DIIM 状态更新。
    4. 计算网络性能指标（加权全局效率）。
    """
    def __init__(self, base_path, disturbance_strategy, sim_start_time='05:30', real_flow_df=None):
        self.base_path = base_path
        self.strategy = disturbance_strategy
        self.real_flow_df = real_flow_df # [新增] 接收真实客流数据 DataFrame
        self.G = nx.MultiGraph() # 使用多重图构建分层网络，允许两个节点间存在多种关系的边
        
        try:
            t_obj = datetime.strptime(sim_start_time, "%H:%M")
            self.sim_start_min = t_obj.hour * 60 + t_obj.minute
            self.sim_start_dt_obj = t_obj # 保存 datetime 对象用于计算
        except: 
            self.sim_start_min = 330
            self.sim_start_dt_obj = datetime(2000, 1, 1, 5, 30)
        
        # 内部缓存与索引映射
        self.id_map_long_to_short = {}
        self.line_stations_index = {}
        self.schedule_rules = [] 
        self.schedule_lookup = {}
        self.station_pos_cache = {}
        self.fast_edge_data = [] # 扁平化的边数据列表，用于极速遍历（避开NX图查询开销）
        self.node_idx_map = {}
        self.node_pos_list = []
        self.node_dists_list = []
        self.node_st_names = []
        
        # 顺序执行初始化流程
        self._load_id_mapping()           # 1. 映射线路ID (Excel -> Dict)
        self._load_json_topology()        # 2. 加载基础拓扑 (JSON)
        self._load_detailed_schedule()    # 3. 加载列车时刻/发车间隔
        self._build_station_index()       # 4. 构建索引
        self._build_graph_topology()      # 5. 构建分层网络图 (Graph Building)
        self._apply_ground_elevation_status() # 6. 标记地上/地下线
        self._load_passenger_flow_data()  # 7. 加载客流权重 (静态初始值)
        self._build_single_layer_network()# 8. 构建单层简化网 (用于辅助计算)
        self._init_diim_matrices()        # 9. 初始化动态影响矩阵 (DIIM Init)
        
        # 韧性恢复参数（设定恢复速度）
        # c(t) 为受损程度，恢复过程遵循指数衰减
        T_rec = 15.0 
        self.k_recovery = (1.0 / T_rec) * math.log(1.0 / 0.01)
        
        self._init_flow_weights()         # 初始化流权重
        self._precompute_static_simulation_data() # 预处理静态数据加速计算
        
        # 计算初始（无故障）状态下的加权网络效率，作为后续归一化的分母
        self.Ef0 = self.calculate_weighted_efficiency(self.sim_start_min, 'Weekday')
        self.congestion_warning_triggered = False

    def _load_id_mapping(self):
        """从Excel加载线路名称与短ID（如：1号线 -> "1"）的映射关系"""
        path = os.path.join(self.base_path, "线路id与名称对应-20251210.xlsx")
        try:
            if os.path.exists(path): df = pd.read_excel(path)
            else: df = pd.read_csv(path + " - Sheet1.csv")
            for _, row in df.iterrows():
                if pd.isna(row['Line ID']): continue
                long_id = str(int(row['Line ID'])) if isinstance(row['Line ID'], float) else str(row['Line ID'])
                line_name = str(row['Line Name'])
                short_id = None
                if "浦江" in line_name: short_id = "PJ"
                elif "磁浮" in line_name: short_id = "Maglev"
                elif "号线" in line_name:
                    match = re.search(r'(\d+)', line_name)
                    if match: short_id = match.group(1)
                if short_id: self.id_map_long_to_short[long_id] = short_id
        except: pass

    def _load_json_topology(self):
        """读取地铁拓扑结构 JSON 文件"""
        path = os.path.join(self.base_path, "subway.json")
        with open(path, "r", encoding="utf-8") as f: self.raw_json = json.load(f)

    def _load_detailed_schedule(self):
        """加载详细的发车间隔规则（区分早晚高峰、工作日/周末、区间车）"""
        path = os.path.join(self.base_path, "line_schedule_detailed.xlsx")
        try:
            df = pd.read_excel(path)
            df['LineID'] = df['LineID'].apply(lambda x: str(int(x)) if isinstance(x, float) else str(x).strip())
            for _, row in df.iterrows():
                s_dt = pd.to_datetime(str(row['Start_Time']), format='%H:%M:%S', errors='coerce')
                e_dt = pd.to_datetime(str(row['End_Time']), format='%H:%M:%S', errors='coerce')
                rule = {
                    'line_id': row['LineID'], 'day_type': str(row['DayType']).strip(),
                    'start_min': s_dt.hour*60+s_dt.minute if pd.notnull(s_dt) else 0,
                    'end_min': e_dt.hour*60+e_dt.minute if pd.notnull(e_dt) else 1440,
                    'from_st': str(row['From_Station']).strip(), 'to_st': str(row['To_Station']).strip(),
                    'interval': float(row['Interval_Sec'])
                }
                self.schedule_rules.append(rule)
                key = (rule['line_id'], rule['day_type'])
                if key not in self.schedule_lookup: self.schedule_lookup[key] = []
                self.schedule_lookup[key].append(rule)
        except: pass

    def _build_station_index(self):
        """构建车站在线路中的顺序索引，用于后续判断某车站是否在区间车的覆盖范围内"""
        for line_info in self.raw_json['l']:
            short_id = self.id_map_long_to_short.get(str(line_info['ls']))
            if not short_id: continue
            st_names = [st['n'] for st in line_info['st']]
            if short_id not in self.line_stations_index: 
                self.line_stations_index[short_id] = []
                self.station_pos_cache[short_id] = {}
            for st in st_names:
                if st not in self.line_stations_index[short_id]: 
                    self.line_stations_index[short_id].append(st)
                    self.station_pos_cache[short_id][st] = len(self.line_stations_index[short_id]) - 1

    def _is_station_in_section(self, line_id, station_name, from_st, to_st):
        """逻辑判定：特定车站是否位于指定的调度区间（from_st 到 to_st）内"""
        line_cache = self.station_pos_cache.get(line_id)
        if not line_cache: return False
        idx_s, idx_f, idx_t = line_cache.get(station_name), line_cache.get(from_st), line_cache.get(to_st)
        if idx_s is None or idx_f is None or idx_t is None: return False
        return min(idx_f, idx_t) <= idx_s <= max(idx_f, idx_t)

    def _build_graph_topology(self):
        """
        构建三维分层模型 (Space-L space)：
        1. 车站层 (Station Layer): 物理车站位置。
        2. 线路层 (Line Layer): 特定线路的站台。
        3. 边 (Edges):
           - boarding: 车站层 <-> 线路层 (进出站)。
           - riding: 线路层 <-> 线路层 (乘车)。
        """
        for line_info in self.raw_json['l']:
            short_id = self.id_map_long_to_short.get(str(line_info['ls']))
            if not short_id: continue
            sts = line_info['st']
            for st in sts:
                pos = tuple(map(float, st['sl'].split(',')))[::-1] # 转为 (lat, lon)
                node_st = f"{st['n']}_车站层"
                node_ln = f"{st['n']}_线路层_{short_id}"
                
                # 添加节点
                if not self.G.has_node(node_st): self.G.add_node(node_st, layer='station', pos=pos, flow=1000)
                self.G.add_node(node_ln, layer='line', pos=pos, line_id=short_id, station_name=st['n'])
                
                # 添加进站边 (Boarding/Alighting Link)
                self.G.add_edge(node_st, node_ln, type='boarding', weight=0)
            
            # 添加运行区间边 (Riding Link)
            for i in range(len(sts) - 1):
                u, v = f"{sts[i]['n']}_线路层_{short_id}", f"{sts[i+1]['n']}_线路层_{short_id}"
                if self.G.has_node(u) and self.G.has_node(v):
                    self.G.add_edge(u, v, type='riding', weight=0, line_id=short_id)

    def _apply_ground_elevation_status(self):
        """识别地铁的地上区间，这对于评估降雨暴露风险非常关键（地下区间受淹水影响，地上区间受雨强影响）"""
        file_path = os.path.join(self.base_path, "上海地铁地上线归纳.txt")
        above_set = set()
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                for line in f:
                    if '：' not in line: continue
                    parts = line.strip().split('：')
                    # 解析线路ID
                    match = re.search(r'(\d+)', parts[0])
                    lid = match.group(1) if match else ("PJ" if "浦江" in parts[0] else None)
                    if not lid: continue
                    # 解析区间
                    for group in parts[1].split('，'):
                        sts = group.strip().split('-')
                        for i in range(len(sts)-1):
                            pair = tuple(sorted((sts[i].strip(), sts[i+1].strip())))
                            above_set.add((lid, pair[0], pair[1]))
        except: pass
        
        # 遍历图中的边，标记 is_above_ground 属性
        for u, v, k, attr in self.G.edges(keys=True, data=True):
            if attr.get('type') == 'riding':
                lid = attr.get('line_id')
                stu, stv = self.G.nodes[u].get('station_name'), self.G.nodes[v].get('station_name')
                pair = tuple(sorted((stu, stv))) if stu and stv else None
                attr['is_above_ground'] = 1 if pair and (lid, pair[0], pair[1]) in above_set else 0

    def _load_passenger_flow_data(self):
        """从外部文件加载客流强度，用于后续加权网络效率的计算（客流大的站点权重更高）"""
        file_path = os.path.join(self.base_path, "进站客流.xlsx")
        p_flow = {}
        sum_f, num_f = 0, 0
        try:
            xls = pd.ExcelFile(file_path)
            for sheet in xls.sheet_names:
                for _, row in xls.parse(sheet, header=None).iterrows():
                    val = ((row[1] or 0) + (row[2] or 0)) / 2
                    p_flow[f"{row[0]}_车站层"] = p_flow.get(f"{row[0]}_车站层", 0) + val
                    sum_f += val; num_f += 1
        except: pass
        
        # 缺失值填充（使用平均值）
        avg_flow = sum_f / max(num_f, 1) or 1000
        avg_deg = np.mean([d for n, d in self.G.degree() if self.G.nodes[n].get('layer')=='station']) or 1
        
        for n, d in self.G.nodes(data=True):
            if d.get('layer') == 'station':
                d['flow'] = p_flow.get(n, (avg_flow/avg_deg)*self.G.degree(n))

    def _build_single_layer_network(self):
        """构建一个简化的单层站点间网络，用于辅助拓扑特征计算"""
        self.single_layer_net = nx.Graph()
        for u, v, attr in self.G.edges(data=True):
            if attr.get('type') == 'riding':
                n1 = self.G.nodes[u].get('station_name', u.split('_')[0])
                n2 = self.G.nodes[v].get('station_name', v.split('_')[0])
                if n1 != n2: self.single_layer_net.add_edge(n1, n2)

    def _init_flow_weights(self):
        """初始化边权：将节点流分配到相邻的进站边和运行边上"""
        ln_weights = {}
        # 1. 进站边权重
        for u, v, k, attr in self.G.edges(keys=True, data=True):
            if attr['type'] == 'boarding':
                st = u if self.G.nodes[u].get('layer')=='station' else v
                ln = v if st == u else u
                w = self.G.nodes[st].get('flow', 1000) / max(1, self.G.degree[st])
                attr['flow_weight'] = w
                ln_weights[ln] = w
        # 2. 运行边权重（取两端节点平均）
        for u, v, k, attr in self.G.edges(keys=True, data=True):
            if attr['type'] == 'riding':
                attr['flow_weight'] = (ln_weights.get(u, 100)+ln_weights.get(v, 100))/2

    def _precompute_static_simulation_data(self):
        """
        [极速仿真优化] 预计算所有不会随时间改变的数据结构。
        将图数据扁平化为 List/Array，避免在 step() 循环中频繁调用 NetworkX 的 lookup 方法。
        """
        nodes = list(self.G.nodes())
        self.node_idx_map = {n: i for i, n in enumerate(nodes)}
        self.node_dists_list = [0.0] * len(nodes)
        for i, n in enumerate(nodes):
            self.node_pos_list.append(self.G.nodes[n]['pos'])
            self.node_st_names.append(self.G.nodes[n].get('station_name', n.split('_')[0]))
        
        target_day_type = 'Weekday'
        for u, v, k in self.edge_list:
            attr = self.G.edges[u, v, k]
            u_idx, v_idx = self.node_idx_map[u], self.node_idx_map[v]
            lid = attr.get('line_id') or self.G.nodes[u].get('line_id') or self.G.nodes[v].get('line_id')
            est = {
                'u_idx': u_idx, 'v_idx': v_idx, 'type': attr['type'], 'line_id': lid,
                'is_above_ground': attr.get('is_above_ground', 0),
                'flow_weight': attr.get('flow_weight', 1.0),
                'boarding_dist': 0.0, 'boarding_node_flow': 1000.0,
                'riding_pos_mid': None, 'riding_applicable_rules': []
            }
            # 预绑定该边适用的调度规则（避免运行时搜索）
            if lid:
                rules = self.schedule_lookup.get((str(lid), target_day_type), [])
                st_u, st_v = self.node_st_names[u_idx], self.node_st_names[v_idx]
                check_st = st_u if attr['type'] == 'boarding' else None
                for r in rules:
                    if check_st:
                        if self._is_station_in_section(str(lid), check_st, r['from_st'], r['to_st']):
                             est['riding_applicable_rules'].append(r)
                    else:
                        if self._is_station_in_section(str(lid), st_u, r['from_st'], r['to_st']) and \
                           self._is_station_in_section(str(lid), st_v, r['from_st'], r['to_st']):
                             est['riding_applicable_rules'].append(r)
            if attr['type'] == 'boarding':
                st = self.node_st_names[u_idx]
                deg = self.single_layer_net.degree[st] if st in self.single_layer_net else 1
                est['boarding_dist'] = deg * 100.0 # 模拟车站内步行距离
                est['boarding_node_flow'] = self.G.nodes[u if self.G.nodes[u]['layer']=='station' else v].get('flow', 1000)
            else:
                p1, p2 = self.G.nodes[u]['pos'], self.G.nodes[v]['pos']
                est['riding_pos_mid'] = ((p1[0]+p2[0])/2, (p1[1]+p2[1])/2)
            self.fast_edge_data.append(est)

    def calculate_weighted_efficiency(self, current_time_min, day_type, line_stats=None):
        """
        计算效率，并包含终极路径追踪。
        """
        node_waters = [self.G.nodes[n].get('water', 0) for n in self.G.nodes()]
        
        # 1. 计算所有边权重
        for i, (u, v, k) in enumerate(self.edge_list):
            sd = self.fast_edge_data[i]
            
            # Rain
            avg_w = (node_waters[sd['u_idx']] + node_waters[sd['v_idx']]) / 2.0
            t_curr, _, _ = self._calc_times_fast(sd, avg_w, current_time_min, line_stats)
            
            # Base
            t_baseline, _, _ = self._calc_times_fast(sd, 0.0, current_time_min, line_stats=None)
            
            w = sd['flow_weight']
            
            # 这里的权重已经是 "时间 / 流量权重"
            weight_rain = (t_curr / w) if w > 0 and t_curr != float('inf') else float('inf')
            weight_base = (t_baseline / w) if w > 0 and t_baseline != float('inf') else float('inf')
            
            self.G.edges[u,v,k]['tau'] = weight_rain
            self.G.edges[u,v,k]['tau_baseline'] = weight_base
            
        st_nodes = [n for n, d in self.G.nodes(data=True) if d.get('layer') == 'station']
        st_set = set(st_nodes)
        N = len(st_nodes)
        if N < 2: return 0.0
        
        sum_inv_curr = 0.0
        sum_inv_base = 0.0
        
        found_anomaly_path = False
        
        # 2. 计算最短路径
        for src in st_nodes:
            # 获取 Rain 场景下的所有路径长度
            try:
                lens_rain = nx.single_source_dijkstra_path_length(self.G, src, weight='tau')
            except: lens_rain = {}
            
            # 获取 Base 场景下的所有路径长度
            try:
                lens_base = nx.single_source_dijkstra_path_length(self.G, src, weight='tau_baseline')
            except: lens_base = {}
            
            # 累加并检查
            for t in st_set:
                if t == src: continue
                
                l_r = lens_rain.get(t, float('inf'))
                l_b = lens_base.get(t, float('inf'))
                
                if l_r > 0: sum_inv_curr += 1.0/l_r
                if l_b > 0: sum_inv_base += 1.0/l_b
                
                # === 捕获异常路径 ===
                # 如果 Rain 距离 < Base 距离，说明 Rain 路径更短（更优），这是不可能的
                if l_r < l_b - 1e-6 and not found_anomaly_path:
                    print(f"\n🔥🔥🔥 [PATH ANOMALY DETECTED] at Time {current_time_min} 🔥🔥🔥")
                    print(f"Source: {src} -> Target: {t}")
                    print(f"Dist Rain: {l_r:.6f} < Dist Base: {l_b:.6f}")
                    
                    # 追踪 Rain 路径
                    try:
                        path = nx.shortest_path(self.G, src, t, weight='tau')
                        print("--- Rain Path Breakdown ---")
                        total_chk = 0
                        for i in range(len(path)-1):
                            u_p, v_p = path[i], path[i+1]
                            # 获取边数据 (取最小权重的那个key)
                            edge_data = self.G.get_edge_data(u_p, v_p)
                            # 找到 weight='tau' 最小的那个 key
                            best_k = min(edge_data, key=lambda k: edge_data[k]['tau'])
                            attr = edge_data[best_k]
                            
                            w_r = attr['tau']
                            w_b = attr['tau_baseline']
                            print(f"   {u_p} -> {v_p} | RainW={w_r:.4f} vs BaseW={w_b:.4f} | {'ERROR' if w_r < w_b else 'OK'}")
                            total_chk += w_r
                        print(f"   Check Sum: {total_chk:.4f}")
                    except:
                        print("   (Could not reconstruct path)")
                    
                    found_anomaly_path = True
        
        Ef = (1/(N*(N-1))) * sum_inv_curr
        Ef0 = (1/(N*(N-1))) * sum_inv_base
        
        return Ef / Ef0 if Ef0 > 0 else 1.0

    def _init_diim_matrices(self):
        """初始化动态进程间影响模型 (DIIM) 矩阵结构"""
        self.edge_list = list(self.G.edges(keys=True))
        self.edge_idx = {e: i for i, e in enumerate(self.edge_list)}
        n = len(self.edge_list)
        self.q_vector = np.zeros(n) # 存储各边的受损失效率 q
        self.A_matrix = sparse.csr_matrix((n, n))
        self.node_to_boarding_indices = {}
        for n_st in [n for n, d in self.G.nodes(data=True) if d['layer']=='station']:
            idxs = []
            for u, v, k in self.G.edges(n_st, keys=True):
                if self.G.edges[u,v,k]['type'] == 'boarding':
                    k_t = (u,v,k) if (u,v,k) in self.edge_idx else (v,u,k)
                    if k_t in self.edge_idx: idxs.append(self.edge_idx[k_t])
            self.node_to_boarding_indices[n_st] = idxs

    def _calc_times_fast(self, static_data, water_depth, current_time_min, line_stats=None):
        """
        计算特定边在特定积水深度下的瞬时通行时间。
        Returns: (当前受灾时间 t_curr, 正常理想时间 t_norm, 调试信息字典 debug_info)
        """
        lookup_time = current_time_min % 1440
        etype = static_data['type']
        
        # 准备调试信息
        debug_info = {}

        lid = static_data['line_id']
        ctx = None
        if lid and line_stats and str(lid) in line_stats:
            st_data = line_stats[str(lid)]
            ctx = {'is_above_ground': static_data.get('is_above_ground', 0), 
                   'max_rain_above': st_data['max_rain_above'], 
                   'max_water_total': st_data['max_water_total']}

        if etype == 'boarding':
            dist = static_data['boarding_dist']
            
            # 1. 速度计算
            speed_factor = self.strategy.get_speed_impact_factor(water_depth, 'boarding', line_context=ctx)
            v_curr = 1.4 * speed_factor
            
            # 记录调试信息
            debug_info['speed_factor'] = speed_factor
            debug_info['v_curr'] = v_curr
            debug_info['dist'] = dist

            if v_curr < 0.01: return float('inf'), 100.0, debug_info
            
            tw_c = dist / v_curr
            tw_n = dist / 1.4
            
            t_b = 30 # 列车停靠时间

            # 2. 等待时间计算
            t_wait_c, t_wait_n = 0, 0
            if static_data['line_id']:
                t_base = 300.0
                for r in static_data['riding_applicable_rules']:
                    if r['start_min'] <= lookup_time < r['end_min']:
                        t_base = r['interval']; break
                
                hw_n = t_base
                # 计算受影响后的发车间隔
                riding_factor = self.strategy.get_speed_impact_factor(water_depth, 'riding', line_context=ctx)
                hw_c = hw_n / max(riding_factor, 0.0001)
                
                # 排队论参数
                lam = max(static_data['boarding_node_flow'] / 3600.0, 0.0001)
                
                # 记录更多调试信息
                debug_info['hw_n'] = hw_n
                debug_info['hw_c'] = hw_c
                debug_info['riding_factor'] = riding_factor
                debug_info['lambda'] = lam

                # [修正开始] 定义单调性约束的排队等待时间计算函数
                def calc_w(EV, L):
                    # 1. 计算饱和度 (Saturation Ratio)
                    # 原分母公式 D = 2 * (1/L - 2*EV)
                    # 饱和状态意味着 1/L = 2*EV => L * 2 * EV = 1
                    # 因此定义 rho = L * 2 * EV
                    rho = L * 2 * EV
                    
                    # 设定安全阈值，超过此阈值认为进入拥堵/过饱和区
                    # 在此区域不再使用原公式（原公式有奇点且非单调），改用幂函数惩罚
                    RHO_THRESH = 0.9

                    # 计算阈值处的理论等待时间 (作为锚点)
                    # 当 rho = 0.9 时:
                    # denom_limit = 2 * (1/L - 2*EV) 
                    #             = 2/L * (1 - 2*EV*L) 
                    #             = 2/L * (1 - 0.9) = 0.2 / L
                    # 这是一个近似计算，为了性能，我们直接用硬逻辑判断
                    
                    if rho < RHO_THRESH:
                        # === 正常区域：使用原排队论公式 ===
                        denom = 2 * ((1.0/L) - 2*EV)
                        # 虽然理论上 rho < 0.9 保证 denom > 0，但加个由衷的兜底
                        if denom <= 1e-6: 
                            val = 1800.0 # 理论上不应走到这里
                        else:
                            val = 2.5*EV + (t_b**2 + 3*EV**2)/denom
                        
                        # 即使在正常区域，也允许适当的上限，但不要强制截断导致非单调
                        # 如果计算值确实很大（极罕见），让它保持很大，不要 clamp 到 1800
                        return min(val, 3600.0) 
                    
                    else:
                        # === 拥堵/过饱和区域：使用单调惩罚函数 ===
                        # 这里的逻辑是：既然已经堵了，那么发车间隔(EV)越大，rho越大，等待时间必须越长。
                        # 我们构造一个平滑过渡函数。
                        # 先算出在 RHO_THRESH (0.9) 时的边界值 T_limit
                        denom_limit = 2 * ((1.0/L) * (1 - RHO_THRESH))
                        t_limit = 2.5*EV + (t_b**2 + 3*EV**2)/denom_limit
                        
                        # 惩罚因子：(当前饱和度 / 阈值饱和度) 的平方
                        # 平方项确保了"越堵越慢"的非线性增长，有效拉开 Rain 和 Base 的差距
                        penalty_factor = (rho / RHO_THRESH) ** 2
                        
                        # 最终时间 = 边界时间 * 惩罚因子
                        # 这样即使 Base 很堵 (rho=0.95), Rain 更堵 (rho=1.1), 
                        # Rain 的结果也一定比 Base 大。
                        final_wait = t_limit * penalty_factor
                        
                        # 即使是非常极端的情况，也建议保留一个物理极限（如 2小时），
                        # 但这个极限必须足够大，以免再次出现 Base > Rain 的截断悖论
                        return min(final_wait, 7200.0) 

                # [修正结束]
                
                t_wait_n = calc_w(hw_n+t_b, lam)
                t_wait_c = calc_w(hw_c+t_b, lam)
                
                debug_info['t_wait_n'] = t_wait_n
                debug_info['t_wait_c'] = t_wait_c

            return tw_c + t_wait_c, tw_n + t_wait_n, debug_info
            
        elif etype == 'riding':
            t_base = 180.0
            for r in static_data['riding_applicable_rules']:
                if r['start_min'] <= lookup_time < r['end_min']:
                    t_base = r['interval']; break
            
            fac = self.strategy.get_speed_impact_factor(water_depth, 'riding', line_context=ctx)
            
            debug_info['t_base_interval'] = t_base
            debug_info['riding_factor'] = fac
            
            if fac <= 0.0001: return float('inf'), t_base, debug_info
            return t_base / fac, t_base, debug_info
            
        return 0, 0, debug_info

    def step(self, t_minute, day_type='Weekday'):
        """
        推进模拟时间步，并包含全网权重暴力一致性检查。
        """
        curr_clk = self.sim_start_min + t_minute
        
        # 1. 动态更新客流 (保持原逻辑)
        if self.real_flow_df is not None:
            curr_abs_time = None
            if hasattr(self.strategy, 'start_dt'):
                curr_abs_time = self.strategy.start_dt + timedelta(minutes=t_minute)
            
            if curr_abs_time:
                lookup_h = curr_abs_time.replace(minute=0, second=0, microsecond=0)
                try:
                    if curr_abs_time.minute == 0 or t_minute == 0:
                        batch = self.real_flow_df[self.real_flow_df['time'] == lookup_h]
                        if not batch.empty:
                            for _, row in batch.iterrows():
                                st_name = row['station']
                                flow_val = (row['inbound'] + row['outbound']) / 2
                                node_st = f"{st_name}_车站层"
                                if self.G.has_node(node_st):
                                    self.G.nodes[node_st]['flow'] = flow_val
                                    
                            for est in self.fast_edge_data:
                                if est['type'] == 'boarding':
                                    st_node_idx = est['u_idx'] if self.G.nodes[list(self.G.nodes())[est['u_idx']]].get('layer') == 'station' else est['v_idx']
                                    st_node_name = list(self.G.nodes())[st_node_idx]
                                    est['boarding_node_flow'] = self.G.nodes[st_node_name].get('flow', 1000)
                except Exception as e:
                    pass

        self.strategy.move_step()
        clat, clon = self.strategy.center_loc
        
        # 2. 更新积水
        for i, (lat, lon) in enumerate(self.node_pos_list):
            self.node_dists_list[i] = self.strategy._haversine(clat, clon, lat, lon)
        for i, n in enumerate(self.G.nodes()):
            w = self.strategy.get_intensity(t_minute, self.G.nodes[n]['pos'], precomputed_dist=self.node_dists_list[i])
            self.G.nodes[n]['water'] = w
            
        # 3. 统计线路风险
        line_stats = {}
        node_waters = [self.G.nodes[n].get('water', 0) for n in self.G.nodes()]
        for sd in self.fast_edge_data:
            if sd['type'] == 'riding' and sd['line_id']:
                lid = str(sd['line_id'])
                if lid not in line_stats: line_stats[lid] = {'max_rain_above': 0.0, 'max_water_total': 0.0}
                aw = (node_waters[sd['u_idx']] + node_waters[sd['v_idx']]) / 2
                if aw > line_stats[lid]['max_water_total']: line_stats[lid]['max_water_total'] = aw
                if sd['is_above_ground']:
                    r = self.strategy.get_instantaneous_intensity(t_minute, sd['riding_pos_mid'])
                    if r > line_stats[lid]['max_rain_above']: line_stats[lid]['max_rain_above'] = r
                    
        # 4. DIIM 更新与全网扫描
        curr_c = np.zeros(len(self.edge_list))
        
        # [暴力扫描] 统计计数器
        scan_count_rain_faster = 0
        scan_total_edges = 0
        scan_debug_msg = ""

        for i, sd in enumerate(self.fast_edge_data):
            aw = (node_waters[sd['u_idx']] + node_waters[sd['v_idx']]) / 2
            
            # 计算 Rain 时间
            tc, tn, _ = self._calc_times_fast(sd, aw, curr_clk, line_stats)
            
            # 计算 Base 时间 (强制无雨)
            tb, _, _ = self._calc_times_fast(sd, 0.0, curr_clk, line_stats=None)
            
            # --- 核心检查点 ---
            # 只要 Rain 比 Base 快超过 0.0001秒，就记录下来
            if tc < tb - 0.0001:
                scan_count_rain_faster += 1
                if scan_count_rain_faster <= 3: # 只记录前3个，防止刷屏
                    scan_debug_msg += f"\n   [BAD EDGE] {self.node_st_names[sd['u_idx']]}->{self.node_st_names[sd['v_idx']]} ({sd['type']}) | Rain={tc:.4f} < Base={tb:.4f}"

            # DIIM 逻辑
            val = 0.0
            if tc == float('inf'): val = 1.0 
            elif tc > 0: val = 1.0 - (tn / tc)
            curr_c[i] = max(0.0, min(1.0, val)) 
            scan_total_edges += 1
        
        # [打印扫描结果]
        if scan_count_rain_faster > 0:
            print(f"\n🚨 [CRITICAL SCAN] Time {t_minute}: Found {scan_count_rain_faster}/{scan_total_edges} edges where Rain is FASTER!")
            print(scan_debug_msg)
        else:
            # 如果没有发现异常边，打印一条确认信息
            # print(f"✅ [SCAN OK] Time {t_minute}: All edges Rain >= Base.")
            pass

        q_next = (1 - self.k_recovery) * self.q_vector + self.k_recovery * curr_c
        self.q_vector = np.clip(q_next, 0, 1)
        
        for i, (u,v,k) in enumerate(self.edge_list): self.G.edges[u,v,k]['q_value'] = self.q_vector[i]
        aff_lines = set()
        for i, q in enumerate(self.q_vector):
            if q > 0.1:
                lid = self.fast_edge_data[i]['line_id']
                if lid: aff_lines.add(lid)
        node_risks = []
        for n, idxs in self.node_to_boarding_indices.items():
            if idxs: node_risks.append(max(self.q_vector[i] for i in idxs))
            else: node_risks.append(0)
        
        self.current_line_stats = line_stats
        avg_water = np.mean(list(node_waters))
        max_n_risk = max(node_risks) if node_risks else 0
        
        avg_q_safe = np.nanmean(self.q_vector) if len(self.q_vector) > 0 else 0.0
        avg_q_safe = np.nan_to_num(avg_q_safe)
        
        return avg_q_safe, avg_water, len(aff_lines), max_n_risk

# ==============================================================================
# 3. 蒙特卡洛数据记录器：收集并保存多次随机实验的数据
# ==============================================================================
class MonteCarloRecorder:
    def __init__(self, time_steps):
        self.times = time_steps
        self.df_eff = pd.DataFrame({'Time': time_steps})
        self.df_risk = pd.DataFrame({'Time': time_steps})
        self.df_water = pd.DataFrame({'Time': time_steps})
        self.df_intensity = pd.DataFrame({'Time': time_steps})
        self.df_lat = pd.DataFrame({'Time': time_steps})
        self.df_lon = pd.DataFrame({'Time': time_steps})
        self.df_aff_lines = pd.DataFrame({'Time': time_steps})
        self.df_max_n_risk = pd.DataFrame({'Time': time_steps})

    def add_run_data(self, run_id, eff, risk, water, inten, lats, lons, aff_ln, max_nr):
        col_name = f"Run_{run_id}"
        self.df_eff[col_name] = eff
        self.df_risk[col_name] = risk
        self.df_water[col_name] = water
        self.df_intensity[col_name] = inten
        self.df_lat[col_name] = lats
        self.df_lon[col_name] = lons
        self.df_aff_lines[col_name] = aff_ln
        self.df_max_n_risk[col_name] = max_nr

    def save_to_excel(self, base_path):
        fname = os.path.join(base_path, f"MonteCarlo_Results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.xlsx")
        print(f"💾 正在导出蒙特卡洛数据到: {fname}")
        try:
            with pd.ExcelWriter(fname, engine='openpyxl') as writer:
                self.df_eff.to_excel(writer, sheet_name='Efficiency', index=False)
                self.df_risk.to_excel(writer, sheet_name='Risk_Avg_Q', index=False)
                self.df_water.to_excel(writer, sheet_name='Water_Level_Avg', index=False)
                self.df_intensity.to_excel(writer, sheet_name='Intensity_Center', index=False)
                self.df_lat.to_excel(writer, sheet_name='Location_Lat', index=False)
                self.df_lon.to_excel(writer, sheet_name='Location_Lon', index=False)
                self.df_aff_lines.to_excel(writer, sheet_name='Affected_Line_Count', index=False)
                self.df_max_n_risk.to_excel(writer, sheet_name='Max_Node_Risk', index=False)
            print("✅ 导出完成。")
        except Exception as e:
            print(f"❌ 导出失败: {e}")

# ==========================================
# 5. [Worker] 独立仿真任务：供多进程并行调用
# 注意：该函数必须定义在顶层，以便 Joblib 进行序列化 (Pickle)
# ==========================================
def simulation_worker(run_id, base_path, intensity, period, rain_dur, sim_dur, start_str, 
                      use_real_data=False, real_db_name=None, real_flow_data=None):
    try:
        # [修改] 支持真实数据模式初始化
        if use_real_data:
            strategy = RealDataRainfall(base_path, db_name=real_db_name, start_datetime_str=start_str)
            # 在真实模式下，我们传入客流 DataFrame
            sys_obj = SubwayDIIMSystem(base_path, strategy, sim_start_time=start_str, real_flow_df=real_flow_data)
        else:
            strategy = ExtremeRainfall(base_path, max_intensity=intensity, 
                                       rainfall_type='chicago', P=period, 
                                       rain_duration=rain_dur)
            
            # [修改] 为 ExtremeRainfall 模式注入客流数据和基准时间
            # 我们将模拟日期固定在无雨的 2025-08-08，以便使用这天的真实客流
            base_date_str = "2025-08-08"
            full_start_str = f"{base_date_str} {start_str}"
            try:
                # 手动给 strategy 绑定 start_dt，以便 SubwayDIIMSystem 能够计算绝对时间并查询客流
                strategy.start_dt = datetime.strptime(full_start_str, "%Y-%m-%d %H:%M")
            except ValueError:
                pass
                
            # 将外部传入的 real_flow_data (08.08的客流) 传给系统
            sys_obj = SubwayDIIMSystem(base_path, strategy, sim_start_time=start_str, real_flow_df=real_flow_data)
            
        t_obj = datetime.strptime(start_str, "%Y-%m-%d %H:%M") if len(start_str) > 5 else datetime.strptime(start_str, "%H:%M")
        start_min = t_obj.hour * 60 + t_obj.minute
        time_steps = list(range(sim_dur))
        r_eff, r_risk, r_water, r_inten, r_lat, r_lon, r_aff, r_mnr = [], [], [], [], [], [], [], []
        
        for t in time_steps:
            avg_q, avg_w, n_lines, max_nr = sys_obj.step(t)
            curr_clk = start_min + t
            # 计算该时间步的网络加权效率
            ef = sys_obj.calculate_weighted_efficiency(curr_clk, 'Weekday', line_stats=getattr(sys_obj, 'current_line_stats', None))
            
            # 对于真实数据，位置固定，返回 0 或默认中心
            c_lat, c_lon = (0, 0)
            if hasattr(sys_obj.strategy, 'center_loc'):
                c_lat, c_lon = sys_obj.strategy.center_loc
                
            # 瞬时雨强，如果是真实数据，传一个中心点坐标去获取，或者取平均？
            # 这里取 (31.23, 121.47) 上海中心点作为参考
            ref_pos = (31.2304, 121.4737)
            inten = sys_obj.strategy.get_instantaneous_intensity(t, ref_pos)
            
            r_eff.append(ef)
            r_risk.append(avg_q)
            r_water.append(avg_w)
            r_inten.append(inten)
            r_lat.append(c_lat)
            r_lon.append(c_lon)
            r_aff.append(n_lines)
            r_mnr.append(max_nr)
        
        # 强制垃圾回收，防止内存泄漏
        del sys_obj
        del strategy
        gc.collect() 
        return (run_id, r_eff, r_risk, r_water, r_inten, r_lat, r_lon, r_aff, r_mnr)
    except Exception as e:
        return (run_id, None, str(e), None, None, None, None, None, None)

# [新增] 数据一致性校验函数
def verify_data_consistency(base_path, flow_df):
    """
    检查 Excel 客流数据、JSON 拓扑数据、DB 降雨数据中的车站名称一致性。
    """
    print("\n🔍 正在进行数据一致性校验...")
    
    # 1. 获取 Excel 中的车站列表
    excel_stations = set(flow_df['station'].unique())
    
    # 2. 获取 JSON 中的车站列表
    json_path = os.path.join(base_path, "subway.json")
    json_stations = set()
    try:
        with open(json_path, "r", encoding="utf-8") as f: 
            data = json.load(f)
            for l in data['l']:
                for st in l['st']:
                    json_stations.add(st['n'])
    except:
        print("⚠️ 无法读取 subway.json，跳过拓扑校验。")

    # 3. 打印比对结果 (Excel vs JSON)
    if json_stations:
        missing_in_json = excel_stations - json_stations
        missing_in_excel = json_stations - excel_stations
        
        print(f"  - Excel 中有 {len(excel_stations)} 个车站")
        print(f"  - JSON 中有 {len(json_stations)} 个车站")
        
        if missing_in_json:
            print(f"⚠️ 警告: Excel 中有 {len(missing_in_json)} 个车站在 JSON 拓扑中找不到 (可能无法更新其客流):")
            print(f"    示例: {list(missing_in_json)[:5]} ...")
        else:
            print("✅ Excel 所有车站均已在 JSON 拓扑中找到。")
            
        if missing_in_excel:
            # 这通常不是问题，因为 Excel 可能只包含部分车站数据
            pass

    # 4. 获取 DB 中的车站列表 (简单检查)
    db_path = os.path.join(base_path, "shanghai_rain_history.db")
    if os.path.exists(db_path):
        try:
            conn = sqlite3.connect(db_path)
            cursor = conn.cursor()
            cursor.execute("SELECT DISTINCT station_name FROM stations") # 假设表名 stations，列名 station_name
            db_stations = set([r[0] for r in cursor.fetchall()])
            conn.close()
            
            print(f"  - DB 中有 {len(db_stations)} 个气象监测站")
            # 气象站和地铁站通常不同名，这里仅做数量确认，不做集合比对
        except:
            print("⚠️ 无法读取数据库中的站点信息，跳过校验。")
    else:
        print("⚠️ 数据库文件不存在。")
    print("--------------------------------------------------\n")


# ==========================================
# 6. 主程序入口 (Joblib 并行执行版)
# ==========================================
def main():
    # === 参数配置区域 ===
    BASE_PATH = "E:\\BaiHao\\20260103算例补充\\输入与输出数据"
    
    # --- 模式开关 ---
    USE_REAL_DATA = False   # [开关] 是否使用 2025.07.29-08.01 的真实数据
    # ----------------
    
    if USE_REAL_DATA:
        # 真实数据模式配置
        MC_ITERATIONS = 1         # 真实数据是确定性的，通常跑 1 次即可 (或者跑多次以测试算法稳定性)
        START_TIME_STR = "2025-07-29 05:00" # 仿真开始时间
        END_TIME_STR   = "2025-08-01 23:00" # 仿真结束时间
        #START_TIME_STR = "2025-07-29 07:00" # 仿真开始时间
        #END_TIME_STR   = "2025-07-29 08:00" # 仿真结束时间
        
        # 计算总分钟数
        t_start = datetime.strptime(START_TIME_STR, "%Y-%m-%d %H:%M")
        t_end   = datetime.strptime(END_TIME_STR, "%Y-%m-%d %H:%M")
        SIM_DURATION = int((t_end - t_start).total_seconds() / 60)
        
        DB_NAME = "shanghai_rain_history.db"
        FLOW_FILE = "shanghai_metro_flow_final_complete.xlsx" # 支持 csv 或 xlsx
        
        print(f"\n🚀 启动真实数据仿真模式 ({START_TIME_STR} -> {END_TIME_STR}, Duration={SIM_DURATION}min)")
        
        # 加载客流数据 (Pre-load flow data)
        flow_path = os.path.join(BASE_PATH, FLOW_FILE)
        print(f"📂 加载客流文件: {flow_path}")
        try:
            if flow_path.endswith('.csv'):
                flow_df = pd.read_csv(flow_path)
            else:
                flow_df = pd.read_excel(flow_path)

            # 1. 基础清洗：转字符串、去空格、统一符号
            flow_df['station'] = flow_df['station'].astype(str).str.strip().str.replace('●', '·').str.replace(' ', '')

            # 2. 特殊处理：将“浦电路”拆分为4号线和6号线，客流平分
            # 筛选出原始名称为 "浦电路" 的所有行
            pudian_mask = (flow_df['station'] == '浦电路')
            
            if pudian_mask.any():
                # 提取这些行
                pudian_rows = flow_df[pudian_mask].copy()
                
                # 构造4号线数据 (取名为 JSON 中对应的 '浦电路(4号线)', 客流除以2)
                pudian_4 = pudian_rows.copy()
                pudian_4['station'] = '浦电路(4号线)'
                pudian_4['inbound'] = pudian_4['inbound'] / 2.0
                pudian_4['outbound'] = pudian_4['outbound'] / 2.0
                
                # 构造6号线数据 (取名为 JSON 中对应的 '浦电路(6号线)', 客流除以2)
                pudian_6 = pudian_rows.copy()
                pudian_6['station'] = '浦电路(6号线)'
                pudian_6['inbound'] = pudian_6['inbound'] / 2.0
                pudian_6['outbound'] = pudian_6['outbound'] / 2.0
                
                # 从原表中删除“浦电路”，并追加拆分后的两部分数据
                flow_df = flow_df[~pudian_mask] # 删除原行
                flow_df = pd.concat([flow_df, pudian_4, pudian_6], ignore_index=True)
                print(f"ℹ️ 已将 {len(pudian_rows)} 条'浦电路'数据拆分为4号线与6号线 (客流平分)。")

            # 【新增修复】 简单的站点名称清洗，解决匹配警告
            def clean_station_name(name):
                name = str(name).strip()
                # 1. 替换特殊符号 '●' 为标准符号 '·' 
                name = name.replace('●', '·').replace(' ', '')
                
                # 2. 修正已知的 Excel 录入笔误及旧站名
                fixes = {
                    # --- 笔误修正 ---
                    '淞淞发路': '淞发路',
                    '淞淞虹路': '淞虹路',
                    '淞淞滨路': '淞滨路',  # [新增] 修正笔误
                    '临洮洮路': '临洮路',  # [新增] 修正笔误
                    '颛颛桥': '颛桥',
                    '佘佘山': '佘山',
                    # --- 站名变更/映射 ---               
                    # --- 暂未开通/无拓扑车站 ---
                    '龙居路': '龙居路',    # 14号线暂缓开通，拓扑中无此站，保留原名即可
                    '严御路': '康恒路',    # [新增] 严御路已更名为康恒路(11号线) 。但是拓扑里面没它。
                }
                return fixes.get(name, name)

            flow_df['station'] = flow_df['station'].apply(clean_station_name)

            # 预处理：转换时间列，方便后续查询
            flow_df['time'] = pd.to_datetime(flow_df['time'])
            
            # 执行一致性校验
            verify_data_consistency(BASE_PATH, flow_df)
            
        except Exception as e:
            print(f"❌ 读取客流文件失败: {e}")
            return

    else:
        # 原始蒙特卡洛模式配置
        MC_ITERATIONS = 100       
        RAIN_P_PERIOD = 10       
        RAIN_DURATION = 180       
        SIM_DURATION = 240        
        START_TIME_STR = "09:00"     
        print(f"\n🚀 启动随机蒙特卡洛仿真模式 (Runs={MC_ITERATIONS})")
        DB_NAME = None
        
        # [修改] 读取 08.08 的基准客流数据 (全量 CSV)
        flow_file_full = "shanghai_metro_flow_with_code_final_20250802-20250831_v4.csv"
        flow_path = os.path.join(BASE_PATH, flow_file_full)
        print(f"📂 加载基准客流文件 (2025.08.08): {flow_path}")
        
        try:
            # 读取 CSV
            full_df = pd.read_csv(flow_path)
            
            # 过滤 2025-08-08 的数据
            # [修改] 原代码的 format='%Y-%m-%d %H' 无法解析 ":00:00" 部分，改为自动推断
            full_df['time'] = pd.to_datetime(full_df['time']) 
            
            target_date = pd.Timestamp("2025-08-08").date()
            flow_df = full_df[full_df['time'].dt.date == target_date].copy()
            
            # 重命名列以匹配 SubwayDIIMSystem
            # csv: inbound, outbound, station_name_mapped
            # system expects: inbound, outbound, station
            flow_df.rename(columns={'station_name_mapped': 'station'}, inplace=True)
            
            # 站点名称清洗 (确保与 if 块中的逻辑一致)
            def clean_station_name(name):
                name = str(name).strip()
                name = name.replace('●', '·').replace(' ', '')
                fixes = {
                    '淞淞发路': '淞发路',
                    '淞淞虹路': '淞虹路',
                    '淞淞滨路': '淞滨路',
                    '临洮洮路': '临洮路', 
                    '颛颛桥': '颛桥',
                    '佘佘山': '佘山',
                    '严御路': '康恒路',
                }
                return fixes.get(name, name)
            flow_df['station'] = flow_df['station'].apply(clean_station_name)
            
            # 处理浦电路拆分 (简单起见，这里复用逻辑)
            pudian_mask = (flow_df['station'] == '浦电路')
            if pudian_mask.any():
                pudian_rows = flow_df[pudian_mask].copy()
                pudian_4 = pudian_rows.copy(); pudian_4['station'] = '浦电路(4号线)'; pudian_4[['inbound','outbound']] /= 2
                pudian_6 = pudian_rows.copy(); pudian_6['station'] = '浦电路(6号线)'; pudian_6[['inbound','outbound']] /= 2
                flow_df = flow_df[~pudian_mask]
                flow_df = pd.concat([flow_df, pudian_4, pudian_6], ignore_index=True)
            
            print(f"✅ 已加载 2025.08.08 客流数据，共 {len(flow_df)} 条记录")
            
        except Exception as e:
            print(f"❌ 读取客流文件失败: {e}")
            flow_df = None

    # N_JOBS 控制并行核心数
    # N_JOBS = 10 if USE_REAL_DATA else 30 # 真实数据读取DB开销大，建议减少并发数或设为1
    N_JOBS = 40

    recorder = MonteCarloRecorder(list(range(SIM_DURATION)))
    
    try:
        # 使用 joblib.Parallel 并行派发 simulation_worker 任务
        results = Parallel(n_jobs=N_JOBS, backend='loky', verbose=10)(
            delayed(simulation_worker)(
                run+1, BASE_PATH, 250, 
                RAIN_P_PERIOD, # period (unused in real mode)
                RAIN_DURATION, # rain_dur (unused in real mode)
                SIM_DURATION, START_TIME_STR,
                use_real_data=USE_REAL_DATA,
                real_db_name=DB_NAME,
                real_flow_data=flow_df # [修改] 将读取到的 flow_df 传递进去
            ) for run in range(MC_ITERATIONS)
        )
        
        # 遍历并行结果并存入记录器
        for res in results:
            if res[1] is None:
                print(f"❌ [Error] Run {res[0]} failed: {res[2]}")
            else:
                recorder.add_run_data(*res)
                
    except Exception as e:
        print(f"❌ [Critical] Parallel execution failed: {e}")

    print("\n🏁 仿真结束，正在保存数据...")
    recorder.save_to_excel(BASE_PATH)

if __name__ == "__main__":
    main()

In [ ]:
import networkx as nx
import json
import math
import pandas as pd
import numpy as np
import random
import os
import re
import gc
import time
import sys
import sqlite3
import warnings
from abc import ABC, abstractmethod
from scipy import sparse
from scipy.spatial.distance import cdist
from scipy.interpolate import griddata
from PIL import Image
from datetime import datetime, timedelta

# [修改 1] 引入 joblib 用于 Jupyter 环境下的并行计算
from joblib import Parallel, delayed 

# 禁用插值产生的可能的警告
warnings.filterwarnings("ignore")

# ==============================================================================
# [全局配置] 并行计算环境设置
# ==============================================================================
# 限制 Numpy/OpenBLAS 的底层线程数。
# 原因：在 Joblib 多进程模式下，如果每个子进程再自动开启多线程进行矩阵运算，
# 会导致 CPU 资源争抢（Oversubscription）甚至死锁，显著降低仿真效率。
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

# ==============================================================================
# 1. 策略模式：定义扰动（降雨）行为
# ==============================================================================
class DisturbanceStrategy(ABC):
    """
    抽象基类：定义系统受到的外部干扰策略。
    
    Methods:
        get_intensity: 获取特定时空的干扰强度。
        get_speed_impact_factor: 获取干扰对系统运行速度的影响系数。
    """
    @abstractmethod
    def get_intensity(self, time_step, position): pass
    @abstractmethod
    def get_speed_impact_factor(self, intensity, edge_type, **kwargs): pass

class ExtremeRainfall(DisturbanceStrategy):
    """
    极端降雨策略模型：模拟一个移动的暴雨中心及其对地铁站点的积水影响。
    
    包含物理过程：
    1. 芝加哥雨型（Chicago Design Storm）生成时间序列降雨。
    2. 移动风暴中心（Moving Storm Center）。
    3. 空间衰减（ARF）计算。
    4. SCS-CN 径流模型计算地表积水。
    """
    def __init__(self, base_path, max_intensity=201.9, rainfall_type='chicago', P=100, r=0.405, rain_duration=180):
        """
        初始化降雨模型参数。
        
        Args:
            max_intensity (float): 降雨峰值强度参数 (mm/h 或相关系数)。
            P (int): 降雨重现期 (Return Period, e.g., 100年一遇)。
            r (float): 雨峰位置系数 (Peak Position parameter)，0-1之间，0.4代表峰值出现在前40%。
            rain_duration (int): 降雨持续总时长 (分钟)。
        """
        # 设定模拟的空间范围（上海经纬度大致范围）
        self.lat_range = (30.6984, 31.5139)
        self.lon_range = (120.849, 121.966)
        
        # 初始化暴雨中心起始位置和移动速度
        self.center_loc, self.velocity = self._init_random_boundary_start()
        
        self.Q0 = max_intensity  # 降雨强度基准
        self.base_path = base_path
        self.rainfall_type = rainfall_type
        self.P_period = P        # 重现期
        self.r_peak = r          # 雨峰系数
        self.rain_duration = rain_duration 
        
        self.chicago_profile = None 
        self.img_array = None
        self._load_drainage_image() # 加载区域排水能力图
        self._distance_cache = {}   # 缓存站点到雨中心的距离，减少重复三角函数计算
        
        # 如果是芝加哥雨型，预计算随时间变化的强度曲线
        if self.rainfall_type == 'chicago': self._precompute_chicago_profile()

    def _init_random_boundary_start(self):
        """
        随机初始化降雨中心。
        逻辑：选择地图四个边界之一作为起点，目标点设为地图中心附近，以此计算移动矢量。
        """
        lat_min, lat_max = self.lat_range
        lon_min, lon_max = self.lon_range
        side = random.choice([0, 1, 2, 3]) # 0:上(北), 1:下(南), 2:左(西), 3:右(东)
        
        # 设定目标点为区域中心附近略带随机偏移
        target_lat = (lat_min + lat_max) / 2 + random.uniform(-0.1, 0.1)
        target_lon = (lon_min + lon_max) / 2 + random.uniform(-0.1, 0.1)
        
        if side == 0:   start_lat, start_lon = lat_max, random.uniform(lon_min, lon_max)
        elif side == 1: start_lat, start_lon = lat_min, random.uniform(lon_min, lon_max)
        elif side == 2: start_lat, start_lon = random.uniform(lat_min, lat_max), lon_min
        else:           start_lat, start_lon = random.uniform(lat_min, lat_max), lon_max
        
        speed_kmh = random.uniform(25, 40) # 降雨系统移动速度 (km/h)
        speed_km_min = speed_kmh / 60.0
        
        # 计算移动单位向量（转换为经纬度增量）
        delta_lat_km = (target_lat - start_lat) * 111.0
        delta_lon_km = (target_lon - start_lon) * 96.0
        dist_km = math.sqrt(delta_lat_km**2 + delta_lon_km**2) or 1
        
        v_lat_deg = ((delta_lat_km / dist_km) * speed_km_min) / 111.0
        v_lon_deg = ((delta_lon_km / dist_km) * speed_km_min) / 96.0
        return (start_lat, start_lon), (v_lat_deg, v_lon_deg)

    def move_step(self):
        """每一分钟移动一次降雨中心，并清空距离缓存（因为中心变了，所有距离需重算）"""
        curr_lat, curr_lon = self.center_loc
        v_lat, v_lon = self.velocity
        self.center_loc = (curr_lat + v_lat, curr_lon + v_lon)
        self._distance_cache = {} 

    def _precompute_chicago_profile(self):
        """
        预计算芝加哥雨型（Chicago Design Storm）的时间分布。
        公式来源：Keifer & Chu (1957)，参数 A, n, b 针对上海地区暴雨强度公式适配。
        """
        times = np.arange(0, self.rain_duration + 1, 1) 
        intensities = []
        t_peak_time = self.rain_duration * self.r_peak
        
        # A, n, b 为上海地区的降雨公式参数（需根据研究区域调整）
        A = 9.581 * (1 + 0.846 * np.log10(self.P_period))
        n = 0.656
        b = 7
        
        for t in times:
            # 峰前时刻公式
            if t <= t_peak_time:
                t_b = t_peak_time - t
                num = A * ((1 - n) * (t_b / self.r_peak) + b)
                den = math.pow((t_b / self.r_peak) + b, 1 + n)
            # 峰后时刻公式
            else:
                t_a = t - t_peak_time
                num = A * ((1 - n) * (t_a / (1 - self.r_peak)) + b)
                den = math.pow((t_a / (1 - self.r_peak)) + b, 1 + n)
            intensities.append(num / den)
        
        # 模拟雨后由于排水不畅导致的“长尾效应”（指数衰减）
        tail_duration = 120  
        if intensities:
            last_val = intensities[-1]
            for t_ex in range(1, tail_duration + 1):
                val = last_val * math.exp(-0.1 * t_ex)
                intensities.append(val if val > 0.001 else 0.0)
        
        # 存储为累计降雨量 (Cumulative Rainfall)，方便后续计算径流
        self.chicago_profile = np.cumsum(intensities)

    def _load_drainage_image(self):
        """加载排水能力栅格图，用于根据地理位置查询排水效率"""
        path = os.path.join(self.base_path, "排水能力_简化.png")
        try:
            img = Image.open(path)
            self.img_array = np.array(img)
        except FileNotFoundError:
            # 如果没图，生成一个随机的（调试用）
            self.img_array = np.random.randint(0, 255, (1000, 1000, 3), dtype=np.uint8)

    def _haversine(self, lat1, lon1, lat2, lon2):
        """计算地球表面两点间的大圆距离 (km)"""
        R = 6371.0 
        dlat = math.radians(lat2 - lat1)
        dlon = math.radians(lon2 - lon1)
        a = math.sin(dlat/2)**2 + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(dlon/2)**2
        return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))

    def _lat_lon_to_pixel(self, lat, lon):
        """经纬度坐标与图像像素坐标的转换"""
        if self.img_array is None: return 0, 0
        h, w = self.img_array.shape[:2]
        lat_min, lat_max = self.lat_range
        lon_min, lon_max = self.lon_range
        px = int((lon - lon_min) / (lon_max - lon_min) * w)
        py = int((lat_max - lat) / (lat_max - lat_min) * h)
        return min(max(px, 0), w - 1), min(max(py, 0), h - 1)

    def _get_drainage_from_image(self, lat, lon):
        """从像素颜色值映射到实际排水能力 (mm/h)"""
        px, py = self._lat_lon_to_pixel(lat, lon)
        # 获取灰度值或R通道值
        r_val = self.img_array[py, px][0] if self.img_array.ndim == 3 else self.img_array[py, px]
        r_norm = r_val / 255.0
        # 简单的阈值映射规则
        if r_val == 0: return 27
        elif r_norm <= 0.9: return 36
        elif r_norm <= 0.99: return 27
        else: return 50

    def _calculate_arf_rainfall(self, distance):
        """空间衰减计算：根据距雨中心距离计算面雨量折减系数 (Area Reduction Factor, ARF)"""
        area = math.pi * (distance**2)
        # ARF 经验公式
        factor = -0.058 * math.log(area) + 0.9949 if area >= 1 else 1.0
        Q_temp = self.Q0 * factor
        # 增加空间随机扰动
        return max(0, Q_temp + np.random.normal(0, 0.3 * abs(Q_temp)))

    def get_instantaneous_intensity(self, time_step, position, precomputed_dist=None):
        """
        获取某时刻、某位置的瞬时降雨强度 (mm/h)。
        主要用于地上线路的能见度/运行风险判断。
        [修改说明] 引入60分钟滑动平均，解决芝加哥雨型瞬时峰值过高的问题。
        """
        if precomputed_dist is not None: dist = precomputed_dist
        else:
            if position in self._distance_cache: dist = self._distance_cache[position]
            else:
                dist = self._haversine(self.center_loc[0], self.center_loc[1], position[0], position[1])
                self._distance_cache[position] = dist
                
        spatial_val = self._calculate_arf_rainfall(dist)
        
        if self.rainfall_type == 'chicago':
            if time_step >= len(self.chicago_profile) - 1: return 0.0
            scale = spatial_val / self.Q0 if self.Q0 > 0 else 0
            
            # [核心修改] 使用 60分钟滑动窗口计算小时雨强
            idx_now = min(int(time_step), len(self.chicago_profile) - 1)
            window_size = 60 
            idx_prev = max(0, idx_now - window_size)
            
            # 这一时段内的总雨量 (mm)
            rain_depth_in_window = self.chicago_profile[idx_now] - self.chicago_profile[idx_prev]
            
            # 时段长度 (min)
            duration = idx_now - idx_prev
            if duration <= 0: return 0.0
            
            # 换算成小时雨强 (mm/h)
            intensity_hourly = (rain_depth_in_window / duration) * 60.0
            
            return max(0, intensity_hourly * scale)
        else:
            # 简单均匀降雨模式
            return max(0, spatial_val / (self.rain_duration / 60.0)) if self.rain_duration > 0 else 0

    def get_intensity(self, time_step, position, precomputed_dist=None):
        """
        核心计算：计算地表净积水深度 (mm)。
        使用了 SCS-CN (Soil Conservation Service Curve Number) 径流模型的简化形式。
        Logic: 积水 = 降雨径流(Runoff) - 城市管网排水(Drainage)
        """
        lat, lon = position
        if precomputed_dist is not None: dist = precomputed_dist
        else:
            if position in self._distance_cache: dist = self._distance_cache[position]
            else:
                dist = self._haversine(self.center_loc[0], self.center_loc[1], lat, lon)
                self._distance_cache[position] = dist
        spatial_val = self._calculate_arf_rainfall(dist)
        
        # 1. 计算累计降雨量 (Cumulative Rainfall)
        if self.rainfall_type == 'chicago':
            scale = spatial_val / self.Q0 if self.Q0 > 0 else 0
            idx = min(int(time_step), len(self.chicago_profile) - 1)
            cum_rain = self.chicago_profile[idx] * scale
        else:
            progress = min(time_step / float(self.rain_duration), 1.0)
            cum_rain = spatial_val * progress
            
        # 2. SCS 径流转换
        S = 15.7 # 潜在最大入渗量 (Potential Maximum Retention)，与土壤/地表类型有关
        # SCS方程：Q = (P - 0.2S)^2 / (P + 0.8S)
        runoff = (cum_rain - 0.05 * S)**2 / (cum_rain + 0.95 * S) if cum_rain > 0.05 * S else 0
        
        # 3. 扣除区域排水
        drainage = (self._get_drainage_from_image(lat, lon) / 60.0) * time_step
        return max(0, runoff - drainage) # 返回值为净积水深度

    def get_speed_impact_factor(self, water_depth, edge_type, line_context=None):
        """
        计算速度折减系数：将物理积水深度转换为对系统运行/步行速度的影响 (0.0 - 1.0)。
        1.0 表示正常速度，0.0 表示完全中断。
        """
        # [修改] 提取线路整体最大积水深度，用于判断是否触发全线停运
        # 如果 max_water_total > 450，意味着线路上某处已严重积水，该线路应整体停运
        max_water_line = 0.0
        if line_context:
            max_water_line = line_context.get('max_water_total', 0.0)

        # [修改] 核心逻辑变更：
        # 如果当前站点水深 > 450mm OR 线路任意位置水深 > 450mm (由 line_context 传入)，
        # 则视为线路瘫痪。
        is_line_failed = (water_depth > 450) or (max_water_line > 450)

        if edge_type == 'boarding': 
            # 进站/步行速度折减 (基于经验研究的步速-水深函数，如 Ishigaki et al.)
            # [修改] 若线路瘫痪（不论是本站淹水还是线路其他站淹水），进站系数设为 0.0001
            if is_line_failed:
                return 0.0001
            
            if water_depth <= 0: return 1.0
            if water_depth <= 250: return max(0.0001, 0.9 - (water_depth / 250.0) * 0.1)
            elif water_depth <= 450: return 0.0001
            else: return 0.0001 # 兜底逻辑，实际上 is_line_failed 已覆盖
            
        elif edge_type == 'riding': 
            # 乘车运行速度折减
            if line_context:
                is_above = line_context.get('is_above_ground', 0)
                max_rain = line_context.get('max_rain_above', 0.0)
                # max_water_line 已经在上面提取了
                
                if is_above == 1: # 地上线主要受瞬时强降雨（能见度）影响
                    if max_rain > 100: return 0.0001 # [修改] 极强降雨停运，因子改为 0.0001
                    elif max_rain >= 80: return 0.5 # 强降雨限速
                else: # 地下线主要受区间进水影响
                    # [修改] 若线路上最大积水超过450mm，列车停运，影响因子变为0.0001
                    if max_water_line > 450: return 0.0001
            return 1.0

# ==============================================================================
# [新增] 真实数据降雨策略类
# ==============================================================================
class RealDataRainfall(DisturbanceStrategy):
    """
    真实数据降雨策略：基于 SQLite 数据库中的历史/实时降雨数据进行空间插值。
    模拟 2025-07-29 等特定日期的真实降雨分布。
    """
    def __init__(self, base_path, db_name="shanghai_rain_history.db", start_datetime_str="2025-07-29 00:00"):
        self.base_path = base_path
        self.db_path = os.path.join(base_path, db_name)
        self.start_dt = datetime.strptime(start_datetime_str, "%Y-%m-%d %H:%M")
        self.center_loc = (31.2304, 121.4737) # 默认中心设为上海市中心，适配接口（真实数据无中心概念）
        
        # 缓存机制，避免每分钟每节点都去查库插值
        self._last_query_time_str = None
        self._cached_grid_interpolator = None # 存储 griddata 的结果或KDTree
        self._cached_df = None
        
        # 上海区域网格参数
        self.SH_LON_MIN, self.SH_LON_MAX = 120.8, 122.3
        self.SH_LAT_MIN, self.SH_LAT_MAX = 30.6, 31.9
        
        self.img_array = None
        self._load_drainage_image()

    def _load_drainage_image(self):
        """复用父类的排水图加载逻辑"""
        path = os.path.join(self.base_path, "排水能力_简化.png")
        try:
            img = Image.open(path)
            self.img_array = np.array(img)
        except:
            self.img_array = np.random.randint(0, 255, (1000, 1000, 3), dtype=np.uint8)

    def _lat_lon_to_pixel(self, lat, lon):
        """复用父类的坐标转换逻辑"""
        if self.img_array is None: return 0, 0
        h, w = self.img_array.shape[:2]
        # 注意：这里需要与 ExtremeRainfall 的范围保持一致，或者使用本类定义的范围
        # 假设排水图是覆盖 self.SH_LAT_MIN/MAX 范围的
        lat_min, lat_max = 30.6984, 31.5139 # 这里沿用 ExtremeRainfall 的范围设定以匹配图片
        lon_min, lon_max = 120.849, 121.966
        px = int((lon - lon_min) / (lon_max - lon_min) * w)
        py = int((lat_max - lat) / (lat_max - lat_min) * h)
        return min(max(px, 0), w - 1), min(max(py, 0), h - 1)

    def _get_drainage_from_image(self, lat, lon):
        """复用父类的排水能力获取逻辑"""
        px, py = self._lat_lon_to_pixel(lat, lon)
        r_val = self.img_array[py, px][0] if self.img_array.ndim == 3 else self.img_array[py, px]
        r_norm = r_val / 255.0
        if r_val == 0: return 27
        elif r_norm <= 0.9: return 36
        elif r_norm <= 0.99: return 27
        else: return 50
    
    # 距离计算方法
    def _haversine(self, lat1, lon1, lat2, lon2):
        """计算地球表面两点间的大圆距离 (km)"""
        R = 6371.0 
        dlat = math.radians(lat2 - lat1)
        dlon = math.radians(lon2 - lon1)
        a = math.sin(dlat/2)**2 + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(dlon/2)**2
        return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))
    
    def move_step(self):
        """真实数据不需要移动风暴中心，此方法留空适配接口"""
        pass

    def _fetch_and_interpolate(self, current_time_str):
        """从数据库读取特定时间的数据并构建插值函数"""
        # 如果时间没变，直接返回缓存
        if current_time_str == self._last_query_time_str and self._cached_df is not None:
            return
        
        self._last_query_time_str = current_time_str
        conn = None
        try:
            conn = sqlite3.connect(self.db_path)
            # 使用 LIKE 模糊匹配时间 (e.g., '2025-07-29 05:00%')
            query = """
                SELECT s.lon, s.lat, r.rain_value 
                FROM rain_data r
                JOIN stations s ON r.station_id = s.station_id
                WHERE r.datetime LIKE ?
            """
            df = pd.read_sql_query(query, conn, params=(f"{current_time_str}%",))
            
            if len(df) < 3:
                # 数据不足时，假设无雨
                self._cached_df = pd.DataFrame({'lon': [], 'lat': [], 'rain_value': []})
            else:
                self._cached_df = df
        except Exception as e:
            # print(f"DB Error: {e}") 
            self._cached_df = pd.DataFrame({'lon': [], 'lat': [], 'rain_value': []})
        finally:
            if conn: conn.close()

    def get_instantaneous_intensity(self, time_step, position):
        """获取瞬时雨强 (mm/h)"""
        # 计算当前绝对时间
        curr_dt = self.start_dt + timedelta(minutes=time_step)
        # 数据库是每小时数据 (e.g. 05:00, 06:00)，这里简单取最近的小时，或者按分钟插值
        # 为了性能，这里取整点小时 (format: YYYY-MM-DD HH)
        # 如果需要更精细，可以修改为 HH:MM 并确保数据库有该精度，或者在两个小时之间做线性插值
        # 这里采用：取当前小时的数据（阶梯状变化）
        target_time_str = curr_dt.strftime("%Y-%m-%d %H") # 匹配数据库的 '2025-07-29 05' 开头
        
        self._fetch_and_interpolate(target_time_str)
        
        if self._cached_df is None or len(self._cached_df) == 0:
            return 0.0
        
        # 使用 Nearest 或 Linear 插值获取特定位置的雨量
        # 由于每次调用 griddata 开销大，这里针对单点计算：
        # 实际上 griddata 适合生成网格。对于单点查询，计算距离最近的几个站做 IDW 更快。
        # 这里简化使用 scipy.interpolate.griddata 对单点插值
        points = self._cached_df[['lat', 'lon']].values
        values = self._cached_df['rain_value'].values
        
        # 注意：griddata 输入是 (points, values, xi)
        # points: (N, 2), xi: (M, 2)
        try:
            val = griddata(points, values, np.array([position]), method='nearest')[0]
            return max(0, val)
        except:
            return 0.0

    def get_intensity(self, time_step, position, precomputed_dist=None):
        """
        计算地表积水深度 (mm)。
        对于真实数据，我们假设数据库里存的是“小时雨量 (mm)”。
        为了计算积水，我们需要累计雨量。
        注意：这是一个简化处理。真实的积水模拟需要连续的时间积分。
        这里我们假设：积水 = 当前时刻瞬时雨强累积 - 排水。
        """
        lat, lon = position
        
        # 获取瞬时雨强 (假设数据库存储的是 mm/h 或者该小时的累计雨量)
        intensity_mm_h = self.get_instantaneous_intensity(time_step, position)
        
        # 计算当前时间步（1分钟）带来的新增雨量
        rain_inc = intensity_mm_h / 60.0 
        
        # 在这里，DisturbanceStrategy 是无状态的(stateless)关于“积水历史”的吗？
        # ExtremeRainfall 类里是用 cumulative profile 算的。
        # 如果我们没有累计曲线，我们只能近似。
        # 方案：假设过去 N 小时的雨都下下来了，扣除排水。
        # 但这非常慢。
        # 妥协方案：SubwayDIIMSystem 的 G.nodes[n]['water'] 是每一帧刷新的。
        # 如果 ExtremeRainfall 是每次重算 total accumulation，那这里也得这么做。
        # 但查询历史所有小时的数据太慢了。
        
        # *修正*：SubwayDIIMSystem.step() 里是直接赋值：
        # self.G.nodes[n]['water'] = self.strategy.get_intensity(...)
        # 所以 get_intensity 必须返回“当前积水深度”。
        
        # 对于 RealData，我们难以每次从 0时刻开始积分。
        # 建议：仅返回瞬时雨强导致的“潜在积水”，或者基于简单的“当前雨强 * 因子”来近似积水风险。
        # 或者，为了更准确，我们在 SimulationSystem 里维护积水状态，而不是在这里计算 total。
        # 但为了不改动 System 的逻辑（System 假设 Strategy 返回积水深度），
        # 我们这里做一个近似： 积水深度 ~= 最近 1 小时的雨量 * 径流系数 - 排水 * 1小时
        
        S = 15.7
        # 假设 effective rain 是过去 60 分钟的累积（即当前 hourly value）
        cum_rain = intensity_mm_h # 粗略假设
        
        runoff = (cum_rain - 0.05 * S)**2 / (cum_rain + 0.95 * S) if cum_rain > 0.05 * S else 0
        
        # 排水能力 (mm/h)
        drainage_rate = self._get_drainage_from_image(lat, lon)
        
        # 估算积水：径流 - 排水（假设排水一直在工作）
        # 这里仅仅是一个状态快照，无法完美模拟时序累积，除非重构 System
        water_depth = max(0, runoff - drainage_rate) 
        
        return water_depth

    def get_speed_impact_factor(self, water_depth, edge_type, line_context=None):
        """复用父类的速度折减逻辑"""
        # [修改] 提取线路整体最大积水深度
        max_water_line = 0.0
        if line_context:
            max_water_line = line_context.get('max_water_total', 0.0)

        # [修改] 判断是否全线停运
        is_line_failed = (water_depth > 450) or (max_water_line > 450)

        if edge_type == 'boarding': 
            # [修改] 若线路瘫痪，进站系数设为 0.0001
            if is_line_failed:
                return 0.0001
                
            if water_depth <= 0: return 1.0
            if water_depth <= 250: return max(0.01, 0.9 - (water_depth / 250.0) * 0.1)
            elif water_depth <= 450: return max(0.0001, 0.8 - ((water_depth - 250) / 200.0) * 0.7)
            else: return 0.0001 
            
        elif edge_type == 'riding': 
            if line_context:
                is_above = line_context.get('is_above_ground', 0)
                max_rain = line_context.get('max_rain_above', 0.0)
                
                if is_above == 1: 
                    if max_rain > 100: return 0.0001  # [修改] 极强降雨停运
                    elif max_rain >= 80: return 0.5 
                else: 
                    if max_water_line > 450: return 0.0001 # [修改] 地下线受淹停运
            return 1.0


# ==============================================================================
# 2. 核心系统：地铁网络拓扑与 DIIM (动态进程间影响模型) 模拟
# ==============================================================================
class SubwayDIIMSystem:
    """
    地铁动态影响模拟系统 (Dynamic Inoperability Input-Output Model System)
    
    主要功能：
    1. 构建双层网络图（Station Layer + Line Layer）。
    2. 加载静态数据（拓扑、时刻表、客流）。
    3. 执行时间步进仿真，计算 DIIM 状态更新。
    4. 计算网络性能指标（加权全局效率）。
    """
    def __init__(self, base_path, disturbance_strategy, sim_start_time='05:30', real_flow_df=None):
        self.base_path = base_path
        self.strategy = disturbance_strategy
        self.real_flow_df = real_flow_df # [新增] 接收真实客流数据 DataFrame
        self.G = nx.MultiGraph() # 使用多重图构建分层网络，允许两个节点间存在多种关系的边
        
        try:
            t_obj = datetime.strptime(sim_start_time, "%H:%M")
            self.sim_start_min = t_obj.hour * 60 + t_obj.minute
            self.sim_start_dt_obj = t_obj # 保存 datetime 对象用于计算
        except: 
            self.sim_start_min = 330
            self.sim_start_dt_obj = datetime(2000, 1, 1, 5, 30)
        
        # 内部缓存与索引映射
        self.id_map_long_to_short = {}
        self.line_stations_index = {}
        self.schedule_rules = [] 
        self.schedule_lookup = {}
        self.station_pos_cache = {}
        self.fast_edge_data = [] # 扁平化的边数据列表，用于极速遍历（避开NX图查询开销）
        self.node_idx_map = {}
        self.node_pos_list = []
        self.node_dists_list = []
        self.node_st_names = []
        
        # 顺序执行初始化流程
        self._load_id_mapping()           # 1. 映射线路ID (Excel -> Dict)
        self._load_json_topology()        # 2. 加载基础拓扑 (JSON)
        self._load_detailed_schedule()    # 3. 加载列车时刻/发车间隔
        self._build_station_index()       # 4. 构建索引
        self._build_graph_topology()      # 5. 构建分层网络图 (Graph Building)
        self._apply_ground_elevation_status() # 6. 标记地上/地下线
        self._load_passenger_flow_data()  # 7. 加载客流权重 (静态初始值)
        self._build_single_layer_network()# 8. 构建单层简化网 (用于辅助计算)
        self._init_diim_matrices()        # 9. 初始化动态影响矩阵 (DIIM Init)
        
        # 韧性恢复参数（设定恢复速度）
        # c(t) 为受损程度，恢复过程遵循指数衰减
        T_rec = 15.0 
        self.k_recovery = (1.0 / T_rec) * math.log(1.0 / 0.01)
        
        self._init_flow_weights()         # 初始化流权重
        self._precompute_static_simulation_data() # 预处理静态数据加速计算
        
        # 计算初始（无故障）状态下的加权网络效率，作为后续归一化的分母
        self.Ef0 = self.calculate_weighted_efficiency(self.sim_start_min, 'Weekday')
        self.congestion_warning_triggered = False

    def _load_id_mapping(self):
        """从Excel加载线路名称与短ID（如：1号线 -> "1"）的映射关系"""
        path = os.path.join(self.base_path, "线路id与名称对应-20251210.xlsx")
        try:
            if os.path.exists(path): df = pd.read_excel(path)
            else: df = pd.read_csv(path + " - Sheet1.csv")
            for _, row in df.iterrows():
                if pd.isna(row['Line ID']): continue
                long_id = str(int(row['Line ID'])) if isinstance(row['Line ID'], float) else str(row['Line ID'])
                line_name = str(row['Line Name'])
                short_id = None
                if "浦江" in line_name: short_id = "PJ"
                elif "磁浮" in line_name: short_id = "Maglev"
                elif "号线" in line_name:
                    match = re.search(r'(\d+)', line_name)
                    if match: short_id = match.group(1)
                if short_id: self.id_map_long_to_short[long_id] = short_id
        except: pass

    def _load_json_topology(self):
        """读取地铁拓扑结构 JSON 文件"""
        path = os.path.join(self.base_path, "subway.json")
        with open(path, "r", encoding="utf-8") as f: self.raw_json = json.load(f)

    def _load_detailed_schedule(self):
        """加载详细的发车间隔规则（区分早晚高峰、工作日/周末、区间车）"""
        path = os.path.join(self.base_path, "line_schedule_detailed.xlsx")
        try:
            df = pd.read_excel(path)
            df['LineID'] = df['LineID'].apply(lambda x: str(int(x)) if isinstance(x, float) else str(x).strip())
            for _, row in df.iterrows():
                s_dt = pd.to_datetime(str(row['Start_Time']), format='%H:%M:%S', errors='coerce')
                e_dt = pd.to_datetime(str(row['End_Time']), format='%H:%M:%S', errors='coerce')
                rule = {
                    'line_id': row['LineID'], 'day_type': str(row['DayType']).strip(),
                    'start_min': s_dt.hour*60+s_dt.minute if pd.notnull(s_dt) else 0,
                    'end_min': e_dt.hour*60+e_dt.minute if pd.notnull(e_dt) else 1440,
                    'from_st': str(row['From_Station']).strip(), 'to_st': str(row['To_Station']).strip(),
                    'interval': float(row['Interval_Sec'])
                }
                self.schedule_rules.append(rule)
                key = (rule['line_id'], rule['day_type'])
                if key not in self.schedule_lookup: self.schedule_lookup[key] = []
                self.schedule_lookup[key].append(rule)
        except: pass

    def _build_station_index(self):
        """构建车站在线路中的顺序索引，用于后续判断某车站是否在区间车的覆盖范围内"""
        for line_info in self.raw_json['l']:
            short_id = self.id_map_long_to_short.get(str(line_info['ls']))
            if not short_id: continue
            st_names = [st['n'] for st in line_info['st']]
            if short_id not in self.line_stations_index: 
                self.line_stations_index[short_id] = []
                self.station_pos_cache[short_id] = {}
            for st in st_names:
                if st not in self.line_stations_index[short_id]: 
                    self.line_stations_index[short_id].append(st)
                    self.station_pos_cache[short_id][st] = len(self.line_stations_index[short_id]) - 1

    def _is_station_in_section(self, line_id, station_name, from_st, to_st):
        """逻辑判定：特定车站是否位于指定的调度区间（from_st 到 to_st）内"""
        line_cache = self.station_pos_cache.get(line_id)
        if not line_cache: return False
        idx_s, idx_f, idx_t = line_cache.get(station_name), line_cache.get(from_st), line_cache.get(to_st)
        if idx_s is None or idx_f is None or idx_t is None: return False
        return min(idx_f, idx_t) <= idx_s <= max(idx_f, idx_t)

    def _build_graph_topology(self):
        """
        构建三维分层模型 (Space-L space)：
        1. 车站层 (Station Layer): 物理车站位置。
        2. 线路层 (Line Layer): 特定线路的站台。
        3. 边 (Edges):
           - boarding: 车站层 <-> 线路层 (进出站)。
           - riding: 线路层 <-> 线路层 (乘车)。
        """
        for line_info in self.raw_json['l']:
            short_id = self.id_map_long_to_short.get(str(line_info['ls']))
            if not short_id: continue
            sts = line_info['st']
            for st in sts:
                pos = tuple(map(float, st['sl'].split(',')))[::-1] # 转为 (lat, lon)
                node_st = f"{st['n']}_车站层"
                node_ln = f"{st['n']}_线路层_{short_id}"
                
                # 添加节点
                if not self.G.has_node(node_st): self.G.add_node(node_st, layer='station', pos=pos, flow=1000)
                self.G.add_node(node_ln, layer='line', pos=pos, line_id=short_id, station_name=st['n'])
                
                # 添加进站边 (Boarding/Alighting Link)
                self.G.add_edge(node_st, node_ln, type='boarding', weight=0)
            
            # 添加运行区间边 (Riding Link)
            for i in range(len(sts) - 1):
                u, v = f"{sts[i]['n']}_线路层_{short_id}", f"{sts[i+1]['n']}_线路层_{short_id}"
                if self.G.has_node(u) and self.G.has_node(v):
                    self.G.add_edge(u, v, type='riding', weight=0, line_id=short_id)

    def _apply_ground_elevation_status(self):
        """识别地铁的地上区间，这对于评估降雨暴露风险非常关键（地下区间受淹水影响，地上区间受雨强影响）"""
        file_path = os.path.join(self.base_path, "上海地铁地上线归纳.txt")
        above_set = set()
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                for line in f:
                    if '：' not in line: continue
                    parts = line.strip().split('：')
                    # 解析线路ID
                    match = re.search(r'(\d+)', parts[0])
                    lid = match.group(1) if match else ("PJ" if "浦江" in parts[0] else None)
                    if not lid: continue
                    # 解析区间
                    for group in parts[1].split('，'):
                        sts = group.strip().split('-')
                        for i in range(len(sts)-1):
                            pair = tuple(sorted((sts[i].strip(), sts[i+1].strip())))
                            above_set.add((lid, pair[0], pair[1]))
        except: pass
        
        # 遍历图中的边，标记 is_above_ground 属性
        for u, v, k, attr in self.G.edges(keys=True, data=True):
            if attr.get('type') == 'riding':
                lid = attr.get('line_id')
                stu, stv = self.G.nodes[u].get('station_name'), self.G.nodes[v].get('station_name')
                pair = tuple(sorted((stu, stv))) if stu and stv else None
                attr['is_above_ground'] = 1 if pair and (lid, pair[0], pair[1]) in above_set else 0

    def _load_passenger_flow_data(self):
        """从外部文件加载客流强度，用于后续加权网络效率的计算（客流大的站点权重更高）"""
        file_path = os.path.join(self.base_path, "进站客流.xlsx")
        p_flow = {}
        sum_f, num_f = 0, 0
        try:
            xls = pd.ExcelFile(file_path)
            for sheet in xls.sheet_names:
                for _, row in xls.parse(sheet, header=None).iterrows():
                    val = ((row[1] or 0) + (row[2] or 0)) / 2
                    p_flow[f"{row[0]}_车站层"] = p_flow.get(f"{row[0]}_车站层", 0) + val
                    sum_f += val; num_f += 1
        except: pass
        
        # 缺失值填充（使用平均值）
        avg_flow = sum_f / max(num_f, 1) or 1000
        avg_deg = np.mean([d for n, d in self.G.degree() if self.G.nodes[n].get('layer')=='station']) or 1
        
        for n, d in self.G.nodes(data=True):
            if d.get('layer') == 'station':
                d['flow'] = p_flow.get(n, (avg_flow/avg_deg)*self.G.degree(n))

    def _build_single_layer_network(self):
        """构建一个简化的单层站点间网络，用于辅助拓扑特征计算"""
        self.single_layer_net = nx.Graph()
        for u, v, attr in self.G.edges(data=True):
            if attr.get('type') == 'riding':
                n1 = self.G.nodes[u].get('station_name', u.split('_')[0])
                n2 = self.G.nodes[v].get('station_name', v.split('_')[0])
                if n1 != n2: self.single_layer_net.add_edge(n1, n2)

    def _init_flow_weights(self):
        """初始化边权：将节点流分配到相邻的进站边和运行边上"""
        ln_weights = {}
        # 1. 进站边权重
        for u, v, k, attr in self.G.edges(keys=True, data=True):
            if attr['type'] == 'boarding':
                st = u if self.G.nodes[u].get('layer')=='station' else v
                ln = v if st == u else u
                w = self.G.nodes[st].get('flow', 1000) / max(1, self.G.degree[st])
                attr['flow_weight'] = w
                ln_weights[ln] = w
        # 2. 运行边权重（取两端节点平均）
        for u, v, k, attr in self.G.edges(keys=True, data=True):
            if attr['type'] == 'riding':
                attr['flow_weight'] = (ln_weights.get(u, 100)+ln_weights.get(v, 100))/2

    def _precompute_static_simulation_data(self):
        """
        [极速仿真优化] 预计算所有不会随时间改变的数据结构。
        将图数据扁平化为 List/Array，避免在 step() 循环中频繁调用 NetworkX 的 lookup 方法。
        """
        nodes = list(self.G.nodes())
        self.node_idx_map = {n: i for i, n in enumerate(nodes)}
        self.node_dists_list = [0.0] * len(nodes)
        for i, n in enumerate(nodes):
            self.node_pos_list.append(self.G.nodes[n]['pos'])
            self.node_st_names.append(self.G.nodes[n].get('station_name', n.split('_')[0]))
        
        target_day_type = 'Weekday'
        for u, v, k in self.edge_list:
            attr = self.G.edges[u, v, k]
            u_idx, v_idx = self.node_idx_map[u], self.node_idx_map[v]
            lid = attr.get('line_id') or self.G.nodes[u].get('line_id') or self.G.nodes[v].get('line_id')
            est = {
                'u_idx': u_idx, 'v_idx': v_idx, 'type': attr['type'], 'line_id': lid,
                'is_above_ground': attr.get('is_above_ground', 0),
                'flow_weight': attr.get('flow_weight', 1.0),
                'boarding_dist': 0.0, 'boarding_node_flow': 1000.0,
                'riding_pos_mid': None, 'riding_applicable_rules': []
            }
            # 预绑定该边适用的调度规则（避免运行时搜索）
            if lid:
                rules = self.schedule_lookup.get((str(lid), target_day_type), [])
                st_u, st_v = self.node_st_names[u_idx], self.node_st_names[v_idx]
                check_st = st_u if attr['type'] == 'boarding' else None
                for r in rules:
                    if check_st:
                        if self._is_station_in_section(str(lid), check_st, r['from_st'], r['to_st']):
                             est['riding_applicable_rules'].append(r)
                    else:
                        if self._is_station_in_section(str(lid), st_u, r['from_st'], r['to_st']) and \
                           self._is_station_in_section(str(lid), st_v, r['from_st'], r['to_st']):
                             est['riding_applicable_rules'].append(r)
            if attr['type'] == 'boarding':
                st = self.node_st_names[u_idx]
                deg = self.single_layer_net.degree[st] if st in self.single_layer_net else 1
                est['boarding_dist'] = deg * 100.0 # 模拟车站内步行距离
                est['boarding_node_flow'] = self.G.nodes[u if self.G.nodes[u]['layer']=='station' else v].get('flow', 1000)
            else:
                p1, p2 = self.G.nodes[u]['pos'], self.G.nodes[v]['pos']
                est['riding_pos_mid'] = ((p1[0]+p2[0])/2, (p1[1]+p2[1])/2)
            self.fast_edge_data.append(est)

    def calculate_weighted_efficiency(self, current_time_min, day_type, line_stats=None):
        """
        计算效率，并包含终极路径追踪。
        """
        node_waters = [self.G.nodes[n].get('water', 0) for n in self.G.nodes()]
        
        # 1. 计算所有边权重
        for i, (u, v, k) in enumerate(self.edge_list):
            sd = self.fast_edge_data[i]
            
            # Rain
            avg_w = (node_waters[sd['u_idx']] + node_waters[sd['v_idx']]) / 2.0
            t_curr, _, _ = self._calc_times_fast(sd, avg_w, current_time_min, line_stats)
            
            # Base
            t_baseline, _, _ = self._calc_times_fast(sd, 0.0, current_time_min, line_stats=None)
            
            w = sd['flow_weight']
            
            # 这里的权重已经是 "时间 / 流量权重"
            weight_rain = (t_curr / w) if w > 0 and t_curr != float('inf') else float('inf')
            weight_base = (t_baseline / w) if w > 0 and t_baseline != float('inf') else float('inf')
            
            self.G.edges[u,v,k]['tau'] = weight_rain
            self.G.edges[u,v,k]['tau_baseline'] = weight_base
            
        st_nodes = [n for n, d in self.G.nodes(data=True) if d.get('layer') == 'station']
        st_set = set(st_nodes)
        N = len(st_nodes)
        if N < 2: return 0.0
        
        sum_inv_curr = 0.0
        sum_inv_base = 0.0
        
        found_anomaly_path = False
        
        # 2. 计算最短路径
        for src in st_nodes:
            # 获取 Rain 场景下的所有路径长度
            try:
                lens_rain = nx.single_source_dijkstra_path_length(self.G, src, weight='tau')
            except: lens_rain = {}
            
            # 获取 Base 场景下的所有路径长度
            try:
                lens_base = nx.single_source_dijkstra_path_length(self.G, src, weight='tau_baseline')
            except: lens_base = {}
            
            # 累加并检查
            for t in st_set:
                if t == src: continue
                
                l_r = lens_rain.get(t, float('inf'))
                l_b = lens_base.get(t, float('inf'))
                
                if l_r > 0: sum_inv_curr += 1.0/l_r
                if l_b > 0: sum_inv_base += 1.0/l_b
                
                # === 捕获异常路径 ===
                # 如果 Rain 距离 < Base 距离，说明 Rain 路径更短（更优），这是不可能的
                if l_r < l_b - 1e-6 and not found_anomaly_path:
                    print(f"\n🔥🔥🔥 [PATH ANOMALY DETECTED] at Time {current_time_min} 🔥🔥🔥")
                    print(f"Source: {src} -> Target: {t}")
                    print(f"Dist Rain: {l_r:.6f} < Dist Base: {l_b:.6f}")
                    
                    # 追踪 Rain 路径
                    try:
                        path = nx.shortest_path(self.G, src, t, weight='tau')
                        print("--- Rain Path Breakdown ---")
                        total_chk = 0
                        for i in range(len(path)-1):
                            u_p, v_p = path[i], path[i+1]
                            # 获取边数据 (取最小权重的那个key)
                            edge_data = self.G.get_edge_data(u_p, v_p)
                            # 找到 weight='tau' 最小的那个 key
                            best_k = min(edge_data, key=lambda k: edge_data[k]['tau'])
                            attr = edge_data[best_k]
                            
                            w_r = attr['tau']
                            w_b = attr['tau_baseline']
                            print(f"   {u_p} -> {v_p} | RainW={w_r:.4f} vs BaseW={w_b:.4f} | {'ERROR' if w_r < w_b else 'OK'}")
                            total_chk += w_r
                        print(f"   Check Sum: {total_chk:.4f}")
                    except:
                        print("   (Could not reconstruct path)")
                    
                    found_anomaly_path = True
        
        Ef = (1/(N*(N-1))) * sum_inv_curr
        Ef0 = (1/(N*(N-1))) * sum_inv_base
        
        return Ef / Ef0 if Ef0 > 0 else 1.0

    def _init_diim_matrices(self):
        """初始化动态进程间影响模型 (DIIM) 矩阵结构"""
        self.edge_list = list(self.G.edges(keys=True))
        self.edge_idx = {e: i for i, e in enumerate(self.edge_list)}
        n = len(self.edge_list)
        self.q_vector = np.zeros(n) # 存储各边的受损失效率 q
        self.A_matrix = sparse.csr_matrix((n, n))
        self.node_to_boarding_indices = {}
        for n_st in [n for n, d in self.G.nodes(data=True) if d['layer']=='station']:
            idxs = []
            for u, v, k in self.G.edges(n_st, keys=True):
                if self.G.edges[u,v,k]['type'] == 'boarding':
                    k_t = (u,v,k) if (u,v,k) in self.edge_idx else (v,u,k)
                    if k_t in self.edge_idx: idxs.append(self.edge_idx[k_t])
            self.node_to_boarding_indices[n_st] = idxs

    def _calc_times_fast(self, static_data, water_depth, current_time_min, line_stats=None):
        """
        计算特定边在特定积水深度下的瞬时通行时间。
        Returns: (当前受灾时间 t_curr, 正常理想时间 t_norm, 调试信息字典 debug_info)
        """
        lookup_time = current_time_min % 1440
        etype = static_data['type']
        
        # 准备调试信息
        debug_info = {}

        lid = static_data['line_id']
        ctx = None
        if lid and line_stats and str(lid) in line_stats:
            st_data = line_stats[str(lid)]
            ctx = {'is_above_ground': static_data.get('is_above_ground', 0), 
                   'max_rain_above': st_data['max_rain_above'], 
                   'max_water_total': st_data['max_water_total']}

        if etype == 'boarding':
            dist = static_data['boarding_dist']
            
            # 1. 速度计算
            speed_factor = self.strategy.get_speed_impact_factor(water_depth, 'boarding', line_context=ctx)
            v_curr = 1.4 * speed_factor
            
            # 记录调试信息
            debug_info['speed_factor'] = speed_factor
            debug_info['v_curr'] = v_curr
            debug_info['dist'] = dist

            if v_curr < 0.01: return float('inf'), 100.0, debug_info
            
            tw_c = dist / v_curr
            tw_n = dist / 1.4
            
            t_b = 30 # 列车停靠时间

            # 2. 等待时间计算
            t_wait_c, t_wait_n = 0, 0
            if static_data['line_id']:
                t_base = 300.0
                for r in static_data['riding_applicable_rules']:
                    if r['start_min'] <= lookup_time < r['end_min']:
                        t_base = r['interval']; break
                
                hw_n = t_base
                # 计算受影响后的发车间隔
                riding_factor = self.strategy.get_speed_impact_factor(water_depth, 'riding', line_context=ctx)
                hw_c = hw_n / max(riding_factor, 0.0001)
                
                # 排队论参数
                lam = max(static_data['boarding_node_flow'] / 3600.0, 0.0001)
                
                # 记录更多调试信息
                debug_info['hw_n'] = hw_n
                debug_info['hw_c'] = hw_c
                debug_info['riding_factor'] = riding_factor
                debug_info['lambda'] = lam

                # [修正开始] 定义单调性约束的排队等待时间计算函数
                def calc_w(EV, L):
                    # 1. 计算饱和度 (Saturation Ratio)
                    # 原分母公式 D = 2 * (1/L - 2*EV)
                    # 饱和状态意味着 1/L = 2*EV => L * 2 * EV = 1
                    # 因此定义 rho = L * 2 * EV
                    rho = L * 2 * EV
                    
                    # 设定安全阈值，超过此阈值认为进入拥堵/过饱和区
                    # 在此区域不再使用原公式（原公式有奇点且非单调），改用幂函数惩罚
                    RHO_THRESH = 0.9

                    # 计算阈值处的理论等待时间 (作为锚点)
                    # 当 rho = 0.9 时:
                    # denom_limit = 2 * (1/L - 2*EV) 
                    #             = 2/L * (1 - 2*EV*L) 
                    #             = 2/L * (1 - 0.9) = 0.2 / L
                    # 这是一个近似计算，为了性能，我们直接用硬逻辑判断
                    
                    if rho < RHO_THRESH:
                        # === 正常区域：使用原排队论公式 ===
                        denom = 2 * ((1.0/L) - 2*EV)
                        # 虽然理论上 rho < 0.9 保证 denom > 0，但加个由衷的兜底
                        if denom <= 1e-6: 
                            val = 1800.0 # 理论上不应走到这里
                        else:
                            val = 2.5*EV + (t_b**2 + 3*EV**2)/denom
                        
                        # 即使在正常区域，也允许适当的上限，但不要强制截断导致非单调
                        # 如果计算值确实很大（极罕见），让它保持很大，不要 clamp 到 1800
                        return min(val, 3600.0) 
                    
                    else:
                        # === 拥堵/过饱和区域：使用单调惩罚函数 ===
                        # 这里的逻辑是：既然已经堵了，那么发车间隔(EV)越大，rho越大，等待时间必须越长。
                        # 我们构造一个平滑过渡函数。
                        # 先算出在 RHO_THRESH (0.9) 时的边界值 T_limit
                        denom_limit = 2 * ((1.0/L) * (1 - RHO_THRESH))
                        t_limit = 2.5*EV + (t_b**2 + 3*EV**2)/denom_limit
                        
                        # 惩罚因子：(当前饱和度 / 阈值饱和度) 的平方
                        # 平方项确保了"越堵越慢"的非线性增长，有效拉开 Rain 和 Base 的差距
                        penalty_factor = (rho / RHO_THRESH) ** 2
                        
                        # 最终时间 = 边界时间 * 惩罚因子
                        # 这样即使 Base 很堵 (rho=0.95), Rain 更堵 (rho=1.1), 
                        # Rain 的结果也一定比 Base 大。
                        final_wait = t_limit * penalty_factor
                        
                        # 即使是非常极端的情况，也建议保留一个物理极限（如 2小时），
                        # 但这个极限必须足够大，以免再次出现 Base > Rain 的截断悖论
                        return min(final_wait, 7200.0) 

                # [修正结束]
                
                t_wait_n = calc_w(hw_n+t_b, lam)
                t_wait_c = calc_w(hw_c+t_b, lam)
                
                debug_info['t_wait_n'] = t_wait_n
                debug_info['t_wait_c'] = t_wait_c

            return tw_c + t_wait_c, tw_n + t_wait_n, debug_info
            
        elif etype == 'riding':
            t_base = 180.0
            for r in static_data['riding_applicable_rules']:
                if r['start_min'] <= lookup_time < r['end_min']:
                    t_base = r['interval']; break
            
            fac = self.strategy.get_speed_impact_factor(water_depth, 'riding', line_context=ctx)
            
            debug_info['t_base_interval'] = t_base
            debug_info['riding_factor'] = fac
            
            if fac <= 0.0001: return float('inf'), t_base, debug_info
            return t_base / fac, t_base, debug_info
            
        return 0, 0, debug_info

    def step(self, t_minute, day_type='Weekday'):
        """
        推进模拟时间步，并包含全网权重暴力一致性检查。
        """
        curr_clk = self.sim_start_min + t_minute
        
        # 1. 动态更新客流 (保持原逻辑)
        if self.real_flow_df is not None:
            curr_abs_time = None
            if hasattr(self.strategy, 'start_dt'):
                curr_abs_time = self.strategy.start_dt + timedelta(minutes=t_minute)
            
            if curr_abs_time:
                lookup_h = curr_abs_time.replace(minute=0, second=0, microsecond=0)
                try:
                    if curr_abs_time.minute == 0 or t_minute == 0:
                        batch = self.real_flow_df[self.real_flow_df['time'] == lookup_h]
                        if not batch.empty:
                            for _, row in batch.iterrows():
                                st_name = row['station']
                                flow_val = (row['inbound'] + row['outbound']) / 2
                                node_st = f"{st_name}_车站层"
                                if self.G.has_node(node_st):
                                    self.G.nodes[node_st]['flow'] = flow_val
                                    
                            for est in self.fast_edge_data:
                                if est['type'] == 'boarding':
                                    st_node_idx = est['u_idx'] if self.G.nodes[list(self.G.nodes())[est['u_idx']]].get('layer') == 'station' else est['v_idx']
                                    st_node_name = list(self.G.nodes())[st_node_idx]
                                    est['boarding_node_flow'] = self.G.nodes[st_node_name].get('flow', 1000)
                except Exception as e:
                    pass

        self.strategy.move_step()
        clat, clon = self.strategy.center_loc
        
        # 2. 更新积水
        for i, (lat, lon) in enumerate(self.node_pos_list):
            self.node_dists_list[i] = self.strategy._haversine(clat, clon, lat, lon)
        for i, n in enumerate(self.G.nodes()):
            w = self.strategy.get_intensity(t_minute, self.G.nodes[n]['pos'], precomputed_dist=self.node_dists_list[i])
            self.G.nodes[n]['water'] = w
            
        # 3. 统计线路风险
        line_stats = {}
        node_waters = [self.G.nodes[n].get('water', 0) for n in self.G.nodes()]
        for sd in self.fast_edge_data:
            if sd['type'] == 'riding' and sd['line_id']:
                lid = str(sd['line_id'])
                if lid not in line_stats: line_stats[lid] = {'max_rain_above': 0.0, 'max_water_total': 0.0}
                aw = (node_waters[sd['u_idx']] + node_waters[sd['v_idx']]) / 2
                if aw > line_stats[lid]['max_water_total']: line_stats[lid]['max_water_total'] = aw
                if sd['is_above_ground']:
                    r = self.strategy.get_instantaneous_intensity(t_minute, sd['riding_pos_mid'])
                    if r > line_stats[lid]['max_rain_above']: line_stats[lid]['max_rain_above'] = r
                    
        # 4. DIIM 更新与全网扫描
        curr_c = np.zeros(len(self.edge_list))
        
        # [暴力扫描] 统计计数器
        scan_count_rain_faster = 0
        scan_total_edges = 0
        scan_debug_msg = ""

        for i, sd in enumerate(self.fast_edge_data):
            aw = (node_waters[sd['u_idx']] + node_waters[sd['v_idx']]) / 2
            
            # 计算 Rain 时间
            tc, tn, _ = self._calc_times_fast(sd, aw, curr_clk, line_stats)
            
            # 计算 Base 时间 (强制无雨)
            tb, _, _ = self._calc_times_fast(sd, 0.0, curr_clk, line_stats=None)
            
            # --- 核心检查点 ---
            # 只要 Rain 比 Base 快超过 0.0001秒，就记录下来
            if tc < tb - 0.0001:
                scan_count_rain_faster += 1
                if scan_count_rain_faster <= 3: # 只记录前3个，防止刷屏
                    scan_debug_msg += f"\n   [BAD EDGE] {self.node_st_names[sd['u_idx']]}->{self.node_st_names[sd['v_idx']]} ({sd['type']}) | Rain={tc:.4f} < Base={tb:.4f}"

            # DIIM 逻辑
            val = 0.0
            if tc == float('inf'): val = 1.0 
            elif tc > 0: val = 1.0 - (tn / tc)
            curr_c[i] = max(0.0, min(1.0, val)) 
            scan_total_edges += 1
        
        # [打印扫描结果]
        if scan_count_rain_faster > 0:
            print(f"\n🚨 [CRITICAL SCAN] Time {t_minute}: Found {scan_count_rain_faster}/{scan_total_edges} edges where Rain is FASTER!")
            print(scan_debug_msg)
        else:
            # 如果没有发现异常边，打印一条确认信息
            # print(f"✅ [SCAN OK] Time {t_minute}: All edges Rain >= Base.")
            pass

        q_next = (1 - self.k_recovery) * self.q_vector + self.k_recovery * curr_c
        self.q_vector = np.clip(q_next, 0, 1)
        
        for i, (u,v,k) in enumerate(self.edge_list): self.G.edges[u,v,k]['q_value'] = self.q_vector[i]
        aff_lines = set()
        for i, q in enumerate(self.q_vector):
            if q > 0.1:
                lid = self.fast_edge_data[i]['line_id']
                if lid: aff_lines.add(lid)
        node_risks = []
        for n, idxs in self.node_to_boarding_indices.items():
            if idxs: node_risks.append(max(self.q_vector[i] for i in idxs))
            else: node_risks.append(0)
        
        self.current_line_stats = line_stats
        avg_water = np.mean(list(node_waters))
        max_n_risk = max(node_risks) if node_risks else 0
        
        avg_q_safe = np.nanmean(self.q_vector) if len(self.q_vector) > 0 else 0.0
        avg_q_safe = np.nan_to_num(avg_q_safe)
        
        return avg_q_safe, avg_water, len(aff_lines), max_n_risk

# ==============================================================================
# 3. 蒙特卡洛数据记录器：收集并保存多次随机实验的数据
# ==============================================================================
class MonteCarloRecorder:
    def __init__(self, time_steps):
        self.times = time_steps
        self.df_eff = pd.DataFrame({'Time': time_steps})
        self.df_risk = pd.DataFrame({'Time': time_steps})
        self.df_water = pd.DataFrame({'Time': time_steps})
        self.df_intensity = pd.DataFrame({'Time': time_steps})
        self.df_lat = pd.DataFrame({'Time': time_steps})
        self.df_lon = pd.DataFrame({'Time': time_steps})
        self.df_aff_lines = pd.DataFrame({'Time': time_steps})
        self.df_max_n_risk = pd.DataFrame({'Time': time_steps})

    def add_run_data(self, run_id, eff, risk, water, inten, lats, lons, aff_ln, max_nr):
        col_name = f"Run_{run_id}"
        self.df_eff[col_name] = eff
        self.df_risk[col_name] = risk
        self.df_water[col_name] = water
        self.df_intensity[col_name] = inten
        self.df_lat[col_name] = lats
        self.df_lon[col_name] = lons
        self.df_aff_lines[col_name] = aff_ln
        self.df_max_n_risk[col_name] = max_nr

    def save_to_excel(self, base_path):
        fname = os.path.join(base_path, f"MonteCarlo_Results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.xlsx")
        print(f"💾 正在导出蒙特卡洛数据到: {fname}")
        try:
            with pd.ExcelWriter(fname, engine='openpyxl') as writer:
                self.df_eff.to_excel(writer, sheet_name='Efficiency', index=False)
                self.df_risk.to_excel(writer, sheet_name='Risk_Avg_Q', index=False)
                self.df_water.to_excel(writer, sheet_name='Water_Level_Avg', index=False)
                self.df_intensity.to_excel(writer, sheet_name='Intensity_Center', index=False)
                self.df_lat.to_excel(writer, sheet_name='Location_Lat', index=False)
                self.df_lon.to_excel(writer, sheet_name='Location_Lon', index=False)
                self.df_aff_lines.to_excel(writer, sheet_name='Affected_Line_Count', index=False)
                self.df_max_n_risk.to_excel(writer, sheet_name='Max_Node_Risk', index=False)
            print("✅ 导出完成。")
        except Exception as e:
            print(f"❌ 导出失败: {e}")

# ==========================================
# 5. [Worker] 独立仿真任务：供多进程并行调用
# 注意：该函数必须定义在顶层，以便 Joblib 进行序列化 (Pickle)
# ==========================================
def simulation_worker(run_id, base_path, intensity, period, rain_dur, sim_dur, start_str, 
                      use_real_data=False, real_db_name=None, real_flow_data=None):
    try:
        # [修改] 支持真实数据模式初始化
        if use_real_data:
            strategy = RealDataRainfall(base_path, db_name=real_db_name, start_datetime_str=start_str)
            # 在真实模式下，我们传入客流 DataFrame
            sys_obj = SubwayDIIMSystem(base_path, strategy, sim_start_time=start_str, real_flow_df=real_flow_data)
        else:
            strategy = ExtremeRainfall(base_path, max_intensity=intensity, 
                                       rainfall_type='chicago', P=period, 
                                       rain_duration=rain_dur)
            
            # [修改] 为 ExtremeRainfall 模式注入客流数据和基准时间
            # 我们将模拟日期固定在无雨的 2025-08-08，以便使用这天的真实客流
            base_date_str = "2025-08-08"
            full_start_str = f"{base_date_str} {start_str}"
            try:
                # 手动给 strategy 绑定 start_dt，以便 SubwayDIIMSystem 能够计算绝对时间并查询客流
                strategy.start_dt = datetime.strptime(full_start_str, "%Y-%m-%d %H:%M")
            except ValueError:
                pass
                
            # 将外部传入的 real_flow_data (08.08的客流) 传给系统
            sys_obj = SubwayDIIMSystem(base_path, strategy, sim_start_time=start_str, real_flow_df=real_flow_data)
            
        t_obj = datetime.strptime(start_str, "%Y-%m-%d %H:%M") if len(start_str) > 5 else datetime.strptime(start_str, "%H:%M")
        start_min = t_obj.hour * 60 + t_obj.minute
        time_steps = list(range(sim_dur))
        r_eff, r_risk, r_water, r_inten, r_lat, r_lon, r_aff, r_mnr = [], [], [], [], [], [], [], []
        
        for t in time_steps:
            avg_q, avg_w, n_lines, max_nr = sys_obj.step(t)
            curr_clk = start_min + t
            # 计算该时间步的网络加权效率
            ef = sys_obj.calculate_weighted_efficiency(curr_clk, 'Weekday', line_stats=getattr(sys_obj, 'current_line_stats', None))
            
            # 对于真实数据，位置固定，返回 0 或默认中心
            c_lat, c_lon = (0, 0)
            if hasattr(sys_obj.strategy, 'center_loc'):
                c_lat, c_lon = sys_obj.strategy.center_loc
                
            # 瞬时雨强，如果是真实数据，传一个中心点坐标去获取，或者取平均？
            # 这里取 (31.23, 121.47) 上海中心点作为参考
            ref_pos = (31.2304, 121.4737)
            inten = sys_obj.strategy.get_instantaneous_intensity(t, ref_pos)
            
            r_eff.append(ef)
            r_risk.append(avg_q)
            r_water.append(avg_w)
            r_inten.append(inten)
            r_lat.append(c_lat)
            r_lon.append(c_lon)
            r_aff.append(n_lines)
            r_mnr.append(max_nr)
        
        # 强制垃圾回收，防止内存泄漏
        del sys_obj
        del strategy
        gc.collect() 
        return (run_id, r_eff, r_risk, r_water, r_inten, r_lat, r_lon, r_aff, r_mnr)
    except Exception as e:
        return (run_id, None, str(e), None, None, None, None, None, None)

# [新增] 数据一致性校验函数
def verify_data_consistency(base_path, flow_df):
    """
    检查 Excel 客流数据、JSON 拓扑数据、DB 降雨数据中的车站名称一致性。
    """
    print("\n🔍 正在进行数据一致性校验...")
    
    # 1. 获取 Excel 中的车站列表
    excel_stations = set(flow_df['station'].unique())
    
    # 2. 获取 JSON 中的车站列表
    json_path = os.path.join(base_path, "subway.json")
    json_stations = set()
    try:
        with open(json_path, "r", encoding="utf-8") as f: 
            data = json.load(f)
            for l in data['l']:
                for st in l['st']:
                    json_stations.add(st['n'])
    except:
        print("⚠️ 无法读取 subway.json，跳过拓扑校验。")

    # 3. 打印比对结果 (Excel vs JSON)
    if json_stations:
        missing_in_json = excel_stations - json_stations
        missing_in_excel = json_stations - excel_stations
        
        print(f"  - Excel 中有 {len(excel_stations)} 个车站")
        print(f"  - JSON 中有 {len(json_stations)} 个车站")
        
        if missing_in_json:
            print(f"⚠️ 警告: Excel 中有 {len(missing_in_json)} 个车站在 JSON 拓扑中找不到 (可能无法更新其客流):")
            print(f"    示例: {list(missing_in_json)[:5]} ...")
        else:
            print("✅ Excel 所有车站均已在 JSON 拓扑中找到。")
            
        if missing_in_excel:
            # 这通常不是问题，因为 Excel 可能只包含部分车站数据
            pass

    # 4. 获取 DB 中的车站列表 (简单检查)
    db_path = os.path.join(base_path, "shanghai_rain_history.db")
    if os.path.exists(db_path):
        try:
            conn = sqlite3.connect(db_path)
            cursor = conn.cursor()
            cursor.execute("SELECT DISTINCT station_name FROM stations") # 假设表名 stations，列名 station_name
            db_stations = set([r[0] for r in cursor.fetchall()])
            conn.close()
            
            print(f"  - DB 中有 {len(db_stations)} 个气象监测站")
            # 气象站和地铁站通常不同名，这里仅做数量确认，不做集合比对
        except:
            print("⚠️ 无法读取数据库中的站点信息，跳过校验。")
    else:
        print("⚠️ 数据库文件不存在。")
    print("--------------------------------------------------\n")


# ==========================================
# 6. 主程序入口 (Joblib 并行执行版)
# ==========================================
def main():
    # === 参数配置区域 ===
    BASE_PATH = "E:\\BaiHao\\20260103算例补充\\输入与输出数据"
    
    # --- 模式开关 ---
    USE_REAL_DATA = False   # [开关] 是否使用 2025.07.29-08.01 的真实数据
    # ----------------
    
    if USE_REAL_DATA:
        # 真实数据模式配置
        MC_ITERATIONS = 1         # 真实数据是确定性的，通常跑 1 次即可 (或者跑多次以测试算法稳定性)
        START_TIME_STR = "2025-07-29 05:00" # 仿真开始时间
        END_TIME_STR   = "2025-08-01 23:00" # 仿真结束时间
        #START_TIME_STR = "2025-07-29 07:00" # 仿真开始时间
        #END_TIME_STR   = "2025-07-29 08:00" # 仿真结束时间
        
        # 计算总分钟数
        t_start = datetime.strptime(START_TIME_STR, "%Y-%m-%d %H:%M")
        t_end   = datetime.strptime(END_TIME_STR, "%Y-%m-%d %H:%M")
        SIM_DURATION = int((t_end - t_start).total_seconds() / 60)
        
        DB_NAME = "shanghai_rain_history.db"
        FLOW_FILE = "shanghai_metro_flow_final_complete.xlsx" # 支持 csv 或 xlsx
        
        print(f"\n🚀 启动真实数据仿真模式 ({START_TIME_STR} -> {END_TIME_STR}, Duration={SIM_DURATION}min)")
        
        # 加载客流数据 (Pre-load flow data)
        flow_path = os.path.join(BASE_PATH, FLOW_FILE)
        print(f"📂 加载客流文件: {flow_path}")
        try:
            if flow_path.endswith('.csv'):
                flow_df = pd.read_csv(flow_path)
            else:
                flow_df = pd.read_excel(flow_path)

            # 1. 基础清洗：转字符串、去空格、统一符号
            flow_df['station'] = flow_df['station'].astype(str).str.strip().str.replace('●', '·').str.replace(' ', '')

            # 2. 特殊处理：将“浦电路”拆分为4号线和6号线，客流平分
            # 筛选出原始名称为 "浦电路" 的所有行
            pudian_mask = (flow_df['station'] == '浦电路')
            
            if pudian_mask.any():
                # 提取这些行
                pudian_rows = flow_df[pudian_mask].copy()
                
                # 构造4号线数据 (取名为 JSON 中对应的 '浦电路(4号线)', 客流除以2)
                pudian_4 = pudian_rows.copy()
                pudian_4['station'] = '浦电路(4号线)'
                pudian_4['inbound'] = pudian_4['inbound'] / 2.0
                pudian_4['outbound'] = pudian_4['outbound'] / 2.0
                
                # 构造6号线数据 (取名为 JSON 中对应的 '浦电路(6号线)', 客流除以2)
                pudian_6 = pudian_rows.copy()
                pudian_6['station'] = '浦电路(6号线)'
                pudian_6['inbound'] = pudian_6['inbound'] / 2.0
                pudian_6['outbound'] = pudian_6['outbound'] / 2.0
                
                # 从原表中删除“浦电路”，并追加拆分后的两部分数据
                flow_df = flow_df[~pudian_mask] # 删除原行
                flow_df = pd.concat([flow_df, pudian_4, pudian_6], ignore_index=True)
                print(f"ℹ️ 已将 {len(pudian_rows)} 条'浦电路'数据拆分为4号线与6号线 (客流平分)。")

            # 【新增修复】 简单的站点名称清洗，解决匹配警告
            def clean_station_name(name):
                name = str(name).strip()
                # 1. 替换特殊符号 '●' 为标准符号 '·' 
                name = name.replace('●', '·').replace(' ', '')
                
                # 2. 修正已知的 Excel 录入笔误及旧站名
                fixes = {
                    # --- 笔误修正 ---
                    '淞淞发路': '淞发路',
                    '淞淞虹路': '淞虹路',
                    '淞淞滨路': '淞滨路',  # [新增] 修正笔误
                    '临洮洮路': '临洮路',  # [新增] 修正笔误
                    '颛颛桥': '颛桥',
                    '佘佘山': '佘山',
                    # --- 站名变更/映射 ---               
                    # --- 暂未开通/无拓扑车站 ---
                    '龙居路': '龙居路',    # 14号线暂缓开通，拓扑中无此站，保留原名即可
                    '严御路': '康恒路',    # [新增] 严御路已更名为康恒路(11号线) 。但是拓扑里面没它。
                }
                return fixes.get(name, name)

            flow_df['station'] = flow_df['station'].apply(clean_station_name)

            # 预处理：转换时间列，方便后续查询
            flow_df['time'] = pd.to_datetime(flow_df['time'])
            
            # 执行一致性校验
            verify_data_consistency(BASE_PATH, flow_df)
            
        except Exception as e:
            print(f"❌ 读取客流文件失败: {e}")
            return

    else:
        # 原始蒙特卡洛模式配置
        MC_ITERATIONS = 100       
        RAIN_P_PERIOD = 25       
        RAIN_DURATION = 180       
        SIM_DURATION = 240        
        START_TIME_STR = "09:00"     
        print(f"\n🚀 启动随机蒙特卡洛仿真模式 (Runs={MC_ITERATIONS})")
        DB_NAME = None
        
        # [修改] 读取 08.08 的基准客流数据 (全量 CSV)
        flow_file_full = "shanghai_metro_flow_with_code_final_20250802-20250831_v4.csv"
        flow_path = os.path.join(BASE_PATH, flow_file_full)
        print(f"📂 加载基准客流文件 (2025.08.08): {flow_path}")
        
        try:
            # 读取 CSV
            full_df = pd.read_csv(flow_path)
            
            # 过滤 2025-08-08 的数据
            # [修改] 原代码的 format='%Y-%m-%d %H' 无法解析 ":00:00" 部分，改为自动推断
            full_df['time'] = pd.to_datetime(full_df['time']) 
            
            target_date = pd.Timestamp("2025-08-08").date()
            flow_df = full_df[full_df['time'].dt.date == target_date].copy()
            
            # 重命名列以匹配 SubwayDIIMSystem
            # csv: inbound, outbound, station_name_mapped
            # system expects: inbound, outbound, station
            flow_df.rename(columns={'station_name_mapped': 'station'}, inplace=True)
            
            # 站点名称清洗 (确保与 if 块中的逻辑一致)
            def clean_station_name(name):
                name = str(name).strip()
                name = name.replace('●', '·').replace(' ', '')
                fixes = {
                    '淞淞发路': '淞发路',
                    '淞淞虹路': '淞虹路',
                    '淞淞滨路': '淞滨路',
                    '临洮洮路': '临洮路', 
                    '颛颛桥': '颛桥',
                    '佘佘山': '佘山',
                    '严御路': '康恒路',
                }
                return fixes.get(name, name)
            flow_df['station'] = flow_df['station'].apply(clean_station_name)
            
            # 处理浦电路拆分 (简单起见，这里复用逻辑)
            pudian_mask = (flow_df['station'] == '浦电路')
            if pudian_mask.any():
                pudian_rows = flow_df[pudian_mask].copy()
                pudian_4 = pudian_rows.copy(); pudian_4['station'] = '浦电路(4号线)'; pudian_4[['inbound','outbound']] /= 2
                pudian_6 = pudian_rows.copy(); pudian_6['station'] = '浦电路(6号线)'; pudian_6[['inbound','outbound']] /= 2
                flow_df = flow_df[~pudian_mask]
                flow_df = pd.concat([flow_df, pudian_4, pudian_6], ignore_index=True)
            
            print(f"✅ 已加载 2025.08.08 客流数据，共 {len(flow_df)} 条记录")
            
        except Exception as e:
            print(f"❌ 读取客流文件失败: {e}")
            flow_df = None

    # N_JOBS 控制并行核心数
    # N_JOBS = 10 if USE_REAL_DATA else 30 # 真实数据读取DB开销大，建议减少并发数或设为1
    N_JOBS = 40

    recorder = MonteCarloRecorder(list(range(SIM_DURATION)))
    
    try:
        # 使用 joblib.Parallel 并行派发 simulation_worker 任务
        results = Parallel(n_jobs=N_JOBS, backend='loky', verbose=10)(
            delayed(simulation_worker)(
                run+1, BASE_PATH, 250, 
                RAIN_P_PERIOD, # period (unused in real mode)
                RAIN_DURATION, # rain_dur (unused in real mode)
                SIM_DURATION, START_TIME_STR,
                use_real_data=USE_REAL_DATA,
                real_db_name=DB_NAME,
                real_flow_data=flow_df # [修改] 将读取到的 flow_df 传递进去
            ) for run in range(MC_ITERATIONS)
        )
        
        # 遍历并行结果并存入记录器
        for res in results:
            if res[1] is None:
                print(f"❌ [Error] Run {res[0]} failed: {res[2]}")
            else:
                recorder.add_run_data(*res)
                
    except Exception as e:
        print(f"❌ [Critical] Parallel execution failed: {e}")

    print("\n🏁 仿真结束，正在保存数据...")
    recorder.save_to_excel(BASE_PATH)

if __name__ == "__main__":
    main()

In [ ]:
import networkx as nx
import json
import math
import pandas as pd
import numpy as np
import random
import os
import re
import gc
import time
import sys
import sqlite3
import warnings
from abc import ABC, abstractmethod
from scipy import sparse
from scipy.spatial.distance import cdist
from scipy.interpolate import griddata
from PIL import Image
from datetime import datetime, timedelta

# [修改 1] 引入 joblib 用于 Jupyter 环境下的并行计算
from joblib import Parallel, delayed 

# 禁用插值产生的可能的警告
warnings.filterwarnings("ignore")

# ==============================================================================
# [全局配置] 并行计算环境设置
# ==============================================================================
# 限制 Numpy/OpenBLAS 的底层线程数。
# 原因：在 Joblib 多进程模式下，如果每个子进程再自动开启多线程进行矩阵运算，
# 会导致 CPU 资源争抢（Oversubscription）甚至死锁，显著降低仿真效率。
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

# ==============================================================================
# 1. 策略模式：定义扰动（降雨）行为
# ==============================================================================
class DisturbanceStrategy(ABC):
    """
    抽象基类：定义系统受到的外部干扰策略。
    
    Methods:
        get_intensity: 获取特定时空的干扰强度。
        get_speed_impact_factor: 获取干扰对系统运行速度的影响系数。
    """
    @abstractmethod
    def get_intensity(self, time_step, position): pass
    @abstractmethod
    def get_speed_impact_factor(self, intensity, edge_type, **kwargs): pass

class ExtremeRainfall(DisturbanceStrategy):
    """
    极端降雨策略模型：模拟一个移动的暴雨中心及其对地铁站点的积水影响。
    
    包含物理过程：
    1. 芝加哥雨型（Chicago Design Storm）生成时间序列降雨。
    2. 移动风暴中心（Moving Storm Center）。
    3. 空间衰减（ARF）计算。
    4. SCS-CN 径流模型计算地表积水。
    """
    def __init__(self, base_path, max_intensity=201.9, rainfall_type='chicago', P=100, r=0.405, rain_duration=180):
        """
        初始化降雨模型参数。
        
        Args:
            max_intensity (float): 降雨峰值强度参数 (mm/h 或相关系数)。
            P (int): 降雨重现期 (Return Period, e.g., 100年一遇)。
            r (float): 雨峰位置系数 (Peak Position parameter)，0-1之间，0.4代表峰值出现在前40%。
            rain_duration (int): 降雨持续总时长 (分钟)。
        """
        # 设定模拟的空间范围（上海经纬度大致范围）
        self.lat_range = (30.6984, 31.5139)
        self.lon_range = (120.849, 121.966)
        
        # 初始化暴雨中心起始位置和移动速度
        self.center_loc, self.velocity = self._init_random_boundary_start()
        
        self.Q0 = max_intensity  # 降雨强度基准
        self.base_path = base_path
        self.rainfall_type = rainfall_type
        self.P_period = P        # 重现期
        self.r_peak = r          # 雨峰系数
        self.rain_duration = rain_duration 
        
        self.chicago_profile = None 
        self.img_array = None
        self._load_drainage_image() # 加载区域排水能力图
        self._distance_cache = {}   # 缓存站点到雨中心的距离，减少重复三角函数计算
        
        # 如果是芝加哥雨型，预计算随时间变化的强度曲线
        if self.rainfall_type == 'chicago': self._precompute_chicago_profile()

    def _init_random_boundary_start(self):
        """
        随机初始化降雨中心。
        逻辑：选择地图四个边界之一作为起点，目标点设为地图中心附近，以此计算移动矢量。
        """
        lat_min, lat_max = self.lat_range
        lon_min, lon_max = self.lon_range
        side = random.choice([0, 1, 2, 3]) # 0:上(北), 1:下(南), 2:左(西), 3:右(东)
        
        # 设定目标点为区域中心附近略带随机偏移
        target_lat = (lat_min + lat_max) / 2 + random.uniform(-0.1, 0.1)
        target_lon = (lon_min + lon_max) / 2 + random.uniform(-0.1, 0.1)
        
        if side == 0:   start_lat, start_lon = lat_max, random.uniform(lon_min, lon_max)
        elif side == 1: start_lat, start_lon = lat_min, random.uniform(lon_min, lon_max)
        elif side == 2: start_lat, start_lon = random.uniform(lat_min, lat_max), lon_min
        else:           start_lat, start_lon = random.uniform(lat_min, lat_max), lon_max
        
        speed_kmh = random.uniform(25, 40) # 降雨系统移动速度 (km/h)
        speed_km_min = speed_kmh / 60.0
        
        # 计算移动单位向量（转换为经纬度增量）
        delta_lat_km = (target_lat - start_lat) * 111.0
        delta_lon_km = (target_lon - start_lon) * 96.0
        dist_km = math.sqrt(delta_lat_km**2 + delta_lon_km**2) or 1
        
        v_lat_deg = ((delta_lat_km / dist_km) * speed_km_min) / 111.0
        v_lon_deg = ((delta_lon_km / dist_km) * speed_km_min) / 96.0
        return (start_lat, start_lon), (v_lat_deg, v_lon_deg)

    def move_step(self):
        """每一分钟移动一次降雨中心，并清空距离缓存（因为中心变了，所有距离需重算）"""
        curr_lat, curr_lon = self.center_loc
        v_lat, v_lon = self.velocity
        self.center_loc = (curr_lat + v_lat, curr_lon + v_lon)
        self._distance_cache = {} 

    def _precompute_chicago_profile(self):
        """
        预计算芝加哥雨型（Chicago Design Storm）的时间分布。
        公式来源：Keifer & Chu (1957)，参数 A, n, b 针对上海地区暴雨强度公式适配。
        """
        times = np.arange(0, self.rain_duration + 1, 1) 
        intensities = []
        t_peak_time = self.rain_duration * self.r_peak
        
        # A, n, b 为上海地区的降雨公式参数（需根据研究区域调整）
        A = 9.581 * (1 + 0.846 * np.log10(self.P_period))
        n = 0.656
        b = 7
        
        for t in times:
            # 峰前时刻公式
            if t <= t_peak_time:
                t_b = t_peak_time - t
                num = A * ((1 - n) * (t_b / self.r_peak) + b)
                den = math.pow((t_b / self.r_peak) + b, 1 + n)
            # 峰后时刻公式
            else:
                t_a = t - t_peak_time
                num = A * ((1 - n) * (t_a / (1 - self.r_peak)) + b)
                den = math.pow((t_a / (1 - self.r_peak)) + b, 1 + n)
            intensities.append(num / den)
        
        # 模拟雨后由于排水不畅导致的“长尾效应”（指数衰减）
        tail_duration = 120  
        if intensities:
            last_val = intensities[-1]
            for t_ex in range(1, tail_duration + 1):
                val = last_val * math.exp(-0.1 * t_ex)
                intensities.append(val if val > 0.001 else 0.0)
        
        # 存储为累计降雨量 (Cumulative Rainfall)，方便后续计算径流
        self.chicago_profile = np.cumsum(intensities)

    def _load_drainage_image(self):
        """加载排水能力栅格图，用于根据地理位置查询排水效率"""
        path = os.path.join(self.base_path, "排水能力_简化.png")
        try:
            img = Image.open(path)
            self.img_array = np.array(img)
        except FileNotFoundError:
            # 如果没图，生成一个随机的（调试用）
            self.img_array = np.random.randint(0, 255, (1000, 1000, 3), dtype=np.uint8)

    def _haversine(self, lat1, lon1, lat2, lon2):
        """计算地球表面两点间的大圆距离 (km)"""
        R = 6371.0 
        dlat = math.radians(lat2 - lat1)
        dlon = math.radians(lon2 - lon1)
        a = math.sin(dlat/2)**2 + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(dlon/2)**2
        return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))

    def _lat_lon_to_pixel(self, lat, lon):
        """经纬度坐标与图像像素坐标的转换"""
        if self.img_array is None: return 0, 0
        h, w = self.img_array.shape[:2]
        lat_min, lat_max = self.lat_range
        lon_min, lon_max = self.lon_range
        px = int((lon - lon_min) / (lon_max - lon_min) * w)
        py = int((lat_max - lat) / (lat_max - lat_min) * h)
        return min(max(px, 0), w - 1), min(max(py, 0), h - 1)

    def _get_drainage_from_image(self, lat, lon):
        """从像素颜色值映射到实际排水能力 (mm/h)"""
        px, py = self._lat_lon_to_pixel(lat, lon)
        # 获取灰度值或R通道值
        r_val = self.img_array[py, px][0] if self.img_array.ndim == 3 else self.img_array[py, px]
        r_norm = r_val / 255.0
        # 简单的阈值映射规则
        if r_val == 0: return 27
        elif r_norm <= 0.9: return 36
        elif r_norm <= 0.99: return 27
        else: return 50

    def _calculate_arf_rainfall(self, distance):
        """空间衰减计算：根据距雨中心距离计算面雨量折减系数 (Area Reduction Factor, ARF)"""
        area = math.pi * (distance**2)
        # ARF 经验公式
        factor = -0.058 * math.log(area) + 0.9949 if area >= 1 else 1.0
        Q_temp = self.Q0 * factor
        # 增加空间随机扰动
        return max(0, Q_temp + np.random.normal(0, 0.3 * abs(Q_temp)))

    def get_instantaneous_intensity(self, time_step, position, precomputed_dist=None):
        """
        获取某时刻、某位置的瞬时降雨强度 (mm/h)。
        主要用于地上线路的能见度/运行风险判断。
        [修改说明] 引入60分钟滑动平均，解决芝加哥雨型瞬时峰值过高的问题。
        """
        if precomputed_dist is not None: dist = precomputed_dist
        else:
            if position in self._distance_cache: dist = self._distance_cache[position]
            else:
                dist = self._haversine(self.center_loc[0], self.center_loc[1], position[0], position[1])
                self._distance_cache[position] = dist
                
        spatial_val = self._calculate_arf_rainfall(dist)
        
        if self.rainfall_type == 'chicago':
            if time_step >= len(self.chicago_profile) - 1: return 0.0
            scale = spatial_val / self.Q0 if self.Q0 > 0 else 0
            
            # [核心修改] 使用 60分钟滑动窗口计算小时雨强
            idx_now = min(int(time_step), len(self.chicago_profile) - 1)
            window_size = 60 
            idx_prev = max(0, idx_now - window_size)
            
            # 这一时段内的总雨量 (mm)
            rain_depth_in_window = self.chicago_profile[idx_now] - self.chicago_profile[idx_prev]
            
            # 时段长度 (min)
            duration = idx_now - idx_prev
            if duration <= 0: return 0.0
            
            # 换算成小时雨强 (mm/h)
            intensity_hourly = (rain_depth_in_window / duration) * 60.0
            
            return max(0, intensity_hourly * scale)
        else:
            # 简单均匀降雨模式
            return max(0, spatial_val / (self.rain_duration / 60.0)) if self.rain_duration > 0 else 0

    def get_intensity(self, time_step, position, precomputed_dist=None):
        """
        核心计算：计算地表净积水深度 (mm)。
        使用了 SCS-CN (Soil Conservation Service Curve Number) 径流模型的简化形式。
        Logic: 积水 = 降雨径流(Runoff) - 城市管网排水(Drainage)
        """
        lat, lon = position
        if precomputed_dist is not None: dist = precomputed_dist
        else:
            if position in self._distance_cache: dist = self._distance_cache[position]
            else:
                dist = self._haversine(self.center_loc[0], self.center_loc[1], lat, lon)
                self._distance_cache[position] = dist
        spatial_val = self._calculate_arf_rainfall(dist)
        
        # 1. 计算累计降雨量 (Cumulative Rainfall)
        if self.rainfall_type == 'chicago':
            scale = spatial_val / self.Q0 if self.Q0 > 0 else 0
            idx = min(int(time_step), len(self.chicago_profile) - 1)
            cum_rain = self.chicago_profile[idx] * scale
        else:
            progress = min(time_step / float(self.rain_duration), 1.0)
            cum_rain = spatial_val * progress
            
        # 2. SCS 径流转换
        S = 15.7 # 潜在最大入渗量 (Potential Maximum Retention)，与土壤/地表类型有关
        # SCS方程：Q = (P - 0.2S)^2 / (P + 0.8S)
        runoff = (cum_rain - 0.05 * S)**2 / (cum_rain + 0.95 * S) if cum_rain > 0.05 * S else 0
        
        # 3. 扣除区域排水
        drainage = (self._get_drainage_from_image(lat, lon) / 60.0) * time_step
        return max(0, runoff - drainage) # 返回值为净积水深度

    def get_speed_impact_factor(self, water_depth, edge_type, line_context=None):
        """
        计算速度折减系数：将物理积水深度转换为对系统运行/步行速度的影响 (0.0 - 1.0)。
        1.0 表示正常速度，0.0 表示完全中断。
        """
        # [修改] 提取线路整体最大积水深度，用于判断是否触发全线停运
        # 如果 max_water_total > 450，意味着线路上某处已严重积水，该线路应整体停运
        max_water_line = 0.0
        if line_context:
            max_water_line = line_context.get('max_water_total', 0.0)

        # [修改] 核心逻辑变更：
        # 如果当前站点水深 > 450mm OR 线路任意位置水深 > 450mm (由 line_context 传入)，
        # 则视为线路瘫痪。
        is_line_failed = (water_depth > 450) or (max_water_line > 450)

        if edge_type == 'boarding': 
            # 进站/步行速度折减 (基于经验研究的步速-水深函数，如 Ishigaki et al.)
            # [修改] 若线路瘫痪（不论是本站淹水还是线路其他站淹水），进站系数设为 0.0001
            if is_line_failed:
                return 0.0001
            
            if water_depth <= 0: return 1.0
            if water_depth <= 250: return max(0.0001, 0.9 - (water_depth / 250.0) * 0.1)
            elif water_depth <= 450: return 0.0001
            else: return 0.0001 # 兜底逻辑，实际上 is_line_failed 已覆盖
            
        elif edge_type == 'riding': 
            # 乘车运行速度折减
            if line_context:
                is_above = line_context.get('is_above_ground', 0)
                max_rain = line_context.get('max_rain_above', 0.0)
                # max_water_line 已经在上面提取了
                
                if is_above == 1: # 地上线主要受瞬时强降雨（能见度）影响
                    if max_rain > 100: return 0.0001 # [修改] 极强降雨停运，因子改为 0.0001
                    elif max_rain >= 80: return 0.5 # 强降雨限速
                else: # 地下线主要受区间进水影响
                    # [修改] 若线路上最大积水超过450mm，列车停运，影响因子变为0.0001
                    if max_water_line > 450: return 0.0001
            return 1.0

# ==============================================================================
# [新增] 真实数据降雨策略类
# ==============================================================================
class RealDataRainfall(DisturbanceStrategy):
    """
    真实数据降雨策略：基于 SQLite 数据库中的历史/实时降雨数据进行空间插值。
    模拟 2025-07-29 等特定日期的真实降雨分布。
    """
    def __init__(self, base_path, db_name="shanghai_rain_history.db", start_datetime_str="2025-07-29 00:00"):
        self.base_path = base_path
        self.db_path = os.path.join(base_path, db_name)
        self.start_dt = datetime.strptime(start_datetime_str, "%Y-%m-%d %H:%M")
        self.center_loc = (31.2304, 121.4737) # 默认中心设为上海市中心，适配接口（真实数据无中心概念）
        
        # 缓存机制，避免每分钟每节点都去查库插值
        self._last_query_time_str = None
        self._cached_grid_interpolator = None # 存储 griddata 的结果或KDTree
        self._cached_df = None
        
        # 上海区域网格参数
        self.SH_LON_MIN, self.SH_LON_MAX = 120.8, 122.3
        self.SH_LAT_MIN, self.SH_LAT_MAX = 30.6, 31.9
        
        self.img_array = None
        self._load_drainage_image()

    def _load_drainage_image(self):
        """复用父类的排水图加载逻辑"""
        path = os.path.join(self.base_path, "排水能力_简化.png")
        try:
            img = Image.open(path)
            self.img_array = np.array(img)
        except:
            self.img_array = np.random.randint(0, 255, (1000, 1000, 3), dtype=np.uint8)

    def _lat_lon_to_pixel(self, lat, lon):
        """复用父类的坐标转换逻辑"""
        if self.img_array is None: return 0, 0
        h, w = self.img_array.shape[:2]
        # 注意：这里需要与 ExtremeRainfall 的范围保持一致，或者使用本类定义的范围
        # 假设排水图是覆盖 self.SH_LAT_MIN/MAX 范围的
        lat_min, lat_max = 30.6984, 31.5139 # 这里沿用 ExtremeRainfall 的范围设定以匹配图片
        lon_min, lon_max = 120.849, 121.966
        px = int((lon - lon_min) / (lon_max - lon_min) * w)
        py = int((lat_max - lat) / (lat_max - lat_min) * h)
        return min(max(px, 0), w - 1), min(max(py, 0), h - 1)

    def _get_drainage_from_image(self, lat, lon):
        """复用父类的排水能力获取逻辑"""
        px, py = self._lat_lon_to_pixel(lat, lon)
        r_val = self.img_array[py, px][0] if self.img_array.ndim == 3 else self.img_array[py, px]
        r_norm = r_val / 255.0
        if r_val == 0: return 27
        elif r_norm <= 0.9: return 36
        elif r_norm <= 0.99: return 27
        else: return 50
    
    # 距离计算方法
    def _haversine(self, lat1, lon1, lat2, lon2):
        """计算地球表面两点间的大圆距离 (km)"""
        R = 6371.0 
        dlat = math.radians(lat2 - lat1)
        dlon = math.radians(lon2 - lon1)
        a = math.sin(dlat/2)**2 + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(dlon/2)**2
        return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))
    
    def move_step(self):
        """真实数据不需要移动风暴中心，此方法留空适配接口"""
        pass

    def _fetch_and_interpolate(self, current_time_str):
        """从数据库读取特定时间的数据并构建插值函数"""
        # 如果时间没变，直接返回缓存
        if current_time_str == self._last_query_time_str and self._cached_df is not None:
            return
        
        self._last_query_time_str = current_time_str
        conn = None
        try:
            conn = sqlite3.connect(self.db_path)
            # 使用 LIKE 模糊匹配时间 (e.g., '2025-07-29 05:00%')
            query = """
                SELECT s.lon, s.lat, r.rain_value 
                FROM rain_data r
                JOIN stations s ON r.station_id = s.station_id
                WHERE r.datetime LIKE ?
            """
            df = pd.read_sql_query(query, conn, params=(f"{current_time_str}%",))
            
            if len(df) < 3:
                # 数据不足时，假设无雨
                self._cached_df = pd.DataFrame({'lon': [], 'lat': [], 'rain_value': []})
            else:
                self._cached_df = df
        except Exception as e:
            # print(f"DB Error: {e}") 
            self._cached_df = pd.DataFrame({'lon': [], 'lat': [], 'rain_value': []})
        finally:
            if conn: conn.close()

    def get_instantaneous_intensity(self, time_step, position):
        """获取瞬时雨强 (mm/h)"""
        # 计算当前绝对时间
        curr_dt = self.start_dt + timedelta(minutes=time_step)
        # 数据库是每小时数据 (e.g. 05:00, 06:00)，这里简单取最近的小时，或者按分钟插值
        # 为了性能，这里取整点小时 (format: YYYY-MM-DD HH)
        # 如果需要更精细，可以修改为 HH:MM 并确保数据库有该精度，或者在两个小时之间做线性插值
        # 这里采用：取当前小时的数据（阶梯状变化）
        target_time_str = curr_dt.strftime("%Y-%m-%d %H") # 匹配数据库的 '2025-07-29 05' 开头
        
        self._fetch_and_interpolate(target_time_str)
        
        if self._cached_df is None or len(self._cached_df) == 0:
            return 0.0
        
        # 使用 Nearest 或 Linear 插值获取特定位置的雨量
        # 由于每次调用 griddata 开销大，这里针对单点计算：
        # 实际上 griddata 适合生成网格。对于单点查询，计算距离最近的几个站做 IDW 更快。
        # 这里简化使用 scipy.interpolate.griddata 对单点插值
        points = self._cached_df[['lat', 'lon']].values
        values = self._cached_df['rain_value'].values
        
        # 注意：griddata 输入是 (points, values, xi)
        # points: (N, 2), xi: (M, 2)
        try:
            val = griddata(points, values, np.array([position]), method='nearest')[0]
            return max(0, val)
        except:
            return 0.0

    def get_intensity(self, time_step, position, precomputed_dist=None):
        """
        计算地表积水深度 (mm)。
        对于真实数据，我们假设数据库里存的是“小时雨量 (mm)”。
        为了计算积水，我们需要累计雨量。
        注意：这是一个简化处理。真实的积水模拟需要连续的时间积分。
        这里我们假设：积水 = 当前时刻瞬时雨强累积 - 排水。
        """
        lat, lon = position
        
        # 获取瞬时雨强 (假设数据库存储的是 mm/h 或者该小时的累计雨量)
        intensity_mm_h = self.get_instantaneous_intensity(time_step, position)
        
        # 计算当前时间步（1分钟）带来的新增雨量
        rain_inc = intensity_mm_h / 60.0 
        
        # 在这里，DisturbanceStrategy 是无状态的(stateless)关于“积水历史”的吗？
        # ExtremeRainfall 类里是用 cumulative profile 算的。
        # 如果我们没有累计曲线，我们只能近似。
        # 方案：假设过去 N 小时的雨都下下来了，扣除排水。
        # 但这非常慢。
        # 妥协方案：SubwayDIIMSystem 的 G.nodes[n]['water'] 是每一帧刷新的。
        # 如果 ExtremeRainfall 是每次重算 total accumulation，那这里也得这么做。
        # 但查询历史所有小时的数据太慢了。
        
        # *修正*：SubwayDIIMSystem.step() 里是直接赋值：
        # self.G.nodes[n]['water'] = self.strategy.get_intensity(...)
        # 所以 get_intensity 必须返回“当前积水深度”。
        
        # 对于 RealData，我们难以每次从 0时刻开始积分。
        # 建议：仅返回瞬时雨强导致的“潜在积水”，或者基于简单的“当前雨强 * 因子”来近似积水风险。
        # 或者，为了更准确，我们在 SimulationSystem 里维护积水状态，而不是在这里计算 total。
        # 但为了不改动 System 的逻辑（System 假设 Strategy 返回积水深度），
        # 我们这里做一个近似： 积水深度 ~= 最近 1 小时的雨量 * 径流系数 - 排水 * 1小时
        
        S = 15.7
        # 假设 effective rain 是过去 60 分钟的累积（即当前 hourly value）
        cum_rain = intensity_mm_h # 粗略假设
        
        runoff = (cum_rain - 0.05 * S)**2 / (cum_rain + 0.95 * S) if cum_rain > 0.05 * S else 0
        
        # 排水能力 (mm/h)
        drainage_rate = self._get_drainage_from_image(lat, lon)
        
        # 估算积水：径流 - 排水（假设排水一直在工作）
        # 这里仅仅是一个状态快照，无法完美模拟时序累积，除非重构 System
        water_depth = max(0, runoff - drainage_rate) 
        
        return water_depth

    def get_speed_impact_factor(self, water_depth, edge_type, line_context=None):
        """复用父类的速度折减逻辑"""
        # [修改] 提取线路整体最大积水深度
        max_water_line = 0.0
        if line_context:
            max_water_line = line_context.get('max_water_total', 0.0)

        # [修改] 判断是否全线停运
        is_line_failed = (water_depth > 450) or (max_water_line > 450)

        if edge_type == 'boarding': 
            # [修改] 若线路瘫痪，进站系数设为 0.0001
            if is_line_failed:
                return 0.0001
                
            if water_depth <= 0: return 1.0
            if water_depth <= 250: return max(0.01, 0.9 - (water_depth / 250.0) * 0.1)
            elif water_depth <= 450: return max(0.0001, 0.8 - ((water_depth - 250) / 200.0) * 0.7)
            else: return 0.0001 
            
        elif edge_type == 'riding': 
            if line_context:
                is_above = line_context.get('is_above_ground', 0)
                max_rain = line_context.get('max_rain_above', 0.0)
                
                if is_above == 1: 
                    if max_rain > 100: return 0.0001  # [修改] 极强降雨停运
                    elif max_rain >= 80: return 0.5 
                else: 
                    if max_water_line > 450: return 0.0001 # [修改] 地下线受淹停运
            return 1.0


# ==============================================================================
# 2. 核心系统：地铁网络拓扑与 DIIM (动态进程间影响模型) 模拟
# ==============================================================================
class SubwayDIIMSystem:
    """
    地铁动态影响模拟系统 (Dynamic Inoperability Input-Output Model System)
    
    主要功能：
    1. 构建双层网络图（Station Layer + Line Layer）。
    2. 加载静态数据（拓扑、时刻表、客流）。
    3. 执行时间步进仿真，计算 DIIM 状态更新。
    4. 计算网络性能指标（加权全局效率）。
    """
    def __init__(self, base_path, disturbance_strategy, sim_start_time='05:30', real_flow_df=None):
        self.base_path = base_path
        self.strategy = disturbance_strategy
        self.real_flow_df = real_flow_df # [新增] 接收真实客流数据 DataFrame
        self.G = nx.MultiGraph() # 使用多重图构建分层网络，允许两个节点间存在多种关系的边
        
        try:
            t_obj = datetime.strptime(sim_start_time, "%H:%M")
            self.sim_start_min = t_obj.hour * 60 + t_obj.minute
            self.sim_start_dt_obj = t_obj # 保存 datetime 对象用于计算
        except: 
            self.sim_start_min = 330
            self.sim_start_dt_obj = datetime(2000, 1, 1, 5, 30)
        
        # 内部缓存与索引映射
        self.id_map_long_to_short = {}
        self.line_stations_index = {}
        self.schedule_rules = [] 
        self.schedule_lookup = {}
        self.station_pos_cache = {}
        self.fast_edge_data = [] # 扁平化的边数据列表，用于极速遍历（避开NX图查询开销）
        self.node_idx_map = {}
        self.node_pos_list = []
        self.node_dists_list = []
        self.node_st_names = []
        
        # 顺序执行初始化流程
        self._load_id_mapping()           # 1. 映射线路ID (Excel -> Dict)
        self._load_json_topology()        # 2. 加载基础拓扑 (JSON)
        self._load_detailed_schedule()    # 3. 加载列车时刻/发车间隔
        self._build_station_index()       # 4. 构建索引
        self._build_graph_topology()      # 5. 构建分层网络图 (Graph Building)
        self._apply_ground_elevation_status() # 6. 标记地上/地下线
        self._load_passenger_flow_data()  # 7. 加载客流权重 (静态初始值)
        self._build_single_layer_network()# 8. 构建单层简化网 (用于辅助计算)
        self._init_diim_matrices()        # 9. 初始化动态影响矩阵 (DIIM Init)
        
        # 韧性恢复参数（设定恢复速度）
        # c(t) 为受损程度，恢复过程遵循指数衰减
        T_rec = 15.0 
        self.k_recovery = (1.0 / T_rec) * math.log(1.0 / 0.01)
        
        self._init_flow_weights()         # 初始化流权重
        self._precompute_static_simulation_data() # 预处理静态数据加速计算
        
        # 计算初始（无故障）状态下的加权网络效率，作为后续归一化的分母
        self.Ef0 = self.calculate_weighted_efficiency(self.sim_start_min, 'Weekday')
        self.congestion_warning_triggered = False

    def _load_id_mapping(self):
        """从Excel加载线路名称与短ID（如：1号线 -> "1"）的映射关系"""
        path = os.path.join(self.base_path, "线路id与名称对应-20251210.xlsx")
        try:
            if os.path.exists(path): df = pd.read_excel(path)
            else: df = pd.read_csv(path + " - Sheet1.csv")
            for _, row in df.iterrows():
                if pd.isna(row['Line ID']): continue
                long_id = str(int(row['Line ID'])) if isinstance(row['Line ID'], float) else str(row['Line ID'])
                line_name = str(row['Line Name'])
                short_id = None
                if "浦江" in line_name: short_id = "PJ"
                elif "磁浮" in line_name: short_id = "Maglev"
                elif "号线" in line_name:
                    match = re.search(r'(\d+)', line_name)
                    if match: short_id = match.group(1)
                if short_id: self.id_map_long_to_short[long_id] = short_id
        except: pass

    def _load_json_topology(self):
        """读取地铁拓扑结构 JSON 文件"""
        path = os.path.join(self.base_path, "subway.json")
        with open(path, "r", encoding="utf-8") as f: self.raw_json = json.load(f)

    def _load_detailed_schedule(self):
        """加载详细的发车间隔规则（区分早晚高峰、工作日/周末、区间车）"""
        path = os.path.join(self.base_path, "line_schedule_detailed.xlsx")
        try:
            df = pd.read_excel(path)
            df['LineID'] = df['LineID'].apply(lambda x: str(int(x)) if isinstance(x, float) else str(x).strip())
            for _, row in df.iterrows():
                s_dt = pd.to_datetime(str(row['Start_Time']), format='%H:%M:%S', errors='coerce')
                e_dt = pd.to_datetime(str(row['End_Time']), format='%H:%M:%S', errors='coerce')
                rule = {
                    'line_id': row['LineID'], 'day_type': str(row['DayType']).strip(),
                    'start_min': s_dt.hour*60+s_dt.minute if pd.notnull(s_dt) else 0,
                    'end_min': e_dt.hour*60+e_dt.minute if pd.notnull(e_dt) else 1440,
                    'from_st': str(row['From_Station']).strip(), 'to_st': str(row['To_Station']).strip(),
                    'interval': float(row['Interval_Sec'])
                }
                self.schedule_rules.append(rule)
                key = (rule['line_id'], rule['day_type'])
                if key not in self.schedule_lookup: self.schedule_lookup[key] = []
                self.schedule_lookup[key].append(rule)
        except: pass

    def _build_station_index(self):
        """构建车站在线路中的顺序索引，用于后续判断某车站是否在区间车的覆盖范围内"""
        for line_info in self.raw_json['l']:
            short_id = self.id_map_long_to_short.get(str(line_info['ls']))
            if not short_id: continue
            st_names = [st['n'] for st in line_info['st']]
            if short_id not in self.line_stations_index: 
                self.line_stations_index[short_id] = []
                self.station_pos_cache[short_id] = {}
            for st in st_names:
                if st not in self.line_stations_index[short_id]: 
                    self.line_stations_index[short_id].append(st)
                    self.station_pos_cache[short_id][st] = len(self.line_stations_index[short_id]) - 1

    def _is_station_in_section(self, line_id, station_name, from_st, to_st):
        """逻辑判定：特定车站是否位于指定的调度区间（from_st 到 to_st）内"""
        line_cache = self.station_pos_cache.get(line_id)
        if not line_cache: return False
        idx_s, idx_f, idx_t = line_cache.get(station_name), line_cache.get(from_st), line_cache.get(to_st)
        if idx_s is None or idx_f is None or idx_t is None: return False
        return min(idx_f, idx_t) <= idx_s <= max(idx_f, idx_t)

    def _build_graph_topology(self):
        """
        构建三维分层模型 (Space-L space)：
        1. 车站层 (Station Layer): 物理车站位置。
        2. 线路层 (Line Layer): 特定线路的站台。
        3. 边 (Edges):
           - boarding: 车站层 <-> 线路层 (进出站)。
           - riding: 线路层 <-> 线路层 (乘车)。
        """
        for line_info in self.raw_json['l']:
            short_id = self.id_map_long_to_short.get(str(line_info['ls']))
            if not short_id: continue
            sts = line_info['st']
            for st in sts:
                pos = tuple(map(float, st['sl'].split(',')))[::-1] # 转为 (lat, lon)
                node_st = f"{st['n']}_车站层"
                node_ln = f"{st['n']}_线路层_{short_id}"
                
                # 添加节点
                if not self.G.has_node(node_st): self.G.add_node(node_st, layer='station', pos=pos, flow=1000)
                self.G.add_node(node_ln, layer='line', pos=pos, line_id=short_id, station_name=st['n'])
                
                # 添加进站边 (Boarding/Alighting Link)
                self.G.add_edge(node_st, node_ln, type='boarding', weight=0)
            
            # 添加运行区间边 (Riding Link)
            for i in range(len(sts) - 1):
                u, v = f"{sts[i]['n']}_线路层_{short_id}", f"{sts[i+1]['n']}_线路层_{short_id}"
                if self.G.has_node(u) and self.G.has_node(v):
                    self.G.add_edge(u, v, type='riding', weight=0, line_id=short_id)

    def _apply_ground_elevation_status(self):
        """识别地铁的地上区间，这对于评估降雨暴露风险非常关键（地下区间受淹水影响，地上区间受雨强影响）"""
        file_path = os.path.join(self.base_path, "上海地铁地上线归纳.txt")
        above_set = set()
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                for line in f:
                    if '：' not in line: continue
                    parts = line.strip().split('：')
                    # 解析线路ID
                    match = re.search(r'(\d+)', parts[0])
                    lid = match.group(1) if match else ("PJ" if "浦江" in parts[0] else None)
                    if not lid: continue
                    # 解析区间
                    for group in parts[1].split('，'):
                        sts = group.strip().split('-')
                        for i in range(len(sts)-1):
                            pair = tuple(sorted((sts[i].strip(), sts[i+1].strip())))
                            above_set.add((lid, pair[0], pair[1]))
        except: pass
        
        # 遍历图中的边，标记 is_above_ground 属性
        for u, v, k, attr in self.G.edges(keys=True, data=True):
            if attr.get('type') == 'riding':
                lid = attr.get('line_id')
                stu, stv = self.G.nodes[u].get('station_name'), self.G.nodes[v].get('station_name')
                pair = tuple(sorted((stu, stv))) if stu and stv else None
                attr['is_above_ground'] = 1 if pair and (lid, pair[0], pair[1]) in above_set else 0

    def _load_passenger_flow_data(self):
        """从外部文件加载客流强度，用于后续加权网络效率的计算（客流大的站点权重更高）"""
        file_path = os.path.join(self.base_path, "进站客流.xlsx")
        p_flow = {}
        sum_f, num_f = 0, 0
        try:
            xls = pd.ExcelFile(file_path)
            for sheet in xls.sheet_names:
                for _, row in xls.parse(sheet, header=None).iterrows():
                    val = ((row[1] or 0) + (row[2] or 0)) / 2
                    p_flow[f"{row[0]}_车站层"] = p_flow.get(f"{row[0]}_车站层", 0) + val
                    sum_f += val; num_f += 1
        except: pass
        
        # 缺失值填充（使用平均值）
        avg_flow = sum_f / max(num_f, 1) or 1000
        avg_deg = np.mean([d for n, d in self.G.degree() if self.G.nodes[n].get('layer')=='station']) or 1
        
        for n, d in self.G.nodes(data=True):
            if d.get('layer') == 'station':
                d['flow'] = p_flow.get(n, (avg_flow/avg_deg)*self.G.degree(n))

    def _build_single_layer_network(self):
        """构建一个简化的单层站点间网络，用于辅助拓扑特征计算"""
        self.single_layer_net = nx.Graph()
        for u, v, attr in self.G.edges(data=True):
            if attr.get('type') == 'riding':
                n1 = self.G.nodes[u].get('station_name', u.split('_')[0])
                n2 = self.G.nodes[v].get('station_name', v.split('_')[0])
                if n1 != n2: self.single_layer_net.add_edge(n1, n2)

    def _init_flow_weights(self):
        """初始化边权：将节点流分配到相邻的进站边和运行边上"""
        ln_weights = {}
        # 1. 进站边权重
        for u, v, k, attr in self.G.edges(keys=True, data=True):
            if attr['type'] == 'boarding':
                st = u if self.G.nodes[u].get('layer')=='station' else v
                ln = v if st == u else u
                w = self.G.nodes[st].get('flow', 1000) / max(1, self.G.degree[st])
                attr['flow_weight'] = w
                ln_weights[ln] = w
        # 2. 运行边权重（取两端节点平均）
        for u, v, k, attr in self.G.edges(keys=True, data=True):
            if attr['type'] == 'riding':
                attr['flow_weight'] = (ln_weights.get(u, 100)+ln_weights.get(v, 100))/2

    def _precompute_static_simulation_data(self):
        """
        [极速仿真优化] 预计算所有不会随时间改变的数据结构。
        将图数据扁平化为 List/Array，避免在 step() 循环中频繁调用 NetworkX 的 lookup 方法。
        """
        nodes = list(self.G.nodes())
        self.node_idx_map = {n: i for i, n in enumerate(nodes)}
        self.node_dists_list = [0.0] * len(nodes)
        for i, n in enumerate(nodes):
            self.node_pos_list.append(self.G.nodes[n]['pos'])
            self.node_st_names.append(self.G.nodes[n].get('station_name', n.split('_')[0]))
        
        target_day_type = 'Weekday'
        for u, v, k in self.edge_list:
            attr = self.G.edges[u, v, k]
            u_idx, v_idx = self.node_idx_map[u], self.node_idx_map[v]
            lid = attr.get('line_id') or self.G.nodes[u].get('line_id') or self.G.nodes[v].get('line_id')
            est = {
                'u_idx': u_idx, 'v_idx': v_idx, 'type': attr['type'], 'line_id': lid,
                'is_above_ground': attr.get('is_above_ground', 0),
                'flow_weight': attr.get('flow_weight', 1.0),
                'boarding_dist': 0.0, 'boarding_node_flow': 1000.0,
                'riding_pos_mid': None, 'riding_applicable_rules': []
            }
            # 预绑定该边适用的调度规则（避免运行时搜索）
            if lid:
                rules = self.schedule_lookup.get((str(lid), target_day_type), [])
                st_u, st_v = self.node_st_names[u_idx], self.node_st_names[v_idx]
                check_st = st_u if attr['type'] == 'boarding' else None
                for r in rules:
                    if check_st:
                        if self._is_station_in_section(str(lid), check_st, r['from_st'], r['to_st']):
                             est['riding_applicable_rules'].append(r)
                    else:
                        if self._is_station_in_section(str(lid), st_u, r['from_st'], r['to_st']) and \
                           self._is_station_in_section(str(lid), st_v, r['from_st'], r['to_st']):
                             est['riding_applicable_rules'].append(r)
            if attr['type'] == 'boarding':
                st = self.node_st_names[u_idx]
                deg = self.single_layer_net.degree[st] if st in self.single_layer_net else 1
                est['boarding_dist'] = deg * 100.0 # 模拟车站内步行距离
                est['boarding_node_flow'] = self.G.nodes[u if self.G.nodes[u]['layer']=='station' else v].get('flow', 1000)
            else:
                p1, p2 = self.G.nodes[u]['pos'], self.G.nodes[v]['pos']
                est['riding_pos_mid'] = ((p1[0]+p2[0])/2, (p1[1]+p2[1])/2)
            self.fast_edge_data.append(est)

    def calculate_weighted_efficiency(self, current_time_min, day_type, line_stats=None):
        """
        计算效率，并包含终极路径追踪。
        """
        node_waters = [self.G.nodes[n].get('water', 0) for n in self.G.nodes()]
        
        # 1. 计算所有边权重
        for i, (u, v, k) in enumerate(self.edge_list):
            sd = self.fast_edge_data[i]
            
            # Rain
            avg_w = (node_waters[sd['u_idx']] + node_waters[sd['v_idx']]) / 2.0
            t_curr, _, _ = self._calc_times_fast(sd, avg_w, current_time_min, line_stats)
            
            # Base
            t_baseline, _, _ = self._calc_times_fast(sd, 0.0, current_time_min, line_stats=None)
            
            w = sd['flow_weight']
            
            # 这里的权重已经是 "时间 / 流量权重"
            weight_rain = (t_curr / w) if w > 0 and t_curr != float('inf') else float('inf')
            weight_base = (t_baseline / w) if w > 0 and t_baseline != float('inf') else float('inf')
            
            self.G.edges[u,v,k]['tau'] = weight_rain
            self.G.edges[u,v,k]['tau_baseline'] = weight_base
            
        st_nodes = [n for n, d in self.G.nodes(data=True) if d.get('layer') == 'station']
        st_set = set(st_nodes)
        N = len(st_nodes)
        if N < 2: return 0.0
        
        sum_inv_curr = 0.0
        sum_inv_base = 0.0
        
        found_anomaly_path = False
        
        # 2. 计算最短路径
        for src in st_nodes:
            # 获取 Rain 场景下的所有路径长度
            try:
                lens_rain = nx.single_source_dijkstra_path_length(self.G, src, weight='tau')
            except: lens_rain = {}
            
            # 获取 Base 场景下的所有路径长度
            try:
                lens_base = nx.single_source_dijkstra_path_length(self.G, src, weight='tau_baseline')
            except: lens_base = {}
            
            # 累加并检查
            for t in st_set:
                if t == src: continue
                
                l_r = lens_rain.get(t, float('inf'))
                l_b = lens_base.get(t, float('inf'))
                
                if l_r > 0: sum_inv_curr += 1.0/l_r
                if l_b > 0: sum_inv_base += 1.0/l_b
                
                # === 捕获异常路径 ===
                # 如果 Rain 距离 < Base 距离，说明 Rain 路径更短（更优），这是不可能的
                if l_r < l_b - 1e-6 and not found_anomaly_path:
                    print(f"\n🔥🔥🔥 [PATH ANOMALY DETECTED] at Time {current_time_min} 🔥🔥🔥")
                    print(f"Source: {src} -> Target: {t}")
                    print(f"Dist Rain: {l_r:.6f} < Dist Base: {l_b:.6f}")
                    
                    # 追踪 Rain 路径
                    try:
                        path = nx.shortest_path(self.G, src, t, weight='tau')
                        print("--- Rain Path Breakdown ---")
                        total_chk = 0
                        for i in range(len(path)-1):
                            u_p, v_p = path[i], path[i+1]
                            # 获取边数据 (取最小权重的那个key)
                            edge_data = self.G.get_edge_data(u_p, v_p)
                            # 找到 weight='tau' 最小的那个 key
                            best_k = min(edge_data, key=lambda k: edge_data[k]['tau'])
                            attr = edge_data[best_k]
                            
                            w_r = attr['tau']
                            w_b = attr['tau_baseline']
                            print(f"   {u_p} -> {v_p} | RainW={w_r:.4f} vs BaseW={w_b:.4f} | {'ERROR' if w_r < w_b else 'OK'}")
                            total_chk += w_r
                        print(f"   Check Sum: {total_chk:.4f}")
                    except:
                        print("   (Could not reconstruct path)")
                    
                    found_anomaly_path = True
        
        Ef = (1/(N*(N-1))) * sum_inv_curr
        Ef0 = (1/(N*(N-1))) * sum_inv_base
        
        return Ef / Ef0 if Ef0 > 0 else 1.0

    def _init_diim_matrices(self):
        """初始化动态进程间影响模型 (DIIM) 矩阵结构"""
        self.edge_list = list(self.G.edges(keys=True))
        self.edge_idx = {e: i for i, e in enumerate(self.edge_list)}
        n = len(self.edge_list)
        self.q_vector = np.zeros(n) # 存储各边的受损失效率 q
        self.A_matrix = sparse.csr_matrix((n, n))
        self.node_to_boarding_indices = {}
        for n_st in [n for n, d in self.G.nodes(data=True) if d['layer']=='station']:
            idxs = []
            for u, v, k in self.G.edges(n_st, keys=True):
                if self.G.edges[u,v,k]['type'] == 'boarding':
                    k_t = (u,v,k) if (u,v,k) in self.edge_idx else (v,u,k)
                    if k_t in self.edge_idx: idxs.append(self.edge_idx[k_t])
            self.node_to_boarding_indices[n_st] = idxs

    def _calc_times_fast(self, static_data, water_depth, current_time_min, line_stats=None):
        """
        计算特定边在特定积水深度下的瞬时通行时间。
        Returns: (当前受灾时间 t_curr, 正常理想时间 t_norm, 调试信息字典 debug_info)
        """
        lookup_time = current_time_min % 1440
        etype = static_data['type']
        
        # 准备调试信息
        debug_info = {}

        lid = static_data['line_id']
        ctx = None
        if lid and line_stats and str(lid) in line_stats:
            st_data = line_stats[str(lid)]
            ctx = {'is_above_ground': static_data.get('is_above_ground', 0), 
                   'max_rain_above': st_data['max_rain_above'], 
                   'max_water_total': st_data['max_water_total']}

        if etype == 'boarding':
            dist = static_data['boarding_dist']
            
            # 1. 速度计算
            speed_factor = self.strategy.get_speed_impact_factor(water_depth, 'boarding', line_context=ctx)
            v_curr = 1.4 * speed_factor
            
            # 记录调试信息
            debug_info['speed_factor'] = speed_factor
            debug_info['v_curr'] = v_curr
            debug_info['dist'] = dist

            if v_curr < 0.01: return float('inf'), 100.0, debug_info
            
            tw_c = dist / v_curr
            tw_n = dist / 1.4
            
            t_b = 30 # 列车停靠时间

            # 2. 等待时间计算
            t_wait_c, t_wait_n = 0, 0
            if static_data['line_id']:
                t_base = 300.0
                for r in static_data['riding_applicable_rules']:
                    if r['start_min'] <= lookup_time < r['end_min']:
                        t_base = r['interval']; break
                
                hw_n = t_base
                # 计算受影响后的发车间隔
                riding_factor = self.strategy.get_speed_impact_factor(water_depth, 'riding', line_context=ctx)
                hw_c = hw_n / max(riding_factor, 0.0001)
                
                # 排队论参数
                lam = max(static_data['boarding_node_flow'] / 3600.0, 0.0001)
                
                # 记录更多调试信息
                debug_info['hw_n'] = hw_n
                debug_info['hw_c'] = hw_c
                debug_info['riding_factor'] = riding_factor
                debug_info['lambda'] = lam

                # [修正开始] 定义单调性约束的排队等待时间计算函数
                def calc_w(EV, L):
                    # 1. 计算饱和度 (Saturation Ratio)
                    # 原分母公式 D = 2 * (1/L - 2*EV)
                    # 饱和状态意味着 1/L = 2*EV => L * 2 * EV = 1
                    # 因此定义 rho = L * 2 * EV
                    rho = L * 2 * EV
                    
                    # 设定安全阈值，超过此阈值认为进入拥堵/过饱和区
                    # 在此区域不再使用原公式（原公式有奇点且非单调），改用幂函数惩罚
                    RHO_THRESH = 0.9

                    # 计算阈值处的理论等待时间 (作为锚点)
                    # 当 rho = 0.9 时:
                    # denom_limit = 2 * (1/L - 2*EV) 
                    #             = 2/L * (1 - 2*EV*L) 
                    #             = 2/L * (1 - 0.9) = 0.2 / L
                    # 这是一个近似计算，为了性能，我们直接用硬逻辑判断
                    
                    if rho < RHO_THRESH:
                        # === 正常区域：使用原排队论公式 ===
                        denom = 2 * ((1.0/L) - 2*EV)
                        # 虽然理论上 rho < 0.9 保证 denom > 0，但加个由衷的兜底
                        if denom <= 1e-6: 
                            val = 1800.0 # 理论上不应走到这里
                        else:
                            val = 2.5*EV + (t_b**2 + 3*EV**2)/denom
                        
                        # 即使在正常区域，也允许适当的上限，但不要强制截断导致非单调
                        # 如果计算值确实很大（极罕见），让它保持很大，不要 clamp 到 1800
                        return min(val, 3600.0) 
                    
                    else:
                        # === 拥堵/过饱和区域：使用单调惩罚函数 ===
                        # 这里的逻辑是：既然已经堵了，那么发车间隔(EV)越大，rho越大，等待时间必须越长。
                        # 我们构造一个平滑过渡函数。
                        # 先算出在 RHO_THRESH (0.9) 时的边界值 T_limit
                        denom_limit = 2 * ((1.0/L) * (1 - RHO_THRESH))
                        t_limit = 2.5*EV + (t_b**2 + 3*EV**2)/denom_limit
                        
                        # 惩罚因子：(当前饱和度 / 阈值饱和度) 的平方
                        # 平方项确保了"越堵越慢"的非线性增长，有效拉开 Rain 和 Base 的差距
                        penalty_factor = (rho / RHO_THRESH) ** 2
                        
                        # 最终时间 = 边界时间 * 惩罚因子
                        # 这样即使 Base 很堵 (rho=0.95), Rain 更堵 (rho=1.1), 
                        # Rain 的结果也一定比 Base 大。
                        final_wait = t_limit * penalty_factor
                        
                        # 即使是非常极端的情况，也建议保留一个物理极限（如 2小时），
                        # 但这个极限必须足够大，以免再次出现 Base > Rain 的截断悖论
                        return min(final_wait, 7200.0) 

                # [修正结束]
                
                t_wait_n = calc_w(hw_n+t_b, lam)
                t_wait_c = calc_w(hw_c+t_b, lam)
                
                debug_info['t_wait_n'] = t_wait_n
                debug_info['t_wait_c'] = t_wait_c

            return tw_c + t_wait_c, tw_n + t_wait_n, debug_info
            
        elif etype == 'riding':
            t_base = 180.0
            for r in static_data['riding_applicable_rules']:
                if r['start_min'] <= lookup_time < r['end_min']:
                    t_base = r['interval']; break
            
            fac = self.strategy.get_speed_impact_factor(water_depth, 'riding', line_context=ctx)
            
            debug_info['t_base_interval'] = t_base
            debug_info['riding_factor'] = fac
            
            if fac <= 0.0001: return float('inf'), t_base, debug_info
            return t_base / fac, t_base, debug_info
            
        return 0, 0, debug_info

    def step(self, t_minute, day_type='Weekday'):
        """
        推进模拟时间步，并包含全网权重暴力一致性检查。
        """
        curr_clk = self.sim_start_min + t_minute
        
        # 1. 动态更新客流 (保持原逻辑)
        if self.real_flow_df is not None:
            curr_abs_time = None
            if hasattr(self.strategy, 'start_dt'):
                curr_abs_time = self.strategy.start_dt + timedelta(minutes=t_minute)
            
            if curr_abs_time:
                lookup_h = curr_abs_time.replace(minute=0, second=0, microsecond=0)
                try:
                    if curr_abs_time.minute == 0 or t_minute == 0:
                        batch = self.real_flow_df[self.real_flow_df['time'] == lookup_h]
                        if not batch.empty:
                            for _, row in batch.iterrows():
                                st_name = row['station']
                                flow_val = (row['inbound'] + row['outbound']) / 2
                                node_st = f"{st_name}_车站层"
                                if self.G.has_node(node_st):
                                    self.G.nodes[node_st]['flow'] = flow_val
                                    
                            for est in self.fast_edge_data:
                                if est['type'] == 'boarding':
                                    st_node_idx = est['u_idx'] if self.G.nodes[list(self.G.nodes())[est['u_idx']]].get('layer') == 'station' else est['v_idx']
                                    st_node_name = list(self.G.nodes())[st_node_idx]
                                    est['boarding_node_flow'] = self.G.nodes[st_node_name].get('flow', 1000)
                except Exception as e:
                    pass

        self.strategy.move_step()
        clat, clon = self.strategy.center_loc
        
        # 2. 更新积水
        for i, (lat, lon) in enumerate(self.node_pos_list):
            self.node_dists_list[i] = self.strategy._haversine(clat, clon, lat, lon)
        for i, n in enumerate(self.G.nodes()):
            w = self.strategy.get_intensity(t_minute, self.G.nodes[n]['pos'], precomputed_dist=self.node_dists_list[i])
            self.G.nodes[n]['water'] = w
            
        # 3. 统计线路风险
        line_stats = {}
        node_waters = [self.G.nodes[n].get('water', 0) for n in self.G.nodes()]
        for sd in self.fast_edge_data:
            if sd['type'] == 'riding' and sd['line_id']:
                lid = str(sd['line_id'])
                if lid not in line_stats: line_stats[lid] = {'max_rain_above': 0.0, 'max_water_total': 0.0}
                aw = (node_waters[sd['u_idx']] + node_waters[sd['v_idx']]) / 2
                if aw > line_stats[lid]['max_water_total']: line_stats[lid]['max_water_total'] = aw
                if sd['is_above_ground']:
                    r = self.strategy.get_instantaneous_intensity(t_minute, sd['riding_pos_mid'])
                    if r > line_stats[lid]['max_rain_above']: line_stats[lid]['max_rain_above'] = r
                    
        # 4. DIIM 更新与全网扫描
        curr_c = np.zeros(len(self.edge_list))
        
        # [暴力扫描] 统计计数器
        scan_count_rain_faster = 0
        scan_total_edges = 0
        scan_debug_msg = ""

        for i, sd in enumerate(self.fast_edge_data):
            aw = (node_waters[sd['u_idx']] + node_waters[sd['v_idx']]) / 2
            
            # 计算 Rain 时间
            tc, tn, _ = self._calc_times_fast(sd, aw, curr_clk, line_stats)
            
            # 计算 Base 时间 (强制无雨)
            tb, _, _ = self._calc_times_fast(sd, 0.0, curr_clk, line_stats=None)
            
            # --- 核心检查点 ---
            # 只要 Rain 比 Base 快超过 0.0001秒，就记录下来
            if tc < tb - 0.0001:
                scan_count_rain_faster += 1
                if scan_count_rain_faster <= 3: # 只记录前3个，防止刷屏
                    scan_debug_msg += f"\n   [BAD EDGE] {self.node_st_names[sd['u_idx']]}->{self.node_st_names[sd['v_idx']]} ({sd['type']}) | Rain={tc:.4f} < Base={tb:.4f}"

            # DIIM 逻辑
            val = 0.0
            if tc == float('inf'): val = 1.0 
            elif tc > 0: val = 1.0 - (tn / tc)
            curr_c[i] = max(0.0, min(1.0, val)) 
            scan_total_edges += 1
        
        # [打印扫描结果]
        if scan_count_rain_faster > 0:
            print(f"\n🚨 [CRITICAL SCAN] Time {t_minute}: Found {scan_count_rain_faster}/{scan_total_edges} edges where Rain is FASTER!")
            print(scan_debug_msg)
        else:
            # 如果没有发现异常边，打印一条确认信息
            # print(f"✅ [SCAN OK] Time {t_minute}: All edges Rain >= Base.")
            pass

        q_next = (1 - self.k_recovery) * self.q_vector + self.k_recovery * curr_c
        self.q_vector = np.clip(q_next, 0, 1)
        
        for i, (u,v,k) in enumerate(self.edge_list): self.G.edges[u,v,k]['q_value'] = self.q_vector[i]
        aff_lines = set()
        for i, q in enumerate(self.q_vector):
            if q > 0.1:
                lid = self.fast_edge_data[i]['line_id']
                if lid: aff_lines.add(lid)
        node_risks = []
        for n, idxs in self.node_to_boarding_indices.items():
            if idxs: node_risks.append(max(self.q_vector[i] for i in idxs))
            else: node_risks.append(0)
        
        self.current_line_stats = line_stats
        avg_water = np.mean(list(node_waters))
        max_n_risk = max(node_risks) if node_risks else 0
        
        avg_q_safe = np.nanmean(self.q_vector) if len(self.q_vector) > 0 else 0.0
        avg_q_safe = np.nan_to_num(avg_q_safe)
        
        return avg_q_safe, avg_water, len(aff_lines), max_n_risk

# ==============================================================================
# 3. 蒙特卡洛数据记录器：收集并保存多次随机实验的数据
# ==============================================================================
class MonteCarloRecorder:
    def __init__(self, time_steps):
        self.times = time_steps
        self.df_eff = pd.DataFrame({'Time': time_steps})
        self.df_risk = pd.DataFrame({'Time': time_steps})
        self.df_water = pd.DataFrame({'Time': time_steps})
        self.df_intensity = pd.DataFrame({'Time': time_steps})
        self.df_lat = pd.DataFrame({'Time': time_steps})
        self.df_lon = pd.DataFrame({'Time': time_steps})
        self.df_aff_lines = pd.DataFrame({'Time': time_steps})
        self.df_max_n_risk = pd.DataFrame({'Time': time_steps})

    def add_run_data(self, run_id, eff, risk, water, inten, lats, lons, aff_ln, max_nr):
        col_name = f"Run_{run_id}"
        self.df_eff[col_name] = eff
        self.df_risk[col_name] = risk
        self.df_water[col_name] = water
        self.df_intensity[col_name] = inten
        self.df_lat[col_name] = lats
        self.df_lon[col_name] = lons
        self.df_aff_lines[col_name] = aff_ln
        self.df_max_n_risk[col_name] = max_nr

    def save_to_excel(self, base_path):
        fname = os.path.join(base_path, f"MonteCarlo_Results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.xlsx")
        print(f"💾 正在导出蒙特卡洛数据到: {fname}")
        try:
            with pd.ExcelWriter(fname, engine='openpyxl') as writer:
                self.df_eff.to_excel(writer, sheet_name='Efficiency', index=False)
                self.df_risk.to_excel(writer, sheet_name='Risk_Avg_Q', index=False)
                self.df_water.to_excel(writer, sheet_name='Water_Level_Avg', index=False)
                self.df_intensity.to_excel(writer, sheet_name='Intensity_Center', index=False)
                self.df_lat.to_excel(writer, sheet_name='Location_Lat', index=False)
                self.df_lon.to_excel(writer, sheet_name='Location_Lon', index=False)
                self.df_aff_lines.to_excel(writer, sheet_name='Affected_Line_Count', index=False)
                self.df_max_n_risk.to_excel(writer, sheet_name='Max_Node_Risk', index=False)
            print("✅ 导出完成。")
        except Exception as e:
            print(f"❌ 导出失败: {e}")

# ==========================================
# 5. [Worker] 独立仿真任务：供多进程并行调用
# 注意：该函数必须定义在顶层，以便 Joblib 进行序列化 (Pickle)
# ==========================================
def simulation_worker(run_id, base_path, intensity, period, rain_dur, sim_dur, start_str, 
                      use_real_data=False, real_db_name=None, real_flow_data=None):
    try:
        # [修改] 支持真实数据模式初始化
        if use_real_data:
            strategy = RealDataRainfall(base_path, db_name=real_db_name, start_datetime_str=start_str)
            # 在真实模式下，我们传入客流 DataFrame
            sys_obj = SubwayDIIMSystem(base_path, strategy, sim_start_time=start_str, real_flow_df=real_flow_data)
        else:
            strategy = ExtremeRainfall(base_path, max_intensity=intensity, 
                                       rainfall_type='chicago', P=period, 
                                       rain_duration=rain_dur)
            
            # [修改] 为 ExtremeRainfall 模式注入客流数据和基准时间
            # 我们将模拟日期固定在无雨的 2025-08-08，以便使用这天的真实客流
            base_date_str = "2025-08-08"
            full_start_str = f"{base_date_str} {start_str}"
            try:
                # 手动给 strategy 绑定 start_dt，以便 SubwayDIIMSystem 能够计算绝对时间并查询客流
                strategy.start_dt = datetime.strptime(full_start_str, "%Y-%m-%d %H:%M")
            except ValueError:
                pass
                
            # 将外部传入的 real_flow_data (08.08的客流) 传给系统
            sys_obj = SubwayDIIMSystem(base_path, strategy, sim_start_time=start_str, real_flow_df=real_flow_data)
            
        t_obj = datetime.strptime(start_str, "%Y-%m-%d %H:%M") if len(start_str) > 5 else datetime.strptime(start_str, "%H:%M")
        start_min = t_obj.hour * 60 + t_obj.minute
        time_steps = list(range(sim_dur))
        r_eff, r_risk, r_water, r_inten, r_lat, r_lon, r_aff, r_mnr = [], [], [], [], [], [], [], []
        
        for t in time_steps:
            avg_q, avg_w, n_lines, max_nr = sys_obj.step(t)
            curr_clk = start_min + t
            # 计算该时间步的网络加权效率
            ef = sys_obj.calculate_weighted_efficiency(curr_clk, 'Weekday', line_stats=getattr(sys_obj, 'current_line_stats', None))
            
            # 对于真实数据，位置固定，返回 0 或默认中心
            c_lat, c_lon = (0, 0)
            if hasattr(sys_obj.strategy, 'center_loc'):
                c_lat, c_lon = sys_obj.strategy.center_loc
                
            # 瞬时雨强，如果是真实数据，传一个中心点坐标去获取，或者取平均？
            # 这里取 (31.23, 121.47) 上海中心点作为参考
            ref_pos = (31.2304, 121.4737)
            inten = sys_obj.strategy.get_instantaneous_intensity(t, ref_pos)
            
            r_eff.append(ef)
            r_risk.append(avg_q)
            r_water.append(avg_w)
            r_inten.append(inten)
            r_lat.append(c_lat)
            r_lon.append(c_lon)
            r_aff.append(n_lines)
            r_mnr.append(max_nr)
        
        # 强制垃圾回收，防止内存泄漏
        del sys_obj
        del strategy
        gc.collect() 
        return (run_id, r_eff, r_risk, r_water, r_inten, r_lat, r_lon, r_aff, r_mnr)
    except Exception as e:
        return (run_id, None, str(e), None, None, None, None, None, None)

# [新增] 数据一致性校验函数
def verify_data_consistency(base_path, flow_df):
    """
    检查 Excel 客流数据、JSON 拓扑数据、DB 降雨数据中的车站名称一致性。
    """
    print("\n🔍 正在进行数据一致性校验...")
    
    # 1. 获取 Excel 中的车站列表
    excel_stations = set(flow_df['station'].unique())
    
    # 2. 获取 JSON 中的车站列表
    json_path = os.path.join(base_path, "subway.json")
    json_stations = set()
    try:
        with open(json_path, "r", encoding="utf-8") as f: 
            data = json.load(f)
            for l in data['l']:
                for st in l['st']:
                    json_stations.add(st['n'])
    except:
        print("⚠️ 无法读取 subway.json，跳过拓扑校验。")

    # 3. 打印比对结果 (Excel vs JSON)
    if json_stations:
        missing_in_json = excel_stations - json_stations
        missing_in_excel = json_stations - excel_stations
        
        print(f"  - Excel 中有 {len(excel_stations)} 个车站")
        print(f"  - JSON 中有 {len(json_stations)} 个车站")
        
        if missing_in_json:
            print(f"⚠️ 警告: Excel 中有 {len(missing_in_json)} 个车站在 JSON 拓扑中找不到 (可能无法更新其客流):")
            print(f"    示例: {list(missing_in_json)[:5]} ...")
        else:
            print("✅ Excel 所有车站均已在 JSON 拓扑中找到。")
            
        if missing_in_excel:
            # 这通常不是问题，因为 Excel 可能只包含部分车站数据
            pass

    # 4. 获取 DB 中的车站列表 (简单检查)
    db_path = os.path.join(base_path, "shanghai_rain_history.db")
    if os.path.exists(db_path):
        try:
            conn = sqlite3.connect(db_path)
            cursor = conn.cursor()
            cursor.execute("SELECT DISTINCT station_name FROM stations") # 假设表名 stations，列名 station_name
            db_stations = set([r[0] for r in cursor.fetchall()])
            conn.close()
            
            print(f"  - DB 中有 {len(db_stations)} 个气象监测站")
            # 气象站和地铁站通常不同名，这里仅做数量确认，不做集合比对
        except:
            print("⚠️ 无法读取数据库中的站点信息，跳过校验。")
    else:
        print("⚠️ 数据库文件不存在。")
    print("--------------------------------------------------\n")


# ==========================================
# 6. 主程序入口 (Joblib 并行执行版)
# ==========================================
def main():
    # === 参数配置区域 ===
    BASE_PATH = "E:\\BaiHao\\20260103算例补充\\输入与输出数据"
    
    # --- 模式开关 ---
    USE_REAL_DATA = False   # [开关] 是否使用 2025.07.29-08.01 的真实数据
    # ----------------
    
    if USE_REAL_DATA:
        # 真实数据模式配置
        MC_ITERATIONS = 1         # 真实数据是确定性的，通常跑 1 次即可 (或者跑多次以测试算法稳定性)
        START_TIME_STR = "2025-07-29 05:00" # 仿真开始时间
        END_TIME_STR   = "2025-08-01 23:00" # 仿真结束时间
        #START_TIME_STR = "2025-07-29 07:00" # 仿真开始时间
        #END_TIME_STR   = "2025-07-29 08:00" # 仿真结束时间
        
        # 计算总分钟数
        t_start = datetime.strptime(START_TIME_STR, "%Y-%m-%d %H:%M")
        t_end   = datetime.strptime(END_TIME_STR, "%Y-%m-%d %H:%M")
        SIM_DURATION = int((t_end - t_start).total_seconds() / 60)
        
        DB_NAME = "shanghai_rain_history.db"
        FLOW_FILE = "shanghai_metro_flow_final_complete.xlsx" # 支持 csv 或 xlsx
        
        print(f"\n🚀 启动真实数据仿真模式 ({START_TIME_STR} -> {END_TIME_STR}, Duration={SIM_DURATION}min)")
        
        # 加载客流数据 (Pre-load flow data)
        flow_path = os.path.join(BASE_PATH, FLOW_FILE)
        print(f"📂 加载客流文件: {flow_path}")
        try:
            if flow_path.endswith('.csv'):
                flow_df = pd.read_csv(flow_path)
            else:
                flow_df = pd.read_excel(flow_path)

            # 1. 基础清洗：转字符串、去空格、统一符号
            flow_df['station'] = flow_df['station'].astype(str).str.strip().str.replace('●', '·').str.replace(' ', '')

            # 2. 特殊处理：将“浦电路”拆分为4号线和6号线，客流平分
            # 筛选出原始名称为 "浦电路" 的所有行
            pudian_mask = (flow_df['station'] == '浦电路')
            
            if pudian_mask.any():
                # 提取这些行
                pudian_rows = flow_df[pudian_mask].copy()
                
                # 构造4号线数据 (取名为 JSON 中对应的 '浦电路(4号线)', 客流除以2)
                pudian_4 = pudian_rows.copy()
                pudian_4['station'] = '浦电路(4号线)'
                pudian_4['inbound'] = pudian_4['inbound'] / 2.0
                pudian_4['outbound'] = pudian_4['outbound'] / 2.0
                
                # 构造6号线数据 (取名为 JSON 中对应的 '浦电路(6号线)', 客流除以2)
                pudian_6 = pudian_rows.copy()
                pudian_6['station'] = '浦电路(6号线)'
                pudian_6['inbound'] = pudian_6['inbound'] / 2.0
                pudian_6['outbound'] = pudian_6['outbound'] / 2.0
                
                # 从原表中删除“浦电路”，并追加拆分后的两部分数据
                flow_df = flow_df[~pudian_mask] # 删除原行
                flow_df = pd.concat([flow_df, pudian_4, pudian_6], ignore_index=True)
                print(f"ℹ️ 已将 {len(pudian_rows)} 条'浦电路'数据拆分为4号线与6号线 (客流平分)。")

            # 【新增修复】 简单的站点名称清洗，解决匹配警告
            def clean_station_name(name):
                name = str(name).strip()
                # 1. 替换特殊符号 '●' 为标准符号 '·' 
                name = name.replace('●', '·').replace(' ', '')
                
                # 2. 修正已知的 Excel 录入笔误及旧站名
                fixes = {
                    # --- 笔误修正 ---
                    '淞淞发路': '淞发路',
                    '淞淞虹路': '淞虹路',
                    '淞淞滨路': '淞滨路',  # [新增] 修正笔误
                    '临洮洮路': '临洮路',  # [新增] 修正笔误
                    '颛颛桥': '颛桥',
                    '佘佘山': '佘山',
                    # --- 站名变更/映射 ---               
                    # --- 暂未开通/无拓扑车站 ---
                    '龙居路': '龙居路',    # 14号线暂缓开通，拓扑中无此站，保留原名即可
                    '严御路': '康恒路',    # [新增] 严御路已更名为康恒路(11号线) 。但是拓扑里面没它。
                }
                return fixes.get(name, name)

            flow_df['station'] = flow_df['station'].apply(clean_station_name)

            # 预处理：转换时间列，方便后续查询
            flow_df['time'] = pd.to_datetime(flow_df['time'])
            
            # 执行一致性校验
            verify_data_consistency(BASE_PATH, flow_df)
            
        except Exception as e:
            print(f"❌ 读取客流文件失败: {e}")
            return

    else:
        # 原始蒙特卡洛模式配置
        MC_ITERATIONS = 100       
        RAIN_P_PERIOD = 50       
        RAIN_DURATION = 180       
        SIM_DURATION = 240        
        START_TIME_STR = "09:00"     
        print(f"\n🚀 启动随机蒙特卡洛仿真模式 (Runs={MC_ITERATIONS})")
        DB_NAME = None
        
        # [修改] 读取 08.08 的基准客流数据 (全量 CSV)
        flow_file_full = "shanghai_metro_flow_with_code_final_20250802-20250831_v4.csv"
        flow_path = os.path.join(BASE_PATH, flow_file_full)
        print(f"📂 加载基准客流文件 (2025.08.08): {flow_path}")
        
        try:
            # 读取 CSV
            full_df = pd.read_csv(flow_path)
            
            # 过滤 2025-08-08 的数据
            # [修改] 原代码的 format='%Y-%m-%d %H' 无法解析 ":00:00" 部分，改为自动推断
            full_df['time'] = pd.to_datetime(full_df['time']) 
            
            target_date = pd.Timestamp("2025-08-08").date()
            flow_df = full_df[full_df['time'].dt.date == target_date].copy()
            
            # 重命名列以匹配 SubwayDIIMSystem
            # csv: inbound, outbound, station_name_mapped
            # system expects: inbound, outbound, station
            flow_df.rename(columns={'station_name_mapped': 'station'}, inplace=True)
            
            # 站点名称清洗 (确保与 if 块中的逻辑一致)
            def clean_station_name(name):
                name = str(name).strip()
                name = name.replace('●', '·').replace(' ', '')
                fixes = {
                    '淞淞发路': '淞发路',
                    '淞淞虹路': '淞虹路',
                    '淞淞滨路': '淞滨路',
                    '临洮洮路': '临洮路', 
                    '颛颛桥': '颛桥',
                    '佘佘山': '佘山',
                    '严御路': '康恒路',
                }
                return fixes.get(name, name)
            flow_df['station'] = flow_df['station'].apply(clean_station_name)
            
            # 处理浦电路拆分 (简单起见，这里复用逻辑)
            pudian_mask = (flow_df['station'] == '浦电路')
            if pudian_mask.any():
                pudian_rows = flow_df[pudian_mask].copy()
                pudian_4 = pudian_rows.copy(); pudian_4['station'] = '浦电路(4号线)'; pudian_4[['inbound','outbound']] /= 2
                pudian_6 = pudian_rows.copy(); pudian_6['station'] = '浦电路(6号线)'; pudian_6[['inbound','outbound']] /= 2
                flow_df = flow_df[~pudian_mask]
                flow_df = pd.concat([flow_df, pudian_4, pudian_6], ignore_index=True)
            
            print(f"✅ 已加载 2025.08.08 客流数据，共 {len(flow_df)} 条记录")
            
        except Exception as e:
            print(f"❌ 读取客流文件失败: {e}")
            flow_df = None

    # N_JOBS 控制并行核心数
    # N_JOBS = 10 if USE_REAL_DATA else 30 # 真实数据读取DB开销大，建议减少并发数或设为1
    N_JOBS = 40

    recorder = MonteCarloRecorder(list(range(SIM_DURATION)))
    
    try:
        # 使用 joblib.Parallel 并行派发 simulation_worker 任务
        results = Parallel(n_jobs=N_JOBS, backend='loky', verbose=10)(
            delayed(simulation_worker)(
                run+1, BASE_PATH, 250, 
                RAIN_P_PERIOD, # period (unused in real mode)
                RAIN_DURATION, # rain_dur (unused in real mode)
                SIM_DURATION, START_TIME_STR,
                use_real_data=USE_REAL_DATA,
                real_db_name=DB_NAME,
                real_flow_data=flow_df # [修改] 将读取到的 flow_df 传递进去
            ) for run in range(MC_ITERATIONS)
        )
        
        # 遍历并行结果并存入记录器
        for res in results:
            if res[1] is None:
                print(f"❌ [Error] Run {res[0]} failed: {res[2]}")
            else:
                recorder.add_run_data(*res)
                
    except Exception as e:
        print(f"❌ [Critical] Parallel execution failed: {e}")

    print("\n🏁 仿真结束，正在保存数据...")
    recorder.save_to_excel(BASE_PATH)

if __name__ == "__main__":
    main()

In [ ]:
import networkx as nx
import json
import math
import pandas as pd
import numpy as np
import random
import os
import re
import gc
import time
import sys
import sqlite3
import warnings
from abc import ABC, abstractmethod
from scipy import sparse
from scipy.spatial.distance import cdist
from scipy.interpolate import griddata
from PIL import Image
from datetime import datetime, timedelta

# [修改 1] 引入 joblib 用于 Jupyter 环境下的并行计算
from joblib import Parallel, delayed 

# 禁用插值产生的可能的警告
warnings.filterwarnings("ignore")

# ==============================================================================
# [全局配置] 并行计算环境设置
# ==============================================================================
# 限制 Numpy/OpenBLAS 的底层线程数。
# 原因：在 Joblib 多进程模式下，如果每个子进程再自动开启多线程进行矩阵运算，
# 会导致 CPU 资源争抢（Oversubscription）甚至死锁，显著降低仿真效率。
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

# ==============================================================================
# 1. 策略模式：定义扰动（降雨）行为
# ==============================================================================
class DisturbanceStrategy(ABC):
    """
    抽象基类：定义系统受到的外部干扰策略。
    
    Methods:
        get_intensity: 获取特定时空的干扰强度。
        get_speed_impact_factor: 获取干扰对系统运行速度的影响系数。
    """
    @abstractmethod
    def get_intensity(self, time_step, position): pass
    @abstractmethod
    def get_speed_impact_factor(self, intensity, edge_type, **kwargs): pass

class ExtremeRainfall(DisturbanceStrategy):
    """
    极端降雨策略模型：模拟一个移动的暴雨中心及其对地铁站点的积水影响。
    
    包含物理过程：
    1. 芝加哥雨型（Chicago Design Storm）生成时间序列降雨。
    2. 移动风暴中心（Moving Storm Center）。
    3. 空间衰减（ARF）计算。
    4. SCS-CN 径流模型计算地表积水。
    """
    def __init__(self, base_path, max_intensity=201.9, rainfall_type='chicago', P=100, r=0.405, rain_duration=180):
        """
        初始化降雨模型参数。
        
        Args:
            max_intensity (float): 降雨峰值强度参数 (mm/h 或相关系数)。
            P (int): 降雨重现期 (Return Period, e.g., 100年一遇)。
            r (float): 雨峰位置系数 (Peak Position parameter)，0-1之间，0.4代表峰值出现在前40%。
            rain_duration (int): 降雨持续总时长 (分钟)。
        """
        # 设定模拟的空间范围（上海经纬度大致范围）
        self.lat_range = (30.6984, 31.5139)
        self.lon_range = (120.849, 121.966)
        
        # 初始化暴雨中心起始位置和移动速度
        self.center_loc, self.velocity = self._init_random_boundary_start()
        
        self.Q0 = max_intensity  # 降雨强度基准
        self.base_path = base_path
        self.rainfall_type = rainfall_type
        self.P_period = P        # 重现期
        self.r_peak = r          # 雨峰系数
        self.rain_duration = rain_duration 
        
        self.chicago_profile = None 
        self.img_array = None
        self._load_drainage_image() # 加载区域排水能力图
        self._distance_cache = {}   # 缓存站点到雨中心的距离，减少重复三角函数计算
        
        # 如果是芝加哥雨型，预计算随时间变化的强度曲线
        if self.rainfall_type == 'chicago': self._precompute_chicago_profile()

    def _init_random_boundary_start(self):
        """
        随机初始化降雨中心。
        逻辑：选择地图四个边界之一作为起点，目标点设为地图中心附近，以此计算移动矢量。
        """
        lat_min, lat_max = self.lat_range
        lon_min, lon_max = self.lon_range
        side = random.choice([0, 1, 2, 3]) # 0:上(北), 1:下(南), 2:左(西), 3:右(东)
        
        # 设定目标点为区域中心附近略带随机偏移
        target_lat = (lat_min + lat_max) / 2 + random.uniform(-0.1, 0.1)
        target_lon = (lon_min + lon_max) / 2 + random.uniform(-0.1, 0.1)
        
        if side == 0:   start_lat, start_lon = lat_max, random.uniform(lon_min, lon_max)
        elif side == 1: start_lat, start_lon = lat_min, random.uniform(lon_min, lon_max)
        elif side == 2: start_lat, start_lon = random.uniform(lat_min, lat_max), lon_min
        else:           start_lat, start_lon = random.uniform(lat_min, lat_max), lon_max
        
        speed_kmh = random.uniform(25, 40) # 降雨系统移动速度 (km/h)
        speed_km_min = speed_kmh / 60.0
        
        # 计算移动单位向量（转换为经纬度增量）
        delta_lat_km = (target_lat - start_lat) * 111.0
        delta_lon_km = (target_lon - start_lon) * 96.0
        dist_km = math.sqrt(delta_lat_km**2 + delta_lon_km**2) or 1
        
        v_lat_deg = ((delta_lat_km / dist_km) * speed_km_min) / 111.0
        v_lon_deg = ((delta_lon_km / dist_km) * speed_km_min) / 96.0
        return (start_lat, start_lon), (v_lat_deg, v_lon_deg)

    def move_step(self):
        """每一分钟移动一次降雨中心，并清空距离缓存（因为中心变了，所有距离需重算）"""
        curr_lat, curr_lon = self.center_loc
        v_lat, v_lon = self.velocity
        self.center_loc = (curr_lat + v_lat, curr_lon + v_lon)
        self._distance_cache = {} 

    def _precompute_chicago_profile(self):
        """
        预计算芝加哥雨型（Chicago Design Storm）的时间分布。
        公式来源：Keifer & Chu (1957)，参数 A, n, b 针对上海地区暴雨强度公式适配。
        """
        times = np.arange(0, self.rain_duration + 1, 1) 
        intensities = []
        t_peak_time = self.rain_duration * self.r_peak
        
        # A, n, b 为上海地区的降雨公式参数（需根据研究区域调整）
        A = 9.581 * (1 + 0.846 * np.log10(self.P_period))
        n = 0.656
        b = 7
        
        for t in times:
            # 峰前时刻公式
            if t <= t_peak_time:
                t_b = t_peak_time - t
                num = A * ((1 - n) * (t_b / self.r_peak) + b)
                den = math.pow((t_b / self.r_peak) + b, 1 + n)
            # 峰后时刻公式
            else:
                t_a = t - t_peak_time
                num = A * ((1 - n) * (t_a / (1 - self.r_peak)) + b)
                den = math.pow((t_a / (1 - self.r_peak)) + b, 1 + n)
            intensities.append(num / den)
        
        # 模拟雨后由于排水不畅导致的“长尾效应”（指数衰减）
        tail_duration = 120  
        if intensities:
            last_val = intensities[-1]
            for t_ex in range(1, tail_duration + 1):
                val = last_val * math.exp(-0.1 * t_ex)
                intensities.append(val if val > 0.001 else 0.0)
        
        # 存储为累计降雨量 (Cumulative Rainfall)，方便后续计算径流
        self.chicago_profile = np.cumsum(intensities)

    def _load_drainage_image(self):
        """加载排水能力栅格图，用于根据地理位置查询排水效率"""
        path = os.path.join(self.base_path, "排水能力_简化.png")
        try:
            img = Image.open(path)
            self.img_array = np.array(img)
        except FileNotFoundError:
            # 如果没图，生成一个随机的（调试用）
            self.img_array = np.random.randint(0, 255, (1000, 1000, 3), dtype=np.uint8)

    def _haversine(self, lat1, lon1, lat2, lon2):
        """计算地球表面两点间的大圆距离 (km)"""
        R = 6371.0 
        dlat = math.radians(lat2 - lat1)
        dlon = math.radians(lon2 - lon1)
        a = math.sin(dlat/2)**2 + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(dlon/2)**2
        return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))

    def _lat_lon_to_pixel(self, lat, lon):
        """经纬度坐标与图像像素坐标的转换"""
        if self.img_array is None: return 0, 0
        h, w = self.img_array.shape[:2]
        lat_min, lat_max = self.lat_range
        lon_min, lon_max = self.lon_range
        px = int((lon - lon_min) / (lon_max - lon_min) * w)
        py = int((lat_max - lat) / (lat_max - lat_min) * h)
        return min(max(px, 0), w - 1), min(max(py, 0), h - 1)

    def _get_drainage_from_image(self, lat, lon):
        """从像素颜色值映射到实际排水能力 (mm/h)"""
        px, py = self._lat_lon_to_pixel(lat, lon)
        # 获取灰度值或R通道值
        r_val = self.img_array[py, px][0] if self.img_array.ndim == 3 else self.img_array[py, px]
        r_norm = r_val / 255.0
        # 简单的阈值映射规则
        if r_val == 0: return 27
        elif r_norm <= 0.9: return 36
        elif r_norm <= 0.99: return 27
        else: return 50

    def _calculate_arf_rainfall(self, distance):
        """空间衰减计算：根据距雨中心距离计算面雨量折减系数 (Area Reduction Factor, ARF)"""
        area = math.pi * (distance**2)
        # ARF 经验公式
        factor = -0.058 * math.log(area) + 0.9949 if area >= 1 else 1.0
        Q_temp = self.Q0 * factor
        # 增加空间随机扰动
        return max(0, Q_temp + np.random.normal(0, 0.3 * abs(Q_temp)))

    def get_instantaneous_intensity(self, time_step, position, precomputed_dist=None):
        """
        获取某时刻、某位置的瞬时降雨强度 (mm/h)。
        主要用于地上线路的能见度/运行风险判断。
        [修改说明] 引入60分钟滑动平均，解决芝加哥雨型瞬时峰值过高的问题。
        """
        if precomputed_dist is not None: dist = precomputed_dist
        else:
            if position in self._distance_cache: dist = self._distance_cache[position]
            else:
                dist = self._haversine(self.center_loc[0], self.center_loc[1], position[0], position[1])
                self._distance_cache[position] = dist
                
        spatial_val = self._calculate_arf_rainfall(dist)
        
        if self.rainfall_type == 'chicago':
            if time_step >= len(self.chicago_profile) - 1: return 0.0
            scale = spatial_val / self.Q0 if self.Q0 > 0 else 0
            
            # [核心修改] 使用 60分钟滑动窗口计算小时雨强
            idx_now = min(int(time_step), len(self.chicago_profile) - 1)
            window_size = 60 
            idx_prev = max(0, idx_now - window_size)
            
            # 这一时段内的总雨量 (mm)
            rain_depth_in_window = self.chicago_profile[idx_now] - self.chicago_profile[idx_prev]
            
            # 时段长度 (min)
            duration = idx_now - idx_prev
            if duration <= 0: return 0.0
            
            # 换算成小时雨强 (mm/h)
            intensity_hourly = (rain_depth_in_window / duration) * 60.0
            
            return max(0, intensity_hourly * scale)
        else:
            # 简单均匀降雨模式
            return max(0, spatial_val / (self.rain_duration / 60.0)) if self.rain_duration > 0 else 0

    def get_intensity(self, time_step, position, precomputed_dist=None):
        """
        核心计算：计算地表净积水深度 (mm)。
        使用了 SCS-CN (Soil Conservation Service Curve Number) 径流模型的简化形式。
        Logic: 积水 = 降雨径流(Runoff) - 城市管网排水(Drainage)
        """
        lat, lon = position
        if precomputed_dist is not None: dist = precomputed_dist
        else:
            if position in self._distance_cache: dist = self._distance_cache[position]
            else:
                dist = self._haversine(self.center_loc[0], self.center_loc[1], lat, lon)
                self._distance_cache[position] = dist
        spatial_val = self._calculate_arf_rainfall(dist)
        
        # 1. 计算累计降雨量 (Cumulative Rainfall)
        if self.rainfall_type == 'chicago':
            scale = spatial_val / self.Q0 if self.Q0 > 0 else 0
            idx = min(int(time_step), len(self.chicago_profile) - 1)
            cum_rain = self.chicago_profile[idx] * scale
        else:
            progress = min(time_step / float(self.rain_duration), 1.0)
            cum_rain = spatial_val * progress
            
        # 2. SCS 径流转换
        S = 15.7 # 潜在最大入渗量 (Potential Maximum Retention)，与土壤/地表类型有关
        # SCS方程：Q = (P - 0.2S)^2 / (P + 0.8S)
        runoff = (cum_rain - 0.05 * S)**2 / (cum_rain + 0.95 * S) if cum_rain > 0.05 * S else 0
        
        # 3. 扣除区域排水
        drainage = (self._get_drainage_from_image(lat, lon) / 60.0) * time_step
        return max(0, runoff - drainage) # 返回值为净积水深度

    def get_speed_impact_factor(self, water_depth, edge_type, line_context=None):
        """
        计算速度折减系数：将物理积水深度转换为对系统运行/步行速度的影响 (0.0 - 1.0)。
        1.0 表示正常速度，0.0 表示完全中断。
        """
        # [修改] 提取线路整体最大积水深度，用于判断是否触发全线停运
        # 如果 max_water_total > 450，意味着线路上某处已严重积水，该线路应整体停运
        max_water_line = 0.0
        if line_context:
            max_water_line = line_context.get('max_water_total', 0.0)

        # [修改] 核心逻辑变更：
        # 如果当前站点水深 > 450mm OR 线路任意位置水深 > 450mm (由 line_context 传入)，
        # 则视为线路瘫痪。
        is_line_failed = (water_depth > 450) or (max_water_line > 450)

        if edge_type == 'boarding': 
            # 进站/步行速度折减 (基于经验研究的步速-水深函数，如 Ishigaki et al.)
            # [修改] 若线路瘫痪（不论是本站淹水还是线路其他站淹水），进站系数设为 0.0001
            if is_line_failed:
                return 0.0001
            
            if water_depth <= 0: return 1.0
            if water_depth <= 250: return max(0.0001, 0.9 - (water_depth / 250.0) * 0.1)
            elif water_depth <= 450: return 0.0001
            else: return 0.0001 # 兜底逻辑，实际上 is_line_failed 已覆盖
            
        elif edge_type == 'riding': 
            # 乘车运行速度折减
            if line_context:
                is_above = line_context.get('is_above_ground', 0)
                max_rain = line_context.get('max_rain_above', 0.0)
                # max_water_line 已经在上面提取了
                
                if is_above == 1: # 地上线主要受瞬时强降雨（能见度）影响
                    if max_rain > 100: return 0.0001 # [修改] 极强降雨停运，因子改为 0.0001
                    elif max_rain >= 80: return 0.5 # 强降雨限速
                else: # 地下线主要受区间进水影响
                    # [修改] 若线路上最大积水超过450mm，列车停运，影响因子变为0.0001
                    if max_water_line > 450: return 0.0001
            return 1.0

# ==============================================================================
# [新增] 真实数据降雨策略类
# ==============================================================================
class RealDataRainfall(DisturbanceStrategy):
    """
    真实数据降雨策略：基于 SQLite 数据库中的历史/实时降雨数据进行空间插值。
    模拟 2025-07-29 等特定日期的真实降雨分布。
    """
    def __init__(self, base_path, db_name="shanghai_rain_history.db", start_datetime_str="2025-07-29 00:00"):
        self.base_path = base_path
        self.db_path = os.path.join(base_path, db_name)
        self.start_dt = datetime.strptime(start_datetime_str, "%Y-%m-%d %H:%M")
        self.center_loc = (31.2304, 121.4737) # 默认中心设为上海市中心，适配接口（真实数据无中心概念）
        
        # 缓存机制，避免每分钟每节点都去查库插值
        self._last_query_time_str = None
        self._cached_grid_interpolator = None # 存储 griddata 的结果或KDTree
        self._cached_df = None
        
        # 上海区域网格参数
        self.SH_LON_MIN, self.SH_LON_MAX = 120.8, 122.3
        self.SH_LAT_MIN, self.SH_LAT_MAX = 30.6, 31.9
        
        self.img_array = None
        self._load_drainage_image()

    def _load_drainage_image(self):
        """复用父类的排水图加载逻辑"""
        path = os.path.join(self.base_path, "排水能力_简化.png")
        try:
            img = Image.open(path)
            self.img_array = np.array(img)
        except:
            self.img_array = np.random.randint(0, 255, (1000, 1000, 3), dtype=np.uint8)

    def _lat_lon_to_pixel(self, lat, lon):
        """复用父类的坐标转换逻辑"""
        if self.img_array is None: return 0, 0
        h, w = self.img_array.shape[:2]
        # 注意：这里需要与 ExtremeRainfall 的范围保持一致，或者使用本类定义的范围
        # 假设排水图是覆盖 self.SH_LAT_MIN/MAX 范围的
        lat_min, lat_max = 30.6984, 31.5139 # 这里沿用 ExtremeRainfall 的范围设定以匹配图片
        lon_min, lon_max = 120.849, 121.966
        px = int((lon - lon_min) / (lon_max - lon_min) * w)
        py = int((lat_max - lat) / (lat_max - lat_min) * h)
        return min(max(px, 0), w - 1), min(max(py, 0), h - 1)

    def _get_drainage_from_image(self, lat, lon):
        """复用父类的排水能力获取逻辑"""
        px, py = self._lat_lon_to_pixel(lat, lon)
        r_val = self.img_array[py, px][0] if self.img_array.ndim == 3 else self.img_array[py, px]
        r_norm = r_val / 255.0
        if r_val == 0: return 27
        elif r_norm <= 0.9: return 36
        elif r_norm <= 0.99: return 27
        else: return 50
    
    # 距离计算方法
    def _haversine(self, lat1, lon1, lat2, lon2):
        """计算地球表面两点间的大圆距离 (km)"""
        R = 6371.0 
        dlat = math.radians(lat2 - lat1)
        dlon = math.radians(lon2 - lon1)
        a = math.sin(dlat/2)**2 + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(dlon/2)**2
        return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))
    
    def move_step(self):
        """真实数据不需要移动风暴中心，此方法留空适配接口"""
        pass

    def _fetch_and_interpolate(self, current_time_str):
        """从数据库读取特定时间的数据并构建插值函数"""
        # 如果时间没变，直接返回缓存
        if current_time_str == self._last_query_time_str and self._cached_df is not None:
            return
        
        self._last_query_time_str = current_time_str
        conn = None
        try:
            conn = sqlite3.connect(self.db_path)
            # 使用 LIKE 模糊匹配时间 (e.g., '2025-07-29 05:00%')
            query = """
                SELECT s.lon, s.lat, r.rain_value 
                FROM rain_data r
                JOIN stations s ON r.station_id = s.station_id
                WHERE r.datetime LIKE ?
            """
            df = pd.read_sql_query(query, conn, params=(f"{current_time_str}%",))
            
            if len(df) < 3:
                # 数据不足时，假设无雨
                self._cached_df = pd.DataFrame({'lon': [], 'lat': [], 'rain_value': []})
            else:
                self._cached_df = df
        except Exception as e:
            # print(f"DB Error: {e}") 
            self._cached_df = pd.DataFrame({'lon': [], 'lat': [], 'rain_value': []})
        finally:
            if conn: conn.close()

    def get_instantaneous_intensity(self, time_step, position):
        """获取瞬时雨强 (mm/h)"""
        # 计算当前绝对时间
        curr_dt = self.start_dt + timedelta(minutes=time_step)
        # 数据库是每小时数据 (e.g. 05:00, 06:00)，这里简单取最近的小时，或者按分钟插值
        # 为了性能，这里取整点小时 (format: YYYY-MM-DD HH)
        # 如果需要更精细，可以修改为 HH:MM 并确保数据库有该精度，或者在两个小时之间做线性插值
        # 这里采用：取当前小时的数据（阶梯状变化）
        target_time_str = curr_dt.strftime("%Y-%m-%d %H") # 匹配数据库的 '2025-07-29 05' 开头
        
        self._fetch_and_interpolate(target_time_str)
        
        if self._cached_df is None or len(self._cached_df) == 0:
            return 0.0
        
        # 使用 Nearest 或 Linear 插值获取特定位置的雨量
        # 由于每次调用 griddata 开销大，这里针对单点计算：
        # 实际上 griddata 适合生成网格。对于单点查询，计算距离最近的几个站做 IDW 更快。
        # 这里简化使用 scipy.interpolate.griddata 对单点插值
        points = self._cached_df[['lat', 'lon']].values
        values = self._cached_df['rain_value'].values
        
        # 注意：griddata 输入是 (points, values, xi)
        # points: (N, 2), xi: (M, 2)
        try:
            val = griddata(points, values, np.array([position]), method='nearest')[0]
            return max(0, val)
        except:
            return 0.0

    def get_intensity(self, time_step, position, precomputed_dist=None):
        """
        计算地表积水深度 (mm)。
        对于真实数据，我们假设数据库里存的是“小时雨量 (mm)”。
        为了计算积水，我们需要累计雨量。
        注意：这是一个简化处理。真实的积水模拟需要连续的时间积分。
        这里我们假设：积水 = 当前时刻瞬时雨强累积 - 排水。
        """
        lat, lon = position
        
        # 获取瞬时雨强 (假设数据库存储的是 mm/h 或者该小时的累计雨量)
        intensity_mm_h = self.get_instantaneous_intensity(time_step, position)
        
        # 计算当前时间步（1分钟）带来的新增雨量
        rain_inc = intensity_mm_h / 60.0 
        
        # 在这里，DisturbanceStrategy 是无状态的(stateless)关于“积水历史”的吗？
        # ExtremeRainfall 类里是用 cumulative profile 算的。
        # 如果我们没有累计曲线，我们只能近似。
        # 方案：假设过去 N 小时的雨都下下来了，扣除排水。
        # 但这非常慢。
        # 妥协方案：SubwayDIIMSystem 的 G.nodes[n]['water'] 是每一帧刷新的。
        # 如果 ExtremeRainfall 是每次重算 total accumulation，那这里也得这么做。
        # 但查询历史所有小时的数据太慢了。
        
        # *修正*：SubwayDIIMSystem.step() 里是直接赋值：
        # self.G.nodes[n]['water'] = self.strategy.get_intensity(...)
        # 所以 get_intensity 必须返回“当前积水深度”。
        
        # 对于 RealData，我们难以每次从 0时刻开始积分。
        # 建议：仅返回瞬时雨强导致的“潜在积水”，或者基于简单的“当前雨强 * 因子”来近似积水风险。
        # 或者，为了更准确，我们在 SimulationSystem 里维护积水状态，而不是在这里计算 total。
        # 但为了不改动 System 的逻辑（System 假设 Strategy 返回积水深度），
        # 我们这里做一个近似： 积水深度 ~= 最近 1 小时的雨量 * 径流系数 - 排水 * 1小时
        
        S = 15.7
        # 假设 effective rain 是过去 60 分钟的累积（即当前 hourly value）
        cum_rain = intensity_mm_h # 粗略假设
        
        runoff = (cum_rain - 0.05 * S)**2 / (cum_rain + 0.95 * S) if cum_rain > 0.05 * S else 0
        
        # 排水能力 (mm/h)
        drainage_rate = self._get_drainage_from_image(lat, lon)
        
        # 估算积水：径流 - 排水（假设排水一直在工作）
        # 这里仅仅是一个状态快照，无法完美模拟时序累积，除非重构 System
        water_depth = max(0, runoff - drainage_rate) 
        
        return water_depth

    def get_speed_impact_factor(self, water_depth, edge_type, line_context=None):
        """复用父类的速度折减逻辑"""
        # [修改] 提取线路整体最大积水深度
        max_water_line = 0.0
        if line_context:
            max_water_line = line_context.get('max_water_total', 0.0)

        # [修改] 判断是否全线停运
        is_line_failed = (water_depth > 450) or (max_water_line > 450)

        if edge_type == 'boarding': 
            # [修改] 若线路瘫痪，进站系数设为 0.0001
            if is_line_failed:
                return 0.0001
                
            if water_depth <= 0: return 1.0
            if water_depth <= 250: return max(0.01, 0.9 - (water_depth / 250.0) * 0.1)
            elif water_depth <= 450: return max(0.0001, 0.8 - ((water_depth - 250) / 200.0) * 0.7)
            else: return 0.0001 
            
        elif edge_type == 'riding': 
            if line_context:
                is_above = line_context.get('is_above_ground', 0)
                max_rain = line_context.get('max_rain_above', 0.0)
                
                if is_above == 1: 
                    if max_rain > 100: return 0.0001  # [修改] 极强降雨停运
                    elif max_rain >= 80: return 0.5 
                else: 
                    if max_water_line > 450: return 0.0001 # [修改] 地下线受淹停运
            return 1.0


# ==============================================================================
# 2. 核心系统：地铁网络拓扑与 DIIM (动态进程间影响模型) 模拟
# ==============================================================================
class SubwayDIIMSystem:
    """
    地铁动态影响模拟系统 (Dynamic Inoperability Input-Output Model System)
    
    主要功能：
    1. 构建双层网络图（Station Layer + Line Layer）。
    2. 加载静态数据（拓扑、时刻表、客流）。
    3. 执行时间步进仿真，计算 DIIM 状态更新。
    4. 计算网络性能指标（加权全局效率）。
    """
    def __init__(self, base_path, disturbance_strategy, sim_start_time='05:30', real_flow_df=None):
        self.base_path = base_path
        self.strategy = disturbance_strategy
        self.real_flow_df = real_flow_df # [新增] 接收真实客流数据 DataFrame
        self.G = nx.MultiGraph() # 使用多重图构建分层网络，允许两个节点间存在多种关系的边
        
        try:
            t_obj = datetime.strptime(sim_start_time, "%H:%M")
            self.sim_start_min = t_obj.hour * 60 + t_obj.minute
            self.sim_start_dt_obj = t_obj # 保存 datetime 对象用于计算
        except: 
            self.sim_start_min = 330
            self.sim_start_dt_obj = datetime(2000, 1, 1, 5, 30)
        
        # 内部缓存与索引映射
        self.id_map_long_to_short = {}
        self.line_stations_index = {}
        self.schedule_rules = [] 
        self.schedule_lookup = {}
        self.station_pos_cache = {}
        self.fast_edge_data = [] # 扁平化的边数据列表，用于极速遍历（避开NX图查询开销）
        self.node_idx_map = {}
        self.node_pos_list = []
        self.node_dists_list = []
        self.node_st_names = []
        
        # 顺序执行初始化流程
        self._load_id_mapping()           # 1. 映射线路ID (Excel -> Dict)
        self._load_json_topology()        # 2. 加载基础拓扑 (JSON)
        self._load_detailed_schedule()    # 3. 加载列车时刻/发车间隔
        self._build_station_index()       # 4. 构建索引
        self._build_graph_topology()      # 5. 构建分层网络图 (Graph Building)
        self._apply_ground_elevation_status() # 6. 标记地上/地下线
        self._load_passenger_flow_data()  # 7. 加载客流权重 (静态初始值)
        self._build_single_layer_network()# 8. 构建单层简化网 (用于辅助计算)
        self._init_diim_matrices()        # 9. 初始化动态影响矩阵 (DIIM Init)
        
        # 韧性恢复参数（设定恢复速度）
        # c(t) 为受损程度，恢复过程遵循指数衰减
        T_rec = 15.0 
        self.k_recovery = (1.0 / T_rec) * math.log(1.0 / 0.01)
        
        self._init_flow_weights()         # 初始化流权重
        self._precompute_static_simulation_data() # 预处理静态数据加速计算
        
        # 计算初始（无故障）状态下的加权网络效率，作为后续归一化的分母
        self.Ef0 = self.calculate_weighted_efficiency(self.sim_start_min, 'Weekday')
        self.congestion_warning_triggered = False

    def _load_id_mapping(self):
        """从Excel加载线路名称与短ID（如：1号线 -> "1"）的映射关系"""
        path = os.path.join(self.base_path, "线路id与名称对应-20251210.xlsx")
        try:
            if os.path.exists(path): df = pd.read_excel(path)
            else: df = pd.read_csv(path + " - Sheet1.csv")
            for _, row in df.iterrows():
                if pd.isna(row['Line ID']): continue
                long_id = str(int(row['Line ID'])) if isinstance(row['Line ID'], float) else str(row['Line ID'])
                line_name = str(row['Line Name'])
                short_id = None
                if "浦江" in line_name: short_id = "PJ"
                elif "磁浮" in line_name: short_id = "Maglev"
                elif "号线" in line_name:
                    match = re.search(r'(\d+)', line_name)
                    if match: short_id = match.group(1)
                if short_id: self.id_map_long_to_short[long_id] = short_id
        except: pass

    def _load_json_topology(self):
        """读取地铁拓扑结构 JSON 文件"""
        path = os.path.join(self.base_path, "subway.json")
        with open(path, "r", encoding="utf-8") as f: self.raw_json = json.load(f)

    def _load_detailed_schedule(self):
        """加载详细的发车间隔规则（区分早晚高峰、工作日/周末、区间车）"""
        path = os.path.join(self.base_path, "line_schedule_detailed.xlsx")
        try:
            df = pd.read_excel(path)
            df['LineID'] = df['LineID'].apply(lambda x: str(int(x)) if isinstance(x, float) else str(x).strip())
            for _, row in df.iterrows():
                s_dt = pd.to_datetime(str(row['Start_Time']), format='%H:%M:%S', errors='coerce')
                e_dt = pd.to_datetime(str(row['End_Time']), format='%H:%M:%S', errors='coerce')
                rule = {
                    'line_id': row['LineID'], 'day_type': str(row['DayType']).strip(),
                    'start_min': s_dt.hour*60+s_dt.minute if pd.notnull(s_dt) else 0,
                    'end_min': e_dt.hour*60+e_dt.minute if pd.notnull(e_dt) else 1440,
                    'from_st': str(row['From_Station']).strip(), 'to_st': str(row['To_Station']).strip(),
                    'interval': float(row['Interval_Sec'])
                }
                self.schedule_rules.append(rule)
                key = (rule['line_id'], rule['day_type'])
                if key not in self.schedule_lookup: self.schedule_lookup[key] = []
                self.schedule_lookup[key].append(rule)
        except: pass

    def _build_station_index(self):
        """构建车站在线路中的顺序索引，用于后续判断某车站是否在区间车的覆盖范围内"""
        for line_info in self.raw_json['l']:
            short_id = self.id_map_long_to_short.get(str(line_info['ls']))
            if not short_id: continue
            st_names = [st['n'] for st in line_info['st']]
            if short_id not in self.line_stations_index: 
                self.line_stations_index[short_id] = []
                self.station_pos_cache[short_id] = {}
            for st in st_names:
                if st not in self.line_stations_index[short_id]: 
                    self.line_stations_index[short_id].append(st)
                    self.station_pos_cache[short_id][st] = len(self.line_stations_index[short_id]) - 1

    def _is_station_in_section(self, line_id, station_name, from_st, to_st):
        """逻辑判定：特定车站是否位于指定的调度区间（from_st 到 to_st）内"""
        line_cache = self.station_pos_cache.get(line_id)
        if not line_cache: return False
        idx_s, idx_f, idx_t = line_cache.get(station_name), line_cache.get(from_st), line_cache.get(to_st)
        if idx_s is None or idx_f is None or idx_t is None: return False
        return min(idx_f, idx_t) <= idx_s <= max(idx_f, idx_t)

    def _build_graph_topology(self):
        """
        构建三维分层模型 (Space-L space)：
        1. 车站层 (Station Layer): 物理车站位置。
        2. 线路层 (Line Layer): 特定线路的站台。
        3. 边 (Edges):
           - boarding: 车站层 <-> 线路层 (进出站)。
           - riding: 线路层 <-> 线路层 (乘车)。
        """
        for line_info in self.raw_json['l']:
            short_id = self.id_map_long_to_short.get(str(line_info['ls']))
            if not short_id: continue
            sts = line_info['st']
            for st in sts:
                pos = tuple(map(float, st['sl'].split(',')))[::-1] # 转为 (lat, lon)
                node_st = f"{st['n']}_车站层"
                node_ln = f"{st['n']}_线路层_{short_id}"
                
                # 添加节点
                if not self.G.has_node(node_st): self.G.add_node(node_st, layer='station', pos=pos, flow=1000)
                self.G.add_node(node_ln, layer='line', pos=pos, line_id=short_id, station_name=st['n'])
                
                # 添加进站边 (Boarding/Alighting Link)
                self.G.add_edge(node_st, node_ln, type='boarding', weight=0)
            
            # 添加运行区间边 (Riding Link)
            for i in range(len(sts) - 1):
                u, v = f"{sts[i]['n']}_线路层_{short_id}", f"{sts[i+1]['n']}_线路层_{short_id}"
                if self.G.has_node(u) and self.G.has_node(v):
                    self.G.add_edge(u, v, type='riding', weight=0, line_id=short_id)

    def _apply_ground_elevation_status(self):
        """识别地铁的地上区间，这对于评估降雨暴露风险非常关键（地下区间受淹水影响，地上区间受雨强影响）"""
        file_path = os.path.join(self.base_path, "上海地铁地上线归纳.txt")
        above_set = set()
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                for line in f:
                    if '：' not in line: continue
                    parts = line.strip().split('：')
                    # 解析线路ID
                    match = re.search(r'(\d+)', parts[0])
                    lid = match.group(1) if match else ("PJ" if "浦江" in parts[0] else None)
                    if not lid: continue
                    # 解析区间
                    for group in parts[1].split('，'):
                        sts = group.strip().split('-')
                        for i in range(len(sts)-1):
                            pair = tuple(sorted((sts[i].strip(), sts[i+1].strip())))
                            above_set.add((lid, pair[0], pair[1]))
        except: pass
        
        # 遍历图中的边，标记 is_above_ground 属性
        for u, v, k, attr in self.G.edges(keys=True, data=True):
            if attr.get('type') == 'riding':
                lid = attr.get('line_id')
                stu, stv = self.G.nodes[u].get('station_name'), self.G.nodes[v].get('station_name')
                pair = tuple(sorted((stu, stv))) if stu and stv else None
                attr['is_above_ground'] = 1 if pair and (lid, pair[0], pair[1]) in above_set else 0

    def _load_passenger_flow_data(self):
        """从外部文件加载客流强度，用于后续加权网络效率的计算（客流大的站点权重更高）"""
        file_path = os.path.join(self.base_path, "进站客流.xlsx")
        p_flow = {}
        sum_f, num_f = 0, 0
        try:
            xls = pd.ExcelFile(file_path)
            for sheet in xls.sheet_names:
                for _, row in xls.parse(sheet, header=None).iterrows():
                    val = ((row[1] or 0) + (row[2] or 0)) / 2
                    p_flow[f"{row[0]}_车站层"] = p_flow.get(f"{row[0]}_车站层", 0) + val
                    sum_f += val; num_f += 1
        except: pass
        
        # 缺失值填充（使用平均值）
        avg_flow = sum_f / max(num_f, 1) or 1000
        avg_deg = np.mean([d for n, d in self.G.degree() if self.G.nodes[n].get('layer')=='station']) or 1
        
        for n, d in self.G.nodes(data=True):
            if d.get('layer') == 'station':
                d['flow'] = p_flow.get(n, (avg_flow/avg_deg)*self.G.degree(n))

    def _build_single_layer_network(self):
        """构建一个简化的单层站点间网络，用于辅助拓扑特征计算"""
        self.single_layer_net = nx.Graph()
        for u, v, attr in self.G.edges(data=True):
            if attr.get('type') == 'riding':
                n1 = self.G.nodes[u].get('station_name', u.split('_')[0])
                n2 = self.G.nodes[v].get('station_name', v.split('_')[0])
                if n1 != n2: self.single_layer_net.add_edge(n1, n2)

    def _init_flow_weights(self):
        """初始化边权：将节点流分配到相邻的进站边和运行边上"""
        ln_weights = {}
        # 1. 进站边权重
        for u, v, k, attr in self.G.edges(keys=True, data=True):
            if attr['type'] == 'boarding':
                st = u if self.G.nodes[u].get('layer')=='station' else v
                ln = v if st == u else u
                w = self.G.nodes[st].get('flow', 1000) / max(1, self.G.degree[st])
                attr['flow_weight'] = w
                ln_weights[ln] = w
        # 2. 运行边权重（取两端节点平均）
        for u, v, k, attr in self.G.edges(keys=True, data=True):
            if attr['type'] == 'riding':
                attr['flow_weight'] = (ln_weights.get(u, 100)+ln_weights.get(v, 100))/2

    def _precompute_static_simulation_data(self):
        """
        [极速仿真优化] 预计算所有不会随时间改变的数据结构。
        将图数据扁平化为 List/Array，避免在 step() 循环中频繁调用 NetworkX 的 lookup 方法。
        """
        nodes = list(self.G.nodes())
        self.node_idx_map = {n: i for i, n in enumerate(nodes)}
        self.node_dists_list = [0.0] * len(nodes)
        for i, n in enumerate(nodes):
            self.node_pos_list.append(self.G.nodes[n]['pos'])
            self.node_st_names.append(self.G.nodes[n].get('station_name', n.split('_')[0]))
        
        target_day_type = 'Weekday'
        for u, v, k in self.edge_list:
            attr = self.G.edges[u, v, k]
            u_idx, v_idx = self.node_idx_map[u], self.node_idx_map[v]
            lid = attr.get('line_id') or self.G.nodes[u].get('line_id') or self.G.nodes[v].get('line_id')
            est = {
                'u_idx': u_idx, 'v_idx': v_idx, 'type': attr['type'], 'line_id': lid,
                'is_above_ground': attr.get('is_above_ground', 0),
                'flow_weight': attr.get('flow_weight', 1.0),
                'boarding_dist': 0.0, 'boarding_node_flow': 1000.0,
                'riding_pos_mid': None, 'riding_applicable_rules': []
            }
            # 预绑定该边适用的调度规则（避免运行时搜索）
            if lid:
                rules = self.schedule_lookup.get((str(lid), target_day_type), [])
                st_u, st_v = self.node_st_names[u_idx], self.node_st_names[v_idx]
                check_st = st_u if attr['type'] == 'boarding' else None
                for r in rules:
                    if check_st:
                        if self._is_station_in_section(str(lid), check_st, r['from_st'], r['to_st']):
                             est['riding_applicable_rules'].append(r)
                    else:
                        if self._is_station_in_section(str(lid), st_u, r['from_st'], r['to_st']) and \
                           self._is_station_in_section(str(lid), st_v, r['from_st'], r['to_st']):
                             est['riding_applicable_rules'].append(r)
            if attr['type'] == 'boarding':
                st = self.node_st_names[u_idx]
                deg = self.single_layer_net.degree[st] if st in self.single_layer_net else 1
                est['boarding_dist'] = deg * 100.0 # 模拟车站内步行距离
                est['boarding_node_flow'] = self.G.nodes[u if self.G.nodes[u]['layer']=='station' else v].get('flow', 1000)
            else:
                p1, p2 = self.G.nodes[u]['pos'], self.G.nodes[v]['pos']
                est['riding_pos_mid'] = ((p1[0]+p2[0])/2, (p1[1]+p2[1])/2)
            self.fast_edge_data.append(est)

    def calculate_weighted_efficiency(self, current_time_min, day_type, line_stats=None):
        """
        计算效率，并包含终极路径追踪。
        """
        node_waters = [self.G.nodes[n].get('water', 0) for n in self.G.nodes()]
        
        # 1. 计算所有边权重
        for i, (u, v, k) in enumerate(self.edge_list):
            sd = self.fast_edge_data[i]
            
            # Rain
            avg_w = (node_waters[sd['u_idx']] + node_waters[sd['v_idx']]) / 2.0
            t_curr, _, _ = self._calc_times_fast(sd, avg_w, current_time_min, line_stats)
            
            # Base
            t_baseline, _, _ = self._calc_times_fast(sd, 0.0, current_time_min, line_stats=None)
            
            w = sd['flow_weight']
            
            # 这里的权重已经是 "时间 / 流量权重"
            weight_rain = (t_curr / w) if w > 0 and t_curr != float('inf') else float('inf')
            weight_base = (t_baseline / w) if w > 0 and t_baseline != float('inf') else float('inf')
            
            self.G.edges[u,v,k]['tau'] = weight_rain
            self.G.edges[u,v,k]['tau_baseline'] = weight_base
            
        st_nodes = [n for n, d in self.G.nodes(data=True) if d.get('layer') == 'station']
        st_set = set(st_nodes)
        N = len(st_nodes)
        if N < 2: return 0.0
        
        sum_inv_curr = 0.0
        sum_inv_base = 0.0
        
        found_anomaly_path = False
        
        # 2. 计算最短路径
        for src in st_nodes:
            # 获取 Rain 场景下的所有路径长度
            try:
                lens_rain = nx.single_source_dijkstra_path_length(self.G, src, weight='tau')
            except: lens_rain = {}
            
            # 获取 Base 场景下的所有路径长度
            try:
                lens_base = nx.single_source_dijkstra_path_length(self.G, src, weight='tau_baseline')
            except: lens_base = {}
            
            # 累加并检查
            for t in st_set:
                if t == src: continue
                
                l_r = lens_rain.get(t, float('inf'))
                l_b = lens_base.get(t, float('inf'))
                
                if l_r > 0: sum_inv_curr += 1.0/l_r
                if l_b > 0: sum_inv_base += 1.0/l_b
                
                # === 捕获异常路径 ===
                # 如果 Rain 距离 < Base 距离，说明 Rain 路径更短（更优），这是不可能的
                if l_r < l_b - 1e-6 and not found_anomaly_path:
                    print(f"\n🔥🔥🔥 [PATH ANOMALY DETECTED] at Time {current_time_min} 🔥🔥🔥")
                    print(f"Source: {src} -> Target: {t}")
                    print(f"Dist Rain: {l_r:.6f} < Dist Base: {l_b:.6f}")
                    
                    # 追踪 Rain 路径
                    try:
                        path = nx.shortest_path(self.G, src, t, weight='tau')
                        print("--- Rain Path Breakdown ---")
                        total_chk = 0
                        for i in range(len(path)-1):
                            u_p, v_p = path[i], path[i+1]
                            # 获取边数据 (取最小权重的那个key)
                            edge_data = self.G.get_edge_data(u_p, v_p)
                            # 找到 weight='tau' 最小的那个 key
                            best_k = min(edge_data, key=lambda k: edge_data[k]['tau'])
                            attr = edge_data[best_k]
                            
                            w_r = attr['tau']
                            w_b = attr['tau_baseline']
                            print(f"   {u_p} -> {v_p} | RainW={w_r:.4f} vs BaseW={w_b:.4f} | {'ERROR' if w_r < w_b else 'OK'}")
                            total_chk += w_r
                        print(f"   Check Sum: {total_chk:.4f}")
                    except:
                        print("   (Could not reconstruct path)")
                    
                    found_anomaly_path = True
        
        Ef = (1/(N*(N-1))) * sum_inv_curr
        Ef0 = (1/(N*(N-1))) * sum_inv_base
        
        return Ef / Ef0 if Ef0 > 0 else 1.0

    def _init_diim_matrices(self):
        """初始化动态进程间影响模型 (DIIM) 矩阵结构"""
        self.edge_list = list(self.G.edges(keys=True))
        self.edge_idx = {e: i for i, e in enumerate(self.edge_list)}
        n = len(self.edge_list)
        self.q_vector = np.zeros(n) # 存储各边的受损失效率 q
        self.A_matrix = sparse.csr_matrix((n, n))
        self.node_to_boarding_indices = {}
        for n_st in [n for n, d in self.G.nodes(data=True) if d['layer']=='station']:
            idxs = []
            for u, v, k in self.G.edges(n_st, keys=True):
                if self.G.edges[u,v,k]['type'] == 'boarding':
                    k_t = (u,v,k) if (u,v,k) in self.edge_idx else (v,u,k)
                    if k_t in self.edge_idx: idxs.append(self.edge_idx[k_t])
            self.node_to_boarding_indices[n_st] = idxs

    def _calc_times_fast(self, static_data, water_depth, current_time_min, line_stats=None):
        """
        计算特定边在特定积水深度下的瞬时通行时间。
        Returns: (当前受灾时间 t_curr, 正常理想时间 t_norm, 调试信息字典 debug_info)
        """
        lookup_time = current_time_min % 1440
        etype = static_data['type']
        
        # 准备调试信息
        debug_info = {}

        lid = static_data['line_id']
        ctx = None
        if lid and line_stats and str(lid) in line_stats:
            st_data = line_stats[str(lid)]
            ctx = {'is_above_ground': static_data.get('is_above_ground', 0), 
                   'max_rain_above': st_data['max_rain_above'], 
                   'max_water_total': st_data['max_water_total']}

        if etype == 'boarding':
            dist = static_data['boarding_dist']
            
            # 1. 速度计算
            speed_factor = self.strategy.get_speed_impact_factor(water_depth, 'boarding', line_context=ctx)
            v_curr = 1.4 * speed_factor
            
            # 记录调试信息
            debug_info['speed_factor'] = speed_factor
            debug_info['v_curr'] = v_curr
            debug_info['dist'] = dist

            if v_curr < 0.01: return float('inf'), 100.0, debug_info
            
            tw_c = dist / v_curr
            tw_n = dist / 1.4
            
            t_b = 30 # 列车停靠时间

            # 2. 等待时间计算
            t_wait_c, t_wait_n = 0, 0
            if static_data['line_id']:
                t_base = 300.0
                for r in static_data['riding_applicable_rules']:
                    if r['start_min'] <= lookup_time < r['end_min']:
                        t_base = r['interval']; break
                
                hw_n = t_base
                # 计算受影响后的发车间隔
                riding_factor = self.strategy.get_speed_impact_factor(water_depth, 'riding', line_context=ctx)
                hw_c = hw_n / max(riding_factor, 0.0001)
                
                # 排队论参数
                lam = max(static_data['boarding_node_flow'] / 3600.0, 0.0001)
                
                # 记录更多调试信息
                debug_info['hw_n'] = hw_n
                debug_info['hw_c'] = hw_c
                debug_info['riding_factor'] = riding_factor
                debug_info['lambda'] = lam

                # [修正开始] 定义单调性约束的排队等待时间计算函数
                def calc_w(EV, L):
                    # 1. 计算饱和度 (Saturation Ratio)
                    # 原分母公式 D = 2 * (1/L - 2*EV)
                    # 饱和状态意味着 1/L = 2*EV => L * 2 * EV = 1
                    # 因此定义 rho = L * 2 * EV
                    rho = L * 2 * EV
                    
                    # 设定安全阈值，超过此阈值认为进入拥堵/过饱和区
                    # 在此区域不再使用原公式（原公式有奇点且非单调），改用幂函数惩罚
                    RHO_THRESH = 0.9

                    # 计算阈值处的理论等待时间 (作为锚点)
                    # 当 rho = 0.9 时:
                    # denom_limit = 2 * (1/L - 2*EV) 
                    #             = 2/L * (1 - 2*EV*L) 
                    #             = 2/L * (1 - 0.9) = 0.2 / L
                    # 这是一个近似计算，为了性能，我们直接用硬逻辑判断
                    
                    if rho < RHO_THRESH:
                        # === 正常区域：使用原排队论公式 ===
                        denom = 2 * ((1.0/L) - 2*EV)
                        # 虽然理论上 rho < 0.9 保证 denom > 0，但加个由衷的兜底
                        if denom <= 1e-6: 
                            val = 1800.0 # 理论上不应走到这里
                        else:
                            val = 2.5*EV + (t_b**2 + 3*EV**2)/denom
                        
                        # 即使在正常区域，也允许适当的上限，但不要强制截断导致非单调
                        # 如果计算值确实很大（极罕见），让它保持很大，不要 clamp 到 1800
                        return min(val, 3600.0) 
                    
                    else:
                        # === 拥堵/过饱和区域：使用单调惩罚函数 ===
                        # 这里的逻辑是：既然已经堵了，那么发车间隔(EV)越大，rho越大，等待时间必须越长。
                        # 我们构造一个平滑过渡函数。
                        # 先算出在 RHO_THRESH (0.9) 时的边界值 T_limit
                        denom_limit = 2 * ((1.0/L) * (1 - RHO_THRESH))
                        t_limit = 2.5*EV + (t_b**2 + 3*EV**2)/denom_limit
                        
                        # 惩罚因子：(当前饱和度 / 阈值饱和度) 的平方
                        # 平方项确保了"越堵越慢"的非线性增长，有效拉开 Rain 和 Base 的差距
                        penalty_factor = (rho / RHO_THRESH) ** 2
                        
                        # 最终时间 = 边界时间 * 惩罚因子
                        # 这样即使 Base 很堵 (rho=0.95), Rain 更堵 (rho=1.1), 
                        # Rain 的结果也一定比 Base 大。
                        final_wait = t_limit * penalty_factor
                        
                        # 即使是非常极端的情况，也建议保留一个物理极限（如 2小时），
                        # 但这个极限必须足够大，以免再次出现 Base > Rain 的截断悖论
                        return min(final_wait, 7200.0) 

                # [修正结束]
                
                t_wait_n = calc_w(hw_n+t_b, lam)
                t_wait_c = calc_w(hw_c+t_b, lam)
                
                debug_info['t_wait_n'] = t_wait_n
                debug_info['t_wait_c'] = t_wait_c

            return tw_c + t_wait_c, tw_n + t_wait_n, debug_info
            
        elif etype == 'riding':
            t_base = 180.0
            for r in static_data['riding_applicable_rules']:
                if r['start_min'] <= lookup_time < r['end_min']:
                    t_base = r['interval']; break
            
            fac = self.strategy.get_speed_impact_factor(water_depth, 'riding', line_context=ctx)
            
            debug_info['t_base_interval'] = t_base
            debug_info['riding_factor'] = fac
            
            if fac <= 0.0001: return float('inf'), t_base, debug_info
            return t_base / fac, t_base, debug_info
            
        return 0, 0, debug_info

    def step(self, t_minute, day_type='Weekday'):
        """
        推进模拟时间步，并包含全网权重暴力一致性检查。
        """
        curr_clk = self.sim_start_min + t_minute
        
        # 1. 动态更新客流 (保持原逻辑)
        if self.real_flow_df is not None:
            curr_abs_time = None
            if hasattr(self.strategy, 'start_dt'):
                curr_abs_time = self.strategy.start_dt + timedelta(minutes=t_minute)
            
            if curr_abs_time:
                lookup_h = curr_abs_time.replace(minute=0, second=0, microsecond=0)
                try:
                    if curr_abs_time.minute == 0 or t_minute == 0:
                        batch = self.real_flow_df[self.real_flow_df['time'] == lookup_h]
                        if not batch.empty:
                            for _, row in batch.iterrows():
                                st_name = row['station']
                                flow_val = (row['inbound'] + row['outbound']) / 2
                                node_st = f"{st_name}_车站层"
                                if self.G.has_node(node_st):
                                    self.G.nodes[node_st]['flow'] = flow_val
                                    
                            for est in self.fast_edge_data:
                                if est['type'] == 'boarding':
                                    st_node_idx = est['u_idx'] if self.G.nodes[list(self.G.nodes())[est['u_idx']]].get('layer') == 'station' else est['v_idx']
                                    st_node_name = list(self.G.nodes())[st_node_idx]
                                    est['boarding_node_flow'] = self.G.nodes[st_node_name].get('flow', 1000)
                except Exception as e:
                    pass

        self.strategy.move_step()
        clat, clon = self.strategy.center_loc
        
        # 2. 更新积水
        for i, (lat, lon) in enumerate(self.node_pos_list):
            self.node_dists_list[i] = self.strategy._haversine(clat, clon, lat, lon)
        for i, n in enumerate(self.G.nodes()):
            w = self.strategy.get_intensity(t_minute, self.G.nodes[n]['pos'], precomputed_dist=self.node_dists_list[i])
            self.G.nodes[n]['water'] = w
            
        # 3. 统计线路风险
        line_stats = {}
        node_waters = [self.G.nodes[n].get('water', 0) for n in self.G.nodes()]
        for sd in self.fast_edge_data:
            if sd['type'] == 'riding' and sd['line_id']:
                lid = str(sd['line_id'])
                if lid not in line_stats: line_stats[lid] = {'max_rain_above': 0.0, 'max_water_total': 0.0}
                aw = (node_waters[sd['u_idx']] + node_waters[sd['v_idx']]) / 2
                if aw > line_stats[lid]['max_water_total']: line_stats[lid]['max_water_total'] = aw
                if sd['is_above_ground']:
                    r = self.strategy.get_instantaneous_intensity(t_minute, sd['riding_pos_mid'])
                    if r > line_stats[lid]['max_rain_above']: line_stats[lid]['max_rain_above'] = r
                    
        # 4. DIIM 更新与全网扫描
        curr_c = np.zeros(len(self.edge_list))
        
        # [暴力扫描] 统计计数器
        scan_count_rain_faster = 0
        scan_total_edges = 0
        scan_debug_msg = ""

        for i, sd in enumerate(self.fast_edge_data):
            aw = (node_waters[sd['u_idx']] + node_waters[sd['v_idx']]) / 2
            
            # 计算 Rain 时间
            tc, tn, _ = self._calc_times_fast(sd, aw, curr_clk, line_stats)
            
            # 计算 Base 时间 (强制无雨)
            tb, _, _ = self._calc_times_fast(sd, 0.0, curr_clk, line_stats=None)
            
            # --- 核心检查点 ---
            # 只要 Rain 比 Base 快超过 0.0001秒，就记录下来
            if tc < tb - 0.0001:
                scan_count_rain_faster += 1
                if scan_count_rain_faster <= 3: # 只记录前3个，防止刷屏
                    scan_debug_msg += f"\n   [BAD EDGE] {self.node_st_names[sd['u_idx']]}->{self.node_st_names[sd['v_idx']]} ({sd['type']}) | Rain={tc:.4f} < Base={tb:.4f}"

            # DIIM 逻辑
            val = 0.0
            if tc == float('inf'): val = 1.0 
            elif tc > 0: val = 1.0 - (tn / tc)
            curr_c[i] = max(0.0, min(1.0, val)) 
            scan_total_edges += 1
        
        # [打印扫描结果]
        if scan_count_rain_faster > 0:
            print(f"\n🚨 [CRITICAL SCAN] Time {t_minute}: Found {scan_count_rain_faster}/{scan_total_edges} edges where Rain is FASTER!")
            print(scan_debug_msg)
        else:
            # 如果没有发现异常边，打印一条确认信息
            # print(f"✅ [SCAN OK] Time {t_minute}: All edges Rain >= Base.")
            pass

        q_next = (1 - self.k_recovery) * self.q_vector + self.k_recovery * curr_c
        self.q_vector = np.clip(q_next, 0, 1)
        
        for i, (u,v,k) in enumerate(self.edge_list): self.G.edges[u,v,k]['q_value'] = self.q_vector[i]
        aff_lines = set()
        for i, q in enumerate(self.q_vector):
            if q > 0.1:
                lid = self.fast_edge_data[i]['line_id']
                if lid: aff_lines.add(lid)
        node_risks = []
        for n, idxs in self.node_to_boarding_indices.items():
            if idxs: node_risks.append(max(self.q_vector[i] for i in idxs))
            else: node_risks.append(0)
        
        self.current_line_stats = line_stats
        avg_water = np.mean(list(node_waters))
        max_n_risk = max(node_risks) if node_risks else 0
        
        avg_q_safe = np.nanmean(self.q_vector) if len(self.q_vector) > 0 else 0.0
        avg_q_safe = np.nan_to_num(avg_q_safe)
        
        return avg_q_safe, avg_water, len(aff_lines), max_n_risk

# ==============================================================================
# 3. 蒙特卡洛数据记录器：收集并保存多次随机实验的数据
# ==============================================================================
class MonteCarloRecorder:
    def __init__(self, time_steps):
        self.times = time_steps
        self.df_eff = pd.DataFrame({'Time': time_steps})
        self.df_risk = pd.DataFrame({'Time': time_steps})
        self.df_water = pd.DataFrame({'Time': time_steps})
        self.df_intensity = pd.DataFrame({'Time': time_steps})
        self.df_lat = pd.DataFrame({'Time': time_steps})
        self.df_lon = pd.DataFrame({'Time': time_steps})
        self.df_aff_lines = pd.DataFrame({'Time': time_steps})
        self.df_max_n_risk = pd.DataFrame({'Time': time_steps})

    def add_run_data(self, run_id, eff, risk, water, inten, lats, lons, aff_ln, max_nr):
        col_name = f"Run_{run_id}"
        self.df_eff[col_name] = eff
        self.df_risk[col_name] = risk
        self.df_water[col_name] = water
        self.df_intensity[col_name] = inten
        self.df_lat[col_name] = lats
        self.df_lon[col_name] = lons
        self.df_aff_lines[col_name] = aff_ln
        self.df_max_n_risk[col_name] = max_nr

    def save_to_excel(self, base_path):
        fname = os.path.join(base_path, f"MonteCarlo_Results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.xlsx")
        print(f"💾 正在导出蒙特卡洛数据到: {fname}")
        try:
            with pd.ExcelWriter(fname, engine='openpyxl') as writer:
                self.df_eff.to_excel(writer, sheet_name='Efficiency', index=False)
                self.df_risk.to_excel(writer, sheet_name='Risk_Avg_Q', index=False)
                self.df_water.to_excel(writer, sheet_name='Water_Level_Avg', index=False)
                self.df_intensity.to_excel(writer, sheet_name='Intensity_Center', index=False)
                self.df_lat.to_excel(writer, sheet_name='Location_Lat', index=False)
                self.df_lon.to_excel(writer, sheet_name='Location_Lon', index=False)
                self.df_aff_lines.to_excel(writer, sheet_name='Affected_Line_Count', index=False)
                self.df_max_n_risk.to_excel(writer, sheet_name='Max_Node_Risk', index=False)
            print("✅ 导出完成。")
        except Exception as e:
            print(f"❌ 导出失败: {e}")

# ==========================================
# 5. [Worker] 独立仿真任务：供多进程并行调用
# 注意：该函数必须定义在顶层，以便 Joblib 进行序列化 (Pickle)
# ==========================================
def simulation_worker(run_id, base_path, intensity, period, rain_dur, sim_dur, start_str, 
                      use_real_data=False, real_db_name=None, real_flow_data=None):
    try:
        # [修改] 支持真实数据模式初始化
        if use_real_data:
            strategy = RealDataRainfall(base_path, db_name=real_db_name, start_datetime_str=start_str)
            # 在真实模式下，我们传入客流 DataFrame
            sys_obj = SubwayDIIMSystem(base_path, strategy, sim_start_time=start_str, real_flow_df=real_flow_data)
        else:
            strategy = ExtremeRainfall(base_path, max_intensity=intensity, 
                                       rainfall_type='chicago', P=period, 
                                       rain_duration=rain_dur)
            
            # [修改] 为 ExtremeRainfall 模式注入客流数据和基准时间
            # 我们将模拟日期固定在无雨的 2025-08-08，以便使用这天的真实客流
            base_date_str = "2025-08-08"
            full_start_str = f"{base_date_str} {start_str}"
            try:
                # 手动给 strategy 绑定 start_dt，以便 SubwayDIIMSystem 能够计算绝对时间并查询客流
                strategy.start_dt = datetime.strptime(full_start_str, "%Y-%m-%d %H:%M")
            except ValueError:
                pass
                
            # 将外部传入的 real_flow_data (08.08的客流) 传给系统
            sys_obj = SubwayDIIMSystem(base_path, strategy, sim_start_time=start_str, real_flow_df=real_flow_data)
            
        t_obj = datetime.strptime(start_str, "%Y-%m-%d %H:%M") if len(start_str) > 5 else datetime.strptime(start_str, "%H:%M")
        start_min = t_obj.hour * 60 + t_obj.minute
        time_steps = list(range(sim_dur))
        r_eff, r_risk, r_water, r_inten, r_lat, r_lon, r_aff, r_mnr = [], [], [], [], [], [], [], []
        
        for t in time_steps:
            avg_q, avg_w, n_lines, max_nr = sys_obj.step(t)
            curr_clk = start_min + t
            # 计算该时间步的网络加权效率
            ef = sys_obj.calculate_weighted_efficiency(curr_clk, 'Weekday', line_stats=getattr(sys_obj, 'current_line_stats', None))
            
            # 对于真实数据，位置固定，返回 0 或默认中心
            c_lat, c_lon = (0, 0)
            if hasattr(sys_obj.strategy, 'center_loc'):
                c_lat, c_lon = sys_obj.strategy.center_loc
                
            # 瞬时雨强，如果是真实数据，传一个中心点坐标去获取，或者取平均？
            # 这里取 (31.23, 121.47) 上海中心点作为参考
            ref_pos = (31.2304, 121.4737)
            inten = sys_obj.strategy.get_instantaneous_intensity(t, ref_pos)
            
            r_eff.append(ef)
            r_risk.append(avg_q)
            r_water.append(avg_w)
            r_inten.append(inten)
            r_lat.append(c_lat)
            r_lon.append(c_lon)
            r_aff.append(n_lines)
            r_mnr.append(max_nr)
        
        # 强制垃圾回收，防止内存泄漏
        del sys_obj
        del strategy
        gc.collect() 
        return (run_id, r_eff, r_risk, r_water, r_inten, r_lat, r_lon, r_aff, r_mnr)
    except Exception as e:
        return (run_id, None, str(e), None, None, None, None, None, None)

# [新增] 数据一致性校验函数
def verify_data_consistency(base_path, flow_df):
    """
    检查 Excel 客流数据、JSON 拓扑数据、DB 降雨数据中的车站名称一致性。
    """
    print("\n🔍 正在进行数据一致性校验...")
    
    # 1. 获取 Excel 中的车站列表
    excel_stations = set(flow_df['station'].unique())
    
    # 2. 获取 JSON 中的车站列表
    json_path = os.path.join(base_path, "subway.json")
    json_stations = set()
    try:
        with open(json_path, "r", encoding="utf-8") as f: 
            data = json.load(f)
            for l in data['l']:
                for st in l['st']:
                    json_stations.add(st['n'])
    except:
        print("⚠️ 无法读取 subway.json，跳过拓扑校验。")

    # 3. 打印比对结果 (Excel vs JSON)
    if json_stations:
        missing_in_json = excel_stations - json_stations
        missing_in_excel = json_stations - excel_stations
        
        print(f"  - Excel 中有 {len(excel_stations)} 个车站")
        print(f"  - JSON 中有 {len(json_stations)} 个车站")
        
        if missing_in_json:
            print(f"⚠️ 警告: Excel 中有 {len(missing_in_json)} 个车站在 JSON 拓扑中找不到 (可能无法更新其客流):")
            print(f"    示例: {list(missing_in_json)[:5]} ...")
        else:
            print("✅ Excel 所有车站均已在 JSON 拓扑中找到。")
            
        if missing_in_excel:
            # 这通常不是问题，因为 Excel 可能只包含部分车站数据
            pass

    # 4. 获取 DB 中的车站列表 (简单检查)
    db_path = os.path.join(base_path, "shanghai_rain_history.db")
    if os.path.exists(db_path):
        try:
            conn = sqlite3.connect(db_path)
            cursor = conn.cursor()
            cursor.execute("SELECT DISTINCT station_name FROM stations") # 假设表名 stations，列名 station_name
            db_stations = set([r[0] for r in cursor.fetchall()])
            conn.close()
            
            print(f"  - DB 中有 {len(db_stations)} 个气象监测站")
            # 气象站和地铁站通常不同名，这里仅做数量确认，不做集合比对
        except:
            print("⚠️ 无法读取数据库中的站点信息，跳过校验。")
    else:
        print("⚠️ 数据库文件不存在。")
    print("--------------------------------------------------\n")


# ==========================================
# 6. 主程序入口 (Joblib 并行执行版)
# ==========================================
def main():
    # === 参数配置区域 ===
    BASE_PATH = "E:\\BaiHao\\20260103算例补充\\输入与输出数据"
    
    # --- 模式开关 ---
    USE_REAL_DATA = False   # [开关] 是否使用 2025.07.29-08.01 的真实数据
    # ----------------
    
    if USE_REAL_DATA:
        # 真实数据模式配置
        MC_ITERATIONS = 1         # 真实数据是确定性的，通常跑 1 次即可 (或者跑多次以测试算法稳定性)
        START_TIME_STR = "2025-07-29 05:00" # 仿真开始时间
        END_TIME_STR   = "2025-08-01 23:00" # 仿真结束时间
        #START_TIME_STR = "2025-07-29 07:00" # 仿真开始时间
        #END_TIME_STR   = "2025-07-29 08:00" # 仿真结束时间
        
        # 计算总分钟数
        t_start = datetime.strptime(START_TIME_STR, "%Y-%m-%d %H:%M")
        t_end   = datetime.strptime(END_TIME_STR, "%Y-%m-%d %H:%M")
        SIM_DURATION = int((t_end - t_start).total_seconds() / 60)
        
        DB_NAME = "shanghai_rain_history.db"
        FLOW_FILE = "shanghai_metro_flow_final_complete.xlsx" # 支持 csv 或 xlsx
        
        print(f"\n🚀 启动真实数据仿真模式 ({START_TIME_STR} -> {END_TIME_STR}, Duration={SIM_DURATION}min)")
        
        # 加载客流数据 (Pre-load flow data)
        flow_path = os.path.join(BASE_PATH, FLOW_FILE)
        print(f"📂 加载客流文件: {flow_path}")
        try:
            if flow_path.endswith('.csv'):
                flow_df = pd.read_csv(flow_path)
            else:
                flow_df = pd.read_excel(flow_path)

            # 1. 基础清洗：转字符串、去空格、统一符号
            flow_df['station'] = flow_df['station'].astype(str).str.strip().str.replace('●', '·').str.replace(' ', '')

            # 2. 特殊处理：将“浦电路”拆分为4号线和6号线，客流平分
            # 筛选出原始名称为 "浦电路" 的所有行
            pudian_mask = (flow_df['station'] == '浦电路')
            
            if pudian_mask.any():
                # 提取这些行
                pudian_rows = flow_df[pudian_mask].copy()
                
                # 构造4号线数据 (取名为 JSON 中对应的 '浦电路(4号线)', 客流除以2)
                pudian_4 = pudian_rows.copy()
                pudian_4['station'] = '浦电路(4号线)'
                pudian_4['inbound'] = pudian_4['inbound'] / 2.0
                pudian_4['outbound'] = pudian_4['outbound'] / 2.0
                
                # 构造6号线数据 (取名为 JSON 中对应的 '浦电路(6号线)', 客流除以2)
                pudian_6 = pudian_rows.copy()
                pudian_6['station'] = '浦电路(6号线)'
                pudian_6['inbound'] = pudian_6['inbound'] / 2.0
                pudian_6['outbound'] = pudian_6['outbound'] / 2.0
                
                # 从原表中删除“浦电路”，并追加拆分后的两部分数据
                flow_df = flow_df[~pudian_mask] # 删除原行
                flow_df = pd.concat([flow_df, pudian_4, pudian_6], ignore_index=True)
                print(f"ℹ️ 已将 {len(pudian_rows)} 条'浦电路'数据拆分为4号线与6号线 (客流平分)。")

            # 【新增修复】 简单的站点名称清洗，解决匹配警告
            def clean_station_name(name):
                name = str(name).strip()
                # 1. 替换特殊符号 '●' 为标准符号 '·' 
                name = name.replace('●', '·').replace(' ', '')
                
                # 2. 修正已知的 Excel 录入笔误及旧站名
                fixes = {
                    # --- 笔误修正 ---
                    '淞淞发路': '淞发路',
                    '淞淞虹路': '淞虹路',
                    '淞淞滨路': '淞滨路',  # [新增] 修正笔误
                    '临洮洮路': '临洮路',  # [新增] 修正笔误
                    '颛颛桥': '颛桥',
                    '佘佘山': '佘山',
                    # --- 站名变更/映射 ---               
                    # --- 暂未开通/无拓扑车站 ---
                    '龙居路': '龙居路',    # 14号线暂缓开通，拓扑中无此站，保留原名即可
                    '严御路': '康恒路',    # [新增] 严御路已更名为康恒路(11号线) 。但是拓扑里面没它。
                }
                return fixes.get(name, name)

            flow_df['station'] = flow_df['station'].apply(clean_station_name)

            # 预处理：转换时间列，方便后续查询
            flow_df['time'] = pd.to_datetime(flow_df['time'])
            
            # 执行一致性校验
            verify_data_consistency(BASE_PATH, flow_df)
            
        except Exception as e:
            print(f"❌ 读取客流文件失败: {e}")
            return

    else:
        # 原始蒙特卡洛模式配置
        MC_ITERATIONS = 100       
        RAIN_P_PERIOD = 100       
        RAIN_DURATION = 180       
        SIM_DURATION = 240        
        START_TIME_STR = "09:00"     
        print(f"\n🚀 启动随机蒙特卡洛仿真模式 (Runs={MC_ITERATIONS})")
        DB_NAME = None
        
        # [修改] 读取 08.08 的基准客流数据 (全量 CSV)
        flow_file_full = "shanghai_metro_flow_with_code_final_20250802-20250831_v4.csv"
        flow_path = os.path.join(BASE_PATH, flow_file_full)
        print(f"📂 加载基准客流文件 (2025.08.08): {flow_path}")
        
        try:
            # 读取 CSV
            full_df = pd.read_csv(flow_path)
            
            # 过滤 2025-08-08 的数据
            # [修改] 原代码的 format='%Y-%m-%d %H' 无法解析 ":00:00" 部分，改为自动推断
            full_df['time'] = pd.to_datetime(full_df['time']) 
            
            target_date = pd.Timestamp("2025-08-08").date()
            flow_df = full_df[full_df['time'].dt.date == target_date].copy()
            
            # 重命名列以匹配 SubwayDIIMSystem
            # csv: inbound, outbound, station_name_mapped
            # system expects: inbound, outbound, station
            flow_df.rename(columns={'station_name_mapped': 'station'}, inplace=True)
            
            # 站点名称清洗 (确保与 if 块中的逻辑一致)
            def clean_station_name(name):
                name = str(name).strip()
                name = name.replace('●', '·').replace(' ', '')
                fixes = {
                    '淞淞发路': '淞发路',
                    '淞淞虹路': '淞虹路',
                    '淞淞滨路': '淞滨路',
                    '临洮洮路': '临洮路', 
                    '颛颛桥': '颛桥',
                    '佘佘山': '佘山',
                    '严御路': '康恒路',
                }
                return fixes.get(name, name)
            flow_df['station'] = flow_df['station'].apply(clean_station_name)
            
            # 处理浦电路拆分 (简单起见，这里复用逻辑)
            pudian_mask = (flow_df['station'] == '浦电路')
            if pudian_mask.any():
                pudian_rows = flow_df[pudian_mask].copy()
                pudian_4 = pudian_rows.copy(); pudian_4['station'] = '浦电路(4号线)'; pudian_4[['inbound','outbound']] /= 2
                pudian_6 = pudian_rows.copy(); pudian_6['station'] = '浦电路(6号线)'; pudian_6[['inbound','outbound']] /= 2
                flow_df = flow_df[~pudian_mask]
                flow_df = pd.concat([flow_df, pudian_4, pudian_6], ignore_index=True)
            
            print(f"✅ 已加载 2025.08.08 客流数据，共 {len(flow_df)} 条记录")
            
        except Exception as e:
            print(f"❌ 读取客流文件失败: {e}")
            flow_df = None

    # N_JOBS 控制并行核心数
    # N_JOBS = 10 if USE_REAL_DATA else 30 # 真实数据读取DB开销大，建议减少并发数或设为1
    N_JOBS = 40

    recorder = MonteCarloRecorder(list(range(SIM_DURATION)))
    
    try:
        # 使用 joblib.Parallel 并行派发 simulation_worker 任务
        results = Parallel(n_jobs=N_JOBS, backend='loky', verbose=10)(
            delayed(simulation_worker)(
                run+1, BASE_PATH, 250, 
                RAIN_P_PERIOD, # period (unused in real mode)
                RAIN_DURATION, # rain_dur (unused in real mode)
                SIM_DURATION, START_TIME_STR,
                use_real_data=USE_REAL_DATA,
                real_db_name=DB_NAME,
                real_flow_data=flow_df # [修改] 将读取到的 flow_df 传递进去
            ) for run in range(MC_ITERATIONS)
        )
        
        # 遍历并行结果并存入记录器
        for res in results:
            if res[1] is None:
                print(f"❌ [Error] Run {res[0]} failed: {res[2]}")
            else:
                recorder.add_run_data(*res)
                
    except Exception as e:
        print(f"❌ [Critical] Parallel execution failed: {e}")

    print("\n🏁 仿真结束，正在保存数据...")
    recorder.save_to_excel(BASE_PATH)

if __name__ == "__main__":
    main()